# Synthetic Experiment — CNN vs Random Forest under Controlled Data-Generating Processes

Virtual species are generated on the real German covariate stack (100 m, EPSG:32632), so the true suitability surface is known. CNN and Random Forest are compared under two data-generating processes — a **pointwise** niche on PC1/PC2 of the covariates and a **configurational** one driven by the spatial arrangement of forest — across sample size, ecological prevalence, resolution and patch size, each in an interpolative (random) and an extrapolative (PC1-band) split.

Primary metric: Spearman ρ against the known truth. Replication unit: virtual species.

## 1 — Setup & Configuration
Imports, paths, experiment constants, and the tuned hyperparameters (read from `keras_tuner_synthetic/`, written by section 7).

In [ ]:
# PROJECT ROOT. Set the environment variable PROJECT_ROOT to the folder that
# holds code/ and data/, or edit the fallback below.
PROJECT_ROOT = os.environ.get("PROJECT_ROOT", os.path.abspath(".."))
import os, json, warnings
# linear algebra runs single-threaded to avoid oversubscription when using multiprocessing + reproducible results:
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import tensorflow as tf

# TARGETED, NOT BLANKET. A bare filterwarnings('ignore') hides sklearn
# convergence warnings and numpy RuntimeWarnings along with the library noise,
# which removes the ability to NOTICE a real problem. Only warnings that have
# been diagnosed and judged harmless are silenced, each with its reason.
warnings.filterwarnings('ignore', message='Mean of empty slice')
warnings.filterwarnings('ignore', message='All-NaN slice encountered')
# ^ both from np.nanmean over blocks that lie entirely outside Germany when
#   resample_env coarsens the raster. Real property of the data, not a bug --
#   the resulting NaN is exactly what the validity masks then exclude.
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
# ^ library churn (TF/Keras/sklearn API moves), not science.
# Everything else -- RuntimeWarning for invalid values, UserWarning,
# ConvergenceWarning -- stays VISIBLE on purpose.
tf.get_logger().setLevel('ERROR')
tf.config.set_visible_devices([], 'GPU')  # CPU is faster than Metal for small CNNs

# ── Paths ─────────────────────────────────────────────────────────────────────
SPECIES     = "sus_scrofa"   # must match SPECIES in data_processing.ipynb
TARGET_RES  = 100            # must match TARGET_RES in data_processing.ipynb
_CODE_OUT   = os.path.join(PROJECT_ROOT, 'code.nosync')
# The covariate stack written by "data_processing.ipynb.nosync Kopie": selected
# layers, log1p where applied, NOT scaled (this notebook scales it, cell 3). A
# separate folder from the old checkpoint_<res>m, which still holds the inputs of
# the v5 run and of the real-data model.
CKPT_DIR    = os.path.join(_CODE_OUT, SPECIES, f"synthetic_stack_{TARGET_RES}m")
FIGURES_DIR = os.path.join(_CODE_OUT, "figures", SPECIES, "synthetic")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Default experiment parameters (change per-cell as needed) ─────────────────
N_POINTS   = 1000    # total sampled points; Section 8 overrides this locally with _S8_N_POINTS=2000
PRES_RATIO = 0.5     # fraction that are presences
PATCH      = 32      # patch size in pixels
N_REPEATS  = 3       # independent training runs per setting (for variance estimate)
EPOCHS     = 80
BATCH_SIZE = 256
SEED       = 42


print('Config OK — TF', tf.__version__)
print(f'N_POINTS={N_POINTS} | PRES_RATIO={PRES_RATIO} | PATCH={PATCH}px | N_REPEATS={N_REPEATS}')

# ── Keras Tuner paths + best HP ─────────────────────────────────────────────
TUNER_DIR = os.path.join(_CODE_OUT, SPECIES, 'keras_tuner_synthetic')
BEST_HP   = {'filters_1': 32, 'filters_2': 16, 'dense_units': 16, 'dropout': 0.2, 'lr': 1e-3}   # Fallback defaults
_hp_file  = os.path.join(TUNER_DIR, 'best_hp_synthetic.json')
_HP_PROV = None       # what the loaded HP were tuned on; checked before every Section-8 sweep
if os.path.exists(_hp_file):
    _hp_d = json.load(open(_hp_file))
    _HP_PROV = _hp_d.pop('_provenance', None)
    BEST_HP.update(_hp_d)   # overwrites Fallback defaults with tuner results
    print('Loaded BEST_HP from file:')
else:
    print('Using default BEST_HP (run the CNN tuner, section 7.1):')
for k, v in BEST_HP.items():
    print(f'  {k}: {v}')

# -- Random Forest best HP (mirrors the CNN block above) ----------------------
# Written by the RF tuner (section 7.2). Loaded here so every downstream section gets
# the tuned RF without re-running the search. The fallback is a sane capped-
# depth config, NOT a copy of any tuned result.
RF_BEST_HP = {'n_estimators': 300, 'max_depth': 6, 'max_features': 'sqrt',
              'min_samples_leaf': 1, 'min_samples_split': 2}    # Fallback defaults
_rf_hp_file = os.path.join(TUNER_DIR, 'best_hp_rf_synthetic.json')
_RF_HP_PROV = None
if os.path.exists(_rf_hp_file):
    _rf_hp_d = json.load(open(_rf_hp_file))
    _RF_HP_PROV = _rf_hp_d.pop('_provenance', None)
    RF_BEST_HP.update(_rf_hp_d)
    print('Loaded RF_BEST_HP from file:')
else:
    print('Using default RF_BEST_HP (run the RF tuner, section 7.2):')
for k, v in RF_BEST_HP.items():
    print(f'  {k}: {v}')



## 2 — Environmental Raster
Loads the covariate stack written by the processing notebook (`synthetic_stack_100m/env_stack_log.npy`: selected layers, log1p where applied, **unscaled**). Landuse is taken out as the raw habitat-quality score — the source of the forest refuge field, not a model input — and the remaining 6 model inputs are min-max scaled to [0, 1], each layer over every pixel where it is defined.

In [ ]:
with open(os.path.join(CKPT_DIR, 'meta.json')) as f:
    meta = json.load(f)
assert meta.get('scaled') is False, (
    'expected the UNSCALED log1p stack from the processing notebook -- this cell '
    'does the scaling itself and must not scale twice')
_env_stack_100m = np.load(os.path.join(CKPT_DIR, meta['STACK_FILE']))   # (H, W, N) float32, unscaled
LAYER_NAMES = meta['LAYER_NAMES']
print(f"Loaded {meta['STACK_FILE']}: {_env_stack_100m.shape}  "
      f"log1p on {meta['log1p_layers']}  (created {meta['created']})")

assert 'Water_Distance' not in LAYER_NAMES, (
    'Water_Distance is excluded at variable selection in the processing notebook -- '
    'this stack predates that change; re-run the processing notebook')

# ── Landuse: extracted HERE, as the raw score, before it leaves the stack ──────
# Landuse is not a model input -- an ordinal class code has no meaningful
# Euclidean distance -- but it is the source of the real-land-use refuge field W
# (8j) and of the orthogonality check (next cell). It is taken out once, here,
# while it is already in memory. It used to be dropped here and re-read from the
# ~2 GB stack file two cells later, by its index in meta['LAYER_NAMES']: a second
# file access, and it made the 8j ecological arm depend on a diagnostic cell
# having run.
# The stack is UNSCALED, so this channel IS the habitat-quality score 0-5.
# No-data (the grid is a bounding box around Germany) is carried as -1, which
# falls out of every np.isin(). Casting NaN to int instead yields INT64_MIN and,
# clipped, an explicit class 0 ('not forest') -- which manufactures a forest edge
# along the entire national border in exactly the field E is built to measure.
_lu_raw = _env_stack_100m[:, :, LAYER_NAMES.index('Landuse')]
_lu_ok  = np.isfinite(_lu_raw)
landuse = np.full(_lu_raw.shape, -1, np.int16)            # -1 = no data
landuse[_lu_ok] = np.round(_lu_raw[_lu_ok]).astype(np.int16)
assert set(np.unique(landuse[_lu_ok]).tolist()) <= {0, 1, 2, 3, 4, 5}, \
    'Landuse is not the raw 0-5 score -- is the stack scaled after all?'
del _lu_raw, _lu_ok

# CLASS LEGEND -- from the processing notebook (section 6f, 'Landuse / Habitat
# Quality (ESA)'). The channel is an ORDINAL HABITAT-QUALITY SCORE, mapped there
# from ESA WorldCover codes:
#   5 = ESA 10 Tree cover          <- the woody/refuge class used here
#   4 = ESA 40 Cropland, ESA 20 Shrubland
#   3 = ESA 90 Herbaceous wetland, ESA 95 Mangroves, ESA 30 Grassland
#   1 = ESA 50 Built-up, ESA 60 Bare/sparse, ESA 100 Moss and lichen
#   0 = ESA 80 Permanent water, ESA 70 Snow and ice
# Score 5 is tree cover ALONE, so {5} is exactly 'forest' -- the refuge class the
# configurational DGP means. The scores themselves are an a-priori hypothesis
# (see the citation notes in the processing notebook), which is why only the
# class-5 indicator is used, and only as the refuge field W.
WOODY_CLASSES = {5}    # = habitat-quality score 5 = ESA 'Tree cover'
print(f"landuse: {int((landuse >= 0).sum()):,} valid px, {int((landuse < 0).sum()):,} no-data (-1), "
      f"forest (score 5) = {float((landuse[landuse >= 0] == 5).mean()):.3f} of valid")

# Landuse is not a model input -- remove it from the stack that is scaled below.
_keep_idx = [i for i, l in enumerate(LAYER_NAMES) if l != 'Landuse']
_env_stack_100m = _env_stack_100m[:, :, _keep_idx]
LAYER_NAMES = [LAYER_NAMES[i] for i in _keep_idx]
print(f'Model inputs: {LAYER_NAMES}')

# ── Min-max scaling to [0, 1]: GLOBAL, per layer ──────────────────────────────
# Each layer's min and max come from EVERY pixel where THAT layer is defined, so
# every value of every layer lies in [0, 1] and no clipping is needed.
# Per layer, not over the pixels valid in all layers at once: ~205,000 pixels
# (0.6% of the land, mostly slivers along the border and coast) lack NDVI but
# have real values in the other layers. They are never sampled (sampling needs
# every layer), but they do occur inside CNN patches near the border -- where 5
# real values and one padded one carry more than 6 padded ones. With a min/max
# taken only over all-layer-valid pixels, those real values landed outside
# [0, 1] (down to -0.03, up to 1.07); per-layer scaling keeps them and keeps
# them in range.
#
# WHY GLOBAL IS RIGHT HERE, AND NOT LEAKAGE. This is not a statistic any model
# fits. It is a fixed, label-free change of units applied identically to the
# whole landscape BEFORE any virtual species exists -- and each species' true
# suitability is itself defined on these scaled covariates (through the PCA in
# 3b). The scaling is therefore part of the data-generating process. Everything
# a model does fit -- padding means, the RF, the CNN, early stopping -- stays on
# the training fold of each split. On top of this, the CNN input is rescaled on
# the training patches only (_s8_train_minmax, section 6).
#
# WHAT IT REPLACES. The previous stack was scaled with min/max taken from the
# training blocks of the REAL-DATA spatial split and clipped to [0, 1]: a split
# this experiment never uses, whose clipping also flattened ~1.5 M Road_Density
# pixels to exactly 1.0.
# one layer at a time: np.nanmin over the whole cube would copy all of it
env_min_100m = np.array([np.nanmin(_env_stack_100m[..., k])
                         for k in range(_env_stack_100m.shape[2])], np.float32)
env_max_100m = np.array([np.nanmax(_env_stack_100m[..., k])
                         for k in range(_env_stack_100m.shape[2])], np.float32)
env_scaled_100m = ((_env_stack_100m - env_min_100m) /
                   (env_max_100m - env_min_100m)).astype(np.float32)
del _env_stack_100m

H100, W100, N_LAYERS = env_scaled_100m.shape
print(f'env_scaled_100m: {env_scaled_100m.shape}  ({H100}x{W100} px, {N_LAYERS} layers, 100m res)')
print(f'Layers: {LAYER_NAMES}')
print(f'Value range: [{np.nanmin(env_scaled_100m):.3f}, {np.nanmax(env_scaled_100m):.3f}]')
assert abs(np.nanmin(env_scaled_100m)) < 1e-6 and abs(np.nanmax(env_scaled_100m) - 1) < 1e-6

### 2.1 — Model Inputs After Scaling
The six channels the models receive, on one common 0–1 colour scale. Landuse is not among them — it leaves the stack before the scaling and is only the source of the forest field.

In [ ]:
# ── The model inputs as the models see them: after scaling ────────────────────
# ONE common colour scale [0, 1] for every panel -- that is the point of the
# figure. After the per-layer min-max each channel spans exactly the same range,
# so no channel enters the CNN with a larger numeric magnitude than another
# merely because of its unit. Landuse is not shown: it leaves the stack before
# the scaling and is not a model input. The configurational arms add the forest
# field W as one more channel at run time; it is not part of this stack either.
_step_sc = max(1, max(H100, W100) // 800)        # downsample for display only
_n_sc, _nc_sc = len(LAYER_NAMES), 3
_nr_sc = -(-_n_sc // _nc_sc)
fig, axes = plt.subplots(_nr_sc, _nc_sc, figsize=(14, _nr_sc * 4.2))
axes = np.atleast_1d(axes).ravel()
for _i_sc, _nm_sc in enumerate(LAYER_NAMES):
    _im_sc = axes[_i_sc].imshow(env_scaled_100m[::_step_sc, ::_step_sc, _i_sc],
                                cmap='viridis', vmin=0, vmax=1, interpolation='nearest')
    axes[_i_sc].set_title(_nm_sc, fontsize=11)
    axes[_i_sc].axis('off')
for _j_sc in range(_n_sc, len(axes)):
    axes[_j_sc].axis('off')
# NO FIGURE TITLE: the thesis caption carries it. The per-panel titles stay --
# they name which channel each map is, which no caption can do.
_cb_sc = fig.colorbar(_im_sc, ax=axes.tolist(), orientation='horizontal',
                      fraction=0.030, pad=0.03)
_cb_sc.set_label('scaled value', fontsize=10)
plt.savefig(os.path.join(FIGURES_DIR, '00_scaled_model_inputs.png'), dpi=130,
            bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"saved 00_scaled_model_inputs.png -- {LAYER_NAMES}, "
      f"range [{np.nanmin(env_scaled_100m):.3f}, {np.nanmax(env_scaled_100m):.3f}]")


### 2.2 — Raster Helpers
Resampling (integer block factors only), validity masks and patch extraction.

In [ ]:
# ============================================================
# Raster helpers used by the tuner and all of Section 8
# ============================================================
# resample_env    block-average the 100 m stack to a coarser grid (integer factors
#                 only -- it refuses 320 m rather than silently delivering 300 m)
# make_valid_mask pixels whose full patch lies inside the valid area
# extract_patches cut PATCH x PATCH windows around point locations
#
# Restored from the original notebook's section-3 cell. That cell also held the
# old per-predictor Gaussian species generator (compute_suitability,
# sample_points), which this notebook no longer uses -- only these three helpers
# are needed, and without them the tuner and Section 8 fail on a NameError.

def resample_env(env_base, target_res_m, base_res_m=100):
    if target_res_m == base_res_m:
        return env_base
    # The factor MUST be an integer: this is a block mean, and a block mean over
    # a fractional number of cells is not defined. The old form rounded silently,
    # so resample_env(env, 320) returned a 300 m grid while every caller went on
    # believing it had 320 m pixels. Refuse instead of rounding.
    _fac = target_res_m / base_res_m
    if abs(_fac - round(_fac)) > 1e-9:
        raise ValueError(
            f'resample_env: {target_res_m} m is not a whole multiple of the '
            f'{base_res_m} m base grid (factor {_fac:.3f}). Block averaging needs '
            f'an integer factor; rounding it would return a different grid than '
            f'the one asked for.')
    factor = int(round(_fac))
    H, W, C = env_base.shape
    H_c, W_c = (H // factor) * factor, (W // factor) * factor
    H_r, W_r = H_c // factor, W_c // factor
    result    = np.empty((H_r, W_r, C), dtype=np.float32)
    for k in range(C):
        # process one layer at a time: peak = 1 layer (~220 MB) instead of full raster (1.34 GB)
        lyr = np.ascontiguousarray(env_base[:H_c, :W_c, k])   # (H_c, W_c)
        result[:, :, k] = np.nanmean(
            lyr.reshape(H_r, factor, W_r, factor), axis=(1, 3))
        del lyr
    return result


def make_valid_mask(env_arr, suit, patch):
    half = patch // 2
    v = np.all(np.isfinite(env_arr), axis=2) & np.isfinite(suit)
    v[:half, :] = False;  v[-half:, :] = False
    v[:, :half] = False;  v[:, -half:] = False
    return v


def extract_patches(env_arr, rows, cols, patch=32):
    half = patch // 2
    C = env_arr.shape[2]
    X = np.empty((len(rows), patch, patch, C), dtype=np.float32)
    for i, (r, c) in enumerate(zip(rows, cols)):
        X[i] = env_arr[r-half:r+half, c-half:c+half, :]
    return X


## 3 — PCA Virtual-Species Generator (`3b`)
A 2-component PCA fitted on 200 000 raster pixels of the 6 model inputs, and the virtual-species generator built on it: a 2-D Gaussian niche on PC1/PC2 with probabilistic occurrence (logistic + Bernoulli). `make_vs(seed)` returns a species' niche parameters, `make_vs_raster(seed)` its full suitability raster. The key `3b` is kept because code names (`_pca3b`, `_vs3b_*`) refer to it.

In [ ]:
# ── Fit PCA on a raster pixel sample (200 K) ──────────────────────────────────
import gc as _gc3b
from sklearn.preprocessing import StandardScaler as _SC3b
from sklearn.decomposition import PCA as _PCA3b

_valid_px3b = np.all(np.isfinite(env_scaled_100m), axis=2)  # boolean mask of valid pixels (all channels non-NaN)
_vr3b, _vc3b = np.where(_valid_px3b)    # row/col indices of valid pixels
_rng3b  = np.random.default_rng(42)     # fixed seed for reproducibility
_idx3b  = _rng3b.choice(len(_vr3b), size=min(200_000, len(_vr3b)), replace=False)   # sample 200 K valid pixels (or fewer if not enough valid pixels)
_E3b    = env_scaled_100m[_vr3b[_idx3b], _vc3b[_idx3b], :]   # (n_samp, C); only valid pixels, all channels
_sc3b   = _SC3b().fit(_E3b)         # standardize features by removing the mean and scaling to unit variance
_Es3b   = _sc3b.transform(_E3b)     # apply the standardization to the sampled pixels
_pca3b  = _PCA3b(n_components=2, random_state=42).fit(_Es3b)    # fit PCA on the standardized sampled pixels
_pcsmp  = _pca3b.transform(_Es3b)                               # project the standardized sampled pixels onto the PCA components (PC1 and PC2)
_vs3b_q10 = np.percentile(_pcsmp, 10, axis=0)    # (2,) — VS param sampling bounds
_vs3b_q90 = np.percentile(_pcsmp, 90, axis=0)
_vs3b_std = _pcsmp.std(axis=0) + 1e-8           # add small epsilon to avoid division by zero in case of zero std
_vs3b_pcsmp = _pcsmp.astype(np.float32)   # (200k, 2) RETAINED, ~1.6 MB
del _E3b, _Es3b, _pcsmp; _gc3b.collect()

print(f'3b PCA  EVR: PC1={_pca3b.explained_variance_ratio_[0]:.3f}  '
      f'PC2={_pca3b.explained_variance_ratio_[1]:.3f}')

# ── Shared PCA niche draw loop ─────────────────────────────────────────────────
# Called by both make_vs_raster (raster context) and make_vs pca-branch (point context).
# RNG call order per attempt: rng.random(2), rng.uniform(0.4,1.5,2), rng.uniform(0.5,2.0).
# This is identical to the pre-refactor loops in both functions → behaviour-preserving.
def _vs_pca_draw(rng, q10, q90, pc_std, pc_scr_valid,
                  max_attempts=50, min_suit_frac=0.01):
    """
    Draw niche parameters until a non-degenerate VS is accepted.
    pc_scr_valid: (N, 2) PC scores for valid points/pixels only.
    Returns (suit_valid, mus, sigs, power, n_attempts) or None on exhaustion.
    """
    for attempt in range(max_attempts):
        mus   = q10 + rng.random(2) * (q90 - q10)   # contains the means for PC1 and PC2 of the virtual species' niche; 
            # drawn randomly (function draws numbers between 0 and 1) from a uniform distribution 
            # between the 10th and 90th percentiles of the PCA scores of the valid pixels. 
        sigs  = pc_std * rng.uniform(0.4, 1.5, 2)   # randomly choose niche widths (standard deviations) for PC1 and PC2 
            # from a uniform distribution between 0.4 and 1.5 times the standard deviation of the PCA scores of the valid pixels.
        power = float(rng.uniform(0.5, 2.0))    # modifies the shape of the Gaussian (1 = standard Gaussian, <1 = flatter, >1 = sharper)
        # calculate suitability values for all valid pixels based on the drawn niche parameters (mus, sigs, power) using a Gaussian function in the PCA space.
        # asks how far away is each pixel from the niche center. closer pixels get higher suitability, and farther pixels get lower suitability.
        sv    = np.exp(-0.5 * (
                    ((pc_scr_valid[:, 0] - mus[0]) / sigs[0])**2 +
                    ((pc_scr_valid[:, 1] - mus[1]) / sigs[1])**2
                )).astype(np.float32)
        sv   **= power  # apply power
        # normalize highest suitability value to 1
        mx = float(sv.max())
        if mx <= 0:
            continue
        sv /= mx
        # frac is the proportion of valid pixels belonging to the high-suitability portion (>= 80th percentile). 
        # If this fraction is greater than or equal to min_suit_frac, the drawn niche parameters are accepted, 
        # and the function returns the suitability values, means, standard deviations, power, and the number of attempts made. 
        # If no valid niche is found after max_attempts, the function returns None.
        frac = float((sv >= np.nanpercentile(sv, 80)).mean())
        if frac >= min_suit_frac:
            return sv, mus, sigs, power, attempt + 1
    return None

# ── make_vs_raster(seed) → (H, W) suitability ─────────────────────────────────
# Transforms valid pixels chunk-by-chunk each call (avoids ~440 MB persistent array).
# Called once per tuner / baseline run — performance is acceptable.
def make_vs_raster(seed, max_attempts=50, min_suit_frac=0.01):
    """
    PCA virtual species as a full (H×W) suitability raster.
    Use this in raster-context calls (Keras tuner, baseline run).
    Section 11+ experiments use make_vs(seed, mode='pca') instead.
    Niche draw logic is shared with make_vs via _vs_pca_draw().
    """
    import gc
    rng   = np.random.default_rng(seed)
    H_, W_ = env_scaled_100m.shape[:2]  # height and width of the environmental raster
    _CHK  = 500_000 # chunk size for processing valid pixels in batches (to avoid memory issues)

    # Compute PC scores for all valid pixels (done once per call, freed on exit)
    _pc01 = np.empty((len(_vr3b), 2), np.float32)   # preallocate array for PCA scores of valid pixels, 2 for PC1 and PC2
    # Process valid pixels in chunks to avoid memory issues
    for _s in range(0, len(_vr3b), _CHK):
        _e = min(_s + _CHK, len(_vr3b))     # end index for the current chunk
        _b = env_scaled_100m[_vr3b[_s:_e], _vc3b[_s:_e], :] # extract the environmental data for the current chunk of valid pixels
        _p = _pca3b.transform(_sc3b.transform(_b))  # apply standardization and PCA transformation to the current chunk of valid pixels
        _pc01[_s:_e] = _p[:, :2].astype(np.float32) # store the PCA scores (PC1 and PC2) for the current chunk of valid pixels
        del _b, _p

    # Call the shared PCA niche draw loop to generate suitability values for the virtual species based on the PCA scores of valid pixels.
    result = _vs_pca_draw(rng, _vs3b_q10, _vs3b_q90, _vs3b_std,
                           _pc01, max_attempts, min_suit_frac)
    del _pc01; gc.collect()

    if result is None:
        return None
    sv, _, _, _, _ = result             # only extract sv and discard the rest
    suit = np.full((H_, W_), np.nan, np.float32)    # create empty raster filled with NaNs
    suit[_vr3b, _vc3b] = sv             # fill the valid pixel locations in the raster with the computed suitability values
    return suit

# ── Smoke check ───────────────────────────────────────────────────────────────
_s3b = make_vs_raster(42)
if _s3b is not None:
    _ok3b = ~np.isnan(_s3b)
    print(f'make_vs_raster smoke: range=[{_s3b[_ok3b].min():.3f},{_s3b[_ok3b].max():.3f}]  '
          f'mean={_s3b[_ok3b].mean():.4f}  n_valid={_ok3b.sum()}  PASS')
    del _s3b
else:
    print('make_vs_raster smoke: FAILED — check env data')

# ── Behaviour-preservation check (run to verify after refactor) ───────────────
# _c1 = make_vs_raster(42); _c2 = make_vs_raster(42)
# assert _c1 is not None and _c2 is not None
# assert float(np.nanmax(np.abs(_c1 - _c2))) == 0.0, 'make_vs_raster not deterministic!'
# del _c1, _c2


# =============================================================================
# SELF-CONTAINED VIRTUAL-SPECIES GENERATOR  (removes Section 8's backward deps)
#
# WHY THIS EXISTS. Section 8 used to call make_vs() from the vs_gen_shared cell,
# which sits ~45 cells LATER in the notebook and itself needs env_pts from the
# 11a cell. Section 8 therefore could not run without first executing two cells
# that belong to a different part of the analysis -- and one of those (c11b) is
# currently broken (LAYER_NAMES is 7-wide while env_pts is 9-wide). This
# definition closes that loop: everything it needs is computed above in THIS
# cell.
#
# WHY IT IS SAFE TO SWAP. Section 8 reads exactly three fields from the returned
# meta -- mus_pc, sigs_pc, power -- and those are drawn from (rng, q10, q90,
# pc_std) alone, all raster quantities defined above. The point cloud entered
# only the acceptance test
#     frac = (sv >= nanpercentile(sv, 80)).mean() >= MIN_SUITABLE_FRAC
# which is ~0.20 by construction and rejects only degenerate draws. Verified
# against the point-based implementation for all 16 seeds of the Section-8
# candidate pool: every seed accepted on attempt 1 and mus_pc / sigs_pc / power
# came out bit-identical. Swapping the acceptance population therefore cannot
# change which species Section 8 draws.
#
# PCA MODE ONLY, ON PURPOSE. The 'original_gaussian' branch needs _GAUSS_IDX,
# q10/q90/std, _LU_IDX and the driver-correlation guard, all of which live in
# c11b -- importing them here would re-create the dependency this removes.
# Section 10d is the only consumer of that mode and it runs after c11b anyway,
# so the later vs_gen_shared definition still serves it.
# =============================================================================
from scipy.optimize import brentq as _vs_brentq

MIN_SUITABLE_FRAC = 0.01
MAX_DRAW_ATTEMPTS = 50


def _vs_logistic(s, c, k):
    return 1.0 / (1.0 + np.exp(-float(k) * (np.asarray(s, np.float64) - float(c))))


def _vs_solve_c(suit_vals, k, target_prev=0.20):
    """Logistic intercept c with mean(logistic(suit, c, k)) == target_prev."""
    a = np.asarray(suit_vals, np.float64)
    def f(c): return _vs_logistic(a, c, k).mean() - target_prev
    try:
        return float(_vs_brentq(f, -0.5, 1.5))  # Brent’s root-finding algorithm
    except Exception:
        return float(np.percentile(a, (1 - target_prev) * 100))


def make_vs(seed, mode='pca', occ='probabilistic', k=10, prevalence=0.20):
    """One virtual species, Leroy-style 2-D Gaussian on PC1/PC2.

    Returns (suit, thresh, meta) -- the same contract the vs_gen_shared cell
    uses, so Section 8 needs no change at the call site.
      suit   (200k,) float32 suitability on the retained PCA sample. Section 8
             ignores the values and only checks that the draw succeeded; the
             surface it actually models is rebuilt per resolution by
             _s8_suit_fn from mus_pc/sigs_pc/power.
      thresh 80th-percentile of suit.
      meta   mode, occ, n_attempts, mus_pc, sigs_pc, power, suitable_frac, and
             logit_c when occ='probabilistic'.
    On exhaustion returns (None, None, meta_with_rejected) -- callers test
    `_r[0] is None`, which is how a degenerate seed is skipped and reported.
    """
    if mode != 'pca':
        raise ValueError(
            f"make_vs (cell 3b) implements mode='pca' only, got {mode!r}. "
            "The 'original_gaussian' branch lives in the vs_gen_shared cell "
            "because it depends on c11b's driver indices.")

    rng   = np.random.default_rng(seed)
    _draw = _vs_pca_draw(rng, _vs3b_q10, _vs3b_q90, _vs3b_std,
                         _vs3b_pcsmp, MAX_DRAW_ATTEMPTS, MIN_SUITABLE_FRAC)
    if _draw is None:
        return None, None, dict(mode=mode, occ=occ,
                                n_attempts=MAX_DRAW_ATTEMPTS, rejected='max_attempts')
    sv, mus, sigs, power, n_att = _draw     # unpack niche params
    thresh    = float(np.nanpercentile(sv, 80))     # highest 20% suitability sample
    suit_frac = float((sv >= thresh).mean())    # fraction of the sampled locations with suit ≥ threshold?
    meta = dict(mode=mode, occ=occ, n_attempts=n_att,       # store species params
                mus_pc=mus.copy(), sigs_pc=sigs.copy(), power=power,
                suitable_frac=suit_frac)
    if occ == 'probabilistic':
        meta['logit_c'] = _vs_solve_c(sv, k, prevalence)    # calculate c for occupancy
    return sv, thresh, meta


make_vs._impl = 'cell3b_raster'   # provenance tag; the later vs_gen_shared
                                  # definition shadows this one if that cell is
                                  # re-run, so Section 8 prints which is live.

_mv3b = make_vs(42, mode='pca')
print(f"make_vs (3b, self-contained) smoke: mus={np.round(_mv3b[2]['mus_pc'],3)} "
      f"power={_mv3b[2]['power']:.3f} attempts={_mv3b[2]['n_attempts']} PASS")
del _mv3b


### 3.1 — Virtual-Species Visualisation
One species' suitability surface, as a map and in PC1/PC2 space.

In [ ]:
# ── PCA virtual-species visualisation ─────────────────────────────────────────
_viz_seed = SEED   # 42 — same fixed seed used throughout
_suit4b   = make_vs_raster(_viz_seed)   # (H, W) suitability raster

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: PCA suitability map
im4b = axes[0].imshow(_suit4b, cmap='YlOrRd', origin='upper', interpolation='none',
                       vmin=0, vmax=1)
axes[0].set_title('PCA Virtual Species — Suitability (100 m)', fontsize=12)
axes[0].axis('off')
cb4b = plt.colorbar(im4b, ax=axes[0], fraction=0.03, pad=0.02)
cb4b.set_label('Suitability [0\u20131]', fontsize=9)

# Right: niche shape in PC1\u00d7PC2 space
# Sample valid raster pixels and colour them by their suitability value.
# The 2-D Gaussian response is visible without re-running the parameter draw.
_n4b   = min(60_000, len(_vr3b))
_rng4b = np.random.default_rng(0)
_sel4b = _rng4b.choice(len(_vr3b), size=_n4b, replace=False)
_E4b   = env_scaled_100m[_vr3b[_sel4b], _vc3b[_sel4b], :]
_P4b   = _pca3b.transform(_sc3b.transform(_E4b))          # (n4b, 2)
_S4b   = _suit4b[_vr3b[_sel4b], _vc3b[_sel4b]]            # suitability for sample
_ok4b  = ~np.isnan(_S4b)
sc4b   = axes[1].scatter(
    _P4b[_ok4b, 0], _P4b[_ok4b, 1],
    c=_S4b[_ok4b], cmap='YlOrRd', s=0.4, alpha=0.5, vmin=0, vmax=1,
)
plt.colorbar(sc4b, ax=axes[1], fraction=0.03, pad=0.02).set_label('Suitability', fontsize=9)
axes[1].set_xlabel(
    f'PC1 ({_pca3b.explained_variance_ratio_[0]:.1%} var)', fontsize=10)
axes[1].set_ylabel(
    f'PC2 ({_pca3b.explained_variance_ratio_[1]:.1%} var)', fontsize=10)
axes[1].set_title('Niche Shape in PCA Space (pixel sample)', fontsize=12)
axes[1].grid(alpha=0.3) # set grid lines for the right subplot, alpha=0.3 for lightness

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '01b_pca_vs_surface.png'), dpi=130, bbox_inches='tight')
plt.show()
del _E4b, _P4b, _S4b, _suit4b
print('PCA-VS visualisation saved.')

## 4 — Landuse / PCA Orthogonality Check
Spearman correlations of local forest cover (P_w) and edge density (E) — the ingredients of the configurational driver — with PC1 and PC2, at the ecological window actually used (r_m = 1600 m) and a smaller one, on an interior-buffered mask. Descriptive only, no p-values. It is the reason the configurational signal is built on edge density, which is near-orthogonal to the niche axes, rather than on cover, which is not.

In [ ]:
## ── Landuse / PC1 / PC2 orthogonality check ───────────────────────────────────
# Needs _pca3b/_sc3b from the PCA generator (section 3), which now runs BEFORE
# this cell -- the notebook executes top to bottom. The guard below only fires
# if the cells are run out of order.
# The check is a sanity check to see if the landuse layer is orthogonal to the first two PCA components 
# of the other environmental layers. If they are not orthogonal, it may indicate that the landuse layer 
# is capturing information that is already present in the other layers, which could lead to redundancy 
# and overfitting in the model. The check is performed by computing the Spearman correlation between 
# the landuse layer and the first two PCA components (PC1 and PC2) of the other environmental layers
import time as _t_lu    # to measure time of code execution
from scipy.ndimage import uniform_filter, minimum_filter
from scipy.stats import spearmanr

if '_pca3b' not in dir() or '_sc3b' not in dir():
    raise RuntimeError('Run cell 3b before this cell -- it fits _pca3b/_sc3b, '
                       'and this check deliberately reuses that fit rather than '
                       'refitting a second PCA on a different pixel subset.')

_t_start = _t_lu.time()
def _lap(label):
    global _t_prev  # treat _t_prev as a global variable to keep track of the previous lap time
    now = _t_lu.time()
    # print time since previous lap + time since start of the run:
    print(f'  [{label}] {now - _t_prev:.1f}s  (total {now - _t_start:.1f}s)', flush=True)
    _t_prev = now
_t_prev = _t_start
print('Starting landuse/pc1/pc2 orthogonality check...', flush=True)

# --- things that need to be supplied ---
# landuse (raw score 0-5, -1 = no data) and WOODY_CLASSES come from the loader
# cell, which extracts them before Landuse leaves the model stack.
if 'landuse' not in dir() or 'WOODY_CLASSES' not in dir():
    raise RuntimeError('Run the loader cell (section 2) first -- it extracts `landuse` '
                       'and defines WOODY_CLASSES.')

# pc1, pc2: the SAME PC1/PC2 projection _s8_suit_fn uses internally (Section
# 8 setup cell) -- _pca3b/_sc3b were fit once on env_scaled_100m
# (cell 3b), so reuse that fit here rather than refitting a new PCA. Mirrors
# _s8_suit_fn's own NaN-masking (project only pixels with all channels
# valid; everything else stays NaN) instead of a fresh implementation.
_H_pc, _W_pc, _C_pc = env_scaled_100m.shape
_Ef_pc = env_scaled_100m.reshape(-1, _C_pc).astype(np.float32, copy=False)  # copy=False avoids a redundant ~1.5GB copy; -1 means "flatten the first two dims" (H*W, C)
_ok_pc = ~np.isnan(_Ef_pc).any(1)   # boolean mask of pixels with all channels valid (True = valid, False = NaN)
_lap('pc reshape + NaN mask')
print(f'  ({int(_ok_pc.sum())}/{len(_ok_pc)} valid pixels going into _pca3b.transform)', flush=True)    # print number of valid pixels for PCA projection
_pc_vals = _pca3b.transform(_sc3b.transform(_Ef_pc[_ok_pc]))    # using the same _pca3b/_sc3b fit as _s8_suit_fn (Section 8) -- avoids refitting PCA on a different subset of pixels
_lap('_sc3b/_pca3b.transform (the actual PCA projection)')
# create a 2D array of pc1, pc2 values, filling invalid pixels with NaN and reshaping to the original height and width (2d):
pc1 = np.full(_H_pc * _W_pc, np.nan, np.float32); pc1[_ok_pc] = _pc_vals[:, 0]; pc1 = pc1.reshape(_H_pc, _W_pc) 
pc2 = np.full(_H_pc * _W_pc, np.nan, np.float32); pc2[_ok_pc] = _pc_vals[:, 1]; pc2 = pc2.reshape(_H_pc, _W_pc)
del _Ef_pc, _ok_pc, _pc_vals
_lap('pc1/pc2 array assembly')

# landuse was loaded from the ORIGINAL (uncropped) raster, so if
# env_scaled_100m has since been through resample_env or any other
# shape-changing step, crop/align landuse to pc1/pc2's shape before use.
assert landuse.shape == pc1.shape == pc2.shape, \
    'landuse/pc1/pc2 shape mismatch -- crop landuse to match if resolution differs from 100m'

# WOODY_CLASSES ({5} = ESA 'Tree cover') and its class legend: see the loader cell.
resolution_m = 100.0     # current raster resolution

# TWO RADII, NOT ONE. The check used to run only at r_m = 300 m, but the shipped
# configurational DGP uses r_m = 1600 m (2*r_m = PATCH*100, the patch/window
# alignment). A check at 300 m therefore did not validate the field the
# experiment actually builds the species on. Both are reported; the 1600 m row
# is the one to cite.
R_M_CHECK = (300.0, 1600.0)

# W: binary refuge field from real land use. `landuse` carries -1 for no-data,
# which is not in WOODY_CLASSES, so no-data lands in the non-woody class here --
# that is why the INTERIOR BUFFER below is not optional.
W = np.isin(landuse, list(WOODY_CLASSES)).astype(np.float32)
_valid_lu = (landuse >= 0) & np.isfinite(pc1) & np.isfinite(pc2)
print(f'  W: realised woody fraction over valid pixels = '
      f'{float(W[_valid_lu].mean()):.4f}', flush=True)
_lap('W (np.isin)')

# b: per-pixel boundary fraction (4-connectivity). Each pixel is compared with
# its left/right and up/down neighbours; every differing neighbour contributes
# one count to BOTH pixels of the pair, and the sum is divided by 4 (the maximum
# number of 4-connected neighbours), giving a per-pixel value in [0, 1].
b = np.zeros_like(W)
b[:, 1:]  += (W[:, 1:] != W[:, :-1]); b[:, :-1] += (W[:, 1:] != W[:, :-1])
b[1:, :]  += (W[1:, :] != W[:-1, :]); b[:-1, :] += (W[1:, :] != W[:-1, :])
b /= 4.0    # normalize to [0, 1]
_lap('b (boundary fraction)')

# ORTHOGONALITY CHECK -- DESCRIPTIVE, NOT A TEST.
# No p-values are printed. With tens of millions of strongly spatially
# autocorrelated pixels the effective sample size is orders of magnitude below
# n, every rho is "significant", and a p-value here would be a statement about
# the pixel count rather than about the landscape. The rho values and n are the
# reportable quantities.
#
# nan_policy='omit' on scipy's spearmanr scales badly at this size, so NaNs are
# masked with plain numpy boolean indexing first.
def _sp_clean(a, bb, mask, label):
    ok = mask.ravel() & np.isfinite(a) & np.isfinite(bb)
    rho = spearmanr(a[ok], bb[ok])[0]
    print(f'    {label}: rho={rho:+.4f}   n={int(ok.sum()):,}')
    return rho

for _r_m in R_M_CHECK:
    win = max(1, int(round((2 * _r_m) / resolution_m)) | 1)   # odd -> has a centre
    # INTERIOR BUFFER (standard treatment for moving-window landscape metrics at
    # a study-area boundary): a pixel whose window is not fully inside the valid
    # area gets a P_w/E that averages over cells which do not exist. Those pixels
    # are dropped rather than zero-filled. Cost at 1600 m: 5.5% of the valid
    # area (19,510 km2 of 355,750), leaving 94.5%.
    _inner = minimum_filter(_valid_lu.view(np.uint8), size=win,
                            mode='constant', cval=0).astype(bool)
    P_w = uniform_filter(W, size=win, mode='nearest')   # composition
    E   = uniform_filter(b, size=win, mode='nearest')   # configuration
    print(f'  r_m={_r_m:.0f} m  window={win}x{win} px  '
          f'inner mask keeps {100.0*_inner.sum()/max(_valid_lu.sum(),1):.2f}% '
          f'of valid pixels', flush=True)
    _sp_clean(E.ravel(),   pc1.ravel(), _inner, 'E   vs PC1')
    _sp_clean(P_w.ravel(), pc1.ravel(), _inner, 'P_w vs PC1')
    _sp_clean(E.ravel(),   pc2.ravel(), _inner, 'E   vs PC2')
    _sp_clean(P_w.ravel(), pc2.ravel(), _inner, 'P_w vs PC2')
    _lap(f'r_m={_r_m:.0f} m block')
    del P_w, E, _inner

print(f'Done. Total: {_t_lu.time() - _t_start:.1f}s', flush=True)


## 5 — CNN Architecture
2× Conv2D + BatchNorm + MaxPooling, GlobalAveragePooling, Dense, sigmoid output; horizontal-flip augmentation, label smoothing 0.05, 1:1 class weights. One fixed architecture for every experiment.

In [ ]:
def build_cnn(patch, n_layers, hp=None):
    if hp is None:
        hp = BEST_HP
    inp = tf.keras.Input(shape=(patch, patch, n_layers))
    x   = tf.keras.layers.RandomFlip('horizontal')(inp)
    for filters in [hp['filters_1'], hp['filters_2']]:
        x = tf.keras.layers.Conv2D(filters, 3, activation='relu', padding='same')(x)    # 3x3 kernel, 
                # same padding to preserve spatial dimensions, filters = number of outputfeature maps
        x = tf.keras.layers.BatchNormalization()(x) # normalize activations to improve training stability and convergence
        x = tf.keras.layers.MaxPooling2D(2)(x)  # downsample by 2x2 pooling, cuts dim roughly in half
                # "Can I make this spatial representation smaller while keeping the strongest signals?"
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)   # global average pooling to reduce spatial dimensions to 1x1
            # "how present is feature X present in this feature map overall?" (1 value per feature map)
    x   = tf.keras.layers.Dense(hp['dense_units'], activation='relu')(x)    # fully connected layer with ReLU activation
    if hp.get('dropout', 0) > 0:        # randomly turns of a fraction of the neurons during training to prevent overfitting
        x = tf.keras.layers.Dropout(hp['dropout'])(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x) # output layer with sigmoid activation for binary classification
    m   = tf.keras.Model(inp, out)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(hp['lr']),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[tf.keras.metrics.AUC(name='auc', num_thresholds=200)],
    )
    return m


## 6 — Experimental Framework
Shared machinery of all sensitivity experiments (code prefix `_s8_`): the virtual-species ensemble and its screening, probabilistic sampling with spatial thinning (one patch diagonal, enforced within and across train/val/test), the interpolative (random) and extrapolative (PC1-band) splits with matched set sizes, the metrics, and the run-level function that fits RF and CNN on identical data. Defines machinery only — no experiment runs in this cell. The CNN input is rescaled per fold on the training patches only (no clipping); the global scaling defines the species, bands and distances. Every sampled point's CNN patch lies entirely inside the study area (at least the 3.2 km configurational window; in patch sweeps the largest patch), in every arm.

In [ ]:
# ============================================================
# SECTION 8 -- Two-Regime Sensitivity Analysis: Setup + 8a
#
# TWO ARMS per sensitivity level:
#   RANDOM  (interpolative): stratified random train/val/test split.
#     Isolates data-condition sensitivity under interpolation.
#   EXTRAP  (extrapolative): fixed PC1-band spatial fold.
#     Training: PC1 < 40th pct (matches dial).  Val: [40,55].  Test: [55,70].
#     Isolates data-condition sensitivity under extrapolation.
#   CONTRAST: how each data condition affects interpolation vs extrapolation,
#     for RF and CNN.
#
# METRICS (primary -> secondary): spearman_true | mae_true | roc_auc.
# MODELS:  RF + CNN (N_CNN_INIT inits) on the SAME sample each rep.
#
# BUG FIXES vs old train_eval():
#   1. Explicit stratified val split (train_test_split x2) -- no validation_split=.
#   2. class_weight={0:1,1:1} -- 1:1
#   3. RF arm on same data as CNN.
#   4. CNN init variance separated: draw data once -> N_CNN_INIT inits per rep.
#   5. Both-classes guard: val must have >= _S8_MIN_VAL pres AND abs for CNN.
#
# CONFOUND TRACKING (extrapolative arm):
#   eds_decorr(test,train) and eds_decorr(val,train) logged per level.
#   Drift with N_POINTS (8a) and resolution (8c-fixed) is expected; document in results.
#
# RUN-ORDER DEPENDENCY (not auto-enforced):
#   Run cell 3b and the build_cnn cell BEFORE Section 8. That is the whole list.
#   Required in scope: _pca3b, _sc3b, _vr3b, _vc3b, make_vs, build_cnn -- all
#   from cell 3b except build_cnn.
#   Section 8 no longer needs c11b0001 or vs_gen_shared: make_vs is defined
#   self-contained in cell 3b (raster acceptance population instead of the 11a
#   point cloud; verified bit-identical over all 16 pool seeds). Those two cells
#   remain for Sections 10d/11/12/14, which use the original_gaussian mode.
# ============================================================
#
# WHY THE CONTRAST IS THE POINT: both arms are run on the *same* sensitivity
# level (same N_POINTS / PRES_RATIO / RESOLUTION / PATCH) and the *same*
# virtual species, so the only thing that differs between RANDOM and EXTRAP is
# how test relates to train in environment space. Any RF-vs-CNN gap that only
# shows up in EXTRAP (not RANDOM) is attributable to extrapolation, not to the
# data condition itself -- that's what "isolates X under interpolation /
# extrapolation" means in the header above.

import time as _t8
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.ensemble import RandomForestClassifier as _RFC
from sklearn.metrics import roc_auc_score as _roc
from sklearn.decomposition import PCA as _PCA
from sklearn.neighbors import NearestNeighbors as _NNB
from sklearn.model_selection import train_test_split as _tts
from scipy.stats import spearmanr as _sp
import pandas as _pd_s8

# -- Config -------------------------------------------------------------------
SMOKE_TEST_S8 = False    # <- set False for full run
_S8_N_REPS    = 1 if SMOKE_TEST_S8 else 3   # independent sampling reps per level, can be overwritten
_S8_N_INIT    = 1 if SMOKE_TEST_S8 else 3   # CNN random inits on same data (init SD)

# SAMPLE SIZE FOR EVERY SECTION-8 EXPERIMENT THAT DOES NOT SWEEP IT.
#
# 2000, not the notebook-wide N_POINTS=1000. Measured in 8i's own N sweep, the
# CNN is effectively blind to configuration below ~1000 points (rho 0.11 / 0.06
# / 0.24 / 0.37 at N=100 / 250 / 500 / 1000) and only reaches 0.63 at 2000 and
# 0.75 at 5000. At N=1000 every other configurational arm -- 8i's patch,
# prevalence and resolution sweeps, and all of 8j -- was therefore measuring the
# CNN in a regime where it cannot yet express the advantage the section exists
# to test.
#
# 2000 also matches the application better: the real sus_scrofa dataset has 3478
# points, so 1000 tested BELOW the setting the thesis is ultimately about.
#
# WHY SECTION-8-SCOPED AND NOT A CHANGE TO N_POINTS: cells outside Section 8
# (10a/10b/10d and the 11/12/14 series) read the global N_POINTS, and their
# results are not ours to move.
#
# WHY IT APPLIES TO THE POINTWISE ARMS TOO: 8i's factorial panel reads alpha=1
# here against alpha=0 from 8a/8b/8e. Raising N only on the configurational side
# would confound the alpha contrast with sample size.
#
# 5000 was considered and rejected: at one patch diagonal of separation the
# training band holds ~5566 points, so 5000 would sit on the geometric limit and
# trigger N reduction at exactly the levels meant to be the reference.
_S8_N_POINTS  = 200 if SMOKE_TEST_S8 else 2000
_S8_N_WIN     = 300     # val/test window size for extrapolative arm
_S8_MIN_VAL   = 8       # both-classes guard: min pres AND abs in val for CNN
_S8_EPOCHS    = 80; _S8_BATCH = 256; _S8_PAT = 12   # CNN training: epochs, batch size, early stopping patience
# Minimum separation between sampled points, in PATCH UNITS. sqrt(2) = one
# patch diagonal (4525 m at 32 px / 100 m) -- reused rather than inventing a second
# constant. Expressed in patches so it scales correctly with resolution:
# a coarser grid means each patch covers more ground and needs more
# separation to keep patches from overlapping.
_S8_SEP_PATCHES = np.sqrt(2.0)

# -- The one Section-8 RF ------------------------------------------------------
# Every RF in Section 8 (8a/8b/8c-fixed/8e/8i/8j, both arms, center- and
# patch-summary variants) is built here. Single factory on purpose: four
# separate _RFC(...) call sites had already drifted from the rest of the
# project (no max_depth -> unlimited depth).
# HP come from the 7a-RF tuner via RF_BEST_HP (loaded in the config cell);
# random_state fixed per the analysis plan (RF uses a fixed seed, CNN carries the
# init-variance budget).
def _s8_rf():
    return _RFC(random_state=42, n_jobs=1, **RF_BEST_HP)    # use only 1 CPU core per fit


# -- Train-only scaling of the CNN input ---------------------------------------
# The stack is min-max scaled GLOBALLY on load (section 2). That scaling defines
# the virtual species -- PCA, truth, PC1 bands and eds_decorr all live on it --
# and it stays. The CNN input is additionally rescaled per channel with the min
# and max of the TRAINING patches only, so no statistic the network sees was
# computed on validation or test pixels, and in the extrapolative arm val/test
# values beyond the training range stay beyond it.
#   NO CLIPPING. Clipping would collapse every test value past the training range
#   onto the boundary -- deleting exactly how far outside it lies, which is the
#   information the extrapolative arm is about. Values outside [0, 1] are harmless
#   for the network (the first layer is a linear convolution).
#   ON THE STACK, NOT THE RAW DATA. Min-max of an already min-max-scaled channel
#   equals min-max of the raw channel (an affine map of an affine map), so the
#   unscaled stack need not be kept in memory.
#   ORDER. Called after _pad: the padding value is the training mean in the global
#   units, and it goes through the same affine map as every other pixel.
#   The RF needs none of this -- its splits are invariant to monotone rescaling.
# A channel that is constant over the training patches is only shifted (range 1).
def _s8_train_minmax(X_tr, *others):
    lo = X_tr.min(axis=(0, 1, 2))
    hi = X_tr.max(axis=(0, 1, 2))
    span = np.where(hi - lo > 0, hi - lo, 1.0).astype(np.float32)
    f = lambda X: ((X - lo) / span).astype(np.float32)
    return (f(X_tr),) + tuple(f(X) for X in others)


# -- Every sampled point's patch lies inside the study area --------------------
# A point is eligible only if a window around it lies entirely inside the area
# where every covariate is defined. The window is the LARGEST of
#   * the point's own patch -- so no CNN patch at a sampled point is padded;
#   * the configurational truth window 2*r_m -- that truth is undefined in this
#     rim, so without it the alpha = 0 reference of cfgGain would include a border
#     strip that its alpha > 0 counterpart cannot, and the two would describe
#     different areas;
#   * the largest patch of the sweep (sweep_patch) -- so every level of a patch
#     sweep samples the same population and the patch effect is not confounded
#     with area.
# The same rule in every run-level function, so every arm draws from one area.
# Cost at 100 m: 5.5% of the valid area at 33 px, 12.5% at 65 px (8e and the 8i
# patch arm). Border and coastal strips leave the population, so results describe
# the interior of the study area.
from scipy.ndimage import minimum_filter as _s8_minf
_S8_PATCH_OK_CACHE = {}


def _s8_patch_buffer_px(resolution, patch, sweep_patch=None):
    win = int(round(2.0 * _S8_CFG['r_m'] / float(resolution))) | 1
    return max(int(patch) | 1, win, (int(sweep_patch) | 1) if sweep_patch else 0)


def _s8_patch_ok(env_arr, size):
    """(H,W) bool: a size x size window around the pixel is inside the valid area.
    Validity depends only on the grid, so cached per (shape, size)."""
    key = (env_arr.shape[:2], int(size))
    if key not in _S8_PATCH_OK_CACHE:
        while len(_S8_PATCH_OK_CACHE) >= 4:
            _S8_PATCH_OK_CACHE.pop(next(iter(_S8_PATCH_OK_CACHE)))
        _S8_PATCH_OK_CACHE[key] = _s8_minf(
            np.all(np.isfinite(env_arr), axis=2).view(np.uint8),
            size=int(size), mode='constant', cval=0).astype(bool)
    return _S8_PATCH_OK_CACHE[key]

# -- Shared PCA virtual species for all Section 8 sub-experiments -------------
# One virtual species is drawn here and reused for EVERY sub-experiment (8a/8b/
# 8c/8e) and every level/rep/arm within them. This is deliberate: since the
# species (and therefore the TRUE suitability surface) is held constant, any
# difference in model skill across levels/arms can only come from the data
# condition being swept -- not from species-to-species variation.
if 'make_vs' not in dir():
    raise RuntimeError('Run cell 3b before Section 8 -- it defines make_vs.')
# WHICH make_vs IS LIVE. The later vs_gen_shared cell defines a superset of this
# function (pca + original_gaussian) and, if it has been run, SHADOWS the cell-3b
# one. For mode='pca' the two were verified to produce bit-identical niches for
# every seed in the candidate pool, so either is correct here -- but a silent
# swap of the generator is worth seeing, not guessing at.
print(f"make_vs implementation: {getattr(make_vs, '_impl', 'vs_gen_shared (point-based)')}")
# -- Configurational niche: drawn PER SPECIES ---------------------------------
# W is the ENVIRONMENT (where forest is) and is the same for every species living
# in a landscape. What differs between species is how they RESPOND to the
# arrangement of that forest: the preferred edge density mu_E and the tolerance
# around it sig_E (both in SD units of the orthogonalised driver E_perp, which is
# unit-variance by construction, so the ranges mean the same in every landscape).
# Before this, mu_E = sig_E = 1.0 for all species, so at alpha = 1 -- where the
# pointwise niche drops out (S_point**0 = 1) -- every "species" had the identical
# truth surface and the ensemble replicated only point draws.
# mu_P stays at the landscape mean (the rule of the mu_P derivation) and w_P stays
# fixed: letting composition vary would re-entangle it with PC1 (P_w vs PC1 rho
# 0.55 on real land use).
# SEPARATE RANDOM STREAM: drawn from (seed, 0xC0F) rather than from make_vs's
# stream, so every pointwise niche -- and with it everything at alpha = 0 -- is
# bit-identical to what it was. The same species gets the same configurational
# niche in every arm (GRF, landuse, corr-len), so the landuse arm is an ecological
# replicate of the same species.
_S8_CFG_NICHE_RANGES = dict(mu_E=(0.5, 1.5), sig_E=(0.6, 1.4))


def _s8_config_niche(seed):
    _r = np.random.default_rng([int(seed), 0xC0F])
    return {k: float(_r.uniform(*v)) for k, v in _S8_CFG_NICHE_RANGES.items()}


_s8_vs_1d, _, _s8_meta = make_vs(SEED, mode='pca', occ='probabilistic',
                                   k=10, prevalence=0.20)
assert _s8_vs_1d is not None, 'S8 VS exhausted attempts'
_s8_meta = {**_s8_meta, **_s8_config_niche(SEED)}

def _s8_suit_fn(env_arr):
    """2-D PCA-Gaussian suitability at any resolution. env_arr is (H,W,C), the env channels."""
    # Needs to work "at any resolution" because 8c-fixed and 8i block-average the
    # raster to coarser grids -- the suitability surface must be recomputed
    # fresh on each resampled array rather than reused from a fixed 100m grid.
    # Projects each pixel's 7 covariates through the SAME PCA/scaler fit once
    # in cell 3b (_pca3b/_sc3b) -- not refit here -- so every resolution/level
    # shares one consistent PC1/PC2 definition of the species' niche.
    H, W, C = env_arr.shape                         # (H,W,C) 
    Ef = env_arr.reshape(-1, C).astype(np.float32)  # (H·W, C), da PCA+Scaler (n_samples, n_features) erwarten
    ok = ~np.isnan(Ef).any(1)                       # .any(1) prüft pro Pixel, True für Pixel, 
                                                    # die in allen 7 Kanälen gültige Werte haben
    s  = np.full(H * W, np.nan, np.float32)
    pc = _pca3b.transform(_sc3b.transform(Ef[ok]))
    s[ok] = np.exp(                                 # 2D-Gauß-Glocke im PCA-Raum, am Optimum 1
        -0.5 * (((pc[:,0] - _s8_meta['mus_pc'][0]) / _s8_meta['sigs_pc'][0])**2 + # z-stand. Abstand zum Optimum
                ((pc[:,1] - _s8_meta['mus_pc'][1]) / _s8_meta['sigs_pc'][1])**2)
    ).astype(np.float32)
    s[ok] **= _s8_meta['power']     # potenziert die Suitability elementweise, Schärfe der Glocke
    mx = float(np.nanmax(s))        # scaling to 1
    if mx > 0: s /= mx
    return s.reshape(H, W)          # reshaping to map format H, W

# This continuous, noiseless surface (0-1 per pixel) IS the "true suitability."
# _s8_run_level() later reads it at the sampled test-point locations (s_te)
# and compares model predictions against it via spearman_true/mae_true -- the
# ground-truth comparison that only virtual species make possible (see the
# Plot-1 schematic in the one-pager: predictions are scored against this
# known surface, not just against the noisy drawn presence/absence labels).
_s8_suit_100m = _s8_suit_fn(env_scaled_100m)
print(f'S8 VS: suit [{np.nanmin(_s8_suit_100m):.3f},{np.nanmax(_s8_suit_100m):.3f}]  '
      f'mean={np.nanmean(_s8_suit_100m):.4f}', flush=True)

# -- Fixed extrapolative fold: PC1-band thresholds ----------------------------
# Derived from cell-3b 200K pixel sample (same RNG seed=42 -> same pixels).
# eds_decorr for this fixed fold (N=1000, 3-trial avg on 100m raster pixels) is
# computed once below, right after _s8_eds_decorr() is defined, and printed --
# see _S8_EDS_TE_TR / _S8_EDS_VL_TR / _S8_EDS_VL_TE. Expected pattern:
# D(val,train)~D(val,test) (early-stopping val is in the same extrapolative
# regime as test); D(test,train) structurally larger (one band further).
# Distance drift with N_POINTS (8a) and resolution (8c-fixed) is logged per level
# (separately, inside _s8_run_level's extrap arm -- decc_te_tr/decc_vl_tr/decc_vl_te).
#
# PC1 (the dominant axis of the shared PCA) is used as the ordering variable
# because it is the single most informative environmental gradient -- slicing
# training/val/test into contiguous PC1 bands (train: low 40%, val: mid band,
# test: high band) gives a well-defined, reproducible "how far is test from
# train in environment space" axis. Train < 40th pct is the EXTRAPOLATIVE
# training pool; val [40,55] sits just past it (early-stopping under mild
# extrapolation); test [55,70] sits further still (the actual extrapolation
# being measured).
_rng_fold = np.random.default_rng(42)
_i_fold   = _rng_fold.choice(len(_vr3b), min(200_000, len(_vr3b)), replace=False)   # sample 200000 indices
_E_fold   = env_scaled_100m[_vr3b[_i_fold], _vc3b[_i_fold], :].astype(np.float32)   # (200K, C) sample of 100m raster pixels
_pc1_fold = _pca3b.transform(_sc3b.transform(_E_fold))[:, 0]    # scale and PCA-transform to get PC1 values
del _E_fold, _i_fold   # free the 200K-pixel sample once PC1 values are extracted
_S8_PC1_TR  = float(np.percentile(_pc1_fold, 40))    # train < 40th pct
_S8_PC1_VAL = float(np.percentile(_pc1_fold, 55))    # val [40,55]
_S8_PC1_TE  = float(np.percentile(_pc1_fold, 70))    # test [55,70]
del _pc1_fold
print(f'Extrap fold PC1: train<{_S8_PC1_TR:.3f} | val<{_S8_PC1_VAL:.3f} | te<{_S8_PC1_TE:.3f}')

# -- Metric + distance helpers ------------------------------------------------
def _s8_eds_decorr(q_E, r_E, K=5):
    """kNN distance ratio between two point sets. Confound-tracking per level.

    WHAT THE PCA STEP DOES: NOTHING TO THE NUMBERS. It is a FULL-RANK rotation
    (n_components = n_features) without whitening, and Euclidean distances are
    invariant under an orthonormal rotation, so both the numerator and the
    denominator are unchanged and so is their ratio. Verified on v6 cells
    against the identical computation with the PCA line removed: max absolute
    difference 3.5e-08, i.e. float32 rounding. The metric is therefore exactly
    the kNN distance ratio on the GLOBALLY MIN-MAX-SCALED covariates, and the
    name "decorr" describes an operation that does not affect the result. The
    line is kept because the column names decc_* and every recorded v6 value
    come from it; removing it would change nothing numerically but would break
    the link to the stored outputs.

    DIFFERENCE TO plain EDS (eds_norm in the sweep scripts): the FEATURE
    SCALING, not decorrelation. eds_norm standardises each feature with a
    StandardScaler FITTED ON THE REFERENCE SET (the training points of that
    cell), so a covariate that varies little inside the training fold gets
    up-weighted; this metric inherits the global min-max scaling fitted once
    over every pixel where the layer is defined, before any species exists.
    Measured on a v6 extrapolative cell: 1.947 here against 2.564 for eds_norm;
    in the interpolative arm the two agree (1.008 against 0.996). A genuine
    correlation correction would need whitening (PCA(whiten=True)) or a
    Mahalanobis distance; neither is what runs here.

    ALWAYS THE ENVIRONMENTAL CHANNELS ONLY -- never the extra W channel that the
    configurational arms add to the model inputs. Three reasons, each
    sufficient on its own:
      1. The metric measures how far test sits from train ALONG THE AXIS THE
         FOLD IS DEFINED ON (PC1 of the env channels). W is orthogonal to
         that by construction -- that is the entire point of the GRF arm -- so
         it adds distance that is the same in train and test and only dilutes
         the ratio. Measured: extrap -6.7%, random +0.2%.
      2. At 100 m W is BINARY with Var = 0.3*0.7 = 0.21, larger than the
         typical min-max-scaled continuous channel, so it would dominate the
         Euclidean kNN distance for reasons unrelated to extrapolation.
      3. W changes character across the resolution sweep (binary at 100 m, a
         continuous sub-pixel proportion once coarsened), so including it
         would make the metric non-comparable BETWEEN levels -- which is
         exactly the direction it is read in.
    (The same argument is why the 6-level Landuse code is not a model input
    either: Euclidean distance on an ordinal class code is meaningless.)
    """
    # eds_decorr measures ENVIRONMENTAL (covariate-space) dissimilarity between
    # a query point set (q_E) and a reference point set (r_E) -- NOT geographic
    # distance. It is the instrument used to check that the two arms actually
    # behave as designed: the extrap arm's D(test,train) should be large (test
    # is genuinely far from train in environment space) and the random arm's
    # should stay small/flat across levels (interpolation, same distribution).
    # The PCA is refit on r_E at full rank: a rotation only, no whitening and no
    # dimensionality reduction -- and therefore distance-preserving (see the
    # docstring). The reference set r_E is the TRAINING points of the cell.
    if len(r_E) < K+2 or len(q_E) < 1: return np.nan    # r_E: training, q_E: test/val
    p  = _PCA(n_components=r_E.shape[1]).fit(r_E)   # full-rank rotation: leaves all distances unchanged
    rp = p.transform(r_E); qp = p.transform(q_E)
    wm = _NNB(n_neighbors=K+1).fit(rp).kneighbors(rp)[0][:, 1:].mean()  # denominator for normalization
    if wm < 1e-10: return np.nan    # avoid division by near-zero
    return float(_NNB(n_neighbors=min(K, len(rp))).fit(rp).kneighbors(qp)[0].mean() / wm)

# -- One-time diagnostic: measure eds_decorr for this fixed fold ------------
# Replaces a hardcoded reference string with an actual computation, using the
# same _s8_eds_decorr() routine the extrap arm uses at runtime (not a second
# implementation). Classifies a pool of 100m raster pixels into the same
# PC1 train/val/test bands via the already-set _S8_PC1_TR/_VAL/_TE thresholds,
# draws N=1000 pixels per band across 3 independent trials, and averages --
# mirroring the "N_POINTS=1000, 3-trial avg" methodology the old hardcoded
# string claimed but never actually ran. One-time at setup, not per-experiment.
_diag_ok2d = np.all(np.isfinite(env_scaled_100m), axis=2)
_diag_vr, _diag_vc = np.where(_diag_ok2d)   # enumerates every valid pixel's (row, col) coordinate
_diag_pc1 = _pca3b.transform(
    _sc3b.transform(env_scaled_100m[_diag_vr, _diag_vc, :].astype(np.float32)))[:, 0]
_diag_tr_idx  = np.where(_diag_pc1 <  _S8_PC1_TR)[0]
_diag_val_idx = np.where((_diag_pc1 >= _S8_PC1_TR)  & (_diag_pc1 < _S8_PC1_VAL))[0]
_diag_te_idx  = np.where((_diag_pc1 >= _S8_PC1_VAL) & (_diag_pc1 < _S8_PC1_TE))[0]

# take the 1000 sampled valid-pixel-list positions → look up their real (row, col) → fetch each pixel's env vector 
# → compute eds_decorr(test,train) and eds_decorr(val,train)
_diag_n = 1000
_diag_d_te, _diag_d_vl, _diag_d_vt = [], [], []
for _trial in range(3):
    _rng_diag = np.random.default_rng(1000 + _trial)
    _tr_sel  = _diag_tr_idx[_rng_diag.choice(len(_diag_tr_idx),  min(_diag_n, len(_diag_tr_idx)),  replace=False)]
    _val_sel = _diag_val_idx[_rng_diag.choice(len(_diag_val_idx), min(_diag_n, len(_diag_val_idx)), replace=False)]
    _te_sel  = _diag_te_idx[_rng_diag.choice(len(_diag_te_idx),  min(_diag_n, len(_diag_te_idx)),  replace=False)]
    _E_tr_d  = env_scaled_100m[_diag_vr[_tr_sel],  _diag_vc[_tr_sel],  :].astype(np.float32)
    _E_val_d = env_scaled_100m[_diag_vr[_val_sel], _diag_vc[_val_sel], :].astype(np.float32)
    _E_te_d  = env_scaled_100m[_diag_vr[_te_sel],  _diag_vc[_te_sel],  :].astype(np.float32)
    _diag_d_te.append(_s8_eds_decorr(_E_te_d,  _E_tr_d))
    _diag_d_vl.append(_s8_eds_decorr(_E_val_d, _E_tr_d))
    _diag_d_vt.append(_s8_eds_decorr(_E_val_d, _E_te_d))

_S8_EDS_TE_TR = float(np.nanmean(_diag_d_te))
_S8_EDS_VL_TR = float(np.nanmean(_diag_d_vl))
_S8_EDS_VL_TE = float(np.nanmean(_diag_d_vt))
del _diag_ok2d, _diag_vr, _diag_vc, _diag_pc1, _diag_tr_idx, _diag_val_idx, _diag_te_idx
print(f'Computed eds_decorr(N={_diag_n}, 3-trial avg): '
      f'D(te,tr)={_S8_EDS_TE_TR:.3f}  D(vl,tr)={_S8_EDS_VL_TR:.3f}  D(vl,te)={_S8_EDS_VL_TE:.3f}')


def _s8_mets(proba, y, suit):
    """spearman_true (primary), mae_true (calibration), roc_auc + auc_oracle (secondary)."""
    # spearman_true: rank correlation of predicted probability vs the TRUE
    #   continuous suitability (suit). Primary metric -- it's the one ground-
    #   truth comparison the drawn labels can't give you, and it's what makes
    #   the virtual-species framework worth using at all.
    # mae_true: mean absolute error of predicted probability vs true suitability.
    #   COMPUTED BUT NOT REPORTED -- it is not comparable between the two models.
    #   Two independent reasons: (1) the CNN trains under label_smoothing=0.05
    #   and the RF does not, so only the CNN's probabilities are shrunk toward
    #   0.5 by construction; (2) both models estimate the posterior of a
    #   BALANCED case-control sample, which is not on the scale of the
    #   max-normalised true suitability it is differenced against. Kept in the
    #   CSV as a raw diagnostic; the thesis reports spearman_true and roc_auc,
    #   both rank-based and unaffected by either problem.
    # roc_auc: AUC against the NOISY DRAWN labels (y) -- the metric a real
    #   study would actually compute, included as a secondary/comparison
    #   point precisely because it's contaminated by the Bernoulli draw noise
    #   that spearman_true sidesteps.
    # auc_oracle: AUC of the true suitability itself (used as a "prediction")
    #   against the drawn labels -- the best possible roc_auc any model could
    #   achieve given the label noise at this prevalence/steepness. The gap
    #   between a model's roc_auc and auc_oracle is the noise ceiling, not
    #   model error.
    # NOTE (deliberate omission): there is no Spearman-form twin of
    #   auc_oracle here. Spearman(drawn labels, truth) is NOT a ceiling on
    #   spearman_true -- AUC is natively defined on (binary labels,
    #   continuous score) so auc_oracle reaches 1.0 for a perfect split,
    #   but Spearman between a 2-valued and a continuous vector is capped
    #   far below 1.0 by binarity alone (~0.87 at prevalence 0.5, ~0.38 at
    #   0.05, with ZERO noise), and that cap moves with prevalence -- so in
    #   8b it would swing ~2.8x for pure metric reasons. Where a genuine
    #   Spearman ceiling exists (information actually destroyed by
    #   coarsening), it lives in 8c-fixed as oracle_spearman_infoloss,
    #   with spearman_rel_ceiling as skill relative to it (a ratio of
    #   correlations, not a share -- see the caveat block in 8c-fixed).
    #   Everywhere else the Spearman ceiling is simply 1.0 and nothing is
    #   reported: spearman_true stands on its own.
    out = {'spearman_true': np.nan, 'mae_true': np.nan, 'roc_auc': np.nan, 'auc_oracle': np.nan}
    ok  = np.isfinite(proba) & np.isfinite(suit)
    if ok.sum() >= 5:
        try: out['spearman_true'] = float(_sp(proba[ok], suit[ok])[0])
        except: pass
        out['mae_true'] = float(np.abs(proba[ok] - suit[ok]).mean())
    ok2 = np.isfinite(proba)
    if ok2.sum() >= 10 and y[ok2].sum() >= 1 and (1-y[ok2]).sum() >= 1: # need at least 1 pres and 1 abs to compute roc_auc
        try: out['roc_auc'] = float(_roc(y[ok2], proba[ok2]))
        except: pass
    ok3 = np.isfinite(suit)
    if ok3.sum() >= 10 and y[ok3].sum() >= 1 and (1-y[ok3]).sum() >= 1:
        try: out['auc_oracle'] = float(_roc(y[ok3], suit[ok3]))
        except: pass
    return out

# -- Extrapolative region masks (deterministic, border-excluded) ---------------

# -- Occurrence + sampling for Section 8 -------------------------------------
# SCOPED DELIBERATELY: this replaces sample_points for Section 8 ONLY. The
# global sample_points is left untouched because Sections 11/12/14 depend on
# its exact behaviour, and this changes results.
#
# TWO CHANGES vs sample_points:
#
# 1. OCCURRENCE IS NOW PROBABILISTIC (literature-standard; Leroy's
#    virtualspecies, Meynard & Kaplan). Suitability is passed through a
#    logistic with steepness k, the intercept c is solved so landscape
#    occupancy equals _S8_OCC_PREV, and occupancy is a Bernoulli draw per
#    pixel. The previous design drew absences *because* the truth said the
#    site was bad, so every absence was genuinely unsuitable and no
#    suitable-but-unoccupied site could exist -- an unrealistically clean
#    discrimination task. Now a good site can come up absent, which is what
#    dispersal limitation and local extinction actually produce.
#
#    Presence/absence BALANCE is unchanged: n_pres/n_abs are still exact
#    quotas, drawn uniformly within the occupied and unoccupied classes
#    (standard balanced case-control sampling). So landscape occupancy
#    (_S8_OCC_PREV) and sample balance (pres_ratio) are two different numbers
#    and both are controlled -- the confound checks keep working.
#
# 2. SPATIAL THINNING. sample_points was a pure lottery with no minimum
#    separation, so ~31% of points had a neighbour closer than one patch
#    width and their CNN patches overlapped by up to ~97%. In the random arm,
#    where train/val/test are split from ONE draw, that is train/test leakage
#    for the CNN. Points are now thinned to a minimum separation, default one
#    patch DIAGONAL -- the same 4525 m (at 32 px / 100 m) rule for spatial splits.
#    Thinning is JOINT over presences and absences: a presence sitting on top
#    of an absence overlaps just as badly as two presences.

# -- Every dropped cell is RECORDED, not just skipped ------------------------
# There were seven `continue` paths in the run-level functions that discarded a
# whole (species, arm, level, rep) cell without emitting a row OR a message. A
# missing row was then indistinguishable from a cell that was never attempted,
# which is exactly the ambiguity an analysis reading only the CSV cannot resolve.
# (Audited on the v4 run: none of them actually fired -- the 6 missing cells
# there were all printed _S8SampleSkip messages -- so this is a latent trap being
# closed, not a repair of known-lost data.)
_S8_DROPS = []


def _s8_drop(sub_exp, level, arm, rep, why):
    """Record AND print a cell that will not be fitted. Never silent."""
    _S8_DROPS.append(dict(sub_exp=sub_exp, level=level, arm=arm, rep=rep, why=why))
    print(f'  DROP ({arm} {sub_exp} lv={level} rep={rep}): {why}', flush=True)


def _s8_drop_report(label=''):
    """Print every cell dropped so far. Call at the end of each sweep."""
    if not _S8_DROPS:
        print(f'  [drops{" " + label if label else ""}: none]', flush=True); return
    print(f'  [drops{" " + label if label else ""}: {len(_S8_DROPS)} cell(s)]', flush=True)
    for _d in _S8_DROPS:
        print(f'    {_d["sub_exp"]} lv={_d["level"]} {_d["arm"]} rep={_d["rep"]}: {_d["why"]}',
              flush=True)


class _S8SampleSkip(RuntimeError):
    """A (species, arm, level) cell that cannot support a valid sample.

    Two things must both be true for a cell to be worth fitting: enough
    occupied/unoccupied pixels to fill the quota, and occupancy that actually
    tracks habitat. A sharply-peaked species whose optimum lies outside the
    extrapolative TRAINING band fails the second -- inside that band its
    suitability is numerically almost constant, so a Bernoulli draw there is
    close to a coin flip and nothing can be learned from it.

    That is a real property of the species-by-fold combination, not an error,
    and it needs the middle path between the two wrong options: crashing a
    multi-hour run, or silently recording a near-zero score that would be read
    as "the model failed" when the truth is "there was nothing to fit". The
    run-level functions catch this, skip the cell, and print it.
    """


_S8_OCC_MODE = 'probabilistic'   # 'probabilistic' | 'quota' (the pre-fix behaviour)
_S8_OCC_K    = 10.0              # logistic steepness; primary k=10
_S8_OCC_PREV = 0.20              # LANDSCAPE occupancy, not the sample balance
_S8_THIN     = True              # spatial thinning on/off
# Occupancy must be distinguishable from a coin flip -- but 'distinguishable'
# depends on how many positives the band affords, so a fixed cutoff is wrong
# at both ends. Measured: with 18 positives a 0.60 cutoff sits 1 SE above
# chance (rejecting on noise); with 100,000 it sits 107 SE above (rejecting
# genuinely learnable weak signal). The test is therefore N SE above chance,
# with a small absolute floor so that a statistically-detectable but
# practically useless signal at huge N still counts as degenerate.
# Cells that PASS but are weak are not hidden: auc_oracle is recorded per row
# and states the achievable ceiling, so the analysis can condition on it.
# Below this many points nothing is learnable, so the cell is skipped rather
# than reported at a meaningless sample size.
_S8_MIN_POINTS = 100
_S8_OCC_AUC_NSE   = 2.0    # SE above chance
_S8_OCC_AUC_FLOOR = 0.52   # practical floor regardless of N


# standard logistic function, used to convert suitability to occupancy probability
# converts each suitability value s into a number between 0 and 1, with a threshold at c and steepness k
# higher suitability -> higher probability of occupancy
def _s8_logistic(s, c, k):
    return 1.0 / (1.0 + np.exp(-k * (np.asarray(s, np.float64) - c)))


# find the right c, taht makes mean(probability) == target_prev, using bisection method
def _s8_solve_c(sv, k, target_prev, max_n=200_000, seed=0):
    """Intercept c such that mean(logistic(sv, c, k)) == target_prev.
    Bisection (monotone decreasing in c), so no scipy.optimize import.
    SOLVED ON A SUBSAMPLE: the intercept is a property of the suitability
    DISTRIBUTION, and 200K pixels pin the mean to ~1e-3. Solving on all valid
    pixels meant 60 logistic evaluations over 35M values -- ~30 s per draw,
    which dominated everything else in the run for no accuracy gain."""
    sv = np.asarray(sv, np.float64)
    if len(sv) > max_n:
        sv = sv[np.random.default_rng(seed).choice(len(sv), max_n, replace=False)]
    # BRACKET FIRST, THEN BISECT. mean(logistic) is monotone DECREASING in c, so
    # the target must lie between the values at the two ends. The old code
    # assumed [-2, 3] always bracketed and silently returned an endpoint when it
    # did not -- a wrong intercept that still looks like a converged answer.
    # The bracket is widened instead of raising: mean(logistic) -> 1 as c -> -inf
    # and -> 0 as c -> +inf, so a non-degenerate suitability distribution always
    # brackets within a few steps and no run is ever lost to this.
    lo, hi = -2.0, 3.0
    for _ in range(40):
        if _s8_logistic(sv, hi, k).mean() <= target_prev <= _s8_logistic(sv, lo, k).mean():
            break
        lo -= 1.0; hi += 1.0
    else:
        raise _S8SampleSkip(
            f'_s8_solve_c: target prevalence {target_prev} unreachable for any c '
            f'in [{lo}, {hi}] at k={k} -- the suitability distribution is degenerate.')
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if _s8_logistic(sv, mid, k).mean() > target_prev:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


# checks whether occupancy is actually related to the suitability surface
def _s8_occ_auc(sv, occ, max_n=50_000, seed=0):
    """AUC of suitability against the DRAWN occupancy -- 'does occupancy
    actually depend on habitat?'. This is the check that catches a degenerate
    species: if the suitability surface is nearly flat, the prevalence
    calibration still hits its target, but by making occupancy essentially
    RANDOM. The occupied-pixel count test cannot see that (there are plenty
    of occupied pixels), so without this a species with no signal would sample
    happily and be scored as a failed model rather than a failed species."""
    n = len(sv)
    idx = (np.random.default_rng(seed).choice(n, max_n, replace=False)  # select pixels
           if n > max_n else np.arange(n))
    s_, o_ = np.asarray(sv)[idx], np.asarray(occ)[idx].astype(bool) # select their suit and occ
    n1, n0 = int(o_.sum()), int((~o_).sum())    # count occ and unocc pixels
    if n1 == 0 or n0 == 0:
        return np.nan, np.nan
    order = np.argsort(s_, kind='mergesort')    # sort suit pixels, kind=mergesort to preserve order of equal elements
    ranks = np.empty(len(s_), np.float64)
    ranks[order] = np.arange(1, len(s_) + 1)    # gives rank according to suit value, 1-based
    _auc = float((ranks[o_].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))   # calculate AUC using ranks of occupied pixels
    _se  = float(np.sqrt((n1 + n0 + 1.0) / (12.0 * n1 * n0)))   # calculate standard error of AUC
    return _auc, _se


# NOTE: es gibt hier BEWUSST keine eigenstaendige _s8_thin()-Funktion mehr.
# Sie existierte als zweite, nie aufgerufene Implementierung neben dem
# Thinner, den _s8_sample_points inline ausfuehrt -- zwei Kopien derselben
# Geometrieregel, die auseinanderlaufen koennen. Der Inline-Thinner unten
# ist die einzige Quelle der Wahrheit.


def _s8_sample_points(suit, valid_mask, n_points, pres_ratio, seed,
                      min_sep_px=0, occ_mode=None, k=None, occ_prev=None,
                      oversample=40, label='', occ_c=None, seed_grid=None):
    """Draw (rows, cols, labels) -- same contract as sample_points.

    Raises with a diagnostic message rather than silently returning a short
    or unbalanced sample: a quietly truncated draw would change n_tr between
    levels and break the confound control everything else depends on."""
    occ_mode = _S8_OCC_MODE if occ_mode is None else occ_mode
    k        = _S8_OCC_K    if k        is None else k
    occ_prev = _S8_OCC_PREV if occ_prev is None else occ_prev

    rng    = np.random.default_rng(seed)
    vr, vc = np.where(valid_mask)       # find all valid pixels in the mask
    if len(vr) == 0:
        raise _S8SampleSkip(f'_s8_sample_points{label}: no valid pixels')
    sv = np.clip(suit[vr, vc].astype(np.float64), 0.0, None)    # extract suitability and clip to [0, inf)
    sv = np.nan_to_num(sv, nan=0.0)     # replacning NaN with 0.0, so that invalid pixels don't contribute to occupancy
    n_pres = max(1, int(n_points * pres_ratio)) # decide on number of pres and abs
    n_abs  = max(1, n_points - n_pres)

    if occ_mode == 'quota':
        # Pre-fix behaviour, kept so the old design can be reproduced exactly.
        pp = np.clip(sv, 1e-8, None); pp /= pp.sum()    # calculates normalized probabilities of pres from suitability
        pa = np.clip(1 - sv, 1e-8, None); pa /= pa.sum()
        pi = rng.choice(len(vr), size=min(n_pres * oversample, len(vr)),    # draws presence candidates
                        replace=False, p=pp)
        ai = rng.choice(len(vr), size=min(n_abs * oversample, len(vr)),
                        replace=False, p=pa)
    else:
        # Probabilistic occupancy, then balanced case-control sampling.
        # The intercept is a LANDSCAPE property, so callers solve it once on the
        # full valid raster and pass it in. Solving it inside each mask instead
        # would force the target occupancy on every band separately -- including
        # an extrapolative training band the species barely occupies, which
        # would invent occupancy there rather than reporting that it is low.
        c = _s8_solve_c(sv, k, occ_prev) if occ_c is None else float(occ_c) # finding c for the logistic function to achieve target occupancy
        p_occ = _s8_logistic(sv, c, k)  # converts suitability to occupancy probability
        occ   = rng.random(len(sv)) < p_occ # Bernoulli draw to determine which pixels are occupied
        n_occ, n_emp = int(occ.sum()), int((~occ).sum())
        # THE HABITAT-AVAILABILITY CHECK. Under a Bernoulli draw the occupied
        # set can be too small to fill the quota -- exactly the failure mode a
        # concentrated S_config produces, and the one nothing checked before.
        if n_occ < n_pres:
            raise _S8SampleSkip(
                f'_s8_sample_points{label}: only {n_occ} occupied pixels for '
                f'{n_pres} presences (landscape occupancy {occ.mean():.4f}, '
                f'target {occ_prev}). The suitability surface is too '
                f'concentrated to support this sample size.')
        if n_emp < n_abs:
            raise _S8SampleSkip(
                f'_s8_sample_points{label}: only {n_emp} unoccupied pixels for '
                f'{n_abs} absences')
        # SECOND habitat check: is occupancy actually driven by habitat? A flat
        # suitability surface still hits the prevalence target, by making
        # occupancy random -- which the count check above cannot detect.
        _auc, _auc_se = _s8_occ_auc(sv, occ)
        _auc_min = max(0.5 + _S8_OCC_AUC_NSE * _auc_se, _S8_OCC_AUC_FLOOR)
        if np.isfinite(_auc) and _auc < _auc_min:
            raise _S8SampleSkip(
                f'_s8_sample_points{label}: occupancy is not distinguishable from '
                f'a coin flip (AUC {_auc:.3f} < {_auc_min:.3f} = max(0.5+'
                f'{_S8_OCC_AUC_NSE:.0f}SE, {_S8_OCC_AUC_FLOOR})). With {int(occ.sum())} '
                f'occupied pixels the null SE is {_auc_se:.4f}. Nothing here is '
                f'learnable, so a score from this cell would measure noise.')
        oi = np.where(occ)[0]; ei = np.where(~occ)[0]   # create candidate pools
        pi = rng.choice(oi, size=min(n_pres * oversample, len(oi)), replace=False)
        ai = rng.choice(ei, size=min(n_abs * oversample, len(ei)), replace=False)

    # Joint thinning: shuffle the two oversampled pools together so neither
    # class is systematically favoured by the greedy order, then take each
    # class up to its quota while enforcing separation against ALL kept points.
    cand   = np.concatenate([pi, ai])
    is_pre = np.concatenate([np.ones(len(pi), bool), np.zeros(len(ai), bool)])  # creates label for each candidate
    perm   = rng.permutation(len(cand))     # randomize the order of candidates
    cand, is_pre = cand[perm], is_pre[perm]
    cr, cc = vr[cand].astype(float), vc[cand].astype(float) # extract raster coordinates of candidates

    # Spatial thinning (see above), for both pres and abs separately
    sep = float(min_sep_px) if _S8_THIN else 0.0
    grid, keep, got_p, got_a = {}, [], 0, 0
    cell = max(sep, 1.0); m2 = sep * sep

    # CROSS-SET SEPARATION (seed_grid).
    # The separation rule used to hold only WITHIN one call. The extrapolative
    # arm draws train, val and test from three different PC1-band masks in three
    # separate calls, and each call started with an empty grid -- so a test point
    # could sit right next to a training point, and their CNN patches overlapped.
    # PC1 bands are ENVIRONMENTAL, not geographic: they interleave spatially, so
    # this is common rather than a corner case. Measured over three species at
    # N=2000, patch 32:
    #     test -> train   2.8% of points with overlapping patches (min 11.5 px)
    #     val  -> train  17.9%                                    (min  4.8 px)
    #     val  -> test    3.9%
    # The val->train figure is the serious one: it let EARLY STOPPING measure the
    # CNN on patches containing training pixels, which the RF (no model
    # selection) is not exposed to -- an asymmetry between the two models being
    # compared. The random arm never had any of this because it draws once and
    # thins jointly before splitting; its measured violation rate is 0.0%.
    #
    # Points from earlier calls are seeded into the grid so they BLOCK candidates
    # without counting toward this call's quota. No information crosses: seeded
    # points only remove candidate positions, they are never returned, never
    # labelled, and never enter any fitted set. The draws stay independent; what
    # changes is that they are now also spatially disjoint, which is the property
    # the split was supposed to have.
    if seed_grid is not None and sep > 0:
        for _pr, _pc in seed_grid:
            grid.setdefault((int(float(_pr) // cell), int(float(_pc) // cell)),
                            []).append((float(_pr), float(_pc)))

    for i in range(len(cand)):
        if is_pre[i] and got_p >= n_pres: continue
        if (not is_pre[i]) and got_a >= n_abs: continue
        r, c_ = cr[i], cc[i]
        if sep > 0:
            gr, gc = int(r // cell), int(c_ // cell)
            clash = False
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    for (pr, pc) in grid.get((gr + dr, gc + dc), ()):
                        if (pr - r) ** 2 + (pc - c_) ** 2 < m2:
                            clash = True; break
                    if clash: break
                if clash: break
            if clash: continue
            grid.setdefault((gr, gc), []).append((r, c_))
        keep.append(i)
        got_p += int(is_pre[i]); got_a += int(not is_pre[i])
        if got_p >= n_pres and got_a >= n_abs: break

    # SEPARATION IS NEVER RELAXED; THE SAMPLE SHRINKS INSTEAD.
    # At coarse resolution a full-quota, non-overlapping draw is geometrically
    # impossible: N=1000 at one patch diagonal needs ~800,000 km2 at 500 m,
    # against ~356,000 km2 of valid Germany. Something has to give, and the
    # choice between "force patch overlap" and "take fewer points" is not
    # symmetric:
    #   - a reduced N is RECORDED (n_tr) and CORRECTABLE -- 8a is a skill-vs-N
    #     curve on the same species and machinery, so a coarse-resolution drop
    #     can be read against what N alone predicts;
    #   - patch overlap is recorded nowhere, has no calibration curve, and
    #     silently flatters the CNN -- the model whose advantage the section
    #     exists to test.
    # So the separation guarantee holds at every level and n_tr carries the
    # cost, visibly.
    #
    # The trim preserves pres_ratio EXACTLY, so a shrunken sample does not also
    # drift in prevalence -- that would turn one confound into two.
    if got_p < n_pres or got_a < n_abs:
        _n_fit = int(min(got_p / max(pres_ratio, 1e-9),         # largest possible sample size preserving prevalence
                         got_a / max(1.0 - pres_ratio, 1e-9)))
        _p_fit = max(1, int(_n_fit * pres_ratio))
        _a_fit = max(1, _n_fit - _p_fit)
        if _n_fit < _S8_MIN_POINTS:
            raise _S8SampleSkip(
                f'_s8_sample_points{label}: at {min_sep_px:.0f} px separation only '
                f'{got_p} presences and {got_a} absences could be placed, giving '
                f'n={_n_fit} < {_S8_MIN_POINTS}. Too few points to fit anything, '
                f'so the cell is skipped rather than reported at a meaningless N.')
        _kp = np.asarray(keep, int) # indices of kept candidates
        _isp = is_pre[_kp]
        _sel = np.concatenate([_kp[_isp][:_p_fit], _kp[~_isp][:_a_fit]])    # selects appropriate number of pres and abs to keep
        keep = _sel                             # indices of candidates to keep after trimming
        print(f'  n reduced{label}: {min_sep_px:.0f}px separation fits '
              f'{_p_fit}+{_a_fit}={_p_fit + _a_fit} of {n_points} requested '
              f'(prevalence held at {pres_ratio:g})', flush=True)

    keep = np.asarray(keep, int)    # positions in candidate list that are kept
    idx  = cand[keep]               # indices in the original raster of the kept candidates
    return (vr[idx].astype(int), vc[idx].astype(int),   # return row,col coord and labels
            is_pre[keep].astype(np.float32))

# -- Species ensemble: the replication unit for all of Section 8 ------------
# WHY THIS REPLACES REPEATED DRAWS. Every sweep used to run one species with
# _S8_N_REPS repeated point draws. Reps within a species share a niche and a
# raster, so they are NOT independent, which left every Section-8 claim resting on n=1
# species. Swapping K reps for K species costs exactly the same number of
# fits and turns pseudo-replication into the real thing.
#
# WHAT VARIES: only the niche. The raster, the PCA, the extrapolative fold
# thresholds (_S8_PC1_*, derived from the PCA and not the species) and every
# data condition stay fixed, so species is the only thing that moves.
#
# HOW THE SWAP WORKS: _s8_suit_fn resolves _s8_meta as a GLOBAL at call time,
# so rebinding _s8_meta re-points the suitability surface without touching
# _s8_suit_fn or any run-level function. _s8_suit_100m is derived from it and
# is refreshed alongside. Both are restored in the finally, so a sweep cell
# that raises (or is interrupted) leaves the notebook as it found it.
_S8_N_SPECIES      = 2 if SMOKE_TEST_S8 else 3    # species ADMITTED per sweep
_S8_SPECIES_POOL_N = 6 if SMOKE_TEST_S8 else 16   # candidates screened (8j admits 8)
# SEED is candidate 0, so the single-species results this supersedes stay
# nested in the ensemble rather than sitting beside it.
_S8_SPECIES_POOL   = [SEED] + [40000 + 7919 * _k for _k in range(1, _S8_SPECIES_POOL_N)]
# TUNING SPECIES: OUTSIDE THE POOL. The hyperparameters are searched once on a
# species of the same construction that never appears in any evaluated sweep, so
# no evaluated species gets a home advantage from having been tuned on (both
# models were tuned on it, but a clean separation costs nothing). Checked on the
# v6 stack: occupancy AUC 0.95 at the 0.20 target -- an ordinary, learnable species.
_S8_TUNE_SEED = 2026
assert _S8_TUNE_SEED not in _S8_SPECIES_POOL, 'the tuning species must not be an evaluated species'


def _s8_species_viable(quiet=False):
    """Is the CURRENTLY-BOUND species modellable in the extrapolative fold?

    WHY SCREEN AT ALL. The extrapolative split is fixed in PC1 space, so a
    species whose niche optimum sits outside the training band has almost no
    suitability variation there: measured on one such draw, occupancy AUC in
    the train band was 0.49 with 453,000 occupied pixels -- precisely
    estimated, and precisely chance. Nothing is learnable in that cell, so the
    whole extrapolative arm for that species is lost. With only a handful of
    species per sweep, losing one to that would leave the extrap arm with too
    few to aggregate over.

    WHAT THIS COSTS. It conditions the population: results generalise over
    "species modellable in this extrapolative fold", not over all draws. That
    is the honest description, and it is also the only population on which
    model skill is even defined -- but it must be REPORTED, so the counts of
    candidates drawn / admitted / rejected are printed.

    Verified against a known-good draw: SEED gives train-band AUC 0.79.
    """
    tr_m, val_m, te_m = _s8_region_masks(env_scaled_100m, PATCH)    # create the 3 spatial bands
    if tr_m is None:
        return True, 'no masks -- not screened'
    global _S8_ENV_OK
    if _S8_ENV_OK is None:          # species-independent; compute once
        _S8_ENV_OK = np.all(np.isfinite(env_scaled_100m), axis=2)   # valid pixels, across all layers
    _ok  = _S8_ENV_OK & np.isfinite(_s8_suit_100m)  # AND have a suit value
    _rng = np.random.default_rng(20260908)

    # SUBSAMPLE EACH BAND BEFORE DRAWING OCCUPANCY. The train band alone is
    # ~14.2M valid pixels; drawing Bernoulli occupancy over it to then feed a
    # 50K-subsampled AUC allocated >100 MB per band per candidate for no gain.
    # Band AUC is a property of the joint DISTRIBUTION of (suitability,
    # occupancy), so a fixed subsample estimates it -- and the SE that
    # _s8_occ_auc returns is then the honest SE of THIS estimate, which is what
    # the adaptive threshold below is built on.
    _NS = 200_000   # limit number of training pixels
    _c  = _s8_solve_c(_s8_suit_100m[_ok], _S8_OCC_K, _S8_OCC_PREV)
    worst, worst_name = np.inf, ''
    for _nm, _m in (('train', tr_m), ('val', val_m), ('test', te_m)):   # test each band
        _br, _bc = np.where(_m & _ok)   # pixel in band AND valid
        if len(_br) < 1000: # unusable
            return False, f'{_nm} band has only {len(_br)} valid px'
        if len(_br) > _NS:
            _pick = _rng.choice(len(_br), _NS, replace=False)   # random select pixels
            _br, _bc = _br[_pick], _bc[_pick]   # keep only those
        _sv  = _s8_suit_100m[_br, _bc].astype(np.float64)   # extract suitability
        _occ = _rng.random(len(_sv)) < _s8_logistic(_sv, _c, _S8_OCC_K) # calculate occupancy 
        _auc, _se = _s8_occ_auc(_sv, _occ, max_n=_NS)
        _lim = max(0.5 + _S8_OCC_AUC_NSE * _se, _S8_OCC_AUC_FLOOR)  # determine minimal acceptable AUC
        if not np.isfinite(_auc) or _auc < _lim:    # reject
            return False, (f'{_nm}-band occupancy AUC {_auc:.3f} < {_lim:.3f} '
                           f'(occupancy {_occ.mean():.3f})')
        if _auc < worst:        # track worst band
            worst, worst_name = _auc, _nm
    return True, f'worst band = {worst_name} (AUC {worst:.3f})'


# manager of your virtual-species ensemble:
# Its job is to:
# 1. take a list of candidate seeds,
# 2. generate a virtual species for each seed,
# 3. optionally check whether that species is viable for the extrapolative experiment,
# 4. keep the first want species that pass,
# 5. give each accepted species a clean index 0, 1, 2, ...,
# 6. and make sure later sweeps use exactly the same admitted species.
def _s8_species_iter(seeds=None, quiet=False, n_admit=None, screen=True):
    """Yield (species_idx, seed) with the module-level species rebound.

    Walks the candidate pool and admits the first `want` species that are
    viable in the extrapolative fold (see _s8_species_viable), so species_idx
    runs 0..K-1 with no gaps and every admitted species contributes to every
    arm. A seed the guarded generator cannot satisfy, or that fails screening,
    is reported and skipped -- never silently substituted, which would make two
    sweeps disagree about what 'species 2' means.
    """
    global _s8_meta, _s8_suit_100m
    pool = _S8_SPECIES_POOL if seeds is None else list(seeds)
    if n_admit is not None: # how many species you want to admit
        want = int(n_admit)
    else:
        want = _S8_N_SPECIES if seeds is None else len(pool)
    _bk_meta, _bk_suit = _s8_meta, _s8_suit_100m    # save current species state

    # ADMISSION IS CACHED ON (pool, want). 8i and 8j each call this, and both
    # must run species drawn from one pool -- otherwise their cfgGain values
    # describe two populations while being reported as one ensemble. Screening
    # is deterministic (fixed rng seed), so a larger `want` admits a SUPERSET of
    # a smaller one; the cache only saves the re-screen for an identical key.
    _ck = (tuple(pool), want)   # set key of candidate pool and number of wanted species
    _cached = _S8_ADMIT_CACHE.get(_ck)  # check if already exists
    if _cached is not None:     # if an ensemble already exists, use that
        pool, screen = list(_cached), False     # cache already contains the admitted species
        print(f'  [species pool: reusing the {len(pool)} already-admitted '
              f'species, so 8i and 8j run the same ensemble]', flush=True)

    admitted, rejected, _admit_seeds = 0, 0, []
    try:
        for _sd in pool:        # loop over candidate seeds
            if admitted >= want:    # stop once enough species have been admitted
                break
            _r = make_vs(_sd, mode='pca', occ='probabilistic',  # generate VS
                         k=_S8_OCC_K, prevalence=_S8_OCC_PREV)
            if _r is None or _r[0] is None:             # reject failed generator
                print(f'  species seed {_sd}: generator exhausted -- skipped',
                      flush=True)
                rejected += 1
                continue
            _s8_meta      = {**_r[2], **_s8_config_niche(_sd)}   # pointwise + configurational niche
            _s8_suit_100m = _s8_suit_fn(env_scaled_100m)    # calculate suitability raster
            if screen:                          # test whether it is a viable species
                _ok_sp, _why = _s8_species_viable(quiet=quiet)
                if not _ok_sp:
                    print(f'  species seed {_sd}: REJECTED -- {_why}', flush=True)
                    rejected += 1
                    continue
            else:
                _why = 'admitted earlier -- not re-screened'
            if not quiet:
                print(f'  -- species {admitted} (seed {_sd}): suit mean='
                      f'{np.nanmean(_s8_suit_100m):.4f} power={_s8_meta["power"]:.3f} '
                      f'| {_why}', flush=True)
            _admit_seeds.append(_sd)    # assign the species index
            yield admitted, _sd
            admitted += 1
    finally:
        _s8_meta, _s8_suit_100m = _bk_meta, _bk_suit    # restores original global species state
        # Cache only a COMPLETE pass. A generator abandoned part-way (an
        # exception, or a `break` in the caller) would otherwise cache a short
        # list and silently shrink every later sweep's ensemble.
        if admitted >= want:
            _S8_ADMIT_CACHE[_ck] = tuple(_admit_seeds)
        print(f'  [species pool: {admitted} admitted, {rejected} rejected, '
              f'{len(pool)} candidates screened]', flush=True)


# Given where this virtual species’ Gaussian niche is centred, 
# how much of its PC1 niche lies in each fixed spatial/environmental band?
def _s8_niche_geometry(meta=None):
    """Where this species' niche sits relative to the FIXED PC1 fold bands.

    DESCRIPTIVE ONLY -- nothing here feeds the split. The bands are set once
    from the PCA and are species-blind, and they stay that way: defining folds
    by distance from the true optimum would make the split a function of the
    ground truth, and would also delete the hardest realistic case (predicting
    into a region where the response turns over) by always centring training on
    the peak. These columns exist so the extrapolative result can be READ
    conditional on niche position, not so the position can be controlled.

    Measured over the 8 admitted species, three structurally different regimes
    occur: optimum inside train (3), inside test (2), and beyond the test band
    entirely (3). That spread is not a defect -- it is why species-level
    replication is needed at all -- but it is invisible in the outputs unless
    recorded, and it is the obvious candidate explanation for the wide
    between-species spread in the extrapolative arm.
    """
    import math as _math
    m = _s8_meta if meta is None else meta      # get species niche params
    mus, sigs = m.get('mus_pc'), m.get('sigs_pc')
    if mus is None or sigs is None:
        return {}
    mu1, sg1 = float(mus[0]), float(max(float(sigs[0]), 1e-9))  # along PC1
    # EFFECTIVE WIDTH. The response is exp(-z^2/2)**power = exp(-power*z^2/2), so
    # the niche is a Gaussian with SD sigma/sqrt(power), not sigma. The band
    # masses below must use that width; with the raw sigma they were wrong for
    # every species with power != 1 (species SEED has power 0.64).
    pw1  = float(m.get('power', 1.0))
    sg1e = sg1 / np.sqrt(max(pw1, 1e-9))

    # What fraction of the Gaussian niche lies to the left of this PC1 value
    def _cdf(x):
        return 0.5 * (1.0 + _math.erf((x - mu1) / (sg1e * _math.sqrt(2.0))))

    # calculate how much niche mass is in each band:
    m_tr = _cdf(_S8_PC1_TR)
    m_vl = _cdf(_S8_PC1_VAL) - _cdf(_S8_PC1_TR)
    m_te = _cdf(_S8_PC1_TE) - _cdf(_S8_PC1_VAL)
    peak = ('train' if mu1 < _S8_PC1_TR else    # determine where the optimum lies
            'val'   if mu1 < _S8_PC1_VAL else
            'test'  if mu1 < _S8_PC1_TE else 'beyond_test')
    return dict(niche_mu_pc1=mu1, niche_sig_pc1=sg1, niche_sig_pc1_eff=sg1e,
                niche_mu_pc2=float(mus[1]), niche_sig_pc2=float(sigs[1]),
                niche_mass_tr=m_tr, niche_mass_val=m_vl, niche_mass_te=m_te,
                niche_mass_beyond=1.0 - m_tr - m_vl - m_te,
                niche_peak_band=peak)


def _s8_tag(rows, species_idx, species_seed):
    """Stamp species identity onto a level's rows. Without this the ensemble
    is unusable: cross-species aggregation needs to know which rows belong to
    which species, and 'rep' no longer carries that.

    Also stamps the niche geometry (see _s8_niche_geometry): read from the
    module-level _s8_meta, which _s8_species_iter has bound to THIS species for
    the duration of the loop body this is called from. Costs nothing -- it is a
    closed-form property of the draw, not a computation over the raster.
    """
    _geo = _s8_niche_geometry()
    for _r in rows:
        _r['species_idx']  = species_idx
        _r['species_seed'] = species_seed
        _r.update(_geo)
    return rows


# -- Configurational-DGP constants: ONE definition for the whole section -----
# 8i and 8j both need these and used to declare their own literals, guarded by
# a cross-check against the 8h cell. With 8h retired that check can never fire,
# which would leave two live sweeps free to drift apart silently -- and two
# arms with different r_m are not comparable, which is the one thing the
# cross-check existed to prevent. Declaring them once removes the failure mode
# instead of detecting it.
#
# r_m: see the derivation in the 8i cell -- spans >= 8 for separability, and
# 2*r_m == PATCH*100 so the truth's window matches the CNN's own
# GlobalAveragePooling window. Changing r_m without re-deriving both is a bug.
# OUTPUT VERSION. Every Section-8 result predates at least one of: the
# probabilistic-occurrence sampler, spatial thinning, the species ensemble,
# the RF-oracle arm. None of the old CSVs is comparable to the new ones, so
# they get a new tag rather than being overwritten -- and the plotting cells
# read the same constant, so a reader can never silently load a stale file.
# v4: N=2000 (the CNN is blind to configuration below ~1000 points),
# 4-level constant-footprint ladder, fixed 100 m truth in the 8i resolution
# arm, niche-geometry and oracle_val columns, and the oracle-model-selection
# arm in 8j. Bumped rather than overwritten so the v3 CSVs survive intact.
#
# v5: the audit fixes. NONE of the v4 numbers are comparable to these, which is
# why the tag moves rather than the files being overwritten:
#   - The two arms now use the SAME set sizes. The random arm used to carve
#     train/val/test out of one n_points draw, so it trained on 68% of n_points
#     (1360 at N=2000) against the extrapolative arm's full 2000. It now draws
#     n_points + 2*_S8_N_WIN and holds out exactly 300/300, so n_tr == n_points
#     and n_vl == n_te == 300 in BOTH arms. The old gap was worth +0.009
#     Spearman to the RF but +0.052 to the CNN -- an arm confound that acted
#     asymmetrically on the two models being compared.
#   - Minimum separation is now enforced ACROSS train/val/test in the
#     extrapolative arm, not only within each set. Measured before the fix:
#     2.8% of test points and 17.9% of val points had CNN patches overlapping a
#     training patch. Now 0.00% in all three pairs, minimum distance exactly on
#     the one-patch-diagonal rule.
#   - The constant-footprint ladder is [100, 200, 400]. The old fourth level,
#     "320 m", was really 300 m: resample_env rounded the block factor, so the
#     footprint was 3000 m and the recorded footprint_m/min_sep_m were wrong by
#     6%. Both call sites now refuse a resolution that is not a whole multiple
#     of the base grid.
#   - P_w and E are computed on an INTERIOR-BUFFERED mask (the window must fit
#     inside the valid area). Without it the no-data exterior acted as an
#     explicit non-woody class and manufactured an edge along the national
#     boundary, in the field E exists to measure. Costs 5.5% of the area.
#   - mu_P is derived from the realised composition field instead of a literal,
#     so the GRF arm (0.300) and the landuse arm (0.409) place the niche at the
#     same position RELATIVE to their own landscape, as the GRF arm always
#     implicitly did.
#   - New always-populated columns: band_frac_tr/val/te, frac_woody_realized,
#     mu_P_requested. Dropped cells are recorded in _S8_DROPS.
#   - Hyperparameters were re-searched with the Section-8 sampler (the tuners
#     previously drew without spatial thinning, so their val_auc was inflated
#     by patch overlap).
# v6: new input stack. WinterTemp is now the real CHELSA BIO11 (coldest-quarter
# mean) -- v5 and everything before silently used BIO1, the ANNUAL mean, through
# a stale cache. The stack is also scaled differently: globally over the study
# area (cell 3) instead of with min/max from the real-data training blocks plus
# clipping. The PCA, and with it every virtual species, changes; no v5 number
# carries over. The v5 CSVs stay on disk under their own tag.
#   Also in v6 (decided after the v6 audit, before any v6 number existed):
#   Elev dropped at variable selection (r = -0.82 with WinterTemp), so 6 env
#   channels; scaling per layer; Roughness computed over valid neighbours only;
#   forest from ESA WorldCover by majority (mode) instead of nearest neighbour;
#   CNN seeded with tf.keras.utils.set_random_seed; ceiling_config_patch scored at
#   each row's test points; hyperparameters tuned on a species outside the pool;
#   CNN input rescaled on the training patches only (_s8_train_minmax);
#   configurational niche (mu_E, sig_E) drawn per species; in the
#   every arm every sampled point's patch lies inside the study area
#   (_s8_patch_ok); 8c-fixed and the 8i resolution arm rebuilt on precise
#   100 m points with a fixed 100 m truth (_s8_run_level_fixedtruth).
_S8_VERSION = 'v6_bio11'

# -- Hyperparameter provenance -------------------------------------------------
# BEST_HP / RF_BEST_HP are read from JSON in section 1 and rewritten by section 7.
# In a full top-to-bottom run section 7 always runs first, but a partial run
# (kernel restart, then only a Section-8 cell) would silently sweep on whatever
# the JSON holds -- e.g. hyperparameters tuned on an earlier stack. The tuners
# therefore stamp what they were tuned ON, and every sweep cell refuses to start
# unless that stamp matches the stack and version loaded now.
def _s8_hp_provenance():
    return {'stack_created': meta['created'], 'stack_layers': list(LAYER_NAMES),
            's8_version': _S8_VERSION, 'tune_seed': int(_S8_TUNE_SEED)}


def _s8_check_hp():
    _want = _s8_hp_provenance()
    for _nm, _prov in (('CNN', _HP_PROV), ('RF', _RF_HP_PROV)):
        if _prov != _want:
            raise RuntimeError(
                f'{_nm} hyperparameters were not tuned on this stack/version '
                f'(stamp {_prov} != current {_want}). Run section 7 (7.1 and 7.2) '
                f'before any Section-8 sweep.')
    print('  [hyperparameters: tuned on this stack and version -- OK]', flush=True)

_S8_CFG = dict(
    r_m         = 1600.0,     # 2*r_m = 3200 m = PATCH(32) * 100 m
    corr_len_m  =  300.0,     # length of the smoothing / sigma of the Gauß core
    # spans = 2*1600/300 = 10.7 -> how many habitat blobs fit into window
    frac_woody  =    0.30,
    mu_P        =    0.30,    # map is 30% woody -> prefers average cover
    sig_P       =    0.20,
    # mu_E / sig_E are NOT here any more: they are drawn per species
    # (_s8_config_niche, _S8_CFG_NICHE_RANGES) and read from _s8_meta.
    w_P         =    0.1,     # S_config = E_term^0.9 * P_w_term^0.1; Weights P and E
    resolution  =  100,
    grf_seed    = 9000,
)
# Bounded LRU cache. Three full-raster bool masks are ~165 MB per
# (resolution, patch) combo at 100 m, and the full run touches ~8 combos --
# unbounded that is over a gigabyte on top of the 2 GB raster. The access
# pattern only ever needs the current combo (reused across species and
# levels within a sweep), so a cap of 2 keeps every hit that matters.
_S8_MASK_CACHE = {}
_S8_MASK_CACHE_MAX = 2
# B7: realised band shares per (H, W, patch). The PC1 thresholds are fixed in
# ABSOLUTE PC units (percentiles of the 100 m pixel distribution) and are then
# applied unchanged to coarsened rasters. That is the right choice -- a band then
# means the same ENVIRONMENT at every resolution, which is what makes the
# extrapolative fold comparable across the sweep -- but it does mean the bands
# stop being 40/15/15 percent once the grid is coarsened, because block
# averaging shrinks the PC1 spread and pulls mass toward the centre. The header
# describes the bands as percentiles; these numbers say what they actually are at
# each level, and they ride along in every output row.
_S8_BAND_FRAC = {}
_S8_ENV_OK = None            # cached all-band finite-env mask (species-independent)
_S8_ADMIT_CACHE = {}         # (pool, want) -> admitted seeds; keeps 8i and 8j identical


def _s8_region_masks(env_arr, patch):
    """Classify EVERY valid pixel by PC1 into train/val/test bands.

    WAS A 100K SUBSAMPLE, AND THAT WAS THE BINDING CONSTRAINT. The masks only
    ever marked pixels drawn into a 100,000-pixel sample, so the extrapolative
    training band contained ~40,000 marked pixels out of ~14,000,000 that
    actually qualify -- about 0.3%. Under the probabilistic sampler that
    starved the band: a species with 0.05% local occupancy had ~7,000 occupied
    sites in reality but only ~18 inside the mask, so the presence quota could
    not be filled and the occupancy-AUC check was being computed from a
    handful of positives, where it is closer to noise than to a measurement.
    Both of the smoke run's skip reasons trace back here rather than to the
    species or the sampler.

    Using every valid pixel costs one PCA transform of the full raster
    (~7 s for 35M rows) and the result depends only on (resolution, patch) --
    never on the species -- so it is cached across the species loop and across
    levels that share a grid. Chunked so the PC scores never materialise for
    the whole raster at once.

    Returns (tr_mask_2d, val_mask_2d, te_mask_2d) or (None, None, None).
    """
    H, W, C = env_arr.shape
    key = (H, W, int(patch))    # check if these masks where already calculated
    if key in _S8_MASK_CACHE:
        return _S8_MASK_CACHE[key]

    half = patch // 2
    ok2d = np.all(np.isfinite(env_arr), axis=2)
    if half > 0:                                    # border exclusion: every
        ok2d[:half, :] = False; ok2d[-half:, :] = False   # sampled point must
        ok2d[:, :half] = False; ok2d[:, -half:] = False   # have a full patch
    vr_, vc_ = np.where(ok2d)   # get coords of all usable pixels
    if len(vr_) == 0:
        return None, None, None

    tr_m = np.zeros((H, W), bool)
    val_m = np.zeros((H, W), bool)
    te_m = np.zeros((H, W), bool)
    _CHK = 2_000_000                # process pixels in Chunks
    for s in range(0, len(vr_), _CHK):
        e = min(s + _CHK, len(vr_)) # define end point
        pc1 = _pca3b.transform(         # scale and project the environmental data to PC1
            _sc3b.transform(env_arr[vr_[s:e], vc_[s:e], :].astype(np.float32)))[:, 0]
        r_, c_ = vr_[s:e], vc_[s:e]
        tr_m[r_[pc1 < _S8_PC1_TR], c_[pc1 < _S8_PC1_TR]] = True # classify pixels according to PC1
        m = (pc1 >= _S8_PC1_TR) & (pc1 < _S8_PC1_VAL); val_m[r_[m], c_[m]] = True
        m = (pc1 >= _S8_PC1_VAL) & (pc1 < _S8_PC1_TE);  te_m[r_[m], c_[m]] = True
        del pc1

    _n_ok = int(ok2d.sum())
    _bf = dict(band_frac_tr=float(tr_m.sum()) / max(_n_ok, 1),
               band_frac_val=float(val_m.sum()) / max(_n_ok, 1),
               band_frac_te=float(te_m.sum()) / max(_n_ok, 1))
    _S8_BAND_FRAC[key] = _bf
    print(f'  region masks @ {H}x{W} patch={patch}: '
          f'train={tr_m.sum():,} val={val_m.sum():,} test={te_m.sum():,} px '
          f'| realised shares {_bf["band_frac_tr"]:.3f}/{_bf["band_frac_val"]:.3f}/'
          f'{_bf["band_frac_te"]:.3f} of valid (100 m nominal: 0.400/0.150/0.150)',
          flush=True)
    while len(_S8_MASK_CACHE) >= _S8_MASK_CACHE_MAX:    # cache the result, i.e. species-independent masks
        _S8_MASK_CACHE.pop(next(iter(_S8_MASK_CACHE)))
    _S8_MASK_CACHE[key] = (tr_m, val_m, te_m)
    return tr_m, val_m, te_m

# -- Core level runner --------------------------------------------------------
# =============================================================================
# CONSTANT-FOOTPRINT RESOLUTION SWEEPS
#
# THE PROBLEM. "Resolution" bundles two things for a CNN: the GRAIN of a pixel
# and the EXTENT the patch covers. Holding PATCH fixed in PIXELS lets extent
# grow with coarsening -- at 1000 m a 32 px patch spans 32 km against 3.2 km at
# 100 m -- so the CNN is handed more context precisely where the comparison is
# meant to get harder. It also inflates the required point separation (one
# patch diagonal), which caps N: at 500 m and 1000 m the extrapolative bands
# cannot hold a non-overlapping sample at all.
#
# THE FIX. Hold the footprint constant in METRES and let PATCH in pixels fall:
# separation becomes constant at PATCH*100*sqrt(2) = 4525 m for every level, so
# N=1000 fits everywhere and the N confound disappears.
#
# WHAT IT COSTS -- STATE THIS IN THE WRITE-UP. It does not remove confounding,
# it moves it, and onto ONE model: PATCH is the CNN's entire spatial input, so
# shrinking it shrinks the CNN's capacity (GlobalAveragePooling sees 64 cells
# at 32 px, 16 at 16 px, 4 at 8 px) while the RF centre-pixel arm is untouched.
# Fixed-pixel-patch flatters the CNN at coarse resolution; constant-footprint
# handicaps it. Neither is neutral, which is why BOTH arms are run: agreement
# in sign makes the resolution conclusion robust to the choice, and a flip is
# itself the finding. 8e's PATCH sweep is the calibration curve for the
# capacity term, exactly as 8a's N curve is for the sample-size term.
#
# WHY THE FOOTPRINT IS 2*r_m. r_m was chosen so the configurational window
# exactly fills the CNN patch at 100 m (2*r_m = PATCH*100). Keeping the
# footprint at that value preserves the patch/window ALIGNMENT at every
# resolution -- the mechanism 8i's patch sweep exists to test, which a
# fixed-pixel-patch resolution sweep silently breaks.
_S8_FOOTPRINT_M = PATCH * 100
assert abs(_S8_FOOTPRINT_M - 2.0 * _S8_CFG['r_m']) < 1e-9, (
    f'footprint {_S8_FOOTPRINT_M} m != 2*r_m {2 * _S8_CFG["r_m"]} m -- the '
    f'patch/window alignment argument no longer holds; fix r_m or PATCH')

# THE ADMISSIBLE SET IS EXACTLY {100, 200, 400}, AND IT IS SMALL FOR A REASON.
# A level must satisfy THREE conditions at once, and an earlier version of this
# ladder checked only the second, which is how a "320 m" level got in that was
# really 300 m:
#   1. It must be a whole multiple of the 100 m base grid. resample_env
#      block-averages, and a block mean needs an integer factor. 320/100 = 3.2
#      silently became factor 3, so the grid was 300 m while every recorded
#      column still said 320 -- footprint 3000 m, not 3200, and a min_sep_m
#      that was wrong by the same 6%.
#   2. It must divide the 3200 m footprint into a whole number of pixels.
#   3. That patch must be >= _S8_MIN_PATCH_PX. 800 m gives 4 px, where two
#      MaxPooling(2) stages leave a 1x1 map and GlobalAveragePooling is
#      degenerate (measured against build_cnn).
# Multiples of 100 that divide 3200: 100/200/400/800/1600/3200 -> 32/16/8/4/2/1
# px. Condition 3 cuts it to 100/200/400. There is NO fourth level at this
# footprint, and 128 m / 160 m (claimed admissible by an earlier comment) fail
# condition 1 outright.
#
# A 320 m level IS reachable by area-weighted aggregation instead of block
# means, since gcd(100, 320) = 20 m. It was rejected anyway: a 320 m pixel spans
# 3.2 base pixels, so the overlap pattern -- and therefore the effective
# smoothing kernel -- shifts from pixel to pixel with period 5. 100/200/400 have
# a uniform kernel and are strictly nested (each 400 m block is an exact union
# of 200 m blocks, each of those of 100 m blocks). In a section whose subject is
# spatial structure, and where E is measured directly off the smoothing, a level
# with a position-dependent kernel is the wrong kind of fourth data point.
_S8_RES_CONSTFOOT = [100, 400] if SMOKE_TEST_S8 else [100, 200, 400]
_S8_MIN_PATCH_PX  = 8

# The fixed-pixel SENSITIVITY ladder, used by 8c-fixed and 8i. Defined here, once,
# so both read the same list and the fit-count print below can read its length
# instead of mirroring it by hand.
RESOLUTION_RANGE = [100, 500] if SMOKE_TEST_S8 else [100, 200, 300, 500, 1000]

def _s8_constfoot_patch(resolution, footprint_m=None):
    """PATCH in pixels holding the CNN's physical footprint constant.

    Raises rather than rounding: a rounded patch changes the footprint by up to
    6% and would silently reintroduce the very confound this removes.
    """
    f = int(_S8_FOOTPRINT_M if footprint_m is None else footprint_m)
    # CONDITION 1 (this was missing, and its absence is what let "320 m" through):
    # the resolution must be a whole multiple of the base grid, or resample_env
    # cannot block-average onto it and quietly rounds the factor instead.
    if int(resolution) % 100:
        raise ValueError(
            f'resolution {resolution} m is not a whole multiple of the 100 m base '
            f'grid -- resample_env would round the block factor and deliver a '
            f'different grid than the label says')
    if f % int(resolution):     # check whether resolution divides footprint exactly
        raise ValueError(
            f'resolution {resolution} m does not divide the {f} m footprint '
            f'into whole pixels ({f / resolution:.2f} px)')
    p = f // int(resolution)    # calculate patch size in pixels
    if p < _S8_MIN_PATCH_PX:
        raise ValueError(
            f'resolution {resolution} m gives PATCH={p} px, below the '
            f'{_S8_MIN_PATCH_PX} px floor: two MaxPooling(2) stages would '
            f'leave a {max(p // 4, 0)}x{max(p // 4, 0)} feature map')
    return int(p)


def _s8_run_level(env_base, resolution, n_points, pres_ratio, patch,
                   sub_exp, level, seed_base, occ_prev=None, n_reps=None,
                   oracle_val=False, patch_buffer_px=None):
    """occ_prev overrides the LANDSCAPE occupancy for this level only
    patch_buffer_px: the largest patch of the sweep, so every level samples the
    same area (see _s8_patch_buffer_px); None outside patch sweeps.
    n_reps overrides _S8_N_REPS for this call only (None = the global).
    Section 8 now replicates over SPECIES rather than over repeated draws
    of one species: reps within a species share a niche and are not
    independent, species are. Callers pass n_reps=1 inside
    _s8_species_iter, so the fit count is unchanged while the replication
    becomes the kind the cross-species statistics can actually use.

    (None = the module default _S8_OCC_PREV). Added for 8b, which sweeps
    ECOLOGICAL prevalence -- the fraction of the landscape the species
    actually occupies -- rather than the presence fraction of the sample.
    Those are different quantities and the SDM literature means the first
    one by "prevalence": species prevalence is a property of the species,
    sample prevalence is a property of your dataset.
    """
    """Both arms x all reps at one sensitivity level. Returns list of row dicts."""
    # env_arr/suit_2d/masks are computed ONCE per level call, shared by every
    # rep and both arms below -- the raster, the true suitability surface, and
    # (for the extrap arm) the spatial train/val/test ZONES do not change
    # within a level. What DOES change per rep is which specific points get
    # sampled from within those zones/from the valid-pixel pool (see the
    # per-rep `seed` below) -- that's the "independent sampling rep."
    env_arr = resample_env(env_base, resolution)
    suit_2d = _s8_suit_fn(env_arr)                  # 2D map with suitability at every pixel
    H, W, C = env_arr.shape
    tr_m, val_m, te_m = _s8_region_masks(env_arr, patch)    # masks for extrapolative arm (fixed per level)
    # every sampled point's patch inside the study area (see _s8_patch_ok)
    _pb = _s8_patch_buffer_px(resolution, patch, patch_buffer_px)
    _patch_ok = _s8_patch_ok(env_arr, _pb)
    if tr_m is not None:
        tr_m, val_m, te_m = tr_m & _patch_ok, val_m & _patch_ok, te_m & _patch_ok
    rows_out = []

    # Landscape occupancy intercept, solved ONCE per level on the full valid
    # raster and reused for every arm. Solving it inside each mask instead
    # would force the target occupancy separately on the extrapolative
    # train/val/test bands -- inventing occupancy in a band the species
    # barely occupies, instead of reporting that it is genuinely low.
    _occ_valid = np.all(np.isfinite(env_arr), axis=2) & np.isfinite(suit_2d)
    _occ_c = _s8_solve_c(suit_2d[_occ_valid],
                         _S8_OCC_K,
                         _S8_OCC_PREV if occ_prev is None else occ_prev)

    for rep in range(_S8_N_REPS if n_reps is None else int(n_reps)):
        seed = seed_base + rep * 997
        for arm in ('random', 'extrap'):
            t0 = _t8.time()

            # -- Split / sample -----------------------------------------------
            def _cln(rr, cc):
                # Center-pixel covariates at (rr,cc), NaN rows dropped -- used
                # only for the eds_decorr distance check below, not for fitting.
                E = env_arr[rr, cc, :].astype(np.float32)
                return E[~np.isnan(E).any(1)]

            # A degenerate (species, arm, level) cell raises _S8SampleSkip from the
            # sampler. Skip it and say so: crashing would lose a multi-hour run,
            # and recording a near-zero score would be read as model failure when
            # the truth is that there was nothing fittable there.
            try:
                if arm == 'random':
                    # INTERPOLATIVE arm: draw one point pool (at the target
                    # pres_ratio) from anywhere in the valid raster, then carve it
                    # into train/val/test via STRATIFIED splits. Because every
                    # split is stratified on the same label vector, train/val/test
                    # all track the same prevalence and come from the same overall
                    # spatial/environmental distribution -- test is "close to"
                    # train by construction, which is the point of this arm.
                    # MATCHED SET SIZES ACROSS THE TWO ARMS (this used to be the
                    # section's biggest confound). The random arm drew n_points and
                    # then CARVED train/val/test out of it, so it trained on 68% of
                    # n_points -- 1360 at N=2000 -- while the extrapolative arm
                    # trained on the full n_points and tested on 300. The arm
                    # contrast is the central comparison of this section, and it was
                    # confounded with a 47% difference in training-set size. Worse,
                    # asymmetrically between the models: read off 8a's own N curve,
                    # moving n_tr from 1360 to 2000 is worth +0.009 Spearman to the
                    # RF but +0.052 to the CNN, the same order as the CNN-RF gaps
                    # being reported.
                    # It now draws n_points + 2*_S8_N_WIN in ONE call, so the joint
                    # thinning still enforces separation across all three sets, then
                    # holds out exactly _S8_N_WIN for val and _S8_N_WIN for test.
                    # Result: n_tr == n_points and n_vl == n_te == _S8_N_WIN, the
                    # same three numbers the extrapolative arm produces --
                    # WHEREVER THE GEOMETRY ALLOWS IT. Measured across the ladders:
                    # the constant-footprint arm (100/200/400 m, the headline) hits
                    # 2000/300/300 in BOTH arms at every level, because separation is
                    # constant at 4525 m there by construction. The fixed-pixel
                    # SENSITIVITY arm cannot: separation grows with the grain
                    # (9.1 km at 200 m, 45.3 km at 1000 m), so graceful N reduction
                    # sets in from 200 m onward and the two arms drift apart again
                    # (extrap 1684 vs random 2000 at 200 m; extrap skipped entirely
                    # at 1000 m). That is a property of the ladder, not of this code,
                    # and it is the reason the fixed-pixel arm is a sensitivity
                    # rather than the headline. n_tr records it per row.
                    _n_draw = int(n_points) + 2 * _S8_N_WIN
                    vm = make_valid_mask(env_arr, suit_2d, patch) & _patch_ok   # whole patch inside the study area
                    if vm.sum() < max(_n_draw, 30):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'valid mask holds {int(vm.sum())} px, need {max(_n_draw, 30)}'); continue
                    r, c, lbl = _s8_sample_points(suit_2d, vm, _n_draw, pres_ratio, seed=seed,
                                                  min_sep_px=patch * _S8_SEP_PATCHES,
                                                  occ_prev=occ_prev, occ_c=_occ_c)
                    lbl = lbl.astype(np.int8)
                    if lbl.sum() < 3 or (1-lbl).sum() < 3:
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'draw has {int(lbl.sum())} pres / {int((1-lbl).sum())} abs, need 3 of each'); continue
                    # THE HOLD-OUT IS FIXED, THE TRAINING FOLD IS THE SWEPT
                    # QUANTITY. Holding out a FRACTION would make n_tr a function of
                    # _S8_N_WIN as well as of n_points, and 8a -- whose entire x
                    # axis is n_points -- would stop measuring what it plots (at
                    # n_points=100 a 40% hold-out leaves n_tr=420, not 100).
                    # So: when the full draw succeeded, hold out exactly
                    # 2*_S8_N_WIN and n_tr == n_points by construction. Only when
                    # graceful N reduction returned fewer points does the hold-out
                    # shrink with it, proportionally, rather than eating the
                    # training fold. n_tr/n_vl/n_te are recorded either way.
                    if len(lbl) >= _n_draw:
                        _hold = 2 * _S8_N_WIN
                    else:
                        _hold = int(round(len(lbl) * (2.0 * _S8_N_WIN) / _n_draw))
                    _hold = max(2, min(_hold, len(lbl) - 4))
                    tr_i, _rest_i = _tts(np.arange(len(lbl)), test_size=_hold,
                                         stratify=lbl, random_state=seed)
                    # Both-classes guard: at low prevalence a split can (rarely)
                    # leave the training fold with only one class, which would make
                    # RF/CNN fitting meaningless -- skip the rep.
                    if len(np.unique(lbl[tr_i])) < 2:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'training fold ended up single-class after the split'); continue
                    val_i, te_i = _tts(_rest_i, test_size=0.5,
                                       stratify=lbl[_rest_i], random_state=seed+1)
                    r_tr=r[tr_i];  c_tr=c[tr_i];  y_tr=lbl[tr_i]
                    r_v =r[val_i]; c_v =c[val_i]; y_v =lbl[val_i]
                    r_te=r[te_i];  c_te=c[te_i];  y_te=lbl[te_i]
                    # eds_decorr in the random arm is expected small/constant (interpolative)
                    d_te = _s8_eds_decorr(_cln(r_te, c_te), _cln(r_tr, c_tr))
                    d_vl = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_tr, c_tr))
                    d_vt = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_te, c_te))  # val-vs-test: flat/small expected here (interpolative, one shared pool)
                else:
                    # EXTRAPOLATIVE arm: train/val/test are drawn from the three
                    # FIXED, disjoint PC1-band zones computed by _s8_region_masks
                    # (low/mid/high PC1 -- see the fold-threshold comment above),
                    # so test is environmentally far from train by construction.
                    if tr_m is None or tr_m.sum() < max(n_points, 10):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'train band holds {0 if tr_m is None else int(tr_m.sum())} px, '
                                 f'need {max(n_points, 10)}'); continue
                    if val_m is None or val_m.sum() < 10 or te_m is None or te_m.sum() < 10:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'val or test band holds fewer than 10 valid px'); continue
                    r_tr, c_tr, lbl_tr = _s8_sample_points(suit_2d, tr_m, n_points, pres_ratio, seed=seed,
                                                           min_sep_px=patch * _S8_SEP_PATCHES,
                                                           occ_prev=occ_prev, occ_c=_occ_c)
                    y_tr = lbl_tr.astype(np.int8)
                    # CROSS-SET SEPARATION (see the seed_grid note in
                    # _s8_sample_points). train/val/test come from three separate
                    # calls on three disjoint PC1-band masks, so without this the
                    # min-separation rule held only WITHIN each set and 2.8% of test
                    # points / 17.9% of val points ended up with CNN patches
                    # overlapping a training patch. Each draw now blocks the ones
                    # after it; the seeded points are never returned or labelled, so
                    # the draws stay independent and only become spatially disjoint.
                    _used_rc = list(zip(r_tr.astype(float), c_tr.astype(float)))
                    n_v = min(_S8_N_WIN, int(val_m.sum()))   # >=10 guaranteed by the guard above
                    # val/test are drawn with the SAME pres_ratio as train rather
                    # than a hardcoded 0.5. Since 8b now sweeps ECOLOGICAL
                    # prevalence instead of sample balance, every sub-experiment
                    # runs at the global PRES_RATIO, so this is currently a no-op
                    # -- kept because hardcoding 0.5 here would silently break any
                    # future arm that does vary the sample balance.
                    #
                    # The LIVE version of the same concern is occ_prev: all three
                    # draws take it, so train, val and test come from a landscape
                    # with the same occupancy. If only train tracked the swept
                    # level, the extrap arm would be tested against a different
                    # ecology than it was trained on and the 8b effect would be
                    # partly a train/test mismatch rather than a prevalence effect.
                    n_t = min(_S8_N_WIN, int(te_m.sum()))   # >=10 guaranteed by the guard above
                    if oracle_val:      # test and val drawn together, then split, so val is closer to train than test is
                        # ORACLE MODEL SELECTION -- an UPPER BOUND, never the headline.
                        # In the primary design val sits BETWEEN train and test, so it is
                        # necessarily closer to train: measured D(val,train)=1.30 against
                        # D(test,train)=1.99. Early stopping therefore selects the epoch that
                        # is best under a shift of 1.30 and the model is then scored at 1.99 --
                        # while the RF, which does no model selection at all, carries no such
                        # penalty. That asymmetry sits in exactly the arm where the CNN does
                        # worst, so the primary design cannot separate "worse at transfer" from
                        # "selected for the wrong shift". This arm bounds that: it is the
                        # standard domain-adaptation contrast of source validation (realistic)
                        # against target/oracle validation (assumes labels in the target domain,
                        # which real transfer does not have) -- the exact analogue of the
                        # RF-oracle arm.
                        #
                        # ONE DRAW, THINNED JOINTLY, THEN SPLIT. A second independent draw from
                        # te_m could return points whose PATCHES overlap test patches, which
                        # would let early stopping peek at test. Drawing both together enforces
                        # one patch diagonal between every val and every test point.
                        _n_comb = n_v + n_t
                        _rc, _cc, _lc = _s8_sample_points(suit_2d, te_m, _n_comb, pres_ratio,
                                                          seed=seed + 200,
                                                          min_sep_px=patch * _S8_SEP_PATCHES,
                                                          occ_prev=occ_prev, occ_c=_occ_c,
                                                          seed_grid=_used_rc)
                        _lc = _lc.astype(np.int8)
                        if len(_lc) < 20 or len(np.unique(_lc)) < 2:
                            _s8_drop(sub_exp, level, arm, rep,
                                     f'oracle-val joint draw returned {len(_lc)} points'); continue
                        _frac_te = min(max(n_t / max(_n_comb, 1), 0.1), 0.9)    # force a 10-90% split if the requested n_t is too small or too large
                        _iv, _it = _tts(np.arange(len(_lc)), test_size=_frac_te,
                                        stratify=_lc, random_state=seed + 300)
                        r_v,  c_v,  y_v  = _rc[_iv], _cc[_iv], _lc[_iv]
                        r_te, c_te, y_te = _rc[_it], _cc[_it], _lc[_it]
                    else:
                        r_v, c_v, lbl_v = _s8_sample_points(suit_2d, val_m, n_v, pres_ratio, seed=seed+100,
                                                            min_sep_px=patch * _S8_SEP_PATCHES,
                                                            occ_prev=occ_prev, occ_c=_occ_c,
                                                            seed_grid=_used_rc)
                        y_v = lbl_v.astype(np.int8)
                        _used_rc += list(zip(r_v.astype(float), c_v.astype(float)))
                        r_te, c_te, lbl_te = _s8_sample_points(suit_2d, te_m, n_t, pres_ratio, seed=seed+200,
                                                               min_sep_px=patch * _S8_SEP_PATCHES,
                                                               occ_prev=occ_prev, occ_c=_occ_c,
                                                               seed_grid=_used_rc)
                        y_te = lbl_te.astype(np.int8)
                    # eds_decorr confound log: expected ~2 (extrapolative by design)
                    d_te = _s8_eds_decorr(_cln(r_te, c_te), _cln(r_tr, c_tr))
                    d_vl = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_tr, c_tr))
                    d_vt = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_te, c_te))  # val-vs-test: checks "val previews test" per level/rep
            except _S8SampleSkip as _skip:
                print(f'  SKIP ({arm} {sub_exp} lv={level} rep={rep}): {_skip}',
                      flush=True)
                continue

            # s_te: the TRUE continuous suitability at the test-point locations
            # -- this is what proba gets compared against for spearman_true/
            # mae_true (see _s8_mets), independent of the noisy drawn labels.
            # INVARIANT: every point that gets FITTED must have a defined truth.
            # This is the check that would have caught the band-mask/NaN mismatch
            # described above, so it stays in permanently rather than being retired
            # once its cause was fixed. It is LOUD BUT NOT FATAL -- a hard assert
            # here would throw away a multi-hour run over a cell that can simply be
            # dropped, which is the same trade-off _S8SampleSkip already makes.
            _bad_truth = sum(int((~np.isfinite(suit_2d[_rr_s, _cc_s])).sum())
                             for _rr_s, _cc_s in ((r_tr, c_tr), (r_v, c_v), (r_te, c_te)))
            if _bad_truth:
                _s8_drop(sub_exp, level, arm, rep,
                         f'{_bad_truth} drawn point(s) have an undefined truth -- the '
                         f'sampling mask and the truth surface have diverged')
                continue
            s_te = suit_2d[r_te, c_te]
            if y_te.sum() < 3 or (1-y_te).sum() < 3:
                _s8_drop(sub_exp, level, arm, rep,
                         f'test fold has {int(y_te.sum())} pres / {int((1-y_te).sum())} abs'); continue
            # base: one row-template per (rep, arm), shared by every model/init
            # fit below. real_prev_tr/te = the ACHIEVED prevalence in the drawn
            # sample (can differ slightly from the target pres_ratio); decc_*
            # = the eds_decorr confound-check values computed just above.
            base = dict(sub_exp=sub_exp, level=level, rep=rep, arm=arm,
                        n_tr=int(len(y_tr)), n_te=int(len(y_te)), n_vl=int(len(y_v)),
                        # False everywhere except the oracle-model-selection bound;
                        # see the oracle_val branch in the extrap sampling above.
                        oracle_val=bool(oracle_val),
                        # patch and footprint are ALWAYS-POPULATED columns for the same reason
                        # resolution is: the constant-footprint arm varies patch WITH resolution, so
                        # neither one alone identifies a level.
                        patch=int(patch),
                        patch_buffer_px=int(_pb),   # window the whole patch must fit in
                        footprint_m=int(patch * resolution),
                        # resolution is its own ALWAYS-POPULATED column. `level` is a convenience
                        # index whose meaning changes between sub-experiments (it is n_points in 8a,
                        # resolution in 8c-fixed, patch in 8e), so it must never be the analytical key.
                        resolution=int(resolution),
                        # The separation ACTUALLY enforced. With graceful N reduction a
                        # level can run at fewer points than requested, so n_tr alone is
                        # ambiguous -- these two columns say why it fell.
                        min_sep_px=float(patch * _S8_SEP_PATCHES),
                        min_sep_m=float(patch * _S8_SEP_PATCHES * resolution),
                        # B7: what the PC1 bands ACTUALLY are on this level's grid.
                        # Fixed in absolute PC units, so they drift away from
                        # 40/15/15 percent as the raster coarsens -- see the note at
                        # _S8_BAND_FRAC. Always populated so a resolution effect can
                        # be read against the band geometry that produced it.
                        **_S8_BAND_FRAC.get((H, W, int(patch)), {}),
                        real_prev_tr=float(y_tr.mean()),
                        real_prev_te=float(y_te.mean()),
                        real_prev_vl=float(y_v.mean()),
                        decc_te_tr=d_te, decc_vl_tr=d_vl, decc_vl_te=d_vt)

            # -- RF ----------------------------------------------------------
            # RF needs no validation split (no early stopping), so it always
            # runs here regardless of the CNN val-guard below. No leakage: RF
            # is fit on the TRAINING fold only, env_arr is already scaled.
            E_tr  = env_arr[r_tr,c_tr,:].astype(np.float32)
            ok_tr = ~np.isnan(E_tr).any(1)
            E_trc = E_tr[ok_tr]
            # No scaler here: env_arr is already min-max scaled upstream, and RF
            # is scale-invariant -- each split is a threshold comparison on one
            # feature, unaffected by any further monotonic per-feature rescaling.
            # A StandardScaler on top would be a no-op on RF's splits/predictions.
            clf   = _s8_rf()
            clf.fit(E_trc, y_tr[ok_tr])    # fit RF on training fold only
            E_te_ = env_arr[r_te,c_te,:].astype(np.float32)
            ok_te = ~np.isnan(E_te_).any(1)
            p_rf  = np.full(len(r_te), np.nan)
            if ok_te.sum() >= 5:
                p_rf[ok_te] = clf.predict_proba(E_te_[ok_te])[:,1]  # predict probabilities for the positive class
            rows_out.append({**base,'model':'rf','cnn_init':0,
                             'early_stop_ep':np.nan,'val_auc_best':np.nan,'train_auc_at_best':np.nan,**_s8_mets(p_rf,y_te,s_te)})

            # -- CNN: both-classes guard on val --------------------------------
            # CNN fitting below uses validation_data=(X_val, y_v) with
            # EarlyStopping/ReduceLROnPlateau monitoring val_auc. If val ends
            # up single-class (both_classes guard fails), val_auc is undefined
            # -- rather than let that silently break early stopping, skip CNN
            # entirely for this (level, rep, arm). This is the low-prevalence
            # failure mode the header's "both-classes guard" refers to: at low
            # PRES_RATIO (8b) or small n_points (8a), a val window can easily
            # end up with too few presences (or too few absences) by chance.
            n_vp=int(y_v.sum()); n_va=int((y_v==0).sum())
            if n_vp < _S8_MIN_VAL or n_va < _S8_MIN_VAL:
                # Routed through the same recorder as every other drop, so the CNN
                # gap shows up in _S8_DROPS rather than only in the console log.
                # NOTE: RF rows for this cell were already appended above -- this
                # drops the CNN only, which is why the reason says so.
                _s8_drop(sub_exp, level, arm, rep,
                         f'val fold has {n_vp} pres / {n_va} abs, below _S8_MIN_VAL='
                         f'{_S8_MIN_VAL} -- CNN skipped (RF row was written)')
                continue

            tr_ch_mean = E_trc.mean(0)
            def _pad(rr, cc):
                # Extract CNN patches and fill any NaN pixel with the TRAINING
                # fold's per-channel mean (convention: padding uses
                # the training-set mean, not zero, since channels are min-max
                # scaled and 0.0 is a real value, not "neutral").
                X = extract_patches(env_arr, rr, cc, patch=patch)
                for ch in range(C):
                    nm = np.isnan(X[...,ch])
                    if nm.any(): X[...,ch][nm] = tr_ch_mean[ch]
                return X

            X_tr  = _pad(r_tr, c_tr)
            X_val = _pad(r_v,  c_v)
            X_te  = _pad(r_te, c_te)
            # CNN input scaled on the TRAINING patches only (see _s8_train_minmax);
            # val/test values beyond the training range are kept, not clipped.
            X_tr, X_val, X_te = _s8_train_minmax(X_tr, X_val, X_te)
            # _pad hat jedes NaN bereits durch das Trainings-Kanalmittel ersetzt,
            # ein Filter hier kann also nie greifen. Statt eines Guards, der
            # Schutz vortaeuscht, steht die Invariante als Assert da.
            assert not np.isnan(X_te).any(), '_pad hat NaNs in den Test-Patches gelassen'

            # Same X_tr/X_val/X_te (i.e. the SAME drawn data) is reused across
            # all _S8_N_INIT random weight initializations below -- only the
            # CNN's random seed changes per init. This is what separates
            # "data-draw spread" (across reps, via seed+rep*997 above) from
            # "CNN init spread" (across ci, via seed+ci*137 here): both are
            # logged as separate rows, so downstream analysis can attribute
            # variance to the right source instead of conflating them.
            cnn_ps = []
            for ci in range(_S8_N_INIT):
                tf.keras.backend.clear_session()
                # tf.keras.utils.set_random_seed, NOT tf.random.set_seed: the latter
                # leaves Keras' own layer seeds unset, and two fits with the same seed
                # then differ by up to 0.067 in predicted probability (measured) --
                # as much as two different seeds. This one seeds Python, NumPy and TF
                # together and makes a fit bit-reproducible on CPU.
                tf.keras.utils.set_random_seed(seed + ci * 137)
                model = build_cnn(patch, C, hp=BEST_HP)
                hist  = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_v),
                    epochs=_S8_EPOCHS, batch_size=_S8_BATCH,
                    class_weight={0:1.0, 1:1.0},
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor='val_auc', patience=_S8_PAT,
                            restore_best_weights=True, mode='max'),
                        tf.keras.callbacks.ReduceLROnPlateau(
                            monitor='val_auc', factor=0.5, patience=5,
                            min_lr=1e-6, mode='max'),
                    ], verbose=0)
                # Fit diagnostics from the history that already exists -- zero extra
                # compute. val_auc_best ~ chance means the fit never learned on the data
                # it was SELECTED on, so its test score is not a measurement of the
                # model; val high but test low means it learned and failed to
                # generalise, which is a real result. Raw numbers only -- the
                # threshold is applied at analysis time (see the plotting cell).
                _vh  = list(hist.history.get('val_auc', []))
                _th  = list(hist.history.get('auc', []))
                _bi  = int(np.argmax(_vh)) if _vh else 0
                n_ep = _bi + 1          # best epoch (restore_best_weights), not the stop epoch
                _val_best = float(_vh[_bi]) if _vh else np.nan
                _tr_best  = float(_th[_bi]) if _bi < len(_th) else np.nan
                p_c  = model.predict(X_te, verbose=0).ravel()    # predict probabilities
                cnn_ps.append(p_c)
                rows_out.append({**base,'model':'cnn','cnn_init':ci,
                                 'early_stop_ep':n_ep,'val_auc_best':_val_best,'train_auc_at_best':_tr_best,**_s8_mets(p_c,y_te,s_te)})
                tf.keras.backend.clear_session()

            # cnn_mean: score the AVERAGE of the N_INIT predicted probabilities
            # once, rather than averaging the per-init scores -- an ensemble
            # point estimate with lower variance than any single init, logged
            # as its own row (cnn_init=-1) alongside the per-init rows above.
            if len(cnn_ps) > 1:
                mp = np.nanmean(np.stack(cnn_ps,0), axis=0) # averages across inits for each test point
                rows_out.append({**base,'model':'cnn_mean','cnn_init':-1,
                                 'early_stop_ep':np.nan,'val_auc_best':np.nan,'train_auc_at_best':np.nan,**_s8_mets(mp,y_te,s_te)})

            print(f'  [{arm}] {sub_exp} lv={level} rep={rep}: '
                  f'n_tr={len(y_tr)} n_te={len(y_te)} '
                  f'D(te,tr)={d_te:.2f} [{_t8.time()-t0:.0f}s]', flush=True)

    return rows_out

# -- Fit count estimate -------------------------------------------------------
# Covers 8a/8b/8c-fixed/8e -- the pointwise sweeps of section 8. 8i and
# 8j are separate cells with their own grids and their own counts.
# Replication is over SPECIES (n_reps=1 each), so the multiplier is
# _S8_N_SPECIES, not _S8_N_REPS. Each pointwise level fits 1 RF + _S8_N_INIT
# CNNs; the configurational cells fit 3 RF instead of 1 and count separately.
# 8b/8c-fixed/8e ranges live in later cells, so their level counts are
# mirrored here rather than read (they are defined further down this cell
# and in later cells, so they are not in scope yet). All four collapse to 2
# levels under SMOKE_TEST_S8. If a range in one of those cells changes, change
# it here too or this print goes quietly wrong.
# 8c-fixed runs BOTH resolution conventions, so its level count is the
# constant-footprint ladder PLUS the fixed-pixel ladder (both in scope above).
_n_res_fixed = len(RESOLUTION_RANGE)     # read, not mirrored -- see above
_n_res_both  = len(_S8_RES_CONSTFOOT) + _n_res_fixed
_s8_lvls = {'8a':       2 if SMOKE_TEST_S8 else 6,
            '8b':       2 if SMOKE_TEST_S8 else 6,
            '8c-fixed': _n_res_both,
            '8e':       2 if SMOKE_TEST_S8 else 4}
_fits_s8 = sum(_s8_lvls.values()) * (1 + _S8_N_INIT) * _S8_N_SPECIES * 2
print(f'\nFit count (8a/8b/8c-fixed/8e): {_fits_s8}')
print(f'  {sum(_s8_lvls.values())} levels x {_S8_N_SPECIES} species x 2 arms '
      f'x (1 RF + {_S8_N_INIT} CNN)')
print(f'  levels per sub-experiment: {_s8_lvls}')
print(f'  8c-fixed runs both resolution conventions: '
      f'{len(_S8_RES_CONSTFOOT)} constant-footprint + {_n_res_fixed} fixed-pixel')
print(f'  This is an UPPER BOUND: coarse fixed-pixel levels whose band cannot '
      f'hold a\n  non-overlapping sample are skipped, and print SKIP when they are.')
print(f'  8i and 8j print their own counts.')
print(f'  SMOKE_TEST_S8 = {SMOKE_TEST_S8}', flush=True)

# =============================================================================
# 8a -- Effect of Sample Size (N_POINTS)
# Sweeps N_POINTS (training sample size) across 100..5000; PRES_RATIO/PATCH
# stay at their global defaults. Tests how much data each model/arm needs
# before spearman_true stabilizes, and whether the RF-CNN gap narrows with N.
# =============================================================================

## 7 — Hyperparameter Tuning
Tuned **once** on the reference condition (patch 32 px, N = 2000, a tuning species drawn outside the evaluation pool, `_S8_TUNE_SEED`) with the same sampler the experiments use, then frozen for every condition — no per-condition tuning. Results go to `keras_tuner_synthetic/`, which section 1 reads.

### 7.1 — CNN: Keras Tuner (Hyperband)
Search space `filters_1/2` ∈ {8, 16, 32}, `dense_units` ∈ {8, 16, 32}, `dropout` ∈ {0.2, 0.3, 0.4}, `lr` ∈ {1e-4, 3e-4, 1e-3}; Hyperband with `max_epochs = 50`, `factor = 3`. The six best configurations are then re-scored over 3 initialisations, and the best **mean** is kept.

In [ ]:
import keras_tuner as kt

# ── Sample baseline synthetic data for HP search ──────────────────────────────
# INTENDED FIX: tuner now uses PCA Gaussian VS (same construction as Section 11+
# experiments). Previously used old per-predictor Gaussian (compute_suitability).

# ── TUNE UNDER THE REGIME THE MODEL IS DEPLOYED IN ───────────────────────────
# This used to draw its points with the GLOBAL sample_points(): deterministic
# suitability-weighted selection, N_POINTS=1000, and NO spatial thinning. That
# last part is the problem. Without a minimum separation, ~31% of drawn points
# have a neighbour closer than one patch width, so their 32x32 patches overlap by
# up to ~97% -- and since the tuner's holdout is a random split of that one pool,
# training and validation patches shared pixels. Hyperband then selected the
# architecture that scored best on a val_auc INFLATED BY THAT LEAKAGE, and those
# hyperparameters were deployed into a Section 8 that samples with probabilistic
# occupancy, one-patch-diagonal thinning and N=2000.
#
# It now draws exactly as Section 8 does, via _s8_sample_points on a PCA species
# of the same construction, at _S8_N_POINTS. That makes the search objective and the
# deployment condition the same thing.
#
# DEPENDENCY: _s8_sample_points / _s8_suit_fn / _S8_TUNE_SEED / _S8_SEP_PATCHES / _S8_N_POINTS
# come from the experimental framework (section 6), which runs before this cell;
# the sampler is shared rather than duplicated here. The product of the search is
# a JSON file that section 1 reads on every later run.
#
# STILL ONE PATCH SIZE AND ONE SPECIES, ON PURPOSE. The analysis plan fixes one CNN
# architecture for the whole project: tuning per patch size or per species would
# confound every comparison with tuning noise. PATCH=32 / species _S8_TUNE_SEED
# (outside the evaluation pool) is the reference condition, and 8e is the calibration curve for what patch size does
# to a FIXED architecture.
for _need in ('_s8_sample_points', '_s8_suit_fn', '_S8_TUNE_SEED', '_S8_SEP_PATCHES', '_S8_N_POINTS'):
    if _need not in dir():
        raise RuntimeError(
            f'{_need} is not defined -- run sections 3 and 6 '
            f'before this tuner, so the search samples exactly as the experiment does.')

# The tuning species (see _S8_TUNE_SEED): same generator, never evaluated.
# _s8_suit_fn reads the niche from the global _s8_meta, so it is bound for this
# one call and restored -- the evaluation species stay untouched.
_tune_vs = make_vs(_S8_TUNE_SEED, mode='pca', occ='probabilistic',
                   k=_S8_OCC_K, prevalence=_S8_OCC_PREV)
assert _tune_vs[0] is not None, 'tuning species: generator exhausted -- pick another _S8_TUNE_SEED'
_meta_bk_t = _s8_meta
try:
    _s8_meta   = _tune_vs[2]
    _suit_tune = _s8_suit_fn(env_scaled_100m)
finally:
    _s8_meta = _meta_bk_t
_suit_t = _suit_tune
_valid_t = (make_valid_mask(env_scaled_100m, _suit_t, PATCH)     # same area rule as Section 8
            & _s8_patch_ok(env_scaled_100m, _s8_patch_buffer_px(100, PATCH)))
_r_t, _c_t, _lbl_t = _s8_sample_points(
    _suit_t, _valid_t, _S8_N_POINTS, PRES_RATIO, seed=SEED,
    min_sep_px=PATCH * _S8_SEP_PATCHES)     # probabilistic occupancy + thinning
_Xp_t = extract_patches(env_scaled_100m, _r_t, _c_t, patch=PATCH)

# 20 % stratified holdout. Split BEFORE padding, so the padding statistic is a
# training-fold quantity and nothing leaks across the split.
_tr_t, _va_t = train_test_split(
    np.arange(len(_lbl_t)), test_size=0.20, stratify=_lbl_t, random_state=42)

# PAD, DO NOT DROP -- and pad exactly as _pad() does inside the run-level
# functions: NaN pixels get the TRAINING fold's per-channel CENTRE-PIXEL mean,
# never 0.0 (the channels are min-max scaled, so 0.0 is the range minimum, not a
# neutral value). This cell used to DROP any patch containing a NaN, which left
# the CNN search on 1921 of 2000 points while the RF search below kept all 2000
# -- so the two were not on the same points after all, and neither matched how
# Section 8 actually handles a NaN pixel. 5.28% of valid centres have at least
# one NaN in their patch, so this is not a rare path.
_E_ctr      = env_scaled_100m[_r_t, _c_t, :].astype(np.float32)
_tr_mask_t  = np.zeros(len(_lbl_t), bool); _tr_mask_t[_tr_t] = True
_tr_ch_mean = _E_ctr[(~np.isnan(_E_ctr).any(1)) & _tr_mask_t].mean(0)
for _ch_t in range(_Xp_t.shape[-1]):
    _nm_t = np.isnan(_Xp_t[..., _ch_t])
    if _nm_t.any():
        _Xp_t[..., _ch_t][_nm_t] = _tr_ch_mean[_ch_t]
assert not np.isnan(_Xp_t).any(), 'padding left NaNs in the tuner patches'

# Scaled on the training patches only, exactly as the run-level functions do
# (_s8_train_minmax), so the search sees the input the experiment deploys.
x_hp_tr, x_hp_va = _s8_train_minmax(_Xp_t[_tr_t], _Xp_t[_va_t])
y_hp_tr, y_hp_va = _lbl_t[_tr_t], _lbl_t[_va_t]
print(f'HP holdout — train: {len(_tr_t)}  val: {len(_va_t)}  '
      f'(drawn as Section 8 draws: probabilistic occupancy, '
      f'{PATCH * _S8_SEP_PATCHES:.1f}px separation, N={_S8_N_POINTS})')
print(f'  pres rate  train: {y_hp_tr.mean():.3f}   val: {y_hp_va.mean():.3f}')

# ── Tunable model ─────────────────────────────────────────────────────────────
def _build_tunable(hp):
    f1 = hp.Choice('filters_1',   [8, 16, 32])
    f2 = hp.Choice('filters_2',   [8, 16, 32])
    du = hp.Choice('dense_units', [8, 16, 32])
    dr = hp.Choice('dropout',     [0.2, 0.3, 0.4])
    lr = hp.Choice('lr',          [1e-4, 3e-4, 1e-3])
    inp = tf.keras.Input(shape=(PATCH, PATCH, N_LAYERS))
    x   = tf.keras.layers.RandomFlip('horizontal')(inp)
    for filters in [f1, f2]:
        x = tf.keras.layers.Conv2D(filters, 3, activation='relu', padding='same')(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(2)(x)
    x   = tf.keras.layers.GlobalAveragePooling2D()(x)
    x   = tf.keras.layers.Dense(du, activation='relu')(x)
    if dr > 0:
        x = tf.keras.layers.Dropout(dr)(x)
    out = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    m   = tf.keras.Model(inp, out)
    m.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[tf.keras.metrics.AUC(name='auc', num_thresholds=200)],
    )
    return m

# ── Hyperband search ──────────────────────────────────────────────────────────
_n_p = int(y_hp_tr.sum()); _n_a = len(y_hp_tr) - _n_p   # number of presence and absence samples in the training set
# class weights: 1:1, also Section 8 trains under a hardcoded {0:1, 1:1} (all three _s8_run_level*
# functions). Any divergence means the hyperparameters were chosen for a
# different objective than the one they are deployed under.
_cw  = {0: 1.0, 1: 1.0}

os.makedirs(TUNER_DIR, exist_ok=True)
print(f'  tuner class balance: {_n_p} pres / {_n_a} abs (ratio {_n_a / max(_n_p, 1):.3f}) -- trained at 1:1')

tuner_syn = kt.Hyperband(
    _build_tunable,
    objective=kt.Objective('val_auc', direction='max'), # select hyperparameters that maximize validation AUC, model trains on Cross-Entropy loss
    max_epochs=50,
    factor=3,       # roughly 3x more training per surviving configuration, while 3x fewer configs continue
    overwrite=True,
    seed=SEED,      # reproducible trial schedule
    directory=TUNER_DIR,
    project_name=f'cnn_sdm_synthetic_{PATCH}px',
)

print(f'\nHyperband: max_epochs=50, factor=3')
print('Search: f1/f2 in [8,16,32], dense in [8,16,32], '
      'dropout in [0.2,0.3,0.4], lr in [1e-4,3e-4,1e-3]')

tf.keras.utils.set_random_seed(SEED)   # every trial's init/shuffle follows from here
tuner_syn.search(
    x_hp_tr, y_hp_tr,
    validation_data=(x_hp_va, y_hp_va),
    class_weight=_cw,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=10, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5, patience=5, min_lr=1e-6, verbose=0),
    ],
    batch_size=BATCH_SIZE,
    verbose=0,
)

# ── Save best HPs ─────────────────────────────────────────────────────────────
_best_kt = tuner_syn.get_best_hyperparameters(1)[0]
BEST_HP  = {k: _best_kt.get(k)
            for k in ['filters_1', 'filters_2', 'dense_units', 'dropout', 'lr']}

# ── RE-SELECT THE TOP CANDIDATES OVER MULTIPLE INITS ─────────────────────────
# Hyperband ranks each configuration on ONE initialisation, and on this problem
# that ranking is inside its own noise: over the fully-trained trials the top
# five span 0.016 val_auc and the top two 0.003, while a single CNN init varies
# far more than that (the init-SD budget recorded throughout Section 8 runs
# 0.10-0.28 in spearman_true). Two searches differing only in 4% of the points
# duly picked different winners. Freezing one architecture for the whole thesis
# on a single-init coin flip contradicts the project's own rule -- "never report
# a single init as a result" -- so the shortlist is re-scored the way every other
# CNN number in this project is produced: _S8_N_INIT independent inits, mean and
# SD reported, and the mean decides.
#
# This does NOT turn into per-condition tuning. It picks ONE architecture, once,
# on the reference condition (PATCH=32, species _S8_TUNE_SEED), which is then held fixed
# everywhere. If the winner is inside one SD of the runner-up, that is stated
# rather than hidden -- the honest reading is then "these architectures are
# indistinguishable here", and the fixed choice is a convention, not a finding.
_N_SHORTLIST = 6
_cands, _seen = [], set()
for _t in tuner_syn.oracle.get_best_trials(num_trials=40):
    _hv = {k: v for k, v in _t.hyperparameters.values.items() if not k.startswith('tuner/')}
    _key = tuple(sorted(_hv.items()))
    if _key in _seen:
        continue
    _seen.add(_key); _cands.append(_hv)
    if len(_cands) >= _N_SHORTLIST:
        break

print(f'\n── Re-scoring the top {len(_cands)} configurations over '
      f'{_S8_N_INIT} inits each ──', flush=True)
print(f'{"f1":>3} {"f2":>3} {"dense":>5} {"drop":>5} {"lr":>7} | {"mean":>7} {"SD":>6}  inits',
      flush=True)
_resel = []
for _ci_hp, _hv in enumerate(_cands):
    _scores = []
    for _init in range(_S8_N_INIT):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED + 9173 * _init)   # see _s8_run_level: seeds Keras too
        _m = build_cnn(PATCH, N_LAYERS, hp=_hv)
        _h = _m.fit(x_hp_tr, y_hp_tr, validation_data=(x_hp_va, y_hp_va),
                    epochs=50, batch_size=BATCH_SIZE, class_weight=_cw,
                    callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor='val_auc', patience=12, restore_best_weights=True,
                        mode='max', verbose=0)], verbose=0)
        _scores.append(float(np.max(_h.history['val_auc'])))
        tf.keras.backend.clear_session()
    _mu = float(np.mean(_scores))
    _sd = float(np.std(_scores, ddof=1)) if len(_scores) > 1 else np.nan   # no SD from one init
    _resel.append((_mu, _sd, _hv, _scores))
    print(f'{_hv["filters_1"]:>3} {_hv["filters_2"]:>3} {_hv["dense_units"]:>5} '
          f'{_hv["dropout"]:>5} {_hv["lr"]:>7} | {_mu:>7.4f} {_sd:>6.4f}  '
          f'{[round(v, 4) for v in _scores]}', flush=True)

_resel.sort(key=lambda r: -r[0])
_best_mu, _best_sd, _best_hp, _ = _resel[0]
BEST_HP = dict(_best_hp)
print(f'\nSelected by MEAN over {_S8_N_INIT} inits: {BEST_HP}')
print(f'  mean val_auc = {_best_mu:.4f} (SD {_best_sd:.4f})')
if len(_resel) > 1:
    _gap = _best_mu - _resel[1][0]
    _pool_sd = float(np.sqrt((_best_sd ** 2 + _resel[1][1] ** 2) / 2.0))
    print(f'  gap to runner-up: {_gap:.4f}; pooled init SD: {_pool_sd:.4f} -> '
          + ('NO noise band (needs _S8_N_INIT > 1), so no verdict.' if not np.isfinite(_pool_sd) else
             'INSIDE the noise band, so the choice is a convention, not a '
             'measured optimum -- report it as such.' if _gap < _pool_sd else
             'outside the noise band, so the choice is supported.'))
print(f'  (Hyperband\'s own single-init pick was '
      f'{ {k: v for k, v in tuner_syn.get_best_hyperparameters(1)[0].values.items() if not k.startswith("tuner/")} })',
      flush=True)

_HP_PROV = _s8_hp_provenance()      # what these HP were tuned on (checked by _s8_check_hp)
with open(os.path.join(TUNER_DIR, 'best_hp_synthetic.json'), 'w') as _f:    # w: write mode
    json.dump({**BEST_HP, '_provenance': _HP_PROV}, _f, indent=2)

print('\nBest HP:')
for k, v in BEST_HP.items():
    print(f'  {k}: {v}')

print('\nTop-5 trials:')
for t in tuner_syn.oracle.get_best_trials(5):
    hpv = t.hyperparameters.values
    print(f"  val_auc={t.score:.4f}  "
          f"f={hpv.get('filters_1')}/{hpv.get('filters_2')}  "
          f"du={hpv.get('dense_units')}  dr={hpv.get('dropout')}  "
          f"lr={hpv.get('lr')}")

tf.keras.backend.clear_session()
del x_hp_tr, y_hp_tr, x_hp_va, y_hp_va, _Xp_t, _lbl_t
print('\nBEST_HP updated — all subsequent CNN calls use these HP.')


### 7.2 — Random Forest
Random search over 60 configurations on the same points and split as 7.1, with centre-pixel features and 1:1 class weights.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ── RF hyperparameter tuner -- mirrors the CNN tuner above (section 7.1) so BOTH
# models in the RF-vs-CNN comparison are tuned ONCE and then frozen, instead
# of CNN being tuned and RF left at ad-hoc/default settings. Uses the SAME
# virtual species / point draw / 80-20 split as the CNN tuner above -- same
# tuning species (_S8_TUNE_SEED), same make_valid_mask/_s8_sample_points calls, same
# train_test_split(random_state=42) -- so these are the SAME POINTS, not
# just "the same construction method". (That claim only became exactly true once
# the CNN tuner started PADDING NaN patches instead of dropping them; before, it
# searched on 1921 points against this cell's 2000.) Only the per-point feature
# representation differs: center-pixel covariates here (RF's input) instead
# of the 32x32 patch (CNN's input) extracted at the identical points above.
#
# Objective: val_auc (ROC AUC on drawn labels of the held-out split) --
# matching the CNN tuner's objective EXACTLY (kt.Objective('val_auc',
# direction='max') above). NOTE: this is NOT this project's stated PRIMARY
# metric elsewhere (spearman_true vs the TRUE continuous suitability) --
# mirroring the CNN tuner's own pre-existing choice here is what makes the
# two searches comparable, not a new inconsistency introduced by this cell.
#
# class_weight during search is 1:1, matching how the tuned RF is actually
# used. The deployed RF sets no class_weight at all (_s8_rf passes only RF_BEST_HP),
# so sklearn's default None applies -- which is exactly 1:1.

# Same change as the CNN tuner above -- see the long note there for why the old
# sample_points() draw made the search objective differ from the deployment
# condition. Identical call, identical seed, so these are still literally the
# SAME POINTS as the CNN tuner's; only the per-point representation differs
# (centre pixel here, 32x32 patch there).
if '_suit_tune' not in dir():
    raise RuntimeError('run the CNN tuner (7.1) first -- it builds the tuning species _suit_tune')
_suit_rf  = _suit_tune      # the tuning species of 7.1, outside the evaluation pool
_valid_rf = (make_valid_mask(env_scaled_100m, _suit_rf, PATCH)   # same valid mask as 7.1
             & _s8_patch_ok(env_scaled_100m, _s8_patch_buffer_px(100, PATCH)))
_r_rf, _c_rf, _lbl_rf = _s8_sample_points(
    _suit_rf, _valid_rf, _S8_N_POINTS, PRES_RATIO, seed=SEED,
    min_sep_px=PATCH * _S8_SEP_PATCHES)
_E_rf  = env_scaled_100m[_r_rf, _c_rf, :]   # extract environmental covariates at sampled points (center pixel values)
_ok_rf = ~np.isnan(_E_rf).any(1)    # filter out any points with NaN values in their environmental covariates
_E_rf, _lbl_rf = _E_rf[_ok_rf], _lbl_rf[_ok_rf] # apply mask

_tr_rf, _va_rf = train_test_split(
    np.arange(len(_E_rf)), test_size=0.20, stratify=_lbl_rf, random_state=42)   # create and divide indices for split
x_hp_tr_rf, y_hp_tr_rf = _E_rf[_tr_rf], _lbl_rf[_tr_rf] # apply split to get training data and labels
x_hp_va_rf, y_hp_va_rf = _E_rf[_va_rf], _lbl_rf[_va_rf]
print(f'RF HP holdout — train: {len(_tr_rf)}  val: {len(_va_rf)}')
print(f'  pres rate  train: {y_hp_tr_rf.mean():.3f}   val: {y_hp_va_rf.mean():.3f}')
print("  (should match the CNN tuner's holdout sizes/prevalence above exactly —"
      " same seed, same draw, same split.)")

# ── Search space ────────────────────────────────────────────────────────────
RF_PARAM_DIST = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [4, 6, 8, 10, None],
    'max_features':      ['sqrt', 'log2', None],
    'min_samples_leaf':  [1, 2, 4, 8],
    'min_samples_split': [2, 5, 10],
}
_full_grid_size = 1
for _v in RF_PARAM_DIST.values():
    _full_grid_size *= len(_v)  # calculate total number of combinations in the hyperparameter grid

# Local smoke flag (no notebook-wide SMOKE_TEST exists at this point) --
# provide a SMOKE_TEST flag, print total fits before launching.
RF_TUNER_SMOKE_TEST = False   # <- set True for a quick wiring check (5 trials instead of 60)
N_TRIALS_RF = 5 if RF_TUNER_SMOKE_TEST else 60

print(f'\n{"SMOKE_TEST MODE" if RF_TUNER_SMOKE_TEST else "FULL RUN"}: '
      f'N_TRIALS={N_TRIALS_RF}  (full grid = {_full_grid_size} combinations, '
      f'{"randomized search over ~%.0f%% of it)" % (100*N_TRIALS_RF/_full_grid_size)}')
print(f'Total model fits: {N_TRIALS_RF}', flush=True)
print('Search space:')
for _k, _v in RF_PARAM_DIST.items():
    print(f'  {_k}: {_v}')

def _sample_rf_hp(rng):     # randomly sample a hyperparameter configuration from the defined search space
    return {k: v[rng.integers(len(v))] for k, v in RF_PARAM_DIST.items()}

_n_p_rf = int(y_hp_tr_rf.sum()); _n_a_rf = len(y_hp_tr_rf) - _n_p_rf    # number of presence and absence samples in the training set
_cw_rf  = {0: 1.0, 1: 1.0}   # 1:1 -- see the note above

print(f'  RF tuner class balance: {_n_p_rf} pres / {_n_a_rf} abs '
      f'(ratio {_n_a_rf / max(_n_p_rf, 1):.3f}) -- searched at 1:1')

# ── Randomized search (single holdout, matching the CNN tuner's own setup) ────
_rf_search_rng = np.random.default_rng(SEED)
rf_trials = []
for _ti in range(N_TRIALS_RF):
    _hp = _sample_rf_hp(_rf_search_rng) # randomly sample a hyperparameter configuration from the defined search space
    _clf = RandomForestClassifier(      # build a Random Forest classifier with the sampled hyperparameters
        n_estimators=_hp['n_estimators'], max_depth=_hp['max_depth'],
        max_features=_hp['max_features'], min_samples_leaf=_hp['min_samples_leaf'],
        min_samples_split=_hp['min_samples_split'],
        class_weight=_cw_rf, random_state=42, n_jobs=1,
    )
    _clf.fit(x_hp_tr_rf, y_hp_tr_rf)    # train the Random Forest classifier on the training data and labels
    _proba = _clf.predict_proba(x_hp_va_rf)[:, 1]   # predict the probabilities of the positive class for the validation data
    _val_auc = float(roc_auc_score(y_hp_va_rf, _proba))     # compute the ROC AUC score for the validation data and predicted probabilities
    rf_trials.append({'val_auc': _val_auc, **_hp})          # store the validation AUC and the hyperparameters in a list of dictionaries

rf_trials.sort(key=lambda t: t['val_auc'], reverse=True)    # find the best configuration by sorting the trials based on validation AUC in descending order
_best_rf = rf_trials[0]
RF_BEST_HP = {k: _best_rf[k] for k in RF_PARAM_DIST}

_RF_HP_PROV = _s8_hp_provenance()   # what these HP were tuned on (checked by _s8_check_hp)
with open(os.path.join(TUNER_DIR, 'best_hp_rf_synthetic.json'), 'w') as _f: # save best hp
    json.dump({**RF_BEST_HP, '_provenance': _RF_HP_PROV}, _f, indent=2)

print(f'\nBest RF HP (val_auc={_best_rf["val_auc"]:.4f}):')
for _k, _v in RF_BEST_HP.items():
    print(f'  {_k}: {_v}')

print('\nTop-5 trials:')
for _t in rf_trials[:5]:
    print(f"  val_auc={_t['val_auc']:.4f}  n_est={_t['n_estimators']}  "
          f"max_depth={_t['max_depth']}  max_features={_t['max_features']}  "
          f"min_leaf={_t['min_samples_leaf']}  min_split={_t['min_samples_split']}")

del x_hp_tr_rf, y_hp_tr_rf, x_hp_va_rf, y_hp_va_rf, _E_rf, _lbl_rf, _suit_rf, _valid_rf, _suit_tune
print(f'\nSaved: {os.path.join(TUNER_DIR, "best_hp_rf_synthetic.json")}')
print('RF_BEST_HP updated — all subsequent RF calls that load this file use these HP.')


## 8 — Sensitivity Experiments
Every experiment runs in two arms — **random** (interpolative) and **extrap** (train, validation and test in successive PC1 bands) — with train/val/test sizes matched between the arms, and replicates over virtual species. Each heading carries the key used in code and output files (e.g. `8a`).

Two earlier experiments were retired and removed: `8c` (resolution with the truth recomputed at every grain — superseded by `8c-fixed`) and `8h` (superseded by `8j`). Code comments that mention them explain design choices, not cells that still exist.

### 8.1 — Sample Size (`8a`)
Training-set size from 100 to 5000 points; validation and test fixed at 300 points each.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
N_POINTS_RANGE = [100, 1000] if SMOKE_TEST_S8 else [100, 250, 500, 1000, 2000, 5000]
_s8a_rows = []
_S8_DROPS.clear()   # per-sweep; _s8_drop_report below lists what this sweep lost

# Replication is over SPECIES, not repeated draws of one species
# (see _s8_species_iter). n_reps=1 keeps the fit count identical
# to the previous 1-species x N-reps design.
for _sp_i, _sp_sd in _s8_species_iter():    # returns species index and seed of admitted species
    _s8a_seed = 4000 + _sp_i * 100000

    for _n in N_POINTS_RANGE:
        print(f'\n8a | N_POINTS={_n}', flush=True)
        _rows_lv = _s8_run_level(env_scaled_100m, 100, _n, PRES_RATIO, PATCH, '8a', _n, _s8a_seed, n_reps=1)
        _s8a_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _s8a_seed += 100

_s8_drop_report('8a')
results_s8a = _pd_s8.DataFrame(_s8a_rows)
print(f'\n8a: {len(results_s8a)} rows', flush=True)
if not results_s8a.empty:
    for arm in ('random','extrap'):
        _s = results_s8a[(results_s8a.arm==arm)&(results_s8a.model=='rf')]
        if not _s.empty:
            print(f'  [{arm}] RF spearman_true={_s.spearman_true.mean():.3f}'
                  f'  roc_auc={_s.roc_auc.mean():.3f}', flush=True)

# Backward compat for Section 9 (expects {level: (mean_auc, std_auc)})
# -- kept only so the older, still-present Section-9 summary-panel cell
# (which predates the two-regime design) doesn't break; not otherwise used
# by the two-regime pipeline in this cell or the Section-8 plotting cell.
results_npoints = {}
for _n in N_POINTS_RANGE:
    _sub = results_s8a[(results_s8a.level==_n)&(results_s8a.arm=='random')&
                       (results_s8a.model.isin(['cnn','rf']))]
    results_npoints[_n] = (float(_sub.roc_auc.mean()) if not _sub.empty else np.nan,
                           float(_sub.roc_auc.std())  if len(_sub)>1 else 0.0)


### 8.2 — Ecological Prevalence (`8b`)
Landscape occupancy from 0.05 to 0.7 at a constant 50 : 50 sample balance.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
# =============================================================================
# 8b -- Effect of ECOLOGICAL PREVALENCE (landscape occupancy)
#
# WHAT CHANGED AND WHY. This arm used to sweep pres_ratio, i.e. the fraction
# of PRESENCES IN THE SAMPLE. Under the probabilistic occurrence design that
# is a property of the sampling protocol, not of the species -- it is 'sample
# prevalence'. The SDM literature means SPECIES prevalence when it reports
# that prevalence degrades model accuracy: the fraction of the landscape the
# species actually occupies. Those are different quantities and only the
# second is ecology, so that is what is swept here.
#
# THE SAMPLE STAYS BALANCED AT PRES_RATIO. Holding the case-control ratio
# fixed while occupancy varies is what isolates the ecological effect: if
# the sample balance moved too, a change in skill could equally be the
# minority class getting smaller, which is a sample-size effect wearing a
# prevalence costume. 
# (For the realistic-but-confounded version, set pres_ratio=_op as well.)
#
# Key interest: AUC vs spearman_true divergence at low prevalence (Lobo 2008).
# Distance drift (extrap arm): expected small -- occupancy changes WHICH
# pixels are occupied, not the geographic spread of the training draw.
# =============================================================================
OCC_PREV_RANGE = [0.05, 0.2] if SMOKE_TEST_S8 else [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]
_s8b_rows = []
_S8_DROPS.clear()   # per-sweep; _s8_drop_report below lists what this sweep lost

# Replication is over SPECIES, not repeated draws of one species
# (see _s8_species_iter). n_reps=1 keeps the fit count identical
# to the previous 1-species x N-reps design.
for _sp_i, _sp_sd in _s8_species_iter():
    _s8b_seed = 5000 + _sp_i * 100000

    for _op in OCC_PREV_RANGE:
        print(f'\n8b | landscape occupancy={_op}  (sample balance held at {PRES_RATIO})',
              flush=True)
        _rows_lv = _s8_run_level(env_scaled_100m, 100, _S8_N_POINTS, PRES_RATIO, PATCH, '8b', _op, _s8b_seed, occ_prev=_op, n_reps=1)
        _s8b_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _s8b_seed += 100

_s8_drop_report('8b')
results_s8b = _pd_s8.DataFrame(_s8b_rows)
print(f'\n8b: {len(results_s8b)} rows', flush=True)
if not results_s8b.empty:
    for arm in ('random','extrap'):
        _s = results_s8b[(results_s8b.arm==arm)&(results_s8b.model=='rf')]
        if not _s.empty:
            print(f'  [{arm}] RF spearman_true={_s.spearman_true.mean():.3f}'
                  f'  roc_auc={_s.roc_auc.mean():.3f}', flush=True)

# Backward compat for Section 9 -- see the matching note in cell 22 (8a).
results_presratio = {}
for _pr in OCC_PREV_RANGE:
    _sub = results_s8b[(results_s8b.level==_pr)&(results_s8b.arm=='random')&
                       (results_s8b.model.isin(['cnn','rf']))]
    results_presratio[_pr] = (float(_sub.roc_auc.mean()) if not _sub.empty else np.nan,
                               float(_sub.roc_auc.std())  if len(_sub)>1 else 0.0)


### 8.3 — Spatial Resolution with Precise Points and a Fixed 100 m Truth (`8c-fixed`)
Can a model trained on exact 100 m locations use coarser covariates without losing information? Truth, occupancy and points stay at 100 m; only the covariates the models see are block-averaged — 100/200/400 m at a constant 3200 m footprint (headline, the **same points and labels at every level**) and 100–1000 m at a fixed 32-px patch (sensitivity). Scored against the 100 m truth at each point; `oracle_spearman_infoloss` is the ceiling — how well the block mean of the truth over the point's coarse cell ranks the points.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
# =============================================================================
# 8c-fixed -- Spatial resolution with PRECISE POINTS and a FIXED 100 m truth
#
# THE QUESTION. The occurrence data are exact locations. Can the models be run
# on COARSER covariates -- to save computation -- without losing what matters,
# or do they need the finest grain available?
#
# DESIGN (_s8_run_level_fixedtruth):
#   * The species lives at 100 m. Its truth is the 100 m surface, occupancy is
#     one Bernoulli draw per 100 m pixel from it, and points are sampled on the
#     100 m grid (PC1 bands of the extrapolative arm too).
#   * Only the COVARIATES the models see are coarsened: each point reads the
#     coarse cell that contains it, and its CNN patch is centred on that cell.
#   * constant_footprint (headline): one eligibility mask for the whole ladder
#     and a separation that is the same at every level (3200 m patch, 4525 m),
#     plus one seed per species and arm, so every level uses the SAME points with
#     the SAME labels -- a paired comparison in which resolution is the only
#     change. Checked after the sweep via a point fingerprint.
#   * fixed_patch_px (sensitivity): a 32 px patch covers more ground as the grain
#     coarsens, the separation grows with it (45 km at 1000 m), so points differ
#     per level and N falls at the coarse end.
#   * Scored against the 100 m truth AT THE POINT.
#   * oracle_spearman_infoloss: Spearman between the block mean of the 100 m
#     truth over the point's coarse cell and the truth at the point -- how well
#     even perfect knowledge of the coarse cell ranks the points. 1.0 at 100 m.
#   * Aggregation noise is part of the design: points whose coarse cells look
#     alike can still differ in truth and label.
#
# WHAT IT REPLACES. The earlier version drew labels from the suitability
# RECOMPUTED on the coarsened covariates -- a species responding to the
# landscape only as seen at the coarse grain -- scored them against the
# block-mean truth, and drew different points at every level. That measured how
# similar the coarse and fine suitability surfaces are, not whether coarse
# modelling of precise data is safe.
# =============================================================================
import hashlib as _hl_fx


def _s8_blockmean_to_coarse_grid(truth_100m, resolution, base_res_m=100):
    """Block mean of a 100 m (H,W) array onto the grid resample_env produces at
    `resolution` (same crop/factor logic), so coarse pixel (i,j) here is coarse
    pixel (i,j) of the coarsened covariates."""
    if resolution == base_res_m:
        return np.asarray(truth_100m, np.float32).copy()
    factor = int(round(resolution / base_res_m))
    H, W = truth_100m.shape
    H_c, W_c = (H // factor) * factor, (W // factor) * factor
    blk = truth_100m[:H_c, :W_c].reshape(H_c // factor, factor, W_c // factor, factor)
    return np.nanmean(blk, axis=(1, 3))


_S8_FIXED_MASK_CACHE = {}


def _s8_fixed_common_mask(env_base, ladder, base_res_m=100):
    """100 m pixels eligible at EVERY (resolution, patch) level of `ladder`:
    valid at 100 m, and at each level the coarse cell containing the pixel passes
    that level's patch rule (_s8_patch_ok). A whole ladder gives the points all
    levels can share; a single level gives that level's own eligible set.
    Species-independent, so cached (a handful of 55 MB masks)."""
    key = (env_base.shape, tuple((int(r), int(p)) for r, p in ladder))
    if key not in _S8_FIXED_MASK_CACHE:
        H, W, _ = env_base.shape
        ok = np.all(np.isfinite(env_base), axis=2)
        for res, patch in ladder:
            f = int(round(res / base_res_m))
            env_c = resample_env(env_base, res)
            okc = _s8_patch_ok(env_c, _s8_patch_buffer_px(res, patch))
            up = np.zeros((H, W), bool)
            up[:okc.shape[0] * f, :okc.shape[1] * f] = np.repeat(np.repeat(okc, f, 0), f, 1)
            ok &= up
            del env_c, okc, up
        while len(_S8_FIXED_MASK_CACHE) >= 8:
            _S8_FIXED_MASK_CACHE.pop(next(iter(_S8_FIXED_MASK_CACHE)))
        _S8_FIXED_MASK_CACHE[key] = ok
    return _S8_FIXED_MASK_CACHE[key]


def _s8_run_level_fixedtruth(env_base, resolution, n_points, pres_ratio, patch,
                             sub_exp, level, seed_base, sample_mask_100, truth_100,
                             occ_prev=None, n_reps=None, cfg=None, extra_cols=None):
    """ONE resolution level: precise 100 m points, fixed 100 m truth, coarsened
    covariates (see the cell header). Same fitting machinery as _s8_run_level:
    matched set sizes, cross-set separation, 1:1 class weights, _S8_N_INIT CNN
    inits, training-mean padding, train-only CNN scaling, both-classes guards.

    sample_mask_100  (H,W) bool, 100 m pixels eligible to be sampled
                     (_s8_fixed_common_mask) -- the caller decides whether the
                     points are shared across levels.
    truth_100        (H,W) 100 m truth: occupancy, labels and scoring.
    cfg              None for the pointwise DGP. For the configurational DGP the
                     dict from _s8_config_truth_100m: the models then also see the
                     forest PROPORTION of the coarse cell as an extra channel, and
                     the patch-summary RF, the oracle-metric RF and the achievable
                     ceiling are added as in _s8_run_level_config.
    extra_cols       constant columns stamped on every row (DGP parameters)."""
    f = int(round(resolution / 100))
    H, W, _ = env_base.shape
    env_arr = resample_env(env_base, resolution)              # what the models see
    valid_c = np.all(np.isfinite(env_arr), axis=2)
    if cfg is not None:
        _v100 = np.all(np.isfinite(env_base), axis=2)
        W_c = _s8_blockmean_to_coarse_grid(
            np.where(_v100, cfg['W_100m'], 0.0).astype(np.float32), resolution).astype(np.float32)
        env_in = np.concatenate([env_arr, W_c[..., None]], axis=2)
        Pw_patch_c, sd_patch_c = _s8_patch_summary_fields(W_c, valid_c, patch)
        Pw_or_c = _s8_blockmean_to_coarse_grid(cfg['Pw_100'], resolution)
        E_or_c  = _s8_blockmean_to_coarse_grid(cfg['E_100'], resolution)
        _ceil_pop, _ceil_pred = _s8_config_ceiling(
            W_c, _s8_blockmean_to_coarse_grid(cfg['S_config_100'], resolution), valid_c, patch)
    else:
        env_in = env_arr
    C_in = env_in.shape[2]
    truth_blk = _s8_blockmean_to_coarse_grid(truth_100, resolution)   # ceiling reference

    # occupancy intercept: a property of the 100 m landscape, solved once
    ok_truth = np.isfinite(truth_100) & np.all(np.isfinite(env_base), axis=2)
    _occ_c = _s8_solve_c(truth_100[ok_truth], _S8_OCC_K,
                         _S8_OCC_PREV if occ_prev is None else occ_prev)
    elig = sample_mask_100 & ok_truth
    tr_b, val_b, te_b = _s8_region_masks(env_base, 0)          # PC1 bands of the 100 m points
    tr_m, val_m, te_m = tr_b & elig, val_b & elig, te_b & elig
    sep100 = patch * resolution / 100.0 * _S8_SEP_PATCHES     # one coarse-patch diagonal, 100 m px
    _pb = _s8_patch_buffer_px(resolution, patch)
    rows_out = []

    for rep in range(_S8_N_REPS if n_reps is None else int(n_reps)):
        seed = seed_base + rep * 997
        for arm in ('random', 'extrap'):
            t0 = _t8.time()

            def _cln(rr, cc):
                # the covariates the models see: the coarse cell containing the point
                E = env_arr[rr // f, cc // f, :].astype(np.float32)
                return E[~np.isnan(E).any(1)]

            try:
                if arm == 'random':
                    # matched set sizes and a fixed hold-out, as in _s8_run_level
                    _n_draw = int(n_points) + 2 * _S8_N_WIN
                    if elig.sum() < max(_n_draw, 30):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'eligible mask holds {int(elig.sum())} px, need {max(_n_draw, 30)}'); continue
                    r, c, lbl = _s8_sample_points(truth_100, elig, _n_draw, pres_ratio, seed=seed,
                                                  min_sep_px=sep100, occ_prev=occ_prev, occ_c=_occ_c)
                    lbl = lbl.astype(np.int8)
                    if lbl.sum() < 3 or (1-lbl).sum() < 3:
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'draw has {int(lbl.sum())} pres / {int((1-lbl).sum())} abs, need 3 of each'); continue
                    if len(lbl) >= _n_draw:
                        _hold = 2 * _S8_N_WIN
                    else:
                        _hold = int(round(len(lbl) * (2.0 * _S8_N_WIN) / _n_draw))
                    _hold = max(2, min(_hold, len(lbl) - 4))
                    tr_i, _rest_i = _tts(np.arange(len(lbl)), test_size=_hold,
                                         stratify=lbl, random_state=seed)
                    if len(np.unique(lbl[tr_i])) < 2:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'training fold ended up single-class after the split'); continue
                    val_i, te_i = _tts(_rest_i, test_size=0.5,
                                       stratify=lbl[_rest_i], random_state=seed+1)
                    r_tr=r[tr_i];  c_tr=c[tr_i];  y_tr=lbl[tr_i]
                    r_v =r[val_i]; c_v =c[val_i]; y_v =lbl[val_i]
                    r_te=r[te_i];  c_te=c[te_i];  y_te=lbl[te_i]
                else:
                    if tr_m.sum() < max(n_points, 10):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'train band holds {int(tr_m.sum())} px, need {max(n_points, 10)}'); continue
                    if val_m.sum() < 10 or te_m.sum() < 10:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'val or test band holds fewer than 10 eligible px'); continue
                    r_tr, c_tr, lbl_tr = _s8_sample_points(truth_100, tr_m, n_points, pres_ratio, seed=seed,
                                                           min_sep_px=sep100, occ_prev=occ_prev, occ_c=_occ_c)
                    y_tr = lbl_tr.astype(np.int8)
                    # cross-set separation, as in _s8_run_level
                    _used_rc = list(zip(r_tr.astype(float), c_tr.astype(float)))
                    r_v, c_v, lbl_v = _s8_sample_points(truth_100, val_m, min(_S8_N_WIN, int(val_m.sum())),
                                                        pres_ratio, seed=seed+100, min_sep_px=sep100,
                                                        occ_prev=occ_prev, occ_c=_occ_c, seed_grid=_used_rc)
                    y_v = lbl_v.astype(np.int8)
                    _used_rc += list(zip(r_v.astype(float), c_v.astype(float)))
                    r_te, c_te, lbl_te = _s8_sample_points(truth_100, te_m, min(_S8_N_WIN, int(te_m.sum())),
                                                           pres_ratio, seed=seed+200, min_sep_px=sep100,
                                                           occ_prev=occ_prev, occ_c=_occ_c, seed_grid=_used_rc)
                    y_te = lbl_te.astype(np.int8)
                d_te = _s8_eds_decorr(_cln(r_te, c_te), _cln(r_tr, c_tr))
                d_vl = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_tr, c_tr))
                d_vt = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_te, c_te))
            except _S8SampleSkip as _skip:
                print(f'  SKIP ({arm} {sub_exp} lv={level} rep={rep}): {_skip}', flush=True)
                continue

            # INVARIANT: every fitted point has a defined 100 m truth (loud, not fatal)
            _bad = sum(int((~np.isfinite(truth_100[_a, _b])).sum())
                       for _a, _b in ((r_tr, c_tr), (r_v, c_v), (r_te, c_te)))
            if _bad:
                _s8_drop(sub_exp, level, arm, rep,
                         f'{_bad} drawn point(s) have an undefined truth'); continue
            if y_te.sum() < 3 or (1-y_te).sum() < 3:
                _s8_drop(sub_exp, level, arm, rep,
                         f'test fold has {int(y_te.sum())} pres / {int((1-y_te).sum())} abs'); continue
            s_te = truth_100[r_te, c_te]                          # truth AT THE POINT
            R = lambda rr: rr // f                                # coarse cell of a 100 m point
            _sig = truth_blk[R(r_te), R(c_te)]
            _okc = np.isfinite(_sig) & np.isfinite(s_te)
            oracle_spearman_infoloss = (float(_sp(_sig[_okc], s_te[_okc])[0])
                                        if _okc.sum() >= 5 else np.nan)
            _fp = _hl_fx.sha1(np.concatenate(
                [r_tr, c_tr, y_tr, r_v, c_v, y_v, r_te, c_te, y_te]).astype(np.int64).tobytes()
                ).hexdigest()[:12]
            base = dict(sub_exp=sub_exp, level=level, rep=rep, arm=arm,
                        n_tr=int(len(y_tr)), n_te=int(len(y_te)), n_vl=int(len(y_v)),
                        patch=int(patch), patch_buffer_px=int(_pb),
                        footprint_m=int(patch * resolution), resolution=int(resolution),
                        sampling_res_m=100,
                        # separation in coarse-patch terms and on the ground
                        min_sep_px=float(patch * _S8_SEP_PATCHES),
                        min_sep_m=float(patch * _S8_SEP_PATCHES * resolution),
                        **_S8_BAND_FRAC.get((H, W, 0), {}),
                        real_prev_tr=float(y_tr.mean()), real_prev_te=float(y_te.mean()),
                        real_prev_vl=float(y_v.mean()),
                        decc_te_tr=d_te, decc_vl_tr=d_vl, decc_vl_te=d_vt,
                        oracle_spearman_infoloss=oracle_spearman_infoloss,
                        points_fingerprint=_fp,
                        **(extra_cols or {}))
            if cfg is not None:
                # achievable ceiling for a patch-limited model, at these test points
                if _ceil_pred is None:
                    base['ceiling_config_patch'] = np.nan
                else:
                    _cp = _ceil_pred[R(r_te), R(c_te)]; _cs = cfg['S_config_100'][r_te, c_te]
                    _okp = np.isfinite(_cp) & np.isfinite(_cs)
                    base['ceiling_config_patch'] = (float(_sp(_cp[_okp], _cs[_okp])[0])
                                                    if _okp.sum() >= 5 else np.nan)
                base['ceiling_config_patch_pop'] = _ceil_pop

            # -- RF (centre cell) -------------------------------------------------
            E_tr  = env_in[R(r_tr), R(c_tr), :].astype(np.float32)
            ok_tr = ~np.isnan(E_tr).any(1)
            E_trc = E_tr[ok_tr]
            clf   = _s8_rf(); clf.fit(E_trc, y_tr[ok_tr])
            E_te_ = env_in[R(r_te), R(c_te), :].astype(np.float32)
            ok_te = ~np.isnan(E_te_).any(1)
            p_rf  = np.full(len(r_te), np.nan)
            if ok_te.sum() >= 5:
                p_rf[ok_te] = clf.predict_proba(E_te_[ok_te])[:, 1]
            rows_out.append({**base, 'model': 'rf', 'cnn_init': 0, 'early_stop_ep': np.nan,
                             'val_auc_best': np.nan, 'train_auc_at_best': np.nan,
                             **_s8_mets(p_rf, y_te, s_te)})
            if cfg is not None:
                for _nm, _fa, _fb in (('rf_patch_summary', Pw_patch_c, sd_patch_c),
                                      ('rf_oracle_metrics', Pw_or_c, E_or_c)):
                    _xtr = np.concatenate([E_tr, np.stack([_fa[R(r_tr), R(c_tr)], _fb[R(r_tr), R(c_tr)]], 1)], 1)
                    _xte = np.concatenate([E_te_, np.stack([_fa[R(r_te), R(c_te)], _fb[R(r_te), R(c_te)]], 1)], 1)
                    _otr = ~np.isnan(_xtr).any(1); _ote = ~np.isnan(_xte).any(1)
                    _p = np.full(len(r_te), np.nan)
                    if _otr.sum() >= 10 and y_tr[_otr].sum() >= 5 and _ote.sum() >= 5:
                        _cl = _s8_rf(); _cl.fit(_xtr[_otr], y_tr[_otr])
                        _p[_ote] = _cl.predict_proba(_xte[_ote])[:, 1]
                    rows_out.append({**base, 'model': _nm, 'cnn_init': 0, 'early_stop_ep': np.nan,
                                     'val_auc_best': np.nan, 'train_auc_at_best': np.nan,
                                     **_s8_mets(_p, y_te, s_te)})

            # -- CNN --------------------------------------------------------------
            n_vp = int(y_v.sum()); n_va = int((y_v == 0).sum())
            if n_vp < _S8_MIN_VAL or n_va < _S8_MIN_VAL:
                _s8_drop(sub_exp, level, arm, rep,
                         f'val fold has {n_vp} pres / {n_va} abs, below _S8_MIN_VAL='
                         f'{_S8_MIN_VAL} -- CNN skipped (RF rows were written)')
                continue
            tr_ch_mean = E_trc.mean(0)

            def _pad(rr, cc):
                X = extract_patches(env_in, R(rr), R(cc), patch=patch)
                for ch in range(C_in):
                    nm = np.isnan(X[..., ch])
                    if nm.any(): X[..., ch][nm] = tr_ch_mean[ch]
                return X

            X_tr, X_val, X_te = _pad(r_tr, c_tr), _pad(r_v, c_v), _pad(r_te, c_te)
            assert not np.isnan(X_te).any(), '_pad left NaNs in the test patches'
            X_tr, X_val, X_te = _s8_train_minmax(X_tr, X_val, X_te)
            cnn_ps = []
            for ci in range(_S8_N_INIT):
                tf.keras.backend.clear_session()
                tf.keras.utils.set_random_seed(seed + ci * 137)   # seeds Keras too (see _s8_run_level)
                model = build_cnn(patch, C_in, hp=BEST_HP)
                hist  = model.fit(
                    X_tr, y_tr, validation_data=(X_val, y_v),
                    epochs=_S8_EPOCHS, batch_size=_S8_BATCH, class_weight={0: 1.0, 1: 1.0},
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor='val_auc', patience=_S8_PAT, restore_best_weights=True, mode='max'),
                        tf.keras.callbacks.ReduceLROnPlateau(
                            monitor='val_auc', factor=0.5, patience=5, min_lr=1e-6, mode='max'),
                    ], verbose=0)
                _vh = list(hist.history.get('val_auc', [])); _th = list(hist.history.get('auc', []))
                _bi = int(np.argmax(_vh)) if _vh else 0
                p_c = model.predict(X_te, verbose=0).ravel()
                cnn_ps.append(p_c)
                rows_out.append({**base, 'model': 'cnn', 'cnn_init': ci, 'early_stop_ep': _bi + 1,
                                 'val_auc_best': float(_vh[_bi]) if _vh else np.nan,
                                 'train_auc_at_best': float(_th[_bi]) if _bi < len(_th) else np.nan,
                                 **_s8_mets(p_c, y_te, s_te)})
                tf.keras.backend.clear_session()
            if len(cnn_ps) > 1:
                mp = np.nanmean(np.stack(cnn_ps, 0), axis=0)
                rows_out.append({**base, 'model': 'cnn_mean', 'cnn_init': -1, 'early_stop_ep': np.nan,
                                 'val_auc_best': np.nan, 'train_auc_at_best': np.nan,
                                 **_s8_mets(mp, y_te, s_te)})
            print(f'  [{arm}] {sub_exp} lv={level} rep={rep}: n_tr={len(y_tr)} n_te={len(y_te)} '
                  f'points={_fp} oracle_spearman_infoloss={oracle_spearman_infoloss:.3f} '
                  f'[{_t8.time()-t0:.0f}s]', flush=True)
    return rows_out


def _s8_fixed_points_report(df):
    """The constant-footprint arm must use the SAME points at every level."""
    if df.empty or 'points_fingerprint' not in df.columns:
        return
    d = df[(df.res_mode == 'constant_footprint') & (df.model == 'rf')]
    g = d.groupby(['species_idx', 'arm'])['points_fingerprint'].nunique()
    bad = g[g > 1]
    print('  same points at every constant-footprint level: '
          + ('YES' if bad.empty else f'NO -- {bad.to_dict()}'), flush=True)


# -- Run across both resolution conventions ------------------------------------
RESOLUTION_RANGE_FIXED = [100, 500] if SMOKE_TEST_S8 else [100, 200, 300, 500, 1000]
_S8CF_ARMS = [
    ('constant_footprint', _S8_RES_CONSTFOOT,      0),
    ('fixed_patch_px',     RESOLUTION_RANGE_FIXED, 50_000),
]
_s8cf_rows = []
_S8_DROPS.clear()

for _sp_i, _sp_sd in _s8_species_iter():
    for _res_mode, _res_levels, _mode_off in _S8CF_ARMS:
        # ONE seed per (species, arm): with the same mask, separation and seed the
        # sampler returns the same points at every constant-footprint level.
        _s8cf_seed = 6500 + _sp_i * 100000 + _mode_off
        _ladder = [(_r, _s8_constfoot_patch(_r) if _res_mode == 'constant_footprint' else PATCH)
                   for _r in _res_levels]
        for _res, _patch_lv in _ladder:
            _mask_cf = _s8_fixed_common_mask(
                env_scaled_100m, _ladder if _res_mode == 'constant_footprint' else [(_res, _patch_lv)])
            print(f'\n8c-fixed [{_res_mode}] | covariates at {_res} m, PATCH={_patch_lv}px '
                  f'(footprint {_res * _patch_lv / 1000:.1f} km); points and truth at 100 m', flush=True)
            _rows_lv = _s8_run_level_fixedtruth(env_scaled_100m, _res, _S8_N_POINTS, PRES_RATIO,
                                                _patch_lv, '8c-fixed', _res, _s8cf_seed,
                                                _mask_cf, _s8_suit_100m, n_reps=1)
            for _r in _rows_lv:
                _r['res_mode'] = _res_mode
            _s8cf_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))

_s8_drop_report('8c-fixed')
results_s8c_fixed = _pd_s8.DataFrame(_s8cf_rows)
_s8_fixed_points_report(results_s8c_fixed)

# -- spearman_rel_ceiling: skill RELATIVE TO the achievable ceiling ----------
# spearman_true / oracle_spearman_infoloss, on the same test points and the
# same reference (the fixed 100 m truth). The ceiling absorbs the loss that
# coarsening inflicts before any model sees the data, so this isolates MODEL
# shortfall and is comparable across resolutions in a way the raw difference is
# not. At 100 m the ceiling is exactly 1.0 and the ratio degenerates to
# spearman_true, which is the boundary check on the whole construction.
#
# IT IS NOT A FRACTION, DESPITE THE OBVIOUS READING. Two reasons, both measured:
#  - A ratio of CORRELATIONS is not a share of signal. The variance-style
#    reading is rho^2 / rho_ceiling^2, which is materially smaller: at
#    8c-fixed / 100 m it is 0.53 where the rho ratio reads 0.72. Both are
#    reported (see spearman_rel_ceiling_sq below) so the text can name which
#    one it means.
#  - It goes NEGATIVE when the model is anti-correlated with truth: 3.7% of
#    8c-fixed rows and 13.9% of the 8i resolution rows. "-0.23 of the
#    recoverable signal captured" is not a sentence; those rows mean the
#    prediction ranks the wrong way, and the name must not imply otherwise.
#    The old name (spearman_rel_ceiling) did imply otherwise.
# Guards:
#  - ceiling <= 0.05 -> NaN. A near-zero ceiling means coarsening destroyed
#    essentially all recoverable signal; the ratio would explode and mean
#    nothing.
#  - NOT clipped at 1.0. The ceiling is a natural reference, not a strict
#    mathematical bound: a model could in principle find some function of the
#    coarse covariates that tracks fine truth slightly better than the coarse
#    truth surface itself does. Measured: never in 8c-fixed, 3.1% of 8i rows.
#    A ratio above 1 is a real signal worth seeing, not an error to hide.
_ceil_cf = results_s8c_fixed['oracle_spearman_infoloss']
results_s8c_fixed['spearman_rel_ceiling'] = (
    results_s8c_fixed['spearman_true'] / _ceil_cf).where(_ceil_cf > 0.05)
# the variance-style companion: this one IS a share, and is the number to quote
# whenever the text says "fraction of the recoverable signal".
results_s8c_fixed['spearman_rel_ceiling_sq'] = (
    (results_s8c_fixed['spearman_true'] ** 2) / (_ceil_cf ** 2)).where(_ceil_cf > 0.05)
print(f'\n8c-fixed: {len(results_s8c_fixed)} rows', flush=True)

# -- Per-level summary: spearman_true (RF, CNN), oracle_spearman_infoloss, mae_true, n --
# Interpretation:
#  - A monotone drop in spearman_true with coarsening is the expected
#    info-loss signal: truth and points are fixed at 100 m, so coarsening the
#    covariates can only destroy information.
#  - oracle_spearman_infoloss minus a model's spearman_true = how much that model
#    loses BEYOND the unavoidable info loss already baked into
#    oracle_spearman_infoloss. If a model's spearman_true tracks oracle_spearman_infoloss down
#    as resolution coarsens, it's using the available coarse information
#    about as well as possible; if it falls FASTER than the oracle, it's
#    losing information the coarse data still contained.
#  - At level=100 (no coarsening), oracle_spearman_infoloss should read exactly 1.000
#    -- the block mean over a 100 m cell is the truth itself; this is a sanity
#    check on the whole mechanism, not a substantive finding.
# REPORTING IS NON-FATAL. The fits above cost hours; a KeyError in a
# summary print must never discard them. The traceback is printed in
# full so the bug is still obvious.
try:
    if not results_s8c_fixed.empty:
        # Split by arm: the two res_modes use DIFFERENT patch sizes at the same
        # resolution, so pooling them would average a 32 px and an 8 px CNN into
        # one 'level' and make the ceiling comparison meaningless.
        print(f'\n{"mode":>19}  {"level":>6}  {"patch":>5}  {"n_eval":>7}  {"oracle_sp":>10}  '
              f'{"RF spear":>9}  {"CNN spear":>10}  {"RF frac":>8}  {"CNN frac":>9}  '
              f'{"RF mae":>8}  {"CNN mae":>8}')
        for _res_mode, _res_levels, _ in _S8CF_ARMS:
          for _res in _res_levels:
            _lvl = results_s8c_fixed[(results_s8c_fixed.level==_res) &
                                     (results_s8c_fixed.res_mode==_res_mode) &
                                     (results_s8c_fixed.arm=='random')]
            if _lvl.empty:
                print(f'{_res_mode:>19}  {_res:>6}  {"-":>5}  {"(level skipped)":>7}')
                continue
            _orc = _lvl.oracle_spearman_infoloss.mean() if not _lvl.empty else np.nan
            _rf  = _lvl[_lvl.model=='rf']
            # cnn_mean only exists when _S8_N_INIT > 1 (averaging one init is
            # meaningless, so _s8_run_level_fixedtruth skips it) -- fall back to the
            # single 'cnn' row in smoke mode, otherwise this whole column reads
            # NaN in exactly the mode used for a first glance at results.
            _cnn = _lvl[_lvl.model=='cnn_mean']
            if _cnn.empty: _cnn = _lvl[_lvl.model=='cnn']
            _n   = int(_rf.n_te.mean()) if not _rf.empty else 0
            _pw  = int(_lvl.patch.iloc[0])
            print(f'{_res_mode:>19}  {_res:>6}  {_pw:>5}  {_n:>7}  {_orc:>10.3f}  '
                  f'{_rf.spearman_true.mean() if not _rf.empty else np.nan:>9.3f}  '
                  f'{_cnn.spearman_true.mean() if not _cnn.empty else np.nan:>10.3f}  '
                  f'{_rf.spearman_rel_ceiling.mean() if not _rf.empty else np.nan:>8.3f}  '
                  f'{_cnn.spearman_rel_ceiling.mean() if not _cnn.empty else np.nan:>9.3f}  '
                  f'{_rf.mae_true.mean() if not _rf.empty else np.nan:>8.3f}  '
                  f'{_cnn.mae_true.mean() if not _cnn.empty else np.nan:>8.3f}')

except Exception as _rep_err:
    import traceback as _tb_rep
    print(f'\n[REPORTING FAILED -- results are intact] {type(_rep_err).__name__}: {_rep_err}', flush=True)
    _tb_rep.print_exc()
# -- Save (new, versioned output; does NOT touch section8_two_regime.csv) ----
_suffix_cf = '_smoke' if SMOKE_TEST_S8 else ''
_s8cf_out = os.path.join(FIGURES_DIR, f'section8c_fixed_truth_{_S8_VERSION}{_suffix_cf}.csv')
if not results_s8c_fixed.empty:
    results_s8c_fixed.to_csv(_s8cf_out, index=False)
    print(f'\nSaved: {_s8cf_out}  ({len(results_s8c_fixed)} rows)', flush=True)
print('(section8_two_regime.csv is not touched by this cell.)')


### 8.4 — Patch Size (`8e`)
Patch 8–64 px at 100 m. The RF sees only the centre pixel, so its curve reflects sampling geometry rather than spatial context. Every level samples the same area — where a 64-px patch fits — so the patch effect is not confounded with the sampled area.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
# =============================================================================
# 8e -- Effect of Patch Size (PATCH)
# All at 100m resolution. Larger patches give the CNN more spatial context
# but reduce valid pixels (border exclusion grows).
# Distance drift (extrap arm): expected small for 8e -- env center-pixel
# values are unchanged; only spatial context (patch) varies.
# =============================================================================
# Sweeps PATCH (CNN input patch size in pixels) at fixed N_POINTS/PRES_RATIO/
# RESOLUTION=100m. Only the CNN's spatial context changes here (RF only ever
# sees the center pixel regardless of patch, via env_arr[r,c,:] in cell 22) --
# so this sub-experiment isolates how much context helps the CNN specifically.
# Larger patch -> _s8_region_masks excludes a wider border (half=patch//2), so
# the pool of valid pixels to sample from shrinks as PATCH grows.
PATCH_RANGE = [8, 32] if SMOKE_TEST_S8 else [8, 16, 32, 64]
_s8e_rows = []
_S8_DROPS.clear()   # per-sweep; _s8_drop_report below lists what this sweep lost

# Replication is over SPECIES, not repeated draws of one species
# (see _s8_species_iter). n_reps=1 keeps the fit count identical
# to the previous 1-species x N-reps design.
for _sp_i, _sp_sd in _s8_species_iter():
    _s8e_seed = 7000 + _sp_i * 100000

    for _p in PATCH_RANGE:
        print(f'\n8e | PATCH={_p}px ({_p*100}m context)', flush=True)
        _rows_lv = _s8_run_level(env_scaled_100m, 100, _S8_N_POINTS, PRES_RATIO, _p, '8e', _p, _s8e_seed, n_reps=1,
                                 patch_buffer_px=max(PATCH_RANGE))   # same area at every patch size
        _s8e_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _s8e_seed += 100

_s8_drop_report('8e')
results_s8e = _pd_s8.DataFrame(_s8e_rows)
print(f'\n8e: {len(results_s8e)} rows', flush=True)
if not results_s8e.empty:
    for arm in ('random','extrap'):
        _s = results_s8e[(results_s8e.arm==arm)&(results_s8e.model=='rf')]
        if not _s.empty:
            print(f'  [{arm}] RF spearman_true={_s.spearman_true.mean():.3f}'
                  f'  roc_auc={_s.roc_auc.mean():.3f}', flush=True)

# Backward compat for Section 9 -- see the matching note in cell 22 (8a).
results_patch = {}
for _p in PATCH_RANGE:
    _sub = results_s8e[(results_s8e.level==_p)&(results_s8e.arm=='random')&
                       (results_s8e.model.isin(['cnn','rf']))]
    results_patch[_p] = (float(_sub.roc_auc.mean()) if not _sub.empty else np.nan,
                         float(_sub.roc_auc.std())  if len(_sub)>1 else 0.0)

# -- Save combined Section 8 output -------------------------------------------
# Concatenates the three pointwise sub-experiments that share this row schema
# (8a/8b/8e) into section8_two_regime.csv -- the file the Section-8 plotting
# cell (and the one-pager's sample-size plot) reads from.
# DELIBERATELY EXCLUDED: 8c-fixed and 8h. Both carry extra columns
# (oracle_spearman_infoloss / spearman_rel_ceiling; alpha / w_source /
# corr_len_m / resolution) and would inject NaN columns here. They write their
# own CSVs -- so they are NOT visible to the plotting cell that reads this file.
import os as _os_s8
_s8_all = _pd_s8.concat(
    [df for df in [results_s8a, results_s8b, results_s8e]
     if not df.empty], ignore_index=True)
_suffix = '_smoke' if SMOKE_TEST_S8 else ''
_s8_out = _os_s8.path.join(FIGURES_DIR, f'section8_two_regime_{_S8_VERSION}{_suffix}.csv')
if not _s8_all.empty:
    _s8_all.to_csv(_s8_out, index=False)
    print(f'\nSaved: {_s8_out}  ({len(_s8_all)} rows)', flush=True)


### 8.5 — Results: Pointwise Experiments
Figures for 8.1–8.4.

In [ ]:
# =============================================================================
# Section 8 -- Plots (run after 8a/8b/8c-fixed/8e)
#
# Restyled to match the 8h plotting cell so the two sections can be read side
# by side: same colourblind-safe palette, same per-rep scatter, same init-SD
# noise band, same fit-health flagging, same confound check, same verdict
# convention. Figures produced:
#   s8_{sexp}_panels.png        : spearman_true | roc_auc, per
#                                 sub-experiment (8a / 8b / 8c-fixed / 8e)
#   s8_cnn_vs_rf.png            : CNN - RF paired contrast vs the noise band
#   s8_8c_fixed_infoloss.png    : 8c-fixed -- skill vs the info-loss ceiling
#   s8_eds_decorr_drift.png     : eds_decorr drift (8a/8c-fixed, extrap arm)
#   s8_extrap_contrast.png      : extrap - random contrast in spearman_true
#
# CONVENTIONS (identical to 8h):
#   colour = MODEL (Okabe-Ito), arm = linestyle or its own panel, never both.
#   Faint dots are individual reps, shown raw -- reps within a level share a
#   species and are NOT independent, so they are never reduced to an SE or CI.
#   Every CNN difference is judged against the CNN init-SD: the SD of
#   spearman_true across random-weight inits fit on the SAME drawn data,
#   averaged over reps. Smaller than that band => reported INCONCLUSIVE.
#   No p-values anywhere, for the nesting reason above.
# =============================================================================
import pandas as pd
import matplotlib.pyplot as _plt8
import matplotlib.ticker as _mtk8

# Suffix must match what the 8e cell SAVED -- this previously hardcoded the
# unsuffixed name, so in smoke mode it silently plotted a stale full-run CSV.
_sfx8p      = '_smoke' if SMOKE_TEST_S8 else ''
_s8_out_csv = os.path.join(FIGURES_DIR, f'section8_two_regime_{_S8_VERSION}{_sfx8p}.csv')
_s8p = pd.read_csv(_s8_out_csv)

# 8c-fixed lives in its own CSV: its extra columns (oracle_spearman_infoloss /
# spearman_rel_ceiling) would inject NaN columns into the combined file, so
# the 8e cell deliberately excludes it. Pooled in here for the shared 3-panel;
# its distinctive columns drive the info-loss figure below.
_s8cf_csv = os.path.join(FIGURES_DIR, f'section8c_fixed_truth_{_S8_VERSION}{_sfx8p}.csv')
_HAS_CF   = os.path.exists(_s8cf_csv)
if _HAS_CF:
    _s8p = pd.concat([_s8p, pd.read_csv(_s8cf_csv)], ignore_index=True)
    print(f'8c-fixed pooled in from {os.path.basename(_s8cf_csv)}')
else:
    print(f'8c-fixed CSV not found ({os.path.basename(_s8cf_csv)}) -- its plots skipped')

# -- ONE res_mode PER FIGURE ---------------------------------------------------
# 8c-fixed runs TWO resolution conventions (see the
# CONSTANT-FOOTPRINT block in the 8a setup cell). They use DIFFERENT patch
# sizes at the same resolution, so a panel keyed on `level` alone would average
# a 32 px CNN with an 8 px one and call the result "300 m". The standard panels
# below therefore show the HEADLINE arm only; _s8p_all keeps both for the
# side-by-side comparison figure, which is where the sensitivity belongs.
_S8_HEADLINE_RES_MODE = 'constant_footprint'
_s8p_all = _s8p.copy()
if 'res_mode' in _s8p.columns:
    _keep8 = _s8p.res_mode.isna() | _s8p.res_mode.isin(
        [_S8_HEADLINE_RES_MODE, 'baseline_100m'])
    print(f'  res_mode present: plotting {_S8_HEADLINE_RES_MODE} as headline '
          f'({int(_keep8.sum())} of {len(_s8p)} rows); '
          f'{sorted(_s8p.res_mode.dropna().unique())} available in _s8p_all')
    _s8p = _s8p[_keep8].reset_index(drop=True)
else:
    print('  no res_mode column (pre-two-arm CSV) -- nothing to filter')

print(f'S8 results loaded: {len(_s8p)} rows')
print(f'  sub_exp={sorted(_s8p.sub_exp.unique())}  arm={sorted(_s8p.arm.unique())}')

# At _S8_N_INIT=1 no 'cnn_mean' row is emitted, so every CNN line would
# silently vanish. Fall back to the per-init rows -- identical at N_INIT=1.
_CN8 = 'cnn_mean' if (_s8p.model == 'cnn_mean').any() else 'cnn'

# Colour = MODEL (Okabe-Ito, colourblind-safe); arm is linestyle only.
_M8 = {'rf':  ('#0072B2', 'RF (center pixel)', 'RF'),
       _CN8:  ('#D55E00', 'CNN (full patch)',  'CNN')}
_ORD8 = ['rf', _CN8]
# ARM IS ENCODED TWICE, DELIBERATELY. Dotted vs dashed can be hard to tell
# apart at figure scale, and the faint per-species dots previously used the
# same colour AND the same marker for both arms, so interpolative and
# extrapolative runs were indistinguishable where they overlapped. Marker
# shape now carries the arm as well as line style.
# NEITHER ARM IS SOLID: a solid line reads as "valid at every x", but skill was
# only measured at the plotted levels. Both styles are broken to say so.
# ENCODING CONVENTION -- used in every Section-8 figure.
#   MODEL  = colour (Okabe-Ito, colourblind-safe) AND marker shape
#   ARM    = line style AND marker fill
# Both dimensions are encoded twice, so neither depends on colour vision alone.
# Marker shape carries the MODEL (not the arm, as before): with four models in
# the 8i figures, colour alone was the only separator and that fails for a
# red-green colourblind reader -- #009E73 and #D55E00 are the classic confusion
# pair. Arm moves to marker FILL, which is orthogonal to shape.
# NEITHER ARM IS SOLID: skill was measured only at the plotted levels, so a
# solid line would imply validity at every x.
_LS8  = {'random': ':',  'extrap': '--'}
_FILL8 = {'random': 'full', 'extrap': 'none'}
_MKM8 = {'rf': 'o', _CN8: '^'}          # model -> marker shape
_ARMS8 = ['random', 'extrap']

_S8EXP = {
    # Titles carry NO code key ('8a' etc.): these figures go into the thesis as
    # they are, and the keys are code identifiers, not the thesis's numbering.
    # 8b's x axis is LANDSCAPE OCCUPANCY -- the level 8b sweeps -- not the
    # presence fraction of the sample, which is held at 0.5 throughout.
    '8a':       ('N training points',   'log',    'Sample size'),
    '8b':       ('Landscape occupancy', 'linear', 'Ecological prevalence'),
    '8c-fixed': ('Resolution (m)',      'log',    'Resolution (fixed 100 m truth)'),
    '8e':       ('Patch size (px)',     'log',    'Patch size'),
}
# The resolution question is carried by 8c-fixed alone: the self-consistent
# variant (key 8c) was retired and removed, because with truth recomputed at each
# resolution its result is the expected one, while the info-loss ceiling of
# 8c-fixed makes the resolution contrast interpretable.
_SEXPS = [s for s in ['8a', '8b', '8c-fixed', '8e'] if s in _s8p.sub_exp.unique()]


# -- CONFOUND CHECK ------------------------------------------------------------
# Before reading any effect off a swept axis, confirm what must NOT vary with
# it doesn't. Unlike 8h, several of these axes are SUPPOSED to move n_tr (8a
# sweeps N directly) or prevalence (8b sweeps it) -- so this prints the range
# and it is on the reader to check that only the intended quantity moves.
print('\n-- confound check: what varies along each swept axis --')
_CONF_COLS = ['n_tr', 'n_te', 'n_vl', 'real_prev_tr', 'real_prev_te']
for _se_c in _SEXPS:
    _g_c = _s8p[(_s8p.sub_exp == _se_c) & (_s8p.model == 'rf')]
    for _arm_c in _ARMS8:
        _a_c = _g_c[_g_c.arm == _arm_c]
        if _a_c.empty: continue
        _mv = []
        for k in _CONF_COLS:
            if k not in _a_c.columns: continue
            lo, hi = float(_a_c[k].min()), float(_a_c[k].max())
            _mv.append(f'{k}={lo:g}' if abs(hi - lo) < 1e-9 else f'{k}={lo:g}..{hi:g} VARIES')
        print(f'  {_se_c:<9} {_arm_c:<7} ' + '  '.join(_mv))
print('  (8a is expected to vary n_tr/n_te/n_vl; 8b is expected to vary'
      ' prevalence.\n   Anything else varying is a confound, not a treatment.)')


# -- CNN FIT HEALTH ----------------------------------------------------------
# Same rule as 8h. Under the null the AUC of a balanced sample has
# SE = sqrt((n+1)/(12*n1*n0)) (Hanley & McNeil). A fit is degenerate when its
# best validation AUC is not _N_SE_ABOVE_CHANCE SE clear of 0.5 -- i.e. it
# never showed skill on the data it was SELECTED on, so its test score is not
# a measurement of the model.
# NOTE: 8b sweeps prevalence, so n1 != n0 there; the balanced form would be
# wrong. real_prev_vl is used to get the actual class counts per row.
# Degenerate fits are REPORTED, NEVER DROPPED -- removing them biases every
# summary upward exactly where the task is hardest.
_N_SE_ABOVE_CHANCE = 2.0
def _val_auc_floor8(n_vl, prev_vl):
    n  = np.asarray(n_vl, float)
    p  = np.clip(np.asarray(prev_vl, float), 1e-6, 1 - 1e-6)
    n1 = np.maximum(n * p, 1.0)
    n0 = np.maximum(n * (1 - p), 1.0)
    return 0.5 + _N_SE_ABOVE_CHANCE * np.sqrt((n + 1.0) / (12.0 * n1 * n0))

_S8_FIT_HEALTH, _DEGEN8 = None, set()
if 'val_auc_best' in _s8p.columns and _s8p.val_auc_best.notna().any():
    _cf8 = _s8p[(_s8p.model == 'cnn') & _s8p.val_auc_best.notna()].copy()
    _cf8['auc_floor']  = _val_auc_floor8(_cf8.n_vl, _cf8.real_prev_vl)
    _cf8['degenerate'] = _cf8.val_auc_best < _cf8.auc_floor
    _cf8['gap']        = _cf8.train_auc_at_best - _cf8.val_auc_best
    _S8_FIT_HEALTH = _cf8.groupby(['sub_exp', 'arm', 'level']).agg(
        n_fits=('degenerate', 'size'), frac_degen=('degenerate', 'mean'),
        val_auc_best=('val_auc_best', 'mean'), train_val_gap=('gap', 'mean'),
        best_epoch=('early_stop_ep', 'mean'), auc_floor=('auc_floor', 'mean'))
    print(f'\n-- CNN fit health (degenerate = val AUC not '
          f'{_N_SE_ABOVE_CHANCE:.0f} SE above chance) --')
    print(_S8_FIT_HEALTH.round(3).to_string())
    _bad8 = _S8_FIT_HEALTH[_S8_FIT_HEALTH.frac_degen > 0.5]
    _DEGEN8 = set(_bad8.index)
    if len(_bad8):
        print(f'\n  WARNING: {len(_bad8)} condition(s) where >50% of CNN fits never '
              f'learned.\n  Those points are drawn hollow with a red x. Read them as '
              f'"the CNN did\n  not fit here", NOT "the CNN scored low here" -- '
              f'different causes, and\n  only the second says anything about model '
              f'capability.')
else:
    print('\n  (no val_auc_best column -- this CSV predates the fit-health '
          'diagnostic;\n   re-run 8a-8e to get CNN degeneracy flagging)')


# REPLICATION UNIT. Section 8 now replicates over SPECIES with one rep each,
# so `rep` is 0 for every row and grouping on it would pool all species into
# one group -- which would inflate the CNN init-SD with between-species
# variance and destroy the pairing in every paired contrast. Older CSVs
# predate the column, so fall back to `rep` for those.
_UNIT = 'species_idx' if 'species_idx' in _s8p.columns else 'rep'
print(f'replication unit: {_UNIT}')

def _init_sd8(sub_exp, arm):
    """CNN init-SD per level: SD across inits within a rep, averaged over reps.
    This is the 'how noisy is a single CNN fit' floor, isolated from data-draw
    variability -- the band every CNN difference is judged against."""
    d = _s8p[(_s8p.sub_exp == sub_exp) & (_s8p.arm == arm) & (_s8p.model == 'cnn')]
    if d.empty:
        return pd.Series(dtype=float)
    return d.groupby(['level', _UNIT])['spearman_true'].std().groupby(level=0).mean()


def _noskill8(sub_exp, arm, model):
    """Levels where a model's TEST AUC is not 2 SE above chance.

    APPLIES TO BOTH MODELS, unlike the CNN degeneracy flag above, and means
    something different. The CNN flag is a TRAINING diagnostic: val AUC never
    cleared chance, so the fit never learned and its test score measures
    nothing. This one is an OUTCOME statement: whatever the model did, the
    result is indistinguishable from guessing on this test set.

    For the RF the flag is confirmatory rather than diagnostic -- the RF is
    deterministic and always fits, so a near-zero score is already unambiguous
    (there is no "failed to train" alternative to rule out). It is drawn anyway
    so that "no skill here" is marked by the same symbol for both models
    instead of being something the reader has to infer for one and read off a
    flag for the other.
    """
    d = _s8p[(_s8p.sub_exp == sub_exp) & (_s8p.arm == arm) & (_s8p.model == model)]
    if d.empty or 'roc_auc' not in d.columns:
        return []
    g = d.groupby('level').agg(auc=('roc_auc', 'mean'), n=('n_te', 'mean'),
                               p=('real_prev_te', 'mean'))
    floor = _val_auc_floor8(g.n, g.p)          # 0.5 + 2 SE, Hanley-McNeil
    return [float(lv) for lv, a, f in zip(g.index, g.auc, floor) if a < f]


def _paired8(sub_exp, arm, m_hi, m_lo):
    """Per-rep paired contrast on the SAME drawn data, then aggregated."""
    d = _s8p[(_s8p.sub_exp == sub_exp) & (_s8p.arm == arm)]
    hi = d[d.model == m_hi].groupby(['level', _UNIT])['spearman_true'].mean()
    lo = d[d.model == m_lo].groupby(['level', _UNIT])['spearman_true'].mean()
    return (hi - lo).dropna()


def _logticks8(ax, sub_exp, scale):
    """On a log axis matplotlib labels decades (10^1, 2x10^1, ...), which for
    levels like 8, 16, 32, 64 is unreadable. Label the ACTUAL swept levels."""
    if scale != 'log':
        return
    lv = sorted(_s8p.loc[_s8p.sub_exp == sub_exp, 'level'].dropna().unique())
    if not lv:
        return
    ax.set_xticks(lv)
    ax.set_xticklabels([f'{v:g}' for v in lv])
    ax.xaxis.set_minor_formatter(_mtk8.NullFormatter())


# -- B9: how many SPECIES actually contributed to each level -----------------
# Species is the replication unit for all of Section 8, so a level carried by
# one species carries no cross-species claim -- but it is drawn exactly like a
# level carried by three, which invites reading it as one. Deleting such levels
# would be worse (a gap is not self-explanatory either), so they are drawn AND
# marked.
# The known case in v4: 8c / fixed_patch_px, where min_sep = patch * sqrt(2) *
# resolution grows with the grain -- 45 km at 1000 m. The extrapolative band
# cannot hold a non-overlapping sample at that separation for ANY species, so
# the cause is geometric, not species-specific, and raising the species count
# would not rescue it. The constant-footprint arm holds n_tr at every level and
# has no such gap, which is why it is the headline.
_NSP_MIN = 3


def _thin_levels8(sub_exp, frame=None):
    """Levels where EITHER model is carried by fewer than _NSP_MIN species.

    A model with NO rows at all for a level counts as 0 species, not as absent:
    a level where the CNN vanished entirely is precisely the case that must not
    pass unremarked. The level list is therefore the UNION over models."""
    _f = _s8p if frame is None else frame
    out = {}
    for _a in _ARMS8:
        _sub = _f[(_f.sub_exp == sub_exp) & (_f.arm == _a)]
        if _sub.empty or 'species_idx' not in _sub.columns:
            continue
        _levels = sorted(_sub.level.unique(), key=float)
        for _m in _ORD8:
            _cnt = (_sub[(_sub.model == _m) & _sub.spearman_true.notna()]
                    .groupby('level')['species_idx'].nunique())
            for _lv in _levels:
                _n = int(_cnt.get(_lv, 0))
                if _n < _NSP_MIN:
                    out.setdefault((_a, float(_lv)), []).append((_M8[_m][2], _n))
    return out


def _guard8(sub_exp):
    """Fraction of (arm, level) reps where CNN ran vs where RF ran -- < 1.0
    means the both-classes val guard skipped CNN for some reps."""
    rf  = _s8p[(_s8p.sub_exp == sub_exp) & (_s8p.model == 'rf')].groupby(['arm', 'level']).size()
    cnn = _s8p[(_s8p.sub_exp == sub_exp) & (_s8p.model == _CN8)].groupby(['arm', 'level']).size()
    return (cnn / rf).fillna(0)


# -- Figs 1-5: per-sub-exp three-panel ---------------------------------------
for _sexp in _SEXPS:
    _xlabel, _xscale, _title = _S8EXP[_sexp]
    # TWO panels, not three. mae_true was dropped from the reporting: the CNN
    # trains under label_smoothing=0.05 and the RF does not, so the CNN's
    # probabilities are shrunk toward 0.5 by construction; and both models
    # estimate a posterior of a BALANCED case-control sample, which is not on
    # the scale of the max-normalised true suitability it was compared against.
    # The column is still computed and kept in the CSV -- it is just not a
    # model-comparable calibration measure, so it is not reported.
    _fig, _axes = _plt8.subplots(1, 2, figsize=(10.5, 4.4))
    _fig.suptitle(_title, fontsize=12, fontweight='bold')
    _gr8 = _guard8(_sexp)
    _thin8 = _thin_levels8(_sexp)
    if _thin8:
        print(f'  species coverage: {_sexp} has level(s) below {_NSP_MIN} species '
              f'-- NOT aggregatable across species, marked in the figure:')
        for (_a_t, _lv_t), _who in sorted(_thin8.items(), key=lambda kv: (kv[0][0], kv[0][1])):
            print(f'    {_a_t} level={_lv_t:g}: ' +
                  ', '.join(f'{_w}={_n}' for _w, _n in _who))

    for _ci, _col8 in enumerate(['spearman_true', 'roc_auc']):
        _ax8 = _axes[_ci]
        for _m in _ORD8:
            _clr, _lbl, _shrt = _M8[_m]
            for _arm in _ARMS8:
                _d = _s8p[(_s8p.sub_exp == _sexp) & (_s8p.model == _m) & (_s8p.arm == _arm)]
                if _d.empty: continue
                _pr = _d.groupby(['level', _UNIT])[_col8].mean()
                _mu = _pr.groupby(level='level').mean()
                _x8 = _mu.index.astype(float).values
                # raw per-rep values (not independent -- shown, not pooled)
                _ax8.plot(_pr.index.get_level_values('level').astype(float),
                          _pr.values, _MKM8[_m], color=_clr, ms=3.4,
                          alpha=0.30, ls='none', fillstyle=_FILL8[_arm], zorder=1)
                _ax8.plot(_x8, _mu.values, ls=_LS8[_arm], marker=_MKM8[_m],
                          color=_clr, ms=5, lw=1.7, fillstyle=_FILL8[_arm],
                          label=f'{_shrt} {_arm}', zorder=3)
                # The CNN init-SD band that used to sit here has been removed: it
                # appeared on one model in one panel only, so the three panels
                # looked like they followed different conventions. The noise band
                # still governs every verdict -- it is drawn in the paired-contrast
                # figure below, which is where a difference is actually judged.
                # flag levels where most CNN fits never learned
                if _m == _CN8 and _DEGEN8:
                    _dg = [a for a in _mu.index if (_sexp, _arm, float(a)) in _DEGEN8]
                    if _dg:
                        _dv = _mu.reindex(_dg)
                        _ax8.plot([float(a) for a in _dg], _dv.values, 'o', ms=9,
                                  mfc='white', mec=_clr, mew=1.8, zorder=4)
                        _ax8.plot([float(a) for a in _dg], _dv.values, 'x', ms=7,
                                  color='crimson', mew=2.0, zorder=5)
        # auc_oracle ceiling on the AUC panel.
        #
        # PLOTTED PER SPECIES, NOT ONLY AS A MEAN. Every species has its OWN
        # ceiling (measured range here: 0.79-1.00), so a per-species dot above
        # the MEAN ceiling line is usually still far below its own -- at 8a
        # N=500 species 2 scores 0.918 against a mean ceiling of 0.910 but an
        # own ceiling of 0.954. Drawing only the mean invites reading that as a
        # model beating the oracle. The faint squares are each species' own
        # ceiling, in the same style as the faint model dots beside them.
        #
        # And the ceiling binds IN EXPECTATION, not per sample: labels are a
        # Bernoulli draw, so every ranker's empirical AUC is a random variable
        # and a model can edge past it by chance. Measured over all 36 RF rows
        # in 8a that happens once, by 0.003, and no row exceeds its own ceiling
        # by even one Hanley-McNeil SE.
        if _col8 == 'roc_auc':
            # uniform no-skill ring for BOTH models (see _noskill8)
            for _mns in _ORD8:
                _clr_ns = _M8[_mns][0]
                for _arm_ns in _ARMS8:
                    _ns = _noskill8(_sexp, _arm_ns, _mns)
                    if not _ns: continue
                    _dns = (_s8p[(_s8p.sub_exp == _sexp) & (_s8p.arm == _arm_ns)
                                 & (_s8p.model == _mns)]
                            .groupby('level')['roc_auc'].mean().reindex(_ns))
                    _ax8.plot([float(v) for v in _ns], _dns.values, 'o', ms=10,
                              mfc='none', mec='0.35', mew=1.3, zorder=6,
                              label='_nolegend_')
            _od = _s8p[(_s8p.sub_exp == _sexp) & (_s8p.model == 'rf')]
            _op = _od.groupby(['level', _UNIT])['auc_oracle'].mean()
            _or8 = _op.groupby(level='level').mean()
            if not _or8.empty:
                _ax8.plot(_op.index.get_level_values('level').astype(float),
                          _op.values, 's', color='#444444', ms=3, alpha=0.28,
                          zorder=1)
                _ax8.plot(_or8.index.astype(float), _or8.values, 'k:', lw=1.3,
                          marker='s', ms=3.5,
                          label='oracle AUC (ceiling, mean)')
        # info-loss ceiling on the Spearman panel -- 8c-fixed only.
        # The AUC panel beside it already carries auc_oracle; this is its
        # counterpart for the primary metric. oracle_spearman_infoloss is what
        # the coarsening left recoverable BEFORE any model ran: Spearman(block
        # mean of the 100 m truth over the point's coarse cell, 100 m truth at
        # the point). A model curve falling parallel to it is losing skill to
        # the lost information, not to the model.
        #
        # DRAWN PER ARM, not pooled. The two arms draw different points, so
        # they have different ceilings (measured: 0.894 random vs 0.798 extrap
        # at 400 m), and the extrapolative arm is missing at 300/500/1000 m
        # because those cells were skipped. One pooled line would silently be
        # random-only at the coarse end and a blend at the fine end, which
        # would read as a ceiling drop that is really a change of composition.
        # Faint squares are the per-species values, in the same style as the
        # faint model dots -- every species has its own ceiling.
        if (_col8 == 'spearman_true' and _sexp == '8c-fixed'
                and 'oracle_spearman_infoloss' in _s8p.columns):
            for _arm_c in _ARMS8:
                _dc8 = _s8p[(_s8p.sub_exp == _sexp) & (_s8p.model == 'rf')
                            & (_s8p.arm == _arm_c)]
                if _dc8.empty: continue
                _pc8 = (_dc8.groupby(['level', _UNIT])['oracle_spearman_infoloss']
                        .mean().dropna())
                if _pc8.empty: continue
                _mc8 = _pc8.groupby(level='level').mean()
                _ax8.plot(_pc8.index.get_level_values('level').astype(float),
                          _pc8.values, 's', color='#444444', ms=3, alpha=0.28,
                          zorder=1)
                _ax8.plot(_mc8.index.astype(float), _mc8.values, ls=':',
                          color='k', lw=1.3, marker='s', ms=3.5,
                          fillstyle=_FILL8[_arm_c],
                          label=f'info-loss ceiling {_arm_c}')

        # grey band = level carried by fewer than _NSP_MIN species (see _thin_levels8).
        # Drawn behind everything and in neutral grey so it never competes with the
        # model/arm encoding, which is already carrying two dimensions.
        for (_a_t, _lv_t) in {k for k in _thin8}:
            _ax8.axvspan(_lv_t * 0.93, _lv_t * 1.07, color='0.85', alpha=0.55,
                         zorder=0, lw=0)
        _ax8.set_xlabel(_xlabel); _ax8.set_xscale(_xscale); _ax8.grid(alpha=0.25)
        _logticks8(_ax8, _sexp, _xscale)

    _axes[0].set_ylabel('Spearman $\\rho$ vs truth')
    _axes[1].set_ylabel('AUC-ROC')
    # handlelength: the default handle is barely longer than the marker, so
    # the marker sat on the dash gap and every style looked solid.
    _axes[0].legend(fontsize=7, ncol=2, handlelength=3.6)
    _axes[1].legend(fontsize=7, handlelength=3.6)

    # Guard-rate annotation -- levels where the both-classes val guard skipped
    # some CNN reps, so a dip (or a missing point) means "fewer reps
    # contributed", not "the CNN did worse". BOTH arms are checked: the
    # previous version looked only at extrap, which missed the real skip in
    # this run (8a random at N=100, where n_vl=12 gives 6 presences against a
    # _S8_MIN_VAL of 8).
    _yl = _axes[0].get_ylim()
    for _arm_g in _ARMS8:
        try:
            _g_a = _gr8.xs(_arm_g, level='arm')
        except Exception:
            continue
        for _lv8, _rate8 in _g_a.items():
            if _rate8 < 0.95:
                _axes[0].annotate(f'{_arm_g[:3]} {_rate8:.0%}',
                                  xy=(float(_lv8), _yl[0] + 0.02 * (_yl[1] - _yl[0])),
                                  fontsize=6, color='darkred', ha='center')
                print(f'  guard: {_sexp} {_arm_g} level={_lv8} -- only '
                      f'{_rate8:.0%} of CNN reps ran')

    _fig.text(0.5, -0.02, 'Faint dots = individual reps (not independent -- shown raw). '
              'Grey ring = AUC not 2 SE above chance (either model). '
              'Hollow + red x = CNN never learned (val AUC at chance). '
              f'Grey band = fewer than {_NSP_MIN} species at that level '
              '(no cross-species claim).',
              ha='center', fontsize=7.5, color='0.35')
    _plt8.tight_layout()
    _fp8 = os.path.join(FIGURES_DIR, f's8_{_sexp}_panels.png')
    _plt8.savefig(_fp8, dpi=150, bbox_inches='tight')
    _plt8.show(); _plt8.close()
    print(f'Saved: {_fp8}')


# -- Fig 6: CNN - RF paired contrast vs the noise band ------------------------
# The "is the difference real" figure, matching 8h's contrast panel. Contrasts
# are formed rep-by-rep on the SAME drawn data before averaging, so the data
# draw cancels. A curve inside the shaded band is not distinguishable from CNN
# restart noise and is reported as inconclusive.
_fig6, _ax6s = _plt8.subplots(1, len(_SEXPS), figsize=(4.0 * len(_SEXPS), 4.2),
                              squeeze=False)
_ax6s = _ax6s[0]
_fig6.suptitle('CNN $-$ RF (paired within rep), judged against the CNN init-SD band',
               fontsize=12, fontweight='bold')
for _ax6, _sexp in zip(_ax6s, _SEXPS):
    _xlabel, _xscale, _title = _S8EXP[_sexp]
    for _arm in _ARMS8:
        _sd = _init_sd8(_sexp, _arm)
        if not _sd.empty and _arm == 'random':
            _xb = _sd.index.astype(float).values
            _ax6.fill_between(_xb, -_sd.values, _sd.values, color='0.55',
                              alpha=0.20, lw=0, label='$\\pm$ CNN init-SD')
        _c = _paired8(_sexp, _arm, _CN8, 'rf')
        if _c.empty: continue
        _mu = _c.groupby(level='level').mean()
        _ax6.plot(_c.index.get_level_values('level').astype(float), _c.values,
                  '^', color='#D55E00', ms=3, alpha=0.28, zorder=1,
                  ls='none', fillstyle=_FILL8[_arm])
        _ax6.plot(_mu.index.astype(float), _mu.values, ls=_LS8[_arm], marker='^',
                  color='#D55E00', ms=5, lw=1.7, fillstyle=_FILL8[_arm],
                  label=f'CNN $-$ RF {_arm}', zorder=3)
    _ax6.axhline(0, color='k', lw=0.9, ls='--', alpha=0.7)
    _ax6.set_xlabel(_xlabel); _ax6.set_xscale(_xscale)
    _logticks8(_ax6, _sexp, _xscale)
    _ax6.set_title(_title, fontsize=10); _ax6.grid(alpha=0.25)
    if _ax6 is _ax6s[0]:
        _ax6.set_ylabel('$\\Delta$ Spearman $\\rho$ (CNN $-$ RF)')
    _ax6.legend(fontsize=7)
_plt8.tight_layout()
_fp6b = os.path.join(FIGURES_DIR, 's8_cnn_vs_rf.png')
_plt8.savefig(_fp6b, dpi=150, bbox_inches='tight'); _plt8.show(); _plt8.close()
print(f'Saved: {_fp6b}')


# -- Fig 7: 8c-fixed -- information loss vs model shortfall -------------------
# Truth, points and labels stay at 100 m; only the covariates coarsen. Two
# causes of a skill drop are separated:
#   oracle_spearman_infoloss = Spearman(block mean of the 100 m truth over the
#     point's coarse cell, 100 m truth at the point) -- the rank information the
#     coarsening itself destroyed before any model.
#   spearman_rel_ceiling  = spearman_true / that ceiling.
# A model tracking the ceiling down (flat spearman_rel_ceiling) is losing skill only
# because the information is gone; a declining spearman_rel_ceiling is a genuine
# model-side coarsening penalty.
if _HAS_CF and '8c-fixed' in _s8p.sub_exp.unique():
    _cf = _s8p[_s8p.sub_exp == '8c-fixed']
    _fig7, _ax7s = _plt8.subplots(1, 2, figsize=(10.5, 4.4))
    _fig7.suptitle('Coarsening against a fixed 100 m truth: '
                   'information loss vs. model shortfall',
                   fontsize=12, fontweight='bold')
    for _col7, _ax7, _ylab7 in [
            ('spearman_true',           _ax7s[0], 'Spearman $\\rho$ vs FIXED 100 m truth'),
            ('spearman_rel_ceiling', _ax7s[1], 'skill / achievable ceiling')]:
        if _col7 not in _cf.columns: continue
        for _m in _ORD8:
            _clr, _lbl, _shrt = _M8[_m]
            for _arm in _ARMS8:
                _d7 = (_cf[(_cf.model == _m) & (_cf.arm == _arm)]
                       .groupby('level')[_col7].mean().dropna())
                if _d7.empty: continue
                _ax7.plot(_d7.index.astype(float), _d7.values, ls=_LS8[_arm],
                          marker=_MKM8[_m], fillstyle=_FILL8[_arm],
                          color=_clr, mec=_clr, ms=6, lw=1.7,
                          label=f'{_shrt} {_arm}')
        _ax7.set_ylabel(_ylab7)
    _ceil7 = _cf[_cf.arm == 'random'].groupby('level')['oracle_spearman_infoloss'].mean().dropna()
    if not _ceil7.empty:
        _ax7s[0].plot(_ceil7.index.astype(float), _ceil7.values, 'k:', lw=1.4,
                      marker='s', ms=4, label='info-loss ceiling')
    _ax7s[0].set_title('Skill vs recoverable ceiling', fontsize=10)
    _ax7s[1].axhline(1.0, color='gray', lw=0.8, ls=':')
    # NOT "share captured": this is a ratio of correlations (and can go
    # negative). The variance-style share is spearman_rel_ceiling_sq.
    _ax7s[1].set_title('Skill relative to the achievable ceiling', fontsize=10)

    for _ax7 in _ax7s:
        _ax7.set_xlabel('Resolution (m)'); _ax7.set_xscale('log')
        _logticks8(_ax7, '8c-fixed', 'log')
        _ax7.grid(alpha=0.25); _ax7.legend(fontsize=7)
    _plt8.tight_layout()
    _fp7 = os.path.join(FIGURES_DIR, 's8_8c_fixed_infoloss.png')
    _plt8.savefig(_fp7, dpi=150, bbox_inches='tight'); _plt8.show(); _plt8.close()
    print(f'Saved: {_fp7}')


# -- Fig 8: eds_decorr drift (8a, 8c-fixed -- extrap arm) --------------------
# The confound check itself, plotted: does D(test,train)/D(val,train) stay near
# its reference value, or drift with the swept axis? Drift means part of the
# extrap-arm sensitivity is a distance-confound effect, not a data-condition one.
#
# THE REFERENCE IS DERIVED FROM THE DATA, AT THE DESIGN POINT. It used to be two
# hardcoded numbers (2.007 / 1.325) from an old N=1000 diagnostic, which stop
# being true the moment the covariates or the scaling change. It is now read off
# the runs themselves, at the condition every other experiment uses: N = the
# design training size (read from 8b, which holds N fixed), 100 m, patch 32.
# WHY NOT N=1000: eds_decorr grows with N, because its normaliser -- the kNN
# distance WITHIN the training set -- shrinks as the points get denser (measured
# on v5: 1.51 at N=100 -> 2.33 at N=5000). The resolution panel runs at the
# design N, so a reference at any other N would sit off its whole curve and show
# a resolution-driven drift that is really just the difference in N.
#
# THE RESOLUTION PANEL READS 8c-fixed. It is the only resolution experiment:
# the self-consistent variant (key 8c) was retired and removed, and 8c-fixed is
# the one the write-up uses.
# 8c-fixed rows are already pooled into _s8p above for the shared panels.
_n_design5 = (int(_s8p.loc[_s8p.sub_exp == '8b', 'n_tr'].median())
              if '8b' in _s8p.sub_exp.unique() else None)
_ref5 = None
if _n_design5 is not None:
    _r5 = _s8p[(_s8p.sub_exp == '8a') & (_s8p.arm == 'extrap') & (_s8p.model == 'rf')
               & (_s8p.level == _n_design5)][['decc_te_tr', 'decc_vl_tr']]
    if not _r5.empty:
        _ref5 = _r5.mean()
print('eds_decorr reference: ' + (f'N={_n_design5}, 100 m: D(te,tr)={_ref5.decc_te_tr:.3f}  '
      f'D(vl,tr)={_ref5.decc_vl_tr:.3f}' if _ref5 is not None else
      'not available (no 8a run at the design N) -- figure drawn without it'))
_fig5, _ax5s = _plt8.subplots(1, 2, figsize=(10, 4))
_fig5.suptitle('eds_decorr drift in the extrapolative arm (distance confound)',
               fontsize=11, fontweight='bold')
for _ax5, _sexp5 in zip(_ax5s, ['8a', '8c-fixed']):
    if _sexp5 not in _s8p.sub_exp.unique():
        _ax5.text(0.5, 0.5, '(no data)', ha='center', va='center', transform=_ax5.transAxes)
        continue
    _ext5 = _s8p[(_s8p.sub_exp == _sexp5) & (_s8p.arm == 'extrap') & (_s8p.model == 'rf')]
    _dg5 = _ext5.groupby('level')[['decc_te_tr', 'decc_vl_tr']].mean()
    _x5 = _dg5.index.astype(float).values
    _ax5.plot(_x5, _dg5.decc_te_tr.values, ls='--', marker='o', color='#0072B2',
              ms=5, label='D(test,train)')
    _ax5.plot(_x5, _dg5.decc_vl_tr.values, ls=':', marker='s', color='#D55E00',
              ms=5, label='D(val,train)')
    if _ref5 is not None:
        _ax5.axhline(_ref5.decc_te_tr, color='#0072B2', lw=0.8, ls=':', alpha=0.6,
                     label=f'D(test,train) at N={_n_design5}, 100 m')
        _ax5.axhline(_ref5.decc_vl_tr, color='#D55E00', lw=0.8, ls=':', alpha=0.6,
                     label=f'D(val,train) at N={_n_design5}, 100 m')
    _ax5.set_xlabel(_S8EXP[_sexp5][0])
    _ax5.set_xscale('log'); _ax5.set_ylabel('eds_decorr')
    _logticks8(_ax5, _sexp5, 'log')   # label the swept levels, not decades
    _ax5.set_title(_S8EXP[_sexp5][2], fontsize=10); _ax5.legend(fontsize=7); _ax5.grid(alpha=0.25)
_plt8.tight_layout()
_fp5 = os.path.join(FIGURES_DIR, 's8_eds_decorr_drift.png')
_plt8.savefig(_fp5, dpi=150, bbox_inches='tight'); _plt8.show(); _plt8.close()
print(f'Saved: {_fp5}')


# -- Fig 9: headline -- extrap minus random contrast -------------------------
# The single-number summary of the two-regime design: for each model and
# sub-experiment, extrap spearman_true minus random. Below zero means
# extrapolation costs skill; the RF-vs-CNN difference in how much it costs is
# what Section 8 is built to show.
_fig9, _ax9s = _plt8.subplots(1, len(_SEXPS), figsize=(4.0 * len(_SEXPS), 4.2),
                              squeeze=False)
_ax9s = _ax9s[0]
_fig9.suptitle('Extrapolation cost: $\\Delta$ spearman_true (extrap $-$ random)',
               fontsize=12, fontweight='bold')
for _ax9, _sexp9 in zip(_ax9s, _SEXPS):
    _xlabel9, _xscale9, _title9 = _S8EXP[_sexp9]
    for _m in _ORD8:
        _clr, _lbl, _shrt = _M8[_m]
        _dfm9 = _s8p[(_s8p.sub_exp == _sexp9) & (_s8p.model == _m)]
        _piv9 = _dfm9.groupby(['arm', 'level'])['spearman_true'].mean().unstack('arm')
        if 'extrap' not in _piv9 or 'random' not in _piv9: continue
        _con9 = (_piv9['extrap'] - _piv9['random']).dropna()
        if _con9.empty: continue
        _ax9.plot(_con9.index.astype(float), _con9.values, marker='o', color=_clr,
                  ms=5, lw=1.7, label=_shrt)
    _ax9.axhline(0, color='gray', lw=0.9, ls=':')
    _ax9.set_xlabel(_xlabel9); _ax9.set_xscale(_xscale9)
    _logticks8(_ax9, _sexp9, _xscale9)
    _ax9.set_title(_title9, fontsize=10); _ax9.grid(alpha=0.25)
    if _ax9 is _ax9s[0]:
        _ax9.set_ylabel('$\\Delta$ Spearman $\\rho$ (extrap $-$ random)')
    _ax9.legend(fontsize=8)
_plt8.tight_layout()
_fp9 = os.path.join(FIGURES_DIR, 's8_extrap_contrast.png')
_plt8.savefig(_fp9, dpi=150, bbox_inches='tight'); _plt8.show(); _plt8.close()
print(f'Saved: {_fp9}')


# -- Verdict table -----------------------------------------------------------
# Applies the noise-band rule explicitly instead of leaving it to the eye.
# INCONCLUSIVE is a valid verdict and is printed as one.
print('\n' + '=' * 96)
print('SECTION 8 VERDICTS -- CNN - RF paired contrast vs the CNN init-SD band')
print('=' * 96)
print(f"{'sub_exp':<10} {'arm':<7} {'level':>9}  {'CNN-RF':>8} {'initSD':>7} "
      f"{'%degen':>7}  verdict")
print('-' * 96)
for _sexp in _SEXPS:
    for _arm in _ARMS8:
        _c = _paired8(_sexp, _arm, _CN8, 'rf')
        if _c.empty: continue
        _sd = _init_sd8(_sexp, _arm)
        _mu = _c.groupby(level='level').mean()
        for _lv in _mu.index:
            _v = float(_mu.loc[_lv]); _b = float(_sd.get(_lv, np.nan))
            _fd = np.nan
            if _S8_FIT_HEALTH is not None and (_sexp, _arm, _lv) in _S8_FIT_HEALTH.index:
                _fd = float(_S8_FIT_HEALTH.loc[(_sexp, _arm, _lv), 'frac_degen'])
            if not np.isfinite(_b):
                _vd = 'no noise band (needs _S8_N_INIT > 1)'
            elif np.isfinite(_fd) and _fd > 0.5:
                _vd = f'UNRELIABLE -- {_fd:.0%} of CNN fits never learned'
            elif abs(_v) < _b:
                _vd = 'INCONCLUSIVE (inside init-SD band)'
            else:
                _vd = 'CNN ahead' if _v > 0 else 'RF ahead'
            _fds = f'{_fd:.0%}' if np.isfinite(_fd) else '  --'
            print(f'{_sexp:<10} {_arm:<7} {float(_lv):>9.4g}  {_v:>8.3f} {_b:>7.3f} '
                  f'{_fds:>7}  {_vd}')
print('=' * 96)
print('Contrasts are paired within rep (same drawn data) before averaging.')
print('No p-values: reps within a level share a species and are not independent.')
# -- ESTIMATOR ROBUSTNESS: mean-of-k vs best-of-k ---------------------------
# cnn_mean averages over ALL inits, failed restarts included. Keeping the
# restart with the best VALIDATION AUC is the estimator a practitioner would
# actually use. Selection touches validation only, never test.
if 'val_auc_best' in _s8p.columns and _s8p.val_auc_best.notna().any():
    _bk8 = (_s8p[_s8p.model == 'cnn'].sort_values('val_auc_best')
              .groupby(['sub_exp', 'arm', 'level', _UNIT], as_index=False).tail(1))
    print('\n' + '=' * 88)
    print('ESTIMATOR ROBUSTNESS -- CNN - RF averaged over levels, both estimators')
    print('=' * 88)
    print(f"{'sub_exp':<10} {'arm':<7} {'mean-of-k':>11} {'best-of-k':>11} "
          f"{'shift':>7}  same sign?")
    print('-' * 88)
    for _se_r in _SEXPS:
        for _arm_r in _ARMS8:
            _rf_r = (_s8p[(_s8p.sub_exp == _se_r) & (_s8p.arm == _arm_r) &
                          (_s8p.model == 'rf')].groupby('level')['spearman_true'].mean())
            _mn_r = (_s8p[(_s8p.sub_exp == _se_r) & (_s8p.arm == _arm_r) &
                          (_s8p.model == _CN8)].groupby('level')['spearman_true'].mean())
            _bk_r = (_bk8[(_bk8.sub_exp == _se_r) & (_bk8.arm == _arm_r)]
                     .groupby('level')['spearman_true'].mean())
            if _rf_r.empty or _mn_r.empty: continue
            _vm = float((_mn_r - _rf_r).mean()); _vb = float((_bk_r - _rf_r).mean())
            _ok = 'yes' if np.sign(_vm) == np.sign(_vb) else 'NO'
            print(f'{_se_r:<10} {_arm_r:<7} {_vm:>11.3f} {_vb:>11.3f} '
                  f'{_vb - _vm:>7.3f}  {_ok}')
    print('=' * 88)

# -- Fig 9: the two resolution conventions, side by side ---------------------
# WHY THIS FIGURE EXISTS. "Resolution" bundles GRAIN with EXTENT for a CNN, and
# the two conventions resolve that bundle in opposite directions:
#   fixed_patch_px      PATCH constant in pixels -> extent GROWS as the grid
#                       coarsens (32 px is 3.2 km at 100 m but 32 km at 1000 m),
#                       which FLATTERS the CNN at coarse resolution, and forces
#                       separation up until N falls or the level is skipped.
#   constant_footprint  PATCH*resolution constant -> extent and separation are
#                       constant and N=1000 holds everywhere, but PATCH in
#                       pixels shrinks, which HANDICAPS the CNN (GAP sees 64
#                       cells at 32 px, 16 at 16 px, 4 at 8 px) while leaving
#                       the RF centre-pixel arm untouched.
# Neither is neutral, so neither identifies the resolution effect alone. What
# this figure asks is whether the CONCLUSION survives the choice: if the CNN-RF
# gap moves the same way under both, the finding is robust to a convention the
# reader might have picked differently. If it flips, that IS the result, and it
# is a more interesting one than either arm on its own.
#
# Per-species lines are drawn thin behind the aggregate deliberately: species
# are the replication unit, and the analysis plan forbids an SE band across them.
if 'res_mode' in _s8p_all.columns and _s8p_all.res_mode.notna().any():
    _cmp9 = _s8p_all[_s8p_all.res_mode.isin(['constant_footprint', 'fixed_patch_px'])]
    _sexp9 = [s for s in ['8c-fixed'] if s in _cmp9.sub_exp.unique()]
    if _sexp9:
        _LS9 = {'constant_footprint': ':', 'fixed_patch_px': '--'}   # never solid
        _fig9, _ax9s = _plt8.subplots(len(_sexp9), 2,
                                      figsize=(11, 4.2 * len(_sexp9)),
                                      squeeze=False)
        # B9 for the SENSITIVITY arm. The headline (constant_footprint) holds
        # n_tr at every level; the fixed-pixel arm does not, and at its coarse end
        # the separation rule makes a non-overlapping sample geometrically
        # impossible -- so levels there can lose a model, or every species. That
        # is a property of the geometry, not of the species pool, and no larger
        # ensemble would recover it.
        for _rm_t in ('constant_footprint', 'fixed_patch_px'):
            _thin9 = _thin_levels8(_sexp9[0], _cmp9[_cmp9.res_mode == _rm_t])
            if _thin9:
                print(f'  species coverage [{_rm_t}]: level(s) below {_NSP_MIN} '
                      f'species -- no cross-species claim there:')
                for (_a_t, _lv_t), _who in sorted(_thin9.items(),
                                                  key=lambda kv: (kv[0][0], kv[0][1])):
                    print(f'    {_a_t} level={_lv_t:g}: ' +
                          ', '.join(f'{_w}={_n} species' for _w, _n in _who))

        for _r9, _se9 in enumerate(_sexp9):
            for _c9, _arm9 in enumerate(_ARMS8):
                _ax9 = _ax9s[_r9][_c9]
                _d9 = _cmp9[(_cmp9.sub_exp == _se9) & (_cmp9.arm == _arm9)]
                for _rm9 in ('constant_footprint', 'fixed_patch_px'):
                    for _m9 in _ORD8:
                        _dd = _d9[(_d9.res_mode == _rm9) & (_d9.model == _m9)]
                        if _dd.empty: continue
                        _clr9, _lbl9, _sh9 = _M8[_m9]
                        _per = _dd.groupby(['level', _UNIT])['spearman_true'].mean()
                        for _u9 in _per.index.get_level_values(_UNIT).unique():
                            _s9 = _per.xs(_u9, level=_UNIT).sort_index()
                            _ax9.plot(_s9.index.astype(float), _s9.values,
                                      ls=_LS9[_rm9], color=_clr9, lw=0.7,
                                      alpha=0.30, zorder=1)
                        _agg = _per.groupby(level='level').median().sort_index()
                        _ax9.plot(_agg.index.astype(float), _agg.values,
                                  ls=_LS9[_rm9], color=_clr9, marker='o', ms=5,
                                  lw=1.9, zorder=3,
                                  label=f'{_sh9} {_rm9.replace("_", " ")}')
                # n_tr annotation: the confound the constant-footprint arm
                # removes. Anchored in AXES fraction, not data coords -- the
                # earlier version read get_ylim() mid-build and the labels were
                # clipped by the later autoscale.
                _nt = (_d9[_d9.res_mode == 'fixed_patch_px']
                       .groupby('level')['n_tr'].mean().sort_index())
                for _lv9, _nv9 in _nt.items():
                    _ax9.annotate(f'{int(_nv9)}', xy=(float(_lv9), 0.015),
                                  xycoords=('data', 'axes fraction'),
                                  fontsize=6, color='#888888', ha='center',
                                  va='bottom', clip_on=False)
                # grey band = fewer than _NSP_MIN species at that level, in EITHER
                # convention (see _thin_levels8).
                for _rm_t in ('constant_footprint', 'fixed_patch_px'):
                    for (_a_t, _lv_t) in _thin_levels8(_se9, _cmp9[(_cmp9.res_mode == _rm_t)
                                                                  & (_cmp9.arm == _arm9)]):
                        _ax9.axvspan(_lv_t * 0.93, _lv_t * 1.07, color='0.85',
                                     alpha=0.55, zorder=0, lw=0)
                _ax9.set_xscale('log')
                # Tick EVERY level actually drawn. _logticks8 reads _s8p, which
                # is filtered to the headline arm, so it would leave the
                # levels that exist in only one convention unlabelled -- points
                # sitting at unnamed x positions.
                _lv9all = sorted(_d9.level.dropna().unique())
                _ax9.set_xticks(_lv9all)
                _ax9.set_xticklabels([f'{v:g}' for v in _lv9all], fontsize=8)
                _ax9.xaxis.set_minor_formatter(_mtk8.NullFormatter())
                _ax9.xaxis.set_minor_locator(_mtk8.NullLocator())
                _ax9.set_xlabel('Resolution (m)   ' + r'$\it{grey}$: $n_{tr}$ fixed-pixel')
                _ax9.set_ylabel('Spearman $\\rho$ vs true suitability')
                _ax9.set_title(f'{_S8EXP[_se9][2]} -- {_arm9} arm', fontsize=10)
                _ax9.grid(alpha=0.25)
                _ax9.legend(fontsize=6.5, loc='best', framealpha=0.85)
        _plt8.tight_layout()
        _fp9 = os.path.join(FIGURES_DIR, 's8_8c_res_mode_comparison.png')
        _plt8.savefig(_fp9, dpi=150, bbox_inches='tight'); _plt8.show(); _plt8.close()
        print(f'Saved: {_fp9}')

        # Does the CNN-RF gap agree in SIGN between the two conventions? That is
        # the whole question the two-arm design was added to answer.
        print('\n-- CNN-RF gap under both resolution conventions --')
        print(f'{"sub_exp":<10} {"arm":<7} {"res":>6} '
              f'{"const_foot":>11} {"fixed_px":>9}  agree')
        for _se9 in _sexp9:
            for _arm9 in _ARMS8:
                _d9 = _cmp9[(_cmp9.sub_exp == _se9) & (_cmp9.arm == _arm9)]
                for _lv9 in sorted(_d9.level.unique()):
                    _g = {}
                    for _rm9 in ('constant_footprint', 'fixed_patch_px'):
                        _x = _d9[(_d9.res_mode == _rm9) & (_d9.level == _lv9)]
                        _rf9 = _x[_x.model == 'rf']['spearman_true'].mean()
                        _cn9 = _x[_x.model == _CN8]['spearman_true'].mean()
                        _g[_rm9] = _cn9 - _rf9
                    _a, _b = _g['constant_footprint'], _g['fixed_patch_px']
                    if not (np.isfinite(_a) and np.isfinite(_b)):
                        _ok9 = 'n/a'
                    else:
                        _ok9 = 'yes' if np.sign(_a) == np.sign(_b) else 'NO'
                    print(f'{_se9:<10} {_arm9:<7} {_lv9:>6.0f} '
                          f'{_a:>11.3f} {_b:>9.3f}  {_ok9}')
        # DERIVED, NOT HARDCODED. The two ladders have changed before (the
        # constant-footprint one lost its bogus "320 m" level), and a literal list
        # here silently went out of date with it.
        _only = {}
        for _rm_o in ('constant_footprint', 'fixed_patch_px'):
            _only[_rm_o] = set(_cmp9.loc[_cmp9.res_mode == _rm_o, 'level'].astype(float))
        _cf_only = sorted(_only['constant_footprint'] - _only['fixed_patch_px'])
        _fp_only = sorted(_only['fixed_patch_px'] - _only['constant_footprint'])
        print('  (levels present in only one convention read NaN by construction: '
              + (f'{[f"{v:g}" for v in _cf_only]} constant-footprint only; ' if _cf_only else '')
              + (f'{[f"{v:g}" for v in _fp_only]} fixed-pixel only' if _fp_only else '')
              + ')')
else:
    print('\n(no res_mode column -- resolution-convention comparison skipped)')




### 8.6 — Configurational DGP: Shared Machinery (`8f`)
A second data-generating process next to the pointwise one: `S = S_point^(1-α) · S_config^α`. `S_config` is built from a binary refuge field `W` — a Gaussian random field (controlled case, `grf`) or real forest cover (`landuse`) — through local cover `P_w` and edge density `E` in a window of fixed radius `r_m`. `E` is orthogonalised against `P_w`, so the configurational signal cannot be recovered from composition. **α = 0 reproduces the pointwise surface bit-for-bit** (asserted in the self-test).

In [ ]:
# =============================================================================
# 8f -- Configurational DGP: shared machinery (Section 1 of 3)
# Adds a SECOND data-generating process alongside the existing pointwise one
# (_s8_suit_fn / _s8_suit_100m, untouched, unmodified, still the null/control
# arm at alpha=0). Nothing here is wired into model-fitting yet -- this cell
# only builds and self-tests the suitability construction; _s8_run_level_config
# (which will use these) comes in a later section.
#
# THREE ARMS this machinery is designed to support (wiring comes later):
#   pointwise: alpha=0, no W            -- the existing experiment, unchanged
#   grf:       alpha>0, W_SOURCE='grf'  -- controlled positive (orthogonal by construction)
#   landuse:   alpha>0, W_SOURCE='landuse' -- ecological validation (real land use)
#
# alpha=0 == null equivalence (load-bearing, not just documentation): since
# S = S_point**(1-alpha) * S_config**alpha, at alpha=0 this is
# S_point**1 * S_config**0. By IEEE 754, x**0 == 1 for ANY x including NaN
# and 0 -- so S_config's shape/NaN-pattern/scale is completely irrelevant at
# alpha=0, and S reduces to *exactly* S_point, bit-for-bit. No epsilon-guards
# needed; this is verified below, not just asserted.
#
# W's TWO ROLES use DIFFERENT coarsening disciplines (this is the key design
# decision from the plan review):
#   - Observed W channel (fed to the models as an extra input): concatenate
#     the fixed 100m binary W onto env_arr BEFORE resample_env -- it rides
#     the existing per-channel block-mean loop and emerges as a continuous
#     sub-pixel PROPORTION at coarser resolutions. No special code; that's
#     what a coarse sensor would see. (Wiring in a later section.)
#   - Truth W (the S_config ingredient, i.e. what P_w/E get computed from):
#       * held at 100 m (the fixed-truth resolution arm, via
#         _s8_config_truth_100m): the truth never leaves the 100 m grid; only
#         the covariates and the W channel the models see are coarsened.
#       * recomputed per level (_s8_run_level_config): block-mean
#         W down to the coarse grid, then RETHRESHOLD at the frac_woody
#         quantile to recover a binary field with the same composition ratio
#         but destroyed fine structure -- _s8_coarsen_W_rethreshold. This
#         rethreshold step IS the fine-mosaic-destruction mechanism, made
#         explicit rather than left implicit in a resample.
# =============================================================================
from scipy.ndimage import gaussian_filter as _s8_gf
from scipy.ndimage import uniform_filter, minimum_filter   # self-contained: don't assume the
                                            # exploratory landuse cell has run
from scipy.stats import spearmanr as _sp   # self-contained: the redundancy
                                          # guard below must not depend on
                                          # cell 27 having been run first.

# -- W generation (GRF source) -------------------------------------------------
def _s8_make_W(shape, corr_len_px, frac_woody, seed):
    """Binary refuge field via Gaussian random field: white noise, smoothed
    at corr_len_px (pixels), thresholded at the frac_woody quantile so the
    realized woody fraction matches frac_woody exactly (by construction, via
    the empirical quantile -- not merely in expectation).
    corr_len_px should be derived from a FIXED ecological scale in metres:
    corr_len_px = corr_len_m / resolution_m. Call this ONCE at 100m -- W is
    never regenerated at coarser resolutions (see the two coarsening helpers
    below for how the same 100m field propagates to each level)."""
    rng    = np.random.default_rng(seed)
    noise  = rng.standard_normal(shape).astype(np.float32)
    smooth = _s8_gf(noise, sigma=corr_len_px)
    thresh = np.percentile(smooth, (1 - frac_woody) * 100)  # value for 70/30 split
    return (smooth >= thresh).astype(np.float32)    # returns binary map

# (_s8_blockmean_to_coarse_grid lives in the 8c-fixed cell, which runs first.)

# -- Truth-W coarsening, discipline 2: SELF-CONSISTENT (8c-style) ------------
def _s8_coarsen_W_rethreshold(W_100m, resolution, frac_woody, base_res_m=100):
    """Block-means the FIXED 100m binary W onto the coarse grid (same
    crop/factor mechanics as resample_env/_s8_blockmean_to_coarse_grid --
    pixel-aligned with env_arr at this resolution), producing a continuous
    0-1 'coarse proportion of W' field, then RETHRESHOLDS at the frac_woody
    quantile to recover a binary field with (approximately) the same overall
    composition ratio as the original -- but with fine-scale arrangement
    destroyed by the intervening averaging. This is the mechanism that lets
    E (edge density) actually respond to coarsening while P_w (composition)
    stays comparatively stable -- exactly the effect the correlation-length
    sweep is meant to expose."""
    if resolution == base_res_m:
        return W_100m.copy()
    factor = int(round(resolution / base_res_m))
    H, W = W_100m.shape
    H_c, W_c = (H // factor) * factor, (W // factor) * factor
    H_r, W_r = H_c // factor, W_c // factor
    blk = W_100m[:H_c, :W_c].reshape(H_r, factor, W_r, factor)
    prop = np.nanmean(blk, axis=(1, 3))
    thresh = np.nanpercentile(prop, (1 - frac_woody) * 100)
    return (prop >= thresh).astype(np.float32)

# -- Composition (P_w) + edge density (E) from a binary W --------------------
def _s8_W_fields(W, valid_mask, resolution_m, r_m):
    """r_m has NO DEFAULT on purpose. It used to default to 300.0, which is
    the value the original (broken) configuration used -- a silent omission
    would have quietly reinstated a setting where configuration is not
    separable from composition. It must be passed, and the sweep cell derives
    it from PATCH so the window matches what the CNN integrates over.

    P_w (local proportion, composition) and E (local edge density,
    configuration) from a binary field W, via uniform_filter over a window
    sized to a FIXED ecological radius r_m (metres): win pixels shrinks as
    resolution coarsens but stays r_m on the ground -- this is what
    separates 'ecological scale' from 'observation scale', per the plan
    review (8c's existing pointwise arm conflates the two; this doesn't).
    NaN handling: W is zero-filled outside valid_mask BEFORE filtering (no
    NaN bleed into the neighbourhood mean, since uniform_filter has no
    NaN-aware mode) -- caller is responsible for re-masking outputs to NaN
    over ~valid_mask afterward; this function is a pure array transform."""
    Wf = np.where(valid_mask, W, 0.0).astype(np.float32)

    # b: per-pixel boundary fraction (4-connectivity) -- same construction
    # as the exploratory landuse/pc1/pc2 orthogonality-check cell.
    b = np.zeros_like(Wf)
    b[:, 1:]  += (Wf[:, 1:] != Wf[:, :-1]); b[:, :-1] += (Wf[:, 1:] != Wf[:, :-1])
    b[1:, :]  += (Wf[1:, :] != Wf[:-1, :]); b[:-1, :] += (Wf[1:, :] != Wf[:-1, :])
    b /= 4.0

    win = max(1, int(round((2 * r_m) / resolution_m)) | 1)
    P_w = uniform_filter(Wf, size=win, mode='nearest')
    E   = uniform_filter(b,  size=win, mode='nearest')
    return P_w, E, win

# -- Sanity check (task item 7): report E's distribution before picking mu_E/sig_E --
def _s8_W_diagnostics(W, valid_mask, resolution_m, r_m, label=''):
    """Prints realized woody fraction and the P_w/E distributions over VALID
    pixels -- run this before committing to mu_E/sig_E, since real land use
    often concentrates edges in thin ecotone bands with big internal blocks
    (E near-zero over most of the map); mu_E/sig_E must sit where the mass
    actually is, not at an a-priori guess."""
    P_w, E, win = _s8_W_fields(W, valid_mask, resolution_m, r_m)
    w_ok = W[valid_mask]; p_ok = P_w[valid_mask]; e_ok = E[valid_mask]
    print(f'[W diagnostics{" " + label if label else ""}] win={win}px  '
          f'realized woody frac={w_ok.mean():.3f}', flush=True)
    print(f'  P_w: mean={p_ok.mean():.3f}  pct[10,50,90]='
          f'{np.percentile(p_ok,[10,50,90]).round(3).tolist()}', flush=True)
    print(f'  E  : mean={e_ok.mean():.3f}  pct[10,50,90]='
          f'{np.percentile(e_ok,[10,50,90]).round(3).tolist()}', flush=True)
    if e_ok.mean() < 0.02:
        print('  WARNING: E is near-zero over most of the map -- set mu_E low '
              '(near the 70-90th percentile printed above), not a naive 0.5 guess.',
              flush=True)
    return P_w, E

# -- Configurational suitability term from P_w/E ------------------------------
def _s8_suit_config_from_fields(P_w, E, valid_mask, mu_P, sig_P, mu_E, sig_E, w_P):
    """Two Gaussian-hump terms (composition, edge density), combined
    MULTIPLICATIVELY -- E_term**(1-w_P) * P_w_term**w_P -- mirroring how
    S_point already multiplies its PC1/PC2 Gaussian terms (conjunctive:
    'edge-like AND, to a lesser degree, cover-like', not an arithmetic
    average of the two). Default w_P should be low (0.1-0.2) or 0, and the
    reason is measured on real land use AT THE RADIUS THE DGP ACTUALLY USES
    (r_m = 1600 m; the check used to be run at 300 m, which is not the shipped
    setting -- see the orthogonality-check cell, which now reports both):
        P_w vs PC1  rho = +0.553     <- NOT orthogonal to the niche axis
        P_w vs PC2  rho = +0.093
        E   vs PC1  rho = +0.140     <- near-orthogonal, which is the point
        E   vs PC2  rho = -0.112
    (v6 stack, forest by majority resampling; the orthogonalised driver E_perp
    is at rho = +0.015 / -0.109 and S_config at +0.044 / -0.074. At r_m = 300 m
    the P_w/PC1 figure is +0.518, so the entanglement gets
    WORSE at the larger, shipped window -- the direction that matters.)
    Weighting P_w up therefore re-entangles the configurational arm with the
    very niche axis it is meant to be orthogonal to, which is why w_P stays at
    0.1 and why E, not P_w, carries the configurational signal. At w_P=0,
    P_w_term**0 == 1 identically (including where P_w is NaN) -- pure-edge,
    by the same IEEE x**0=1 rule the alpha=0 null-equivalence relies on.
    Returns (H,W) float32, NaN outside valid_mask, NOT yet renormalized to
    max=1 (caller does that, matching _s8_suit_fn's own convention, so both
    surfaces are combined pre-normalization consistently)."""
    # calculates distance from the optimum, standardised by sigma, then applies 
    # the Gaussian exp(-0.5 * z^2) to get a [0,1] term, i.e. a suitability.
    E_term   = np.exp(-0.5 * ((E   - mu_E) / sig_E) ** 2).astype(np.float32)
    P_w_term = np.exp(-0.5 * ((P_w - mu_P) / sig_P) ** 2).astype(np.float32)
    S_config = (E_term ** (1 - w_P)) * (P_w_term ** w_P)    # weighting of the two terms
    S_config = np.where(valid_mask, S_config, np.nan).astype(np.float32)
    mx = np.nanmax(S_config)
    if mx > 0:
        S_config = S_config / mx
    return S_config

# -- Blend: S = S_point^(1-alpha) * S_config^alpha ----------------------------
def _s8_suit_blend(S_point, S_config, alpha):
    """alpha=0 -> exactly S_point (bit-for-bit, verified below); alpha=1 ->
    exactly S_config. Geometric mean in between. No renormalization here --
    both inputs are already max-normalized to 1 by their own constructors
    (_s8_suit_fn / _s8_suit_config_from_fields), and a product of two
    already-normalized [0,1] surfaces stays in [0,1] without needing a
    second rescale."""
    if alpha == 0:
        return S_point.copy()
    if alpha == 1:
        return S_config.copy()
    return (S_point ** (1 - alpha)) * (S_config ** alpha)

# =============================================================================
# Self-test (cheap, no model fitting) -- confirms the machinery above before
# anything gets wired into _s8_run_level_config in a later section.
# =============================================================================
# SCOPE OF THIS SELF-TEST: it exercises the BLEND and COARSENING invariants
# (alpha=0 / alpha=1 bit-for-bit, grid shapes, spatial coherence), all of
# which hold for any S_config regardless of how the driver was built. The
# driver used below is RAW E -- the pre-fix construction -- purely because
# _s8_orthogonalize_E is defined further down this cell and is not in scope
# yet. That is NOT the shipped DGP: the experiment drives S_config off the
# orthogonalised driver with mu_E=sig_E=1.0, and the second self-test at the
# end of this cell covers that path, including the same blend identities.
print('--- 8f self-test: configurational DGP machinery ---', flush=True)

_s8f_valid = ~np.isnan(_s8_suit_100m)
_s8f_corr_len_m  = 300.0     # matches the shipped setting (was 900.0, the
                             # pre-fix value; spans was 0.7 there, i.e. the
                             # window sat inside one habitat blob)
_s8f_frac_woody  = 0.30
_s8f_W100 = _s8_make_W(_s8_suit_100m.shape, _s8f_corr_len_m / 100.0, _s8f_frac_woody, seed=123) # binary map
print(f'W (100m): shape={_s8f_W100.shape}  woody frac={_s8f_W100[_s8f_valid].mean():.3f}',
      flush=True)

# calculate landscape properties (composition, edge density) at 100m, for the config DGP
_s8f_Pw, _s8f_E = _s8_W_diagnostics(_s8f_W100, _s8f_valid, resolution_m=100.0,
                                     r_m=1600.0, label='(grf, 100m)')

# Pick mu_E/sig_E from the diagnostic's own printed distribution (70th pct),
# not an a-priori guess -- per the sanity-check requirement.
_s8f_mu_E  = float(np.percentile(_s8f_E[_s8f_valid], 70))
_s8f_sig_E = float(_s8f_E[_s8f_valid].std() + 1e-6)
_s8f_mu_P, _s8f_sig_P, _s8f_w_P = 0.30, 0.20, 0.1

_s8f_S_config = _s8_suit_config_from_fields(
    _s8f_Pw, _s8f_E, _s8f_valid, _s8f_mu_P, _s8f_sig_P, _s8f_mu_E, _s8f_sig_E, _s8f_w_P)
print(f'S_config (alpha=1 map): [{np.nanmin(_s8f_S_config):.3f},'
      f'{np.nanmax(_s8f_S_config):.3f}]  mean={np.nanmean(_s8f_S_config):.4f}', flush=True)
# Flag speckle vs coherent bands (task item 7): a coherent config surface
# should be spatially autocorrelated at roughly corr_len_px -- cheap proxy:
# row-lag-1 correlation of S_config with itself shifted one row. Near 0
# would mean speckle; a coherent mosaic should be strongly positive.
# NOTE the mask: BOTH the row-i and row-i+1 slices must be masked by the
# INTERSECTION of their validities. Masking the shifted slice with row i
# alone (the earlier version) leaks a NaN in wherever row i is valid but
# row i+1 is not -- which happens all along the raster border -- and a
# single NaN makes np.corrcoef return NaN for the whole matrix, which is
# exactly what it printed before this fix.
#
# for each pair of adjacent rows, only consider pixels that are valid in BOTH rows
_s8f_lag_ok = _s8f_valid[:-1, :] & _s8f_valid[1:, :]
# take the upper pixel from every valid vertical pair, and the lower pixel from every 
# valid vertical pair (ravel turns them into 1D arrays), and compute the correlation.
# Ergo: When one pixel has a high suitability, does the pixel directly below it also 
# tend to have high suitability?
_s8f_real_lag1 = np.corrcoef(_s8f_S_config[:-1, :][_s8f_lag_ok].ravel(),
                              _s8f_S_config[1:,  :][_s8f_lag_ok].ravel())[0, 1]
print(f'S_config row-lag-1 autocorrelation (valid pixels): {_s8f_real_lag1:.3f} '
      f'(near 0 would indicate speckle, not coherent bands; n={int(_s8f_lag_ok.sum())} pairs)', flush=True)

# alpha=0 null-equivalence: verified, not just asserted in a comment.
_s8f_S_a0 = _s8_suit_blend(_s8_suit_100m, _s8f_S_config, alpha=0.0)
_s8f_max_abs_diff = np.nanmax(np.abs(_s8f_S_a0 - _s8_suit_100m))
print(f'alpha=0 bit-for-bit check: max|S(alpha=0) - S_point| = {_s8f_max_abs_diff:.2e} '
      f'(must be exactly 0.0)', flush=True)
assert _s8f_max_abs_diff == 0.0, 'alpha=0 does not reproduce S_point exactly'

_s8f_S_a1 = _s8_suit_blend(_s8_suit_100m, _s8f_S_config, alpha=1.0)
assert np.array_equal(_s8f_S_a1, _s8f_S_config, equal_nan=True), \
    'alpha=1 does not reproduce S_config exactly'
print('alpha=1 bit-for-bit check: PASS', flush=True)

_s8f_S_mid = _s8_suit_blend(_s8_suit_100m, _s8f_S_config, alpha=0.5)
print(f'S(alpha=0.5): [{np.nanmin(_s8f_S_mid):.3f},{np.nanmax(_s8f_S_mid):.3f}]  '
      f'mean={np.nanmean(_s8f_S_mid):.4f}', flush=True)

# Coarsening-discipline checks at one resolution (200m) -- both paths should
# run without error and produce the SAME-SHAPE grid as resample_env would.
_s8f_test_res = 200
_s8f_env200_shape = resample_env(env_scaled_100m, _s8f_test_res).shape[:2]

_s8f_W_fixed_blockmean = _s8_blockmean_to_coarse_grid(_s8f_S_config, _s8f_test_res)
print(f'_s8_blockmean_to_coarse_grid @ {_s8f_test_res}m: shape={_s8f_W_fixed_blockmean.shape}  '
      f'expected={_s8f_env200_shape}  match={_s8f_W_fixed_blockmean.shape == _s8f_env200_shape}',
      flush=True)
assert _s8f_W_fixed_blockmean.shape == _s8f_env200_shape

_s8f_W_rethresh = _s8_coarsen_W_rethreshold(_s8f_W100, _s8f_test_res, _s8f_frac_woody)
print(f'_s8_coarsen_W_rethreshold @ {_s8f_test_res}m: shape={_s8f_W_rethresh.shape}  '
      f'expected={_s8f_env200_shape}  match={_s8f_W_rethresh.shape == _s8f_env200_shape}  '
      f'realized woody frac={np.nanmean(_s8f_W_rethresh):.3f} (target {_s8f_frac_woody})',
      flush=True)
assert _s8f_W_rethresh.shape == _s8f_env200_shape

print('--- 8f self-test PASSED ---', flush=True)

# Ceiling on how much of the configurational driver may be reachable from
# composition. 0.15 is deliberately close to the measured floor (0.00-0.09
# across the feasible parameter grid) rather than a loose tolerance: the
# original construction sat at 0.975, so anything near even 0.3 means the
# arm is not testing configuration.
_S8_MAX_REDUNDANCY = 0.15

# -- Orthogonalising CONFIGURATION against COMPOSITION ------------------------
def _s8_orthogonalize_E(E, P_w, valid_mask, n_bins=50):
    """Conditional z-score of E given P_w:  (E - E[E|P_w]) / SD[E|P_w].

    WHY THIS EXISTS -- the configuration arm did not work without it.
    Edge density and local cover are not independent for a thresholded
    field: E peaks at intermediate P_w and falls away toward both extremes,
    so a smooth POINTWISE function of P_w alone reproduces most of E.
    Measured on this generator at the original settings, composition alone
    recovered E at rho=0.81 and the resulting S_config at rho=0.975 -- the
    "configurational" surface was ~97% reachable without ever reading an
    arrangement, so the arm could not test what it claimed to test.

    A plain correlation check does NOT catch this: rho(E, P_w) is only 0.44,
    because the dependence is non-monotone (a hump, not a slope). It has to
    be tested against the conditional MEAN -- that is _s8_redundancy below,
    and it is why that guard is now asserted before every config run.

    Removing the conditional mean alone is NOT sufficient (it left rho=0.54):
    E's conditional SPREAD also varies with P_w, and the Gaussian hump in
    _s8_suit_config_from_fields turns that spread back into a P_w signal.
    Both moments must go, hence a z-score rather than a residual.

    PRECONDITION -- this only works if the ecological window actually spans
    several correlation lengths. At spans = 2*r_m/corr_len < 1 the window
    sits inside a single habitat blob, there is no arrangement in it to
    measure, and orthogonalisation cannot manufacture what is not there
    (measured: residual redundancy 0.16 at spans=0.7, 0.000 at spans>=8).
    The sweep cell asserts spans >= 8.

    Returns float32, NaN outside valid_mask or in under-populated strata.
    Unit variance by construction -- so mu_E/sig_E are now on a KNOWN scale
    (1.0 = one SD of configuration) rather than an empirical percentile read
    off one arbitrary setting and reused where it does not apply."""
    #
    # remove the part of E that can be explained by P_w, including both 
    # its mean relationship and its changing variability. 
    #
    # Conditional z-score of E given P_w:  (E - E[E|P_w]) / SD[E|P_w] means:
    # For a given amount of woody cover P_w, ask what E normally looks like. 
    # Then express the actual E as a number of standard deviations above or 
    # below that expectation.
    E = np.asarray(E, np.float32)
    P = np.asarray(P_w, np.float32)
    ok = valid_mask & np.isfinite(E) & np.isfinite(P)
    out = np.full(E.shape, np.nan, np.float32)  # create NaN output map
    if not ok.any():    # is at least one True in ok?
        return out
    pv = P[ok]  # keep only valid P_w values, 1D array
    # create bin boundaries (in P values) from quantiles to divide P into approx. equal groups:
    qs = np.unique(np.quantile(pv, np.linspace(0.0, 1.0, n_bins + 1)))
    if qs.size < 3:                       # P_w degenerate -> nothing to remove
        return out
    qs[-1] = np.nextafter(qs[-1], np.inf)   # slightly incease last boundary to evade edge case
    nb  = qs.size - 1
    # assign every valid pixel to a bin:
    # assign within which boundaries a value lies, from 0, then clipped to number of bins
    idx = np.clip(np.digitize(pv, qs) - 1, 0, nb - 1)   # index = which bin a pixel belongs to
    ev  = E[ok].astype(np.float64)  # extract the valid E values
    cnt = np.bincount(idx, minlength=nb)    # count how many pixels are in each bin
    s1  = np.bincount(idx, weights=ev,      minlength=nb)   # calculate sum of E in each bin
    s2  = np.bincount(idx, weights=ev * ev, minlength=nb)   # calculate the sum of squared E in each bin
    # Calculate the mean of E within each P_w bin (the conditional mean SD[E|P_w]):
    with np.errstate(invalid='ignore', divide='ignore'):
        mu = s1 / cnt
        sd = np.sqrt(np.clip(s2 / cnt - mu ** 2, 0.0, None))
    # strata too small to estimate two moments, or with no spread, are dropped
    bad = (cnt < 50) | ~np.isfinite(mu) | ~np.isfinite(sd) | (sd < 1e-12)
    # orthogonalization, i.e. how many sd away from mean for that bin:
    z = np.full(ev.shape, np.nan)
    good = ~bad[idx]
    z[good] = (ev[good] - mu[idx][good]) / sd[idx][good]
    out[ok] = z.astype(np.float32)
    return out


def _s8_redundancy(y, x, valid_mask, n_bins=50, max_n=2_000_000, seed=0):
    """|Spearman| of y against the best POINTWISE function of x alone.

    Bin x into quantile strata, replace each y by its stratum mean, and
    correlate that back against y. The stratum mean IS the conditional
    expectation E[y|x], so this is an upper bound on what ANY model holding
    x -- with unlimited flexibility, monotone or not -- can reconstruct of y.

    Read it as: 0 = y is unreachable from x (what we want of configuration
    given composition); 1 = y is entirely a function of x (what the original
    construction accidentally had, at 0.975).

    Subsamples to max_n pixels: the ranking is stable well below a full
    55M-pixel raster and spearmanr sorts, so the full pass is needlessly
    slow for a diagnostic."""
    #
    # How much of y can be reconstructed if I am only allowed to look at x?
    # y: thing you want to reconstruct/explain
    # x: predictor you are allowed to use
    #
    ok = valid_mask & np.isfinite(y) & np.isfinite(x)
    n  = int(ok.sum())
    if n < 1000:
        return np.nan
    xv, yv = np.asarray(x)[ok], np.asarray(y)[ok]   # extract valid values
    if n > max_n:
        sel = np.random.default_rng(seed).choice(n, max_n, replace=False)
        xv, yv = xv[sel], yv[sel]
    qs = np.unique(np.quantile(xv, np.linspace(0.0, 1.0, n_bins + 1)))  # create quantile bins for x
    if qs.size < 3:
        return np.nan
    qs[-1] = np.nextafter(qs[-1], np.inf)
    nb  = qs.size - 1
    idx = np.clip(np.digitize(xv, qs) - 1, 0, nb - 1)
    cnt = np.bincount(idx, minlength=nb)
    s1  = np.bincount(idx, weights=yv.astype(np.float64), minlength=nb)
    with np.errstate(invalid='ignore', divide='ignore'):    # calculate the average y in each x bin
        f = s1 / cnt
    f[~np.isfinite(f)] = np.nanmean(yv) # handle invalid bins
    # give every pixel its bin's predicted y (f[idx]), then calculate spearman with actual y (yv)
    r = _sp(f[idx], yv)[0]
    return float(abs(r)) if np.isfinite(r) else np.nan  # high r: high reconstruction possible


def _s8_patch_summary_fields(W_channel, valid_mask, patch):
    """Mean and SD of the model's OWN W channel over a patch-sized window.

    THIS REPLACES AN ORACLE FEATURE. The patch-summary RF previously received
    Pw_2d -- literally one of the two arrays passed to
    _s8_suit_config_from_fields to BUILD the truth, at the truth's own
    ecological radius r_m, sampled at the same pixel. That is circular: the
    model was handed an argument of the function generating its target, and
    its reaching the oracle ceiling was close to arithmetic rather than a
    finding. It also silently assumed knowledge of r_m, which no real model
    has.

    What it gets instead is computed from exactly what the CNN receives --
    the W channel of env_arr8 -- summarised over a patch-sized box, with no
    reference to r_m and no reference to the truth. uniform_filter at
    (patch|1) evaluated at a pixel IS the mean over the patch centred there,
    so this is the patch summary, computed once per level rather than per
    point. (The window is forced odd to stay centred; for an even `patch`
    that is one pixel wider than the CNN's own patch, which changes nothing
    material and keeps the two exactly concentric.)

    The control is honest in both directions, measured at the shipped
    setting: it recovers composition at rho=1.000 and the orthogonalised
    configurational driver at rho=0.001 -- a real composition model, and
    provably capped below configuration, which is the ceiling the CNN has to
    beat for a configurational claim to hold.

    ON THE 1.000 -- that is not the old leak returning. Because r_m is set so
    2*r_m = PATCH*100 m (see the sweep cell), the patch window and the
    ecological window are the same size, so this NUMERICALLY equals P_w even
    though it is computed from the model's own input and refers only to
    `patch`, never to r_m or to the truth. The difference from the old
    feature is provenance, not value, and the effect is to make the control
    the STRONGEST possible composition model rather than a weakened one --
    which makes the CNN-vs-control contrast conservative. It costs nothing
    here precisely because the driver is orthogonal to composition: perfect
    composition knowledge still buys 0.001 of it.

    NOTE on the SD column: at 100 m the channel is binary, so
    Var(W)=P_w-P_w^2 makes SD a deterministic transform of the mean and it
    adds no information (kept anyway -- it pre-empts the "why no spread
    feature" objection, and the identity is what makes the cap provable).
    At coarser resolutions resample_env averages W, the channel is no longer
    binary, the identity no longer holds, and SD becomes genuinely
    informative. Both regimes are fine; the cap argument is only needed at
    100 m, where it holds exactly."""
    win = int(patch) | 1    # make win odd
    # prepare W raster for filtering operation:
    Wf  = np.where(valid_mask, np.nan_to_num(W_channel, nan=0.0), 0.0).astype(np.float32)
    m   = uniform_filter(Wf, size=win, mode='nearest')  # calculate moving average
    m2  = uniform_filter(Wf * Wf, size=win, mode='nearest') # calculate moving average of W**2
    # calculate SD from the variance; clip forces tiny negative values to zero:
    sd  = np.sqrt(np.clip(m2 - m * m, 0.0, None)).astype(np.float32)
    m   = np.where(valid_mask, m,  np.nan).astype(np.float32)
    sd  = np.where(valid_mask, sd, np.nan).astype(np.float32)
    return m, sd



# -- Standard composition-controlled landscape metrics (for comparison) -------
def _s8_landscape_metrics(W, win):
    """Moving-window Aggregation Index and CLUMPY for the woody class.

    WHY THESE ARE HERE. They are the obvious literature alternative to the
    conditional z-score used as the actual configurational driver: both are
    standard FRAGSTATS configuration metrics that are described as
    controlling for composition by construction, and "use the published
    metric" is a fair reviewer question. They are computed here so the answer
    is measured rather than asserted -- see _s8_driver_redundancy_table.

    AI (He et al. 2000): like-adjacencies as a fraction of the most that
    n cells of the class could possibly have, where the maximum is the
    single-class packing bound
        m = floor(sqrt(n)), r = n - m^2
        max = 2m(m-1)                if r == 0
              2m(m-1) + 2r - 1       if n <= m(m+1)
              2m(m-1) + 2r - 2       otherwise
    CLUMPY (McGarigal): (G - P) / P     if G < P and P < 0.5
                        (G - P) / (1-P) otherwise
    with G the like-adjacency proportion and P the class proportion.

    IMPLEMENTATION NOTE -- these are moving-window reimplementations, not
    FRAGSTATS itself. Each adjacency is attributed to the left/top cell of
    its pair and summed with uniform_filter, and window borders use 'nearest'
    like every other field here. Both are approximations at the window edge.
    They are adequate for the comparison being drawn (residual dependence of
    0.3-0.5 versus 0.002) but should not be quoted as FRAGSTATS values.

    Returns (AI, CLUMPY), float32, NaN where undefined (empty windows)."""
    W  = np.asarray(W, np.float32)
    n  = float(win * win)
    # find horizontally and vertically adjacent woody cells:
    h  = np.zeros_like(W); h[:, :-1] = W[:, :-1] * W[:, 1:]
    v  = np.zeros_like(W); v[:-1, :] = W[:-1, :] * W[1:, :]
    # count different-class boundaries (opposite of above):
    hu = np.zeros_like(W); hu[:, :-1] = (W[:, :-1] != W[:, 1:])
    vu = np.zeros_like(W); vu[:-1, :] = (W[:-1, :] != W[1:, :])
    # convert these into moving-window counts. h+v tells us where woody pixels 
    # touch other woody pixels; uniform_filter calculates the local mean;
    # *n converts the mean back into an approx count od adjacencies in the window
    g_ii = uniform_filter(h + v,   size=win, mode='nearest') * n
    g_ik = uniform_filter(hu + vu, size=win, mode='nearest') * n    # woody-non-woody boundaries
    P    = uniform_filter(W,       size=win, mode='nearest')    # local woody proportion
    n_i  = P * n                                                # number of woody cells
    # calculate the theoretical maximum aggregation, i.e. given a number of woody cells, 
    # what is the most possible adjacency between them if I arrange them as compactly as possible:
    m    = np.floor(np.sqrt(np.clip(n_i, 0.0, None)))   # edge length of biggest compact square
    r    = n_i - m * m                                  # rest of the cells
    base = 2.0 * m * (m - 1.0)      # neighbors in mxm square; 2*m+(m-1) is the exact number of
    # edges within a m*m grid. rest of cells are put along an edge of that square_
    mx   = np.where(r <= 0, base,
           np.where(n_i <= m * (m + 1.0), base + 2.0 * r - 1.0, base + 2.0 * r - 2.0))
    with np.errstate(invalid='ignore', divide='ignore'):
        AI  = np.where(mx > 0, g_ii / mx, np.nan)   # actual number of same pixel naeighbors / max possible number
        tot = g_ii + g_ik                           # sum of forest-forest and forest-open neigbors
        G   = np.where(tot > 0, g_ii / tot, np.nan) # fraction of forest-forest neigbors of tot
        # CLUMPY = the deviation (G - P), rescaled to [-1, +1].
        #
        # WHY TWO DENOMINATORS. At a fixed P the deviation is asymmetric: it can
        # only fall to -P (when G = 0, every woody cell isolated) but can rise to
        # +(1 - P) (when G = 1, one solid block). Dividing by the distance to the
        # bound ON THAT SIDE is what makes 0 -> -1 and 1 -> +1 exactly.
        #
        # WHY THE EXTRA `P < 0.5`. It is NOT redundant with `G < P`. The /P branch
        # is reserved for a class that is BOTH a minority AND more scattered than
        # its own share would imply; once the class is the majority the (1 - P)
        # denominator is used regardless of where G sits. Worked examples, all
        # computed with this exact code:
        #
        #   6x6 grid, 4 woody cells as one 2x2 block   P=0.111 G=0.333
        #       G < P is False -> second branch: (0.333-0.111)/0.889 = +0.250
        #   6x6 grid, SAME 4 cells but isolated in the corners   P=0.111 G=0.000
        #       G < P is True, P < 0.5 True -> first branch: (0-0.111)/0.111 = -1.000
        #       (identical composition, opposite arrangement, opposite sign)
        #   4x4 grid, woody ring around a 2x2 hole   P=0.750 G=0.600
        #       G < P is True BUT P < 0.5 is False -> second branch anyway:
        #       (0.600-0.750)/0.250 = -0.600
        #
        # CAVEAT ON THE ZERO POINT. The comparison is against P, but for a truly
        # random field the expected G is P/(2-P), not P: a random adjacency is
        # woody-woody with probability P^2 and a boundary with probability
        # 2P(1-P), so G = P^2 / (P^2 + 2P(1-P)) = P/(2-P). CLUMPY therefore does
        # NOT read 0 on a random map (measured here: -0.34 to -0.54 across
        # P = 0.1..0.7). That is a property of McGarigal's published formula, not
        # of this reimplementation. It does not affect the use made of CL here:
        # _s8_driver_redundancy_table only feeds it to _s8_redundancy, which asks
        # how much of CL a pointwise function of P_w can reconstruct on the REAL
        # W field -- the zero point never enters that comparison.
        CL  = np.where((G < P) & (P < 0.5),                     # minority AND more scattered than its share
                       (G - P) / np.maximum(P, 1e-9),           # -> normalise against the distance down to 0
                       (G - P) / np.maximum(1.0 - P, 1e-9))     # -> otherwise against the distance up to 1
        # 0 = matches the P baseline; -1 = maximally dispersed; +1 = maximally clumped
    return AI.astype(np.float32), CL.astype(np.float32)


def _s8_driver_redundancy_table(W, valid_mask, resolution_m, r_m, label=''):
    """THE table that justifies the configurational driver, measured.

    Reports, for one W field, how much of each candidate configuration
    variable a model holding only local composition could reconstruct
    (_s8_redundancy: correlation against the best pointwise function of P_w,
    so non-monotone dependence is caught -- a plain correlation is not
    enough here and is what missed the original confound).

    The result to carry into the thesis is that "composition-controlled by
    construction" does not survive measurement: AI removes roughly a quarter
    of the redundancy in raw edge density and CLUMPY a little over half,
    both far above the 0.15 usability threshold, while conditional
    standardisation reaches ~0.00. Reporting this is what turns the choice of
    a bespoke driver from an unexplained deviation into an evidenced one.

    Returns the dict so callers can log or tabulate it."""
    P_w, E, win = _s8_W_fields(W, valid_mask, resolution_m, r_m=r_m)
    AI, CL = _s8_landscape_metrics(np.where(valid_mask, W, 0.0).astype(np.float32), win)
    E_perp = _s8_orthogonalize_E(E, P_w, valid_mask)
    out = {}
    for name, arr in (('raw edge density E',        E),
                      ('Aggregation Index (AI)',    AI),
                      ('CLUMPY',                    CL),
                      ('conditional z-score E_perp', E_perp)):
        ok = valid_mask & np.isfinite(arr)
        out[name] = _s8_redundancy(arr, P_w, ok)
    print(f'  [driver redundancy vs composition {label}] window={win}px', flush=True)
    for k, val in out.items():
        flag = 'USABLE' if val <= _S8_MAX_REDUNDANCY else 'too redundant'
        print(f'    {k:<28} {val:6.3f}   {flag}', flush=True)
    return out


def _s8_config_ceiling(W_channel, S_config, valid_mask, patch,
                       n_bins=14, max_n=2_000_000, seed=0):
    """Best Spearman against S_config attainable by ANY model that can only
    integrate over its own patch.

    WHY IT IS NEEDED. spearman_true is not interpretable on its own in this
    arm: the configurational driver is deliberately subtle, so a model
    scoring 0.40 might be capturing 42% of what is there or 95% of it, and
    those are opposite conclusions. This supplies the denominator.

    HOW. The two things a patch-limited model can compute about W are its
    local cover and its local edge density, both averaged over the patch
    (GlobalAveragePooling2D is exactly a patch-wide mean, so this is the
    scale the CNN natively works at). Binning S_config on the joint strata
    of those two gives E[S_config | patch statistics] -- an upper bound on
    any function of them, monotone or not, with unlimited flexibility.

    WHERE IT IS EVALUATED. The conditional expectation is a property of the
    LANDSCAPE and is estimated on the whole valid raster, once per level. But
    the Spearman it is compared with (spearman_true) is computed on 300
    balanced case-control TEST points, and Spearman depends on the population
    it is computed over: on uniform pixels this predictor reaches 0.950 at the
    shipped setting, on 300-point case-control test sets only 0.925 +- 0.012
    (measured, both arms). Dividing a test-set Spearman by a landscape-wide one
    understated captured/achievable by ~3%. So this returns the predictor as a
    MAP, and the run-level function scores it at each row's own test points --
    numerator and denominator then come from the same points.

    EXACTLY WHAT IT BOUNDS. Recovery of S_config. At alpha == 1 that is the
    scoring target itself; for alpha < 1 the target also carries S_point, so
    this is a COMPONENT ceiling only and must never be divided into
    spearman_true there (the plotting cells form the ratio only at alpha == 1).

    Returns (ceiling_landscape, pred_2d):
      ceiling_landscape  the population value on uniform pixels (kept as a
                         diagnostic column, ceiling_config_patch_pop)
      pred_2d            E[S_config | patch strata] per pixel, NaN elsewhere
    or (nan, None) if the surface is degenerate."""
    win = int(patch) | 1
    Wf  = np.where(valid_mask, np.nan_to_num(W_channel, nan=0.0), 0.0).astype(np.float32)
    b = np.zeros_like(Wf)
    b[:, 1:]  += (Wf[:, 1:] != Wf[:, :-1]); b[:, :-1] += (Wf[:, 1:] != Wf[:, :-1])
    b[1:, :]  += (Wf[1:, :] != Wf[:-1, :]); b[:-1, :] += (Wf[1:, :] != Wf[:-1, :])
    b /= 4.0
    E_patch = uniform_filter(b,  size=win, mode='nearest')
    P_patch = uniform_filter(Wf, size=win, mode='nearest')

    ok = valid_mask & np.isfinite(S_config) & np.isfinite(E_patch) & np.isfinite(P_patch)
    n  = int(ok.sum())
    if n < 1000:
        return np.nan, None
    ys_all = np.asarray(S_config)[ok]
    xs_all = [E_patch[ok], P_patch[ok]]
    # Stratum EDGES from a subsample, as before; every valid pixel is then
    # assigned to its stratum so the predictor exists wherever a test point can
    # fall. The stratum MEANS now come from all valid pixels rather than the
    # subsample, which moves the landscape value by ~1e-4 (measured) -- noise.
    sel = (np.random.default_rng(seed).choice(n, max_n, replace=False)
           if n > max_n else np.arange(n))
    nb_all = 1
    codes = np.zeros(n, np.int64)
    for x in xs_all:
        q = np.unique(np.quantile(x[sel], np.linspace(0.0, 1.0, n_bins + 1)))
        if q.size < 3:
            return np.nan, None
        q[-1] = np.nextafter(q[-1], np.inf)
        nb = q.size - 1
        codes = codes * nb + np.clip(np.digitize(x, q) - 1, 0, nb - 1)
        nb_all *= nb
    # conditional mean of S_config per stratum, over ALL valid pixels
    cnt = np.bincount(codes, minlength=nb_all)
    mu  = np.bincount(codes, weights=ys_all.astype(np.float64),
                      minlength=nb_all) / np.maximum(cnt, 1)
    pred_all = mu[codes]
    r = _sp(pred_all[sel], ys_all[sel])[0]
    pred_2d = np.full(np.shape(S_config), np.nan, np.float32)
    pred_2d[ok] = pred_all.astype(np.float32)
    return (float(abs(r)) if np.isfinite(r) else np.nan), pred_2d

# -- Self-test: orthogonalisation, the guard, and the honest control ----------
# Small synthetic field (not the real raster) so this stays cheap enough to run
# on every cell execution. It asserts the three properties the configuration
# arm depends on -- if any regresses, 8h silently stops testing configuration,
# which is exactly what happened before these were in place.
_st_W = _s8_make_W((800, 800), 300.0 / 100.0, 0.30, seed=9000)
_st_v = np.ones((800, 800), bool)
_st_P, _st_E, _st_win = _s8_W_fields(_st_W, _st_v, 100.0, r_m=1600.0)
_st_in = np.zeros((800, 800), bool)
_st_in[_st_win:-_st_win, _st_win:-_st_win] = True          # drop the filter border
_st_Ep = _s8_orthogonalize_E(_st_E, _st_P, _st_in)
_st_ok = _st_in & np.isfinite(_st_Ep)

_st_raw  = _s8_redundancy(_st_E,  _st_P, _st_in)
_st_orth = _s8_redundancy(_st_Ep, _st_P, _st_ok)
_st_pm, _st_psd = _s8_patch_summary_fields(_st_W, _st_v, 32)
_st_comp = _s8_redundancy(_st_P,  _st_pm, _st_ok)          # control reaches composition
_st_cfg  = _s8_redundancy(_st_Ep, _st_pm, _st_ok)          # control cannot reach config

print('--- 8f self-test: configuration/composition separation ---')
print(f'  raw E recoverable from P_w      : {_st_raw:.3f}   (the confound, unfixed)')
print(f'  E_perp recoverable from P_w     : {_st_orth:.3f}   (limit {_S8_MAX_REDUNDANCY})')
print(f'  E_perp moments                  : mean={np.nanmean(_st_Ep[_st_ok]):+.4f} '
      f'sd={np.nanstd(_st_Ep[_st_ok]):.4f}   (hump uses mu_E/sig_E on this scale)')
print(f'  patch summary -> composition    : {_st_comp:.3f}   (want HIGH: real control)')
print(f'  patch summary -> configuration  : {_st_cfg:.3f}   (want ~0: provably capped)')

assert _st_raw > 0.5, (
    f'raw E is only {_st_raw:.3f} recoverable from P_w -- the known confound is '
    f'absent, so this self-test is no longer testing anything. Check the generator.')
assert _st_orth <= _S8_MAX_REDUNDANCY, (
    f'orthogonalisation left {_st_orth:.3f} recoverable from composition')
assert abs(float(np.nanmean(_st_Ep[_st_ok]))) < 0.05, 'E_perp mean drifted from 0'
assert abs(float(np.nanstd(_st_Ep[_st_ok])) - 1.0) < 0.05, 'E_perp sd drifted from 1'
assert _st_comp > 0.5,  f'patch summary reaches composition at only {_st_comp:.3f}'
assert _st_cfg  < 0.15, f'patch summary reaches configuration at {_st_cfg:.3f} -- the '\
                        f'composition control is not capped and the ladder is invalid'

# The comparison that justifies using a bespoke driver instead of a published
# one. Printed on every run so the number is reproducible and citable rather
# than a claim in prose. ~1s on this field.
_st_tab = _s8_driver_redundancy_table(_st_W, _st_in, 100.0, 1600.0,
                                      label='(800px synthetic GRF, corr_len 300m)')
# _st_tab creates does not output the different metrics, but how much they would reduce 
# redundancy with respect to composition. Therefore the comparison with _S8_MAX_REDUNDANCY
assert _st_tab['Aggregation Index (AI)'] > _S8_MAX_REDUNDANCY, (
    'AI now passes the redundancy limit -- if that is real, the bespoke driver '
    'is no longer needed and the published metric should be preferred. '
    'Re-check before changing anything.')
# The blend identities, re-checked on the SHIPPED construction (the earlier
# self-test checks them on the raw-E surface). alpha=0 must return S_point
# untouched even though E_perp introduces NaN that raw E did not have --
# _s8_suit_blend short-circuits on alpha==0, so NaN in S_config cannot leak
# into the null arm. That is the property the pointwise baseline rests on.
#
# create S_config using composition and orthogonalized configuration
_st_S  = _s8_suit_config_from_fields(_st_P, _st_Ep, _st_ok, 0.30, 0.20, 1.0, 1.0, 0.1)
# create a deliberately simple pointwise suitability:
_st_pt = np.where(_st_ok, np.linspace(0, 1, _st_S.size).reshape(_st_S.shape), np.nan
                  ).astype(np.float32)
_st_a0 = _s8_suit_blend(_st_pt, _st_S, alpha=0.0)
_st_a1 = _s8_suit_blend(_st_pt, _st_S, alpha=1.0)
assert np.array_equal(_st_a0, _st_pt, equal_nan=True), \
    'alpha=0 no longer reproduces S_point once the driver carries NaN'
assert np.array_equal(_st_a1, _st_S, equal_nan=True), \
    'alpha=1 no longer reproduces S_config'
print(f'  blend identities on the shipped driver: PASS '
      f'(NaN in S_config: {np.isnan(_st_S).mean():.3f} of pixels)')
print('  8f self-test PASS', flush=True)
del _st_W, _st_v, _st_P, _st_E, _st_Ep, _st_ok, _st_in, _st_pm, _st_psd


### 8.7 — Configurational DGP: Run-Level Function (`8g`)
`_s8_run_level_config`: the run-level function for the configurational DGP. The models see one extra input channel (`W`). Next to the centre-pixel RF and the CNN it fits a patch-summary RF (composition-only control) and an oracle-metric RF (information upper bound), and it computes the achievable ceiling for a patch-limited model. Only points whose whole patch lies inside the study area are sampled, and each species has its own configurational niche (`mu_E`, `sig_E`). The fixed-truth resolution arm of 8i runs through `_s8_run_level_fixedtruth` (8c-fixed cell).

In [ ]:
# =============================================================================
# _s8_run_level_config -- configurational-DGP variant of _s8_run_level
# Used by every configurational arm of 8i and 8j except the fixed-truth
# resolution arm. Structurally a sibling of _s8_run_level: the per-rep/per-arm
# loop body (both-classes guards, 1:1 class
# weights, N_CNN_INIT reps, eds_decorr confound tracking) is REUSED
# UNCHANGED from _s8_run_level, not reimplemented -- the differences are:
#   1. env_arr gains an extra channel (W) for the CNN and center-RF's inputs.
#   2. Sampling/scoring uses S = S_point^(1-alpha) * S_config^alpha (via 8f's
#      _s8_suit_blend) instead of _s8_suit_fn's output alone.
#   3. A THIRD model, 'rf_patch_summary', runs alongside center-RF ('rf')
#      and CNN every rep/arm -- same features as center-RF plus P_w/local-SD
#      of W at the point locations. All three always run together on the
#      same drawn points/split, so the center-RF < patch-summary-RF < CNN
#      ladder is a genuinely paired comparison, not three separate runs.
# mu_P/sig_P/mu_E/sig_E/w_P are threaded through as explicit parameters, not
# hardcoded here or in the 8f machinery -- retunable from wherever an arm is
# defined without touching this function or the 8f cell.
# =============================================================================
def _s8_config_truth_100m(env_base, W_100m, r_m, sig_P, mu_E, sig_E, w_P):
    """The configurational truth on the 100 m grid, built exactly as
    _s8_run_level_config builds it at 100 m: window of radius r_m, interior
    buffer, mu_P from the realised field, E orthogonalised against P_w, and the
    same redundancy guard. Used by the fixed-truth resolution arm (8i), which
    keeps this truth at 100 m and coarsens only the covariates."""
    _valid = np.all(np.isfinite(env_base), axis=2)
    Pw, E, win = _s8_W_fields(W_100m, _valid, resolution_m=100.0, r_m=r_m)
    inner = minimum_filter(_valid.view(np.uint8), size=win,
                           mode='constant', cval=0).astype(bool)
    mu_P = float(np.nanmean(Pw[inner]))
    Ep = _s8_orthogonalize_E(E, Pw, inner)
    ok = inner & np.isfinite(Ep)
    red = _s8_redundancy(Ep, Pw, ok)
    assert not np.isfinite(red) or red <= _S8_MAX_REDUNDANCY, (
        f'config driver is {red:.3f} recoverable from composition (limit '
        f'{_S8_MAX_REDUNDANCY}) at r_m={r_m:.0f}')
    S = _s8_suit_config_from_fields(Pw, Ep, ok, mu_P, sig_P, mu_E, sig_E, w_P)
    return dict(S_config_100=S, Pw_100=Pw, E_100=E, W_100m=W_100m, mu_P=mu_P,
                redundancy=red, frac_woody_realized=float(np.nanmean(W_100m[_valid])))


def _s8_run_level_config(env_base, resolution, n_points, pres_ratio, patch,
                          sub_exp, level, seed_base,
                          W_100m, w_source, corr_len_m, alpha, truth_discipline,
                          mu_P, sig_P, mu_E, sig_E, w_P,
                          r_m, frac_woody=0.30, n_reps=None, occ_prev=None,
                         oracle_val=False, patch_buffer_px=None):
    """Both arms x all reps, ONE sensitivity level, configurational DGP.
    r_m has NO DEFAULT: it used to default to 300.0, the value the original
    (broken) configuration used, so a silent omission would have quietly
    reinstated a setting where configuration is not separable from
    composition. The sweep cells derive it from PATCH and pass it explicitly.

    n_reps overrides _S8_N_REPS for this call only. Added for the species
    ensemble (8j), where SPECIES is the replication unit and one rep per
    species is the correct design -- reps within a species are not
    independent, species are. Defaults to None = use the global, so every
    existing caller is unaffected.

    Mirrors _s8_run_level's docstring/contract exactly (same return shape:
    list of row dicts) -- see that function for the shared design rationale
    (env_arr/masks computed once per level, reps vary only which points get
    drawn).

    W_100m: the FIXED 100m binary refuge field (from _s8_make_W for the grf
      source, or np.isin(landuse, WOODY_CLASSES) for the landuse source --
      built by the caller, once, outside any level/resolution loop).
    w_source: 'grf' or 'landuse' -- provenance label only (this function
      doesn't branch on it), recorded in every output row so grf/landuse
      rows are never ambiguous downstream.
    corr_len_m: the GRF correlation length in metres that produced W_100m
      (None for w_source='landuse', where there is no such parameter) --
      recorded, not used here; _s8_make_W is called by the caller, before
      this function, not inside it.
    alpha: config-fraction in [0,1]. alpha=0 -> S is bit-for-bit suit_2d
      (verified in the 8f self-test), i.e. this function reproduces
      _s8_run_level's result exactly at alpha=0 (modulo the harmless extra
      W input channel the models also see -- see NOTE below).
    truth_discipline: must be 'self_consistent' (recorded per row): the truth
        is built on this level's own grid. The fixed-truth resolution arm --
        truth pinned at 100 m, covariates coarsened -- runs through
        _s8_run_level_fixedtruth (8c-fixed cell) instead.
    r_m, frac_woody: ecological window radius (metres) and target woody
      fraction -- same meaning as in the 8f cell's self-test.

    NOTE on alpha=0: at alpha=0 the SAMPLING/SCORING surface S equals
    suit_2d exactly (8f-verified), so which points get drawn and how they
    score against truth is identical to _s8_run_level. The one thing that
    is NOT identical even at alpha=0 is that both models still receive the
    extra W input channel (env_arr8 instead of env_arr7) -- this is
    intentional: it isolates "does an uninformative-at-alpha=0 extra
    channel change anything" as its own falsifiable check, separate from
    "is the null arm's suitability surface identical" (which IS guaranteed).
    If you want a byte-for-byte rerun of _s8_run_level itself, call
    _s8_run_level directly -- that remains the actual pointwise arm.

    patch_buffer_px: the largest patch of a patch sweep, so every level samples
      the same area (see _s8_patch_buffer_px); None otherwise.
    """
    assert 0.0 <= alpha <= 1.0
    assert truth_discipline == 'self_consistent', (
        'the fixed-truth resolution arm runs through _s8_run_level_fixedtruth')

    # -- Suitability-relevant array: UNCHANGED from _s8_run_level ------------
    env_arr = resample_env(env_base, resolution)         # (H,W,C) -- identical call, identical result
    H, W_, C = env_arr.shape                             # C env channels; suitability/masks never see W
    suit_2d = _s8_suit_fn(env_arr)                        # UNCHANGED pointwise term

    # -- Model-input array: env + W, concatenated BEFORE resample_env --------
    # W is zero-filled at the 100m validity mask (never NaN by construction
    # after this), so concatenating it never shrinks the valid-pixel pool
    # relative to the env-only case -- region masks / point sampling
    # below still key off env_arr's own env-channel validity, not env_arr_ext's.
    _valid_100m   = np.all(np.isfinite(env_base), axis=2)
    W_100m_filled = np.where(_valid_100m, W_100m, 0.0).astype(np.float32)
    env_base_ext  = np.concatenate([env_base, W_100m_filled[..., None]], axis=2)  # (H100,W100,C+1)
    env_arr8      = resample_env(env_base_ext, resolution)   # coarsens env(C)+W(1) together
    # env_arr8[...,:7] == env_arr exactly: resample_env processes each
    # channel independently (per-channel loop, no cross-channel mixing), so
    # stacking W alongside the env channels cannot perturb their own
    # coarsened values -- this is a property of resample_env's own
    # implementation, not an assumption; see Section 2 verification.
    C8 = env_arr8.shape[2]                                # == C + 1

    # -- Truth W / S_config on THIS level's grid ('self_consistent') ----------
    # W is coarsened and rethresholded to the level's grid (a copy at 100 m), and
    # P_w, E, E_perp and S_config are computed there. Every arm that uses this
    # function runs at 100 m. The resolution arm, which keeps the truth at 100 m
    # while the covariates coarsen, runs through _s8_run_level_fixedtruth.
    # E_perp: CONFIGURATION with COMPOSITION divided out (see
    # _s8_orthogonalize_E), built on the same grid as E itself.
    # INTERIOR BUFFER (_s8_inner_mask). P_w and E are MOVING-WINDOW means, so a
    # pixel whose window is not entirely inside the valid area averages over
    # cells that do not exist. The raster is a bounding box around Germany and
    # 35.7% of it is no-data, so without this the entire national boundary gets a
    # fabricated non-woody exterior -- which manufactures a woody/non-woody EDGE
    # along it, in the very field E is built to measure. The standard treatment
    # for moving-window landscape metrics at a study-area boundary is to drop
    # those pixels rather than fill them; that is what this does. Measured cost at
    # r_m=1600 m (33x33 px window): 5.48% of the valid area, 19,510 of 355,750
    # km2, leaving 94.52%. The rim is NOT evenly distributed over the
    # extrapolative bands -- it is 7.18% of the train band, 4.76% of val and
    # 3.48% of test -- which is why the band masks have to be intersected with
    # isfinite(S) below rather than assumed to follow.
    # Applied in BOTH W sources, so the GRF and landuse arms share one analysis
    # mask and stay comparable.
    # At alpha=0 nothing changes: _s8_suit_blend returns S_point.copy() and never
    # touches S_config, so the pointwise null arm keeps the full mask and its
    # bit-for-bit equivalence with _s8_run_level is preserved.
    # mu_P FROM THE REALISED FIELD, not from a literal: the composition optimum
    # sits at this landscape's own mean cover (GRF 0.30 by construction, real land
    # use ~0.41), so both W sources describe the same niche relative to their own
    # landscape. A literal 0.30 put the landuse optimum 0.55 SD below its mean.
    W_level      = _s8_coarsen_W_rethreshold(W_100m, resolution, frac_woody)
    _valid_level = np.all(np.isfinite(env_arr), axis=2)
    Pw_2d, E_2d, _win_t = _s8_W_fields(W_level, _valid_level, resolution_m=resolution, r_m=r_m)
    _inner_level = minimum_filter(_valid_level.view(np.uint8), size=_win_t,
                                  mode='constant', cval=0).astype(bool)
    _mu_P_eff    = float(np.nanmean(Pw_2d[_inner_level]))
    Eperp_2d     = _s8_orthogonalize_E(E_2d, Pw_2d, _inner_level)
    _ok_level    = _inner_level & np.isfinite(Eperp_2d)
    _redund      = _s8_redundancy(Eperp_2d, Pw_2d, _ok_level)
    S_config_2d  = _s8_suit_config_from_fields(
        Pw_2d, Eperp_2d, _ok_level, _mu_P_eff, sig_P, mu_E, sig_E, w_P)
    # oracle-metric RF features: the truth's own ingredients, on the same grid
    Pw_oracle_2d, E_oracle_2d = Pw_2d, E_2d

    # GUARD -- the check that the original construction lacked. If the
    # configurational driver is still reachable from composition, the arm
    # cannot test configuration and the run is not worth doing; fail loudly
    # rather than produce an uninterpretable result. Redundancy is driven by
    # spans = 2*r_m/corr_len, so a breach here means the PARAMETERS are
    # wrong (window inside a single habitat blob), not the code.
    if alpha > 0 and np.isfinite(_redund):
        assert _redund <= _S8_MAX_REDUNDANCY, (
            f'config driver is {_redund:.3f} recoverable from composition '
            f'(limit {_S8_MAX_REDUNDANCY}) at r_m={r_m:.0f} -- raise r_m or '
            f'lower corr_len so the window spans >= 8 correlation lengths')

    S = _s8_suit_blend(suit_2d, S_config_2d, alpha)   # replaces suit_2d for sampling + scoring, below

    # -- Patch-summary features: computed from the MODEL'S OWN INPUT --------
    # Previously these were Pw_2d and sqrt(Pw_2d - Pw_2d^2) -- i.e. one of
    # the two arrays passed above to BUILD S_config, at the truth's own
    # radius r_m, read at the same pixel. That handed the RF an argument of
    # the function generating its target, so its reaching the oracle ceiling
    # was near-arithmetic rather than a finding, and it presumed knowledge of
    # r_m that no real model has. They now come from env_arr8's W channel --
    # exactly what the CNN receives -- summarised over a patch-sized box,
    # with no reference to r_m and none to the truth. See
    # _s8_patch_summary_fields.
    #
    # This also removes the old truth-discipline coupling: because the
    # features no longer read the truth, they need not inherit its
    # discipline -- they follow the model input in both branches by
    # construction, which is strictly what we want.
    Pw_patch_2d, sd_patch_2d = _s8_patch_summary_fields(
        env_arr8[..., -1], np.all(np.isfinite(env_arr), axis=2), patch)

    # Ceiling for this level: the most any patch-limited model could recover
    # of S_config. Supplies the denominator that makes spearman_true readable
    # -- 0.40 against a 0.95 ceiling and 0.40 against a 0.45 ceiling are
    # opposite findings. Computed once per level (~5s); it does not depend on
    # arm, rep or model. NaN in the pointwise arm, which has no S_config.
    # The PREDICTOR is built once per level (it depends on W, r_m, resolution
    # and the hump parameters, none of which vary by arm or rep); the reported
    # ceiling_config_patch is that predictor scored at each row's own test
    # points, further down -- see _s8_config_ceiling for why.
    if alpha > 0:
        _ceiling_cfg, _ceil_pred_2d = _s8_config_ceiling(
            env_arr8[..., -1], S_config_2d, np.all(np.isfinite(env_arr), axis=2), patch)
    else:
        _ceiling_cfg, _ceil_pred_2d = np.nan, None

    # -- Region masks: UNCHANGED, env channels only -> identical eligible-pixel
    # pool to _s8_run_level, so config-arm point LOCATIONS are drawn from
    # the same population as the pointwise arm; only the surface they're
    # drawn/scored against (S vs suit_2d) and what the models can see
    # (env_arr8 vs env_arr) differ.
    tr_m, val_m, te_m = _s8_region_masks(env_arr, patch)    # drawn from PC1 bands

    # THE BAND MASKS MUST ALSO REQUIRE A DEFINED TRUTH.
    # _s8_region_masks keys off the env-channel validity alone, which was IDENTICAL
    # to "S is finite" until the interior buffer arrived. It no longer is: S_config
    # is NaN in a window-wide rim, so 7.18% of the train band, 4.76% of val and
    # 3.48% of test are eligible by the mask while having no defined suitability.
    # Measured consequence of NOT intersecting: 112 of 2600 drawn points (4.31%)
    # were fitted with an undefined truth -- 95 in train, 10 in val, 7 in test,
    # and every one of them an ABSENCE, exactly as the mechanism predicts. _s8_sample_points substitutes 0.0
    # for a NaN suitability -- so such a pixel can never be drawn as a PRESENCE,
    # but it can very easily be drawn as an ABSENCE. It would then be TRAINED ON
    # with a truth value that does not exist, and simultaneously dropped from
    # spearman_true by _s8_mets's isfinite guard: the fitted set and the scored set
    # would quietly disagree. The random arm never had this, because
    # make_valid_mask already intersects with isfinite(S).
    #
    # At alpha=0 this is a no-op by construction -- S is suit_2d, and both it and
    # the band masks are finite exactly where all env channels are -- which is
    # asserted rather than assumed, so the null arm stays comparable to
    # _s8_run_level point-for-point.
    _S_ok = np.isfinite(S)
    if tr_m is not None:
        if alpha == 0:
            assert (np.array_equal(tr_m & _S_ok, tr_m)
                    and np.array_equal(val_m & _S_ok, val_m)
                    and np.array_equal(te_m & _S_ok, te_m)), (
                'alpha=0: isfinite(S) shrank a band mask, so the null arm would no '
                'longer draw from the same population as _s8_run_level')
        _n_before = (int(tr_m.sum()), int(val_m.sum()), int(te_m.sum()))
        tr_m, val_m, te_m = tr_m & _S_ok, val_m & _S_ok, te_m & _S_ok
        _n_after = (int(tr_m.sum()), int(val_m.sum()), int(te_m.sum()))
        if _n_after != _n_before:
            print(f'  bands intersected with isfinite(S): '
                  f'train {_n_before[0]:,}->{_n_after[0]:,} '
                  f'val {_n_before[1]:,}->{_n_after[1]:,} '
                  f'test {_n_before[2]:,}->{_n_after[2]:,} '
                  f'({100.0 * (1 - _n_after[0] / max(_n_before[0], 1)):.2f}% of the '
                  f'train band is interior-buffer rim)', flush=True)
    # EVERY PATCH INSIDE THE STUDY AREA -- the rule of _s8_run_level
    # (_s8_patch_ok), applied at every alpha, so alpha = 0 here still draws
    # exactly _s8_run_level's population. For alpha > 0 it also keeps the
    # model's W channel -- 0 outside Germany -- from showing an artificial forest
    # edge along the border: 7-10% of eligible points had such a patch wherever
    # the patch exceeded the truth window (measured). No padding value for W
    # would be neutral (any constant makes an edge, reflection makes mirrored
    # forest), so those points are excluded instead of their exterior invented.
    _pb = _s8_patch_buffer_px(resolution, patch, patch_buffer_px)
    _patch_ok = _s8_patch_ok(env_arr, _pb)
    if tr_m is not None:
        _nb = (int(tr_m.sum()), int(val_m.sum()), int(te_m.sum()))
        tr_m, val_m, te_m = tr_m & _patch_ok, val_m & _patch_ok, te_m & _patch_ok
        _na = (int(tr_m.sum()), int(val_m.sum()), int(te_m.sum()))
        if _na != _nb:
            print(f'  patch buffer {_pb} px: bands train {_nb[0]:,}->{_na[0]:,} '
                  f'val {_nb[1]:,}->{_na[1]:,} test {_nb[2]:,}->{_na[2]:,}', flush=True)
    rows_out = []

    # Landscape occupancy intercept, solved ONCE per level on the full valid
    # raster and reused for every arm. Solving it inside each mask instead
    # would force the target occupancy separately on the extrapolative
    # train/val/test bands -- inventing occupancy in a band the species
    # barely occupies, instead of reporting that it is genuinely low.
    _occ_valid = np.all(np.isfinite(env_arr), axis=2) & np.isfinite(S)
    _occ_c = _s8_solve_c(S[_occ_valid],
                         _S8_OCC_K,
                         _S8_OCC_PREV if occ_prev is None else occ_prev)

    for rep in range(_S8_N_REPS if n_reps is None else int(n_reps)):
        seed = seed_base + rep * 997
        for arm in ('random', 'extrap'):
            t0 = _t8.time()

            def _cln(rr, cc):
                E = env_arr[rr, cc, :].astype(np.float32)   # UNCHANGED: eds_decorr stays env-channel-only
                return E[~np.isnan(E).any(1)]

            # A degenerate (species, arm, level) cell raises _S8SampleSkip from the
            # sampler. Skip it and say so: crashing would lose a multi-hour run,
            # and recording a near-zero score would be read as model failure when
            # the truth is that there was nothing fittable there.
            try:
                if arm == 'random':
                    # MATCHED SET SIZES ACROSS THE TWO ARMS (this used to be the
                    # section's biggest confound). The random arm drew n_points and
                    # then CARVED train/val/test out of it, so it trained on 68% of
                    # n_points -- 1360 at N=2000 -- while the extrapolative arm
                    # trained on the full n_points and tested on 300. The arm
                    # contrast is the central comparison of this section, and it was
                    # confounded with a 47% difference in training-set size. Worse,
                    # asymmetrically between the models: read off 8a's own N curve,
                    # moving n_tr from 1360 to 2000 is worth +0.009 Spearman to the
                    # RF but +0.052 to the CNN, the same order as the CNN-RF gaps
                    # being reported.
                    # It now draws n_points + 2*_S8_N_WIN in ONE call, so the joint
                    # thinning still enforces separation across all three sets, then
                    # holds out exactly _S8_N_WIN for val and _S8_N_WIN for test.
                    # Result: n_tr == n_points and n_vl == n_te == _S8_N_WIN, the
                    # same three numbers the extrapolative arm produces --
                    # WHEREVER THE GEOMETRY ALLOWS IT. Measured across the ladders:
                    # the constant-footprint arm (100/200/400 m, the headline) hits
                    # 2000/300/300 in BOTH arms at every level, because separation is
                    # constant at 4525 m there by construction. The fixed-pixel
                    # SENSITIVITY arm cannot: separation grows with the grain
                    # (9.1 km at 200 m, 45.3 km at 1000 m), so graceful N reduction
                    # sets in from 200 m onward and the two arms drift apart again
                    # (extrap 1684 vs random 2000 at 200 m; extrap skipped entirely
                    # at 1000 m). That is a property of the ladder, not of this code,
                    # and it is the reason the fixed-pixel arm is a sensitivity
                    # rather than the headline. n_tr records it per row.
                    _n_draw = int(n_points) + 2 * _S8_N_WIN
                    vm = make_valid_mask(env_arr, S, patch)      # S replaces suit_2d
                    vm &= _patch_ok                              # whole patch inside the study area
                    if vm.sum() < max(_n_draw, 30):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'valid mask holds {int(vm.sum())} px, need {max(_n_draw, 30)}'); continue
                    r, c, lbl = _s8_sample_points(S, vm, _n_draw, pres_ratio, seed=seed,
                                                  min_sep_px=patch * _S8_SEP_PATCHES,
                                                  occ_prev=occ_prev, occ_c=_occ_c)
                    lbl = lbl.astype(np.int8)
                    if lbl.sum() < 3 or (1-lbl).sum() < 3:
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'draw has {int(lbl.sum())} pres / {int((1-lbl).sum())} abs, need 3 of each'); continue
                    # THE HOLD-OUT IS FIXED, THE TRAINING FOLD IS THE SWEPT
                    # QUANTITY. Holding out a FRACTION would make n_tr a function of
                    # _S8_N_WIN as well as of n_points, and 8a -- whose entire x
                    # axis is n_points -- would stop measuring what it plots (at
                    # n_points=100 a 40% hold-out leaves n_tr=420, not 100).
                    # So: when the full draw succeeded, hold out exactly
                    # 2*_S8_N_WIN and n_tr == n_points by construction. Only when
                    # graceful N reduction returned fewer points does the hold-out
                    # shrink with it, proportionally, rather than eating the
                    # training fold. n_tr/n_vl/n_te are recorded either way.
                    if len(lbl) >= _n_draw:
                        _hold = 2 * _S8_N_WIN
                    else:
                        _hold = int(round(len(lbl) * (2.0 * _S8_N_WIN) / _n_draw))
                    _hold = max(2, min(_hold, len(lbl) - 4))
                    tr_i, _rest_i = _tts(np.arange(len(lbl)), test_size=_hold,
                                         stratify=lbl, random_state=seed)
                    # Both-classes guard: at low prevalence a split can (rarely)
                    # leave the training fold with only one class, which would make
                    # RF/CNN fitting meaningless -- skip the rep.
                    if len(np.unique(lbl[tr_i])) < 2:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'training fold ended up single-class after the split'); continue
                    val_i, te_i = _tts(_rest_i, test_size=0.5,
                                       stratify=lbl[_rest_i], random_state=seed+1)
                    r_tr=r[tr_i];  c_tr=c[tr_i];  y_tr=lbl[tr_i]
                    r_v =r[val_i]; c_v =c[val_i]; y_v =lbl[val_i]
                    r_te=r[te_i];  c_te=c[te_i];  y_te=lbl[te_i]
                    d_te = _s8_eds_decorr(_cln(r_te, c_te), _cln(r_tr, c_tr))
                    d_vl = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_tr, c_tr))
                    d_vt = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_te, c_te))
                else:
                    if tr_m is None or tr_m.sum() < max(n_points, 10):
                        _s8_drop(sub_exp, level, arm, rep,
                                 f'train band holds {0 if tr_m is None else int(tr_m.sum())} px, '
                                 f'need {max(n_points, 10)}'); continue
                    if val_m is None or val_m.sum() < 10 or te_m is None or te_m.sum() < 10:
                        _s8_drop(sub_exp, level, arm, rep,
                                 'val or test band holds fewer than 10 valid px'); continue
                    r_tr, c_tr, lbl_tr = _s8_sample_points(S, tr_m, n_points, pres_ratio, seed=seed,
                                                           min_sep_px=patch * _S8_SEP_PATCHES,
                                                           occ_prev=occ_prev, occ_c=_occ_c)
                    y_tr = lbl_tr.astype(np.int8)
                    # CROSS-SET SEPARATION (see the seed_grid note in
                    # _s8_sample_points). train/val/test come from three separate
                    # calls on three disjoint PC1-band masks, so without this the
                    # min-separation rule held only WITHIN each set and 2.8% of test
                    # points / 17.9% of val points ended up with CNN patches
                    # overlapping a training patch. Each draw now blocks the ones
                    # after it; the seeded points are never returned or labelled, so
                    # the draws stay independent and only become spatially disjoint.
                    _used_rc = list(zip(r_tr.astype(float), c_tr.astype(float)))
                    n_v = min(_S8_N_WIN, int(val_m.sum()))
                    n_t = min(_S8_N_WIN, int(te_m.sum()))
                    if oracle_val:
                        # ORACLE MODEL SELECTION -- an UPPER BOUND, never the headline.
                        # In the primary design val sits BETWEEN train and test, so it is
                        # necessarily closer to train: measured D(val,train)=1.30 against
                        # D(test,train)=1.99. Early stopping therefore selects the epoch that
                        # is best under a shift of 1.30 and the model is then scored at 1.99 --
                        # while the RF, which does no model selection at all, carries no such
                        # penalty. That asymmetry sits in exactly the arm where the CNN does
                        # worst, so the primary design cannot separate "worse at transfer" from
                        # "selected for the wrong shift". This arm bounds that: it is the
                        # standard domain-adaptation contrast of source validation (realistic)
                        # against target/oracle validation (assumes labels in the target domain,
                        # which real transfer does not have) -- the exact analogue of the
                        # RF-oracle arm.
                        #
                        # ONE DRAW, THINNED JOINTLY, THEN SPLIT. A second independent draw from
                        # te_m could return points whose PATCHES overlap test patches, which
                        # would let early stopping peek at test. Drawing both together enforces
                        # one patch diagonal between every val and every test point.
                        _n_comb = n_v + n_t
                        _rc, _cc, _lc = _s8_sample_points(S, te_m, _n_comb, pres_ratio,
                                                          seed=seed + 200,
                                                          min_sep_px=patch * _S8_SEP_PATCHES,
                                                          occ_prev=occ_prev, occ_c=_occ_c,
                                                          seed_grid=_used_rc)
                        _lc = _lc.astype(np.int8)
                        if len(_lc) < 20 or len(np.unique(_lc)) < 2:
                            _s8_drop(sub_exp, level, arm, rep,
                                     f'oracle-val joint draw returned {len(_lc)} points'); continue
                        _frac_te = min(max(n_t / max(_n_comb, 1), 0.1), 0.9)
                        _iv, _it = _tts(np.arange(len(_lc)), test_size=_frac_te,
                                        stratify=_lc, random_state=seed + 300)
                        r_v,  c_v,  y_v  = _rc[_iv], _cc[_iv], _lc[_iv]
                        r_te, c_te, y_te = _rc[_it], _cc[_it], _lc[_it]
                    else:
                        r_v, c_v, lbl_v = _s8_sample_points(S, val_m, n_v, pres_ratio, seed=seed+100,
                                                            min_sep_px=patch * _S8_SEP_PATCHES,
                                                            occ_prev=occ_prev, occ_c=_occ_c,
                                                            seed_grid=_used_rc)
                        y_v = lbl_v.astype(np.int8)
                        _used_rc += list(zip(r_v.astype(float), c_v.astype(float)))
                        r_te, c_te, lbl_te = _s8_sample_points(S, te_m, n_t, pres_ratio, seed=seed+200,
                                                               min_sep_px=patch * _S8_SEP_PATCHES,
                                                               occ_prev=occ_prev, occ_c=_occ_c,
                                                               seed_grid=_used_rc)
                        y_te = lbl_te.astype(np.int8)
                    d_te = _s8_eds_decorr(_cln(r_te, c_te), _cln(r_tr, c_tr))
                    d_vl = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_tr, c_tr))
                    d_vt = _s8_eds_decorr(_cln(r_v,  c_v),  _cln(r_te, c_te))
            except _S8SampleSkip as _skip:
                print(f'  SKIP ({arm} {sub_exp} lv={level} rep={rep}): {_skip}',
                      flush=True)
                continue

            # INVARIANT: every point that gets FITTED must have a defined truth.
            # This is the check that would have caught the band-mask/NaN mismatch
            # described above, so it stays in permanently rather than being retired
            # once its cause was fixed. It is LOUD BUT NOT FATAL -- a hard assert
            # here would throw away a multi-hour run over a cell that can simply be
            # dropped, which is the same trade-off _S8SampleSkip already makes.
            _bad_truth = sum(int((~np.isfinite(S[_rr_s, _cc_s])).sum())
                             for _rr_s, _cc_s in ((r_tr, c_tr), (r_v, c_v), (r_te, c_te)))
            if _bad_truth:
                _s8_drop(sub_exp, level, arm, rep,
                         f'{_bad_truth} drawn point(s) have an undefined truth -- the '
                         f'sampling mask and the truth surface have diverged')
                continue
            s_te = S[r_te, c_te]   # was suit_2d[r_te, c_te] in _s8_run_level
            if y_te.sum() < 3 or (1-y_te).sum() < 3:
                _s8_drop(sub_exp, level, arm, rep,
                         f'test fold has {int(y_te.sum())} pres / {int((1-y_te).sum())} abs'); continue

            # Achievable ceiling AT THESE TEST POINTS: the patch-limited predictor
            # against S_config, on exactly the points spearman_true is computed on.
            # Signed, not abs(): on 300 points a weak predictor can come out
            # negative, and hiding that would inflate the ceiling; the ratio in the
            # plotting cells is NaN'd below 0.05 anyway.
            if _ceil_pred_2d is None:
                _ceiling_row = np.nan
            else:
                _cp = _ceil_pred_2d[r_te, c_te]; _cs = S_config_2d[r_te, c_te]
                _okc = np.isfinite(_cp) & np.isfinite(_cs)
                _ceiling_row = (float(_sp(_cp[_okc], _cs[_okc])[0])
                                if _okc.sum() >= 5 else np.nan)

            # NOTE: there is deliberately no Spearman-form oracle here.
            # See _s8_mets in the Section-8 setup cell for why (Spearman
            # against binary labels is capped by binarity, not by noise,
            # so it is not a ceiling on spearman_true).
            base = dict(sub_exp=sub_exp, level=level, resolution=resolution, rep=rep, arm=arm,
                        n_tr=int(len(y_tr)), n_te=int(len(y_te)), n_vl=int(len(y_v)),
                        # False everywhere except the oracle-model-selection bound;
                        # see the oracle_val branch in the extrap sampling above.
                        oracle_val=bool(oracle_val),
                        # patch and footprint are ALWAYS-POPULATED columns for the same reason
                        # resolution is: the constant-footprint arm varies patch WITH resolution, so
                        # neither one alone identifies a level.
                        patch=int(patch),
                        patch_buffer_px=int(_pb),   # window the whole patch must fit in
                        footprint_m=int(patch * resolution),
                        # The separation ACTUALLY enforced. With graceful N reduction a
                        # level can run at fewer points than requested, so n_tr alone is
                        # ambiguous -- these two columns say why it fell.
                        min_sep_px=float(patch * _S8_SEP_PATCHES),
                        min_sep_m=float(patch * _S8_SEP_PATCHES * resolution),
                        # B7: what the PC1 bands ACTUALLY are on this level's grid.
                        # Fixed in absolute PC units, so they drift away from
                        # 40/15/15 percent as the raster coarsens -- see the note at
                        # _S8_BAND_FRAC. Always populated so a resolution effect can
                        # be read against the band geometry that produced it.
                        **_S8_BAND_FRAC.get((H, W_, int(patch)), {}),   # W_ here: W is the refuge field
                        real_prev_tr=float(y_tr.mean()),
                        real_prev_te=float(y_te.mean()),
                        real_prev_vl=float(y_v.mean()),
                        decc_te_tr=d_te, decc_vl_tr=d_vl, decc_vl_te=d_vt,
                        alpha=alpha, truth_discipline=truth_discipline,
                        w_source=w_source, corr_len_m=corr_len_m,
                        # mu_P as ACTUALLY USED (derived from the realised field --
                        # see the truth_discipline branch), alongside the value the
                        # caller requested, so a row is never ambiguous about which
                        # niche it describes.
                        mu_P=_mu_P_eff, mu_P_requested=mu_P,
                        sig_P=sig_P, mu_E=mu_E, sig_E=sig_E, w_P=w_P,
                        r_m=r_m,
                        # frac_woody: the caller's TARGET (it drives the rethreshold
                        # in _s8_coarsen_W_rethreshold) and what the field actually
                        # has. For the landuse source these differ a lot -- 0.30
                        # requested against 0.409 realised in v5 -- and every row used to
                        # record only the first.
                        frac_woody=frac_woody,
                        frac_woody_realized=float(np.nanmean(W_100m[_valid_100m])),
                        ceiling_config_patch=_ceiling_row,
                        ceiling_config_patch_pop=_ceiling_cfg)

            # -- RF: identical fit/predict call to _s8_run_level; feature
            # matrix is (n, C+1) instead of (n, C) via env_arr8 -- W is just an
            # extra column, no special-casing needed (RF is feature-count
            # agnostic; the earlier StandardScaler-removal reasoning for RF
            # scale-invariance applies to this extra column too).
            E_tr  = env_arr8[r_tr,c_tr,:].astype(np.float32)
            ok_tr = ~np.isnan(E_tr).any(1)
            E_trc = E_tr[ok_tr]
            clf   = _s8_rf()
            clf.fit(E_trc, y_tr[ok_tr])
            E_te_ = env_arr8[r_te,c_te,:].astype(np.float32)
            ok_te = ~np.isnan(E_te_).any(1)
            p_rf  = np.full(len(r_te), np.nan)
            if ok_te.sum() >= 5:
                p_rf[ok_te] = clf.predict_proba(E_te_[ok_te])[:,1]
            rows_out.append({**base,'model':'rf','cnn_init':0,
                             'early_stop_ep':np.nan,'val_auc_best':np.nan,
                             'train_auc_at_best':np.nan,**_s8_mets(p_rf,y_te,s_te)})

            # -- RF (patch-summary): center-RF's own features (E_tr/E_te_,
            # unfiltered) plus P_w and local SD of W at the same points --
            # composition-aware, but (per the note above) provably blind to
            # arrangement. Fit independently of center-RF (its own NaN mask,
            # since the two extra columns can be invalid at slightly
            # different points than env_arr8's own NaN pattern) -- same
            # train/test SPLIT (same r_tr/c_tr/y_tr/r_te/c_te/y_te) as every
            # other model this rep/arm, just its own valid-row subset,
            # exactly like center-RF/CNN already have independent ok_*
            # masks from each other.
            _ps_tr_extra = np.stack([Pw_patch_2d[r_tr, c_tr], sd_patch_2d[r_tr, c_tr]], axis=1)
            _ps_te_extra = np.stack([Pw_patch_2d[r_te, c_te], sd_patch_2d[r_te, c_te]], axis=1)
            E_tr_ps = np.concatenate([E_tr, _ps_tr_extra], axis=1)   # (n, C+3)
            E_te_ps = np.concatenate([E_te_, _ps_te_extra], axis=1)
            ok_tr_ps = ~np.isnan(E_tr_ps).any(1)
            ok_te_ps = ~np.isnan(E_te_ps).any(1)
            p_rf_ps  = np.full(len(r_te), np.nan)
            if ok_tr_ps.sum() >= 10 and y_tr[ok_tr_ps].sum() >= 5 and ok_te_ps.sum() >= 5:
                clf_ps = _s8_rf()
                clf_ps.fit(E_tr_ps[ok_tr_ps], y_tr[ok_tr_ps])
                p_rf_ps[ok_te_ps] = clf_ps.predict_proba(E_te_ps[ok_te_ps])[:,1]
            rows_out.append({**base,'model':'rf_patch_summary','cnn_init':0,
                             'early_stop_ep':np.nan,'val_auc_best':np.nan,
                             'train_auc_at_best':np.nan,**_s8_mets(p_rf_ps,y_te,s_te)})

            # -- RF (oracle metrics): centre-pixel features PLUS the two
            # landscape metrics the truth is actually built from -- local
            # cover P_w and local edge density E, at the truth's own radius.
            #
            # WHAT THIS IS AND IS NOT. It is NOT a realistic competitor: it
            # presumes knowledge of r_m that no analyst has. It IS the
            # informational upper bound the CNN is measured against, made
            # concrete as a fitted model. The CNN can in principle compute
            # both fields from its own patch -- that is exactly what
            # ceiling_config_patch measures -- so this arm holds INFORMATION
            # constant and varies only whether the model is handed the
            # features or must learn them. The gap CNN-vs-oracle is therefore
            # the cost of having to discover what to compute, and the gap
            # oracle-vs-ceiling is what is available but not learnable from
            # this many training points.
            #
            # RAW E, NOT E_perp. Handing it the orthogonalised driver would be
            # circular -- that is the literal array passed to build the truth.
            # E and P_w are standard landscape metrics any ecologist could
            # compute, and since E_perp is a deterministic function of the
            # two, an RF can still in principle recover the truth from them.
            _or_tr = np.stack([Pw_oracle_2d[r_tr, c_tr], E_oracle_2d[r_tr, c_tr]], axis=1)
            _or_te = np.stack([Pw_oracle_2d[r_te, c_te], E_oracle_2d[r_te, c_te]], axis=1)
            E_tr_or = np.concatenate([E_tr,  _or_tr], axis=1)   # (n, C+3)
            E_te_or = np.concatenate([E_te_, _or_te], axis=1)
            ok_tr_or = ~np.isnan(E_tr_or).any(1)
            ok_te_or = ~np.isnan(E_te_or).any(1)
            p_rf_or  = np.full(len(r_te), np.nan)
            if ok_tr_or.sum() >= 10 and y_tr[ok_tr_or].sum() >= 5 and ok_te_or.sum() >= 5:
                clf_or = _s8_rf()
                clf_or.fit(E_tr_or[ok_tr_or], y_tr[ok_tr_or])
                p_rf_or[ok_te_or] = clf_or.predict_proba(E_te_or[ok_te_or])[:,1]
            rows_out.append({**base,'model':'rf_oracle_metrics','cnn_init':0,
                             'early_stop_ep':np.nan,'val_auc_best':np.nan,
                             'train_auc_at_best':np.nan,
                             **_s8_mets(p_rf_or,y_te,s_te)})

            n_vp=int(y_v.sum()); n_va=int((y_v==0).sum())
            if n_vp < _S8_MIN_VAL or n_va < _S8_MIN_VAL:
                # Routed through the same recorder as every other drop, so the CNN
                # gap shows up in _S8_DROPS rather than only in the console log.
                # NOTE: RF rows for this cell were already appended above -- this
                # drops the CNN only, which is why the reason says so.
                _s8_drop(sub_exp, level, arm, rep,
                         f'val fold has {n_vp} pres / {n_va} abs, below _S8_MIN_VAL='
                         f'{_S8_MIN_VAL} -- CNN skipped (RF row was written)')
                continue

            tr_ch_mean = E_trc.mean(0)
            def _pad(rr, cc):
                # extract_patches on env_arr8 (C+1 channels) instead of
                # env_arr (C) -- extract_patches derives C from
                # env_arr.shape[2] already, no change needed there.
                X = extract_patches(env_arr8, rr, cc, patch=patch)
                for ch in range(C8):
                    nm = np.isnan(X[...,ch])
                    if nm.any(): X[...,ch][nm] = tr_ch_mean[ch]
                return X

            X_tr  = _pad(r_tr, c_tr)
            X_val = _pad(r_v,  c_v)
            X_te  = _pad(r_te, c_te)
            # CNN input scaled on the TRAINING patches only (see _s8_train_minmax);
            # val/test values beyond the training range are kept, not clipped.
            X_tr, X_val, X_te = _s8_train_minmax(X_tr, X_val, X_te)
            # _pad has already replaced every NaN with the training-fold channel
            # mean, so a filter here can never fire. The invariant is asserted
            # instead of a guard that only looks like protection.
            assert not np.isnan(X_te).any(), '_pad left NaNs in the test patches'

            cnn_ps = []
            for ci in range(_S8_N_INIT):
                tf.keras.backend.clear_session()
                # tf.keras.utils.set_random_seed, NOT tf.random.set_seed: the latter
                # leaves Keras' own layer seeds unset, and two fits with the same seed
                # then differ by up to 0.067 in predicted probability (measured) --
                # as much as two different seeds. This one seeds Python, NumPy and TF
                # together and makes a fit bit-reproducible on CPU.
                tf.keras.utils.set_random_seed(seed + ci * 137)
                model = build_cnn(patch, C8, hp=BEST_HP)   # C8 == C + 1; build_cnn already channel-count-agnostic
                hist  = model.fit(
                    X_tr, y_tr,
                    validation_data=(X_val, y_v),
                    epochs=_S8_EPOCHS, batch_size=_S8_BATCH,
                    class_weight={0:1.0, 1:1.0},
                    callbacks=[
                        tf.keras.callbacks.EarlyStopping(
                            monitor='val_auc', patience=_S8_PAT,
                            restore_best_weights=True, mode='max'),
                        tf.keras.callbacks.ReduceLROnPlateau(
                            monitor='val_auc', factor=0.5, patience=5,
                            min_lr=1e-6, mode='max'),
                    ], verbose=0)
                # Fit diagnostics, read off the history that already exists -- zero extra
                # compute. These separate the two ways a CNN row can be uninformative:
                #   val_auc_best ~ 0.5            -> never learned anything, on the very
                #       data it was selected on. The row is a failed fit, not a weak model.
                #   val_auc_best high, test low   -> learned, then failed to generalise.
                #       That is a real result about extrapolation, not a broken run.
                # Thresholds are deliberately NOT applied here: raw numbers go in the
                # output so the cutoff can be stated and revised at analysis time
                # without re-running anything.
                _vh  = list(hist.history.get('val_auc', []))
                _th  = list(hist.history.get('auc', []))
                _bi  = int(np.argmax(_vh)) if _vh else 0
                n_ep = _bi + 1          # best epoch (restore_best_weights), not the stop epoch
                _val_best = float(_vh[_bi]) if _vh else np.nan
                _tr_best  = float(_th[_bi]) if _bi < len(_th) else np.nan
                p_c  = model.predict(X_te, verbose=0).ravel()
                cnn_ps.append(p_c)
                rows_out.append({**base,'model':'cnn','cnn_init':ci,
                                 'early_stop_ep':n_ep,
                                 'val_auc_best':_val_best,'train_auc_at_best':_tr_best,
                                 **_s8_mets(p_c,y_te,s_te)})
                tf.keras.backend.clear_session()

            if len(cnn_ps) > 1:
                mp = np.nanmean(np.stack(cnn_ps,0), axis=0)
                rows_out.append({**base,'model':'cnn_mean','cnn_init':-1,
                                 'early_stop_ep':np.nan,
                                 'val_auc_best':np.nan,'train_auc_at_best':np.nan,
                                 **_s8_mets(mp,y_te,s_te)})

            print(f'  [{arm}] {sub_exp} lv={level} rep={rep} alpha={alpha:.2f}: '
                  f'n_tr={len(y_tr)} n_te={len(y_te)} '
                  f'D(te,tr)={d_te:.2f} [{_t8.time()-t0:.0f}s]', flush=True)

    return rows_out


### 8.8 — Pointwise Axes under the Configurational DGP (`8i`)
Repeats the axes of 8.1–8.4 at α = 1, completing the factorial with their α = 0 counterparts. The patch axis holds `r_m` fixed, so the achievable ceiling peaks where the patch just covers the ecological window — a mechanistic test rather than a descriptive sweep.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
# =============================================================================
# 8i -- The 8a / 8e axes re-run under the CONFIGURATIONAL DGP (alpha = 1)
#
# WHY. 8a-8e sweep sample size and patch size in the POINTWISE regime, where
# the center-pixel RF holds exactly the right features and wins everywhere.
# 8h shows the CNN wins in the configurational regime, but only at one setting
# of N and PATCH. These two arms complete the factorial: the alpha=0 halves
# already exist (8a and 8e), so running the alpha=1 halves gives the
# interaction -- "does the sample-size requirement differ between regimes?" --
# without any new baseline.
#
# STANDALONE BY DESIGN. This cell does NOT require re-running the 8h sweep.
# It needs only the shared machinery (8f, cell 40) and _s8_run_level_config
# (8g), re-derives the handful of constants it shares with 8h, and CROSS-
# ASSERTS them against 8h's values when that cell has been run in the same
# session -- so it can never silently drift from the arm it is compared to.
# It writes its own CSV; 8h's output is untouched.
#
# THE PATCH ARM CARRIES A PRE-REGISTERED PREDICTION. r_m is held FIXED while
# PATCH varies, so the ratio window/patch moves and the achievable ceiling is
# a PEAKED function of patch size, maximised where the patch just covers the
# ecological window. Measured in advance on a synthetic field at these exact
# settings:
#     PATCH   window/patch   predicted ceiling
#         8           4.00               0.209
#        16           2.00               0.391
#        32           1.00               0.930
#        64           0.50               0.383
# Small patches cannot see the neighbourhood the truth depends on; large ones
# dilute it, because GlobalAveragePooling2D averages over the whole patch and
# convolutions are translation-equivariant, so the network cannot down-weight
# the periphery. A CNN skill curve that peaks at the predicted patch size is
# mechanistic evidence that the model is integrating the ecological
# neighbourhood -- much harder for a confound to mimic than a monotone trend.
# ceiling_config_patch is recomputed per level on the REAL raster, so the
# prediction lands in the output as data rather than as a claim in a comment.
#
# HOLDING r_m FIXED IS THE POINT. Scaling r_m with PATCH would preserve the
# alignment and flatten the curve -- that is a different experiment (pure
# receptive-field capacity). Fixed r_m is also the ecologically honest one: an
# animal's neighbourhood does not change because the modeller changed a patch
# size.
# =============================================================================
import os as _os_8i

# -- Constants shared with 8h -------------------------------------------------
# Constants come from _S8_CFG (cell 27) so 8i and 8j cannot drift apart.
_S8I_SEED        = 9000
_S8I_R_M = _S8_CFG['r_m']
_S8I_CORR_LEN_M = _S8_CFG['corr_len_m']
_S8I_FRAC_WOODY = _S8_CFG['frac_woody']
_S8I_MU_P, _S8I_SIG_P = _S8_CFG['mu_P'], _S8_CFG['sig_P']
# mu_E / sig_E: per species, read from _s8_meta at each call (_s8_config_niche).
_S8I_W_P = _S8_CFG['w_P']
_S8I_ALPHA       = 1.0               # fully configurational; alpha=0 half is 8a/8e
_S8I_RESOLUTION = _S8_CFG['resolution']

# -- W field ------------------------------------------------------------------
# Built unconditionally. 8h is retired, so there is no earlier W to reuse, and
# _s8_make_W is deterministic in (shape, corr_len_px, frac, seed) -- 8i and 8j
# therefore share a bit-identical field without either depending on the other.
_W_8i = _s8_make_W(_s8_suit_100m.shape, _S8I_CORR_LEN_M / 100.0,
                   _S8I_FRAC_WOODY, seed=_S8I_SEED)
print(f'8i: built the GRF W field (corr_len={_S8I_CORR_LEN_M:.0f} m, '
      f'seed={_S8I_SEED})', flush=True)

# -- Sweep grids: the SAME level sets as 8a / 8e ------------------------------
# Identical to 8a's N_POINTS_RANGE and 8e's PATCH_RANGE so each level pairs
# one-to-one with its existing alpha=0 counterpart.
N_POINTS_RANGE_CFG = ([100, 1000] if SMOKE_TEST_S8 else [100, 250, 500, 1000, 2000, 5000])
# Same level sets as 8b and 8c-fixed, so each level pairs one-to-one with its
# alpha=0 counterpart.
OCC_PREV_RANGE_CFG     = ([0.05, 0.2] if SMOKE_TEST_S8
                          else [0.05, 0.1, 0.2, 0.3, 0.5, 0.7])
# Resolution stops where the habitat field itself would stop existing:
# corr_len is 300 m, so a 500 m grid puts it below one pixel and the arm
# would measure the destruction of W rather than anything about the models.
RESOLUTION_RANGE_CFG_A1 = ([100, 200] if SMOKE_TEST_S8 else [100, 200, 300])
PATCH_RANGE_CFG    = ([8, 32]     if SMOKE_TEST_S8 else [8, 16, 32, 64])

# -- Preconditions ------------------------------------------------------------
# spans = 2*r_m/corr_len must stay >= 8 (it does not depend on patch or N, so
# one check covers every level here).
_spans_8i = 2.0 * _S8I_R_M / _S8I_CORR_LEN_M
print(f'8i: spans = {_spans_8i:.1f}  (window {2*_S8I_R_M:.0f} m / corr_len '
      f'{_S8I_CORR_LEN_M:.0f} m)', flush=True)
assert _spans_8i >= 8.0, (
    f'spans {_spans_8i:.1f} < 8: the window sits inside a single habitat blob '
    f'and configuration is not separable from composition')
# NOTE: the window/patch check that 8h asserts is deliberately NOT asserted
# here -- breaking that alignment is the treatment in the patch arm, and the
# per-level ceiling column is what records its cost.
print(f'8i: window/patch ratio across the patch arm: '
      f'{[f"{2*_S8I_R_M/(p*100.0):.2f}" for p in PATCH_RANGE_CFG]}'
      f'   (1.00 = aligned; the sweep deliberately moves off it)', flush=True)

# -- Seed bases, one per arm -------------------------------------------------
# 1000 apart so the four arms cannot interleave, and well clear of 8j's
# 40000-block. The per-species offset applied at each loop is +100000.
_S8I_SEED_PATCH = _S8I_SEED + 20000     # 29000
_S8I_SEED_NPTS  = _S8I_SEED + 24000     # 33000
_S8I_SEED_PREV  = _S8I_SEED + 28000     # 37000
_S8I_SEED_RES   = _S8I_SEED + 29000     # 38000

# -- Fit count + seed plan ----------------------------------------------------
# Replication is over SPECIES (n_reps=1 per species), not repeated draws, so
# the count is levels x species, not levels x reps. Each config level fits
# 3 RF (center / patch-summary / oracle) + _S8_N_INIT CNNs.
_n8i_levels = (len(PATCH_RANGE_CFG) + len(N_POINTS_RANGE_CFG)
               + len(OCC_PREV_RANGE_CFG) + len(RESOLUTION_RANGE_CFG_A1)
               + len(_S8_RES_CONSTFOOT))   # resolution runs BOTH res_modes
_fits_8i = (3 + _S8_N_INIT) * _S8_N_SPECIES * 2 * _n8i_levels
print(f'\n8i levels -- patch={len(PATCH_RANGE_CFG)} n_points={len(N_POINTS_RANGE_CFG)} '
      f'prevalence={len(OCC_PREV_RANGE_CFG)} '
      f'resolution={len(RESOLUTION_RANGE_CFG_A1)}+{len(_S8_RES_CONSTFOOT)} '
      f'(total {_n8i_levels})')
print(f'  (3 RF + {_S8_N_INIT} CNN) x {_S8_N_SPECIES} species x 2 arms '
      f'x {_n8i_levels} levels')
print(f'Total model fits: {_fits_8i}', flush=True)

# SEED DISTINCTNESS IS VERIFIED, NOT RE-DERIVED.
# This used to be a second, parallel construction of the seed layout, asserted
# against itself: if a loop below changed its base or its stride, the plan kept
# agreeing with the plan while the actual draws collided. Every loop now RECORDS
# the seed it really used, and the assert at the END of this cell checks those.
# (The resolution arm records one seed per species and convention: its levels
# share their points on purpose.)
# Arm bases are 1000+ apart and the species offset is 100000, so no
# (arm, species, level) cell should collide.
_seeds_used_8i = []

# =============================================================================
# Arm 1 -- PATCH sweep at alpha = 1 (r_m FIXED: the alignment mechanism test)
# =============================================================================
_S8_DROPS.clear()   # per-cell; _s8_drop_report at the end lists what 8i lost
_s8i_patch_rows = []

# Replication is over SPECIES, not repeated draws of one species
# (see _s8_species_iter). n_reps=1 keeps the fit count identical
# to the previous 1-species x N-reps design.
for _sp_i, _sp_sd in _s8_species_iter():
    _s8i_seed = _S8I_SEED_PATCH + _sp_i * 100000
    for _p8i in PATCH_RANGE_CFG:
        print(f'\n8i-patch | PATCH={_p8i}px ({_p8i*100} m)  window/patch='
              f'{2*_S8I_R_M/(_p8i*100.0):.2f}  (grf, alpha={_S8I_ALPHA})', flush=True)
        _rows_lv = _s8_run_level_config(env_scaled_100m, resolution=_S8I_RESOLUTION, 
                                        n_points=_S8_N_POINTS, pres_ratio=PRES_RATIO, patch=_p8i, 
                                        sub_exp='8i_patch_grf_alpha1', level=_p8i, seed_base=_s8i_seed,
                                        patch_buffer_px=max(PATCH_RANGE_CFG),   # same population at every patch level 
                                        W_100m=_W_8i, w_source='grf', corr_len_m=_S8I_CORR_LEN_M, 
                                        alpha=_S8I_ALPHA, truth_discipline='self_consistent', 
                                        mu_P=_S8I_MU_P, sig_P=_S8I_SIG_P, mu_E=_s8_meta['mu_E'], 
                                        sig_E=_s8_meta['sig_E'], w_P=_S8I_W_P, r_m=_S8I_R_M, 
                                        frac_woody=_S8I_FRAC_WOODY, n_reps=1)
        _s8i_patch_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _seeds_used_8i.append(_s8i_seed); _s8i_seed += 100
results_s8i_patch = _pd_s8.DataFrame(_s8i_patch_rows)
print(f'\n8i-patch: {len(results_s8i_patch)} rows', flush=True)

# =============================================================================
# Arm 2 -- SAMPLE SIZE sweep at alpha = 1 (patch fixed at the aligned value)
# =============================================================================
# PATCH stays at its global default here, i.e. the aligned setting, so this arm
# varies only N and pairs level-for-level with 8a.
_s8i_npts_rows = []

# Replication is over SPECIES, not repeated draws of one species
# (see _s8_species_iter). n_reps=1 keeps the fit count identical
# to the previous 1-species x N-reps design.
for _sp_i, _sp_sd in _s8_species_iter():
    _s8i_seed = _S8I_SEED_NPTS + _sp_i * 100000
    for _n8i_pts in N_POINTS_RANGE_CFG:
        print(f'\n8i-npoints | N={_n8i_pts} (grf, alpha={_S8I_ALPHA}, patch={PATCH})',
              flush=True)
        _rows_lv = _s8_run_level_config(env_scaled_100m, resolution=_S8I_RESOLUTION, n_points=_n8i_pts, 
                                        pres_ratio=PRES_RATIO, patch=PATCH, sub_exp='8i_npoints_grf_alpha1', 
                                        level=_n8i_pts, seed_base=_s8i_seed, W_100m=_W_8i, w_source='grf', 
                                        corr_len_m=_S8I_CORR_LEN_M, alpha=_S8I_ALPHA, 
                                        truth_discipline='self_consistent', mu_P=_S8I_MU_P, sig_P=_S8I_SIG_P, 
                                        mu_E=_s8_meta['mu_E'], sig_E=_s8_meta['sig_E'], w_P=_S8I_W_P, r_m=_S8I_R_M, 
                                        frac_woody=_S8I_FRAC_WOODY, n_reps=1)
        _s8i_npts_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _seeds_used_8i.append(_s8i_seed); _s8i_seed += 100
results_s8i_npts = _pd_s8.DataFrame(_s8i_npts_rows)
print(f'\n8i-npoints: {len(results_s8i_npts)} rows', flush=True)

# =============================================================================
# Arm 3 -- ECOLOGICAL PREVALENCE at alpha = 1 (the alpha=1 half of 8b)
# =============================================================================
# Sweeps LANDSCAPE OCCUPANCY, exactly as 8b now does, with the sample balance
# held at PRES_RATIO. Pairs level-for-level with 8b, so the two together answer
# "does rarity cost the CNN more or less than it costs the RF, and does that
# differ between regimes".
_s8i_prev_rows = []
for _sp_i, _sp_sd in _s8_species_iter():
    _s8i_seed = _S8I_SEED_PREV + _sp_i * 100000
    for _op8i in OCC_PREV_RANGE_CFG:
        print(f'\n8i-prevalence | occupancy={_op8i} (grf, alpha={_S8I_ALPHA}, '
              f'sample balance {PRES_RATIO})', flush=True)
        _rows_lv = _s8_run_level_config(
            env_scaled_100m, resolution=_S8I_RESOLUTION, n_points=_S8_N_POINTS,
            pres_ratio=PRES_RATIO, patch=PATCH,
            sub_exp='8i_prevalence_grf_alpha1', level=_op8i, seed_base=_s8i_seed,
            W_100m=_W_8i, w_source='grf', corr_len_m=_S8I_CORR_LEN_M,
            alpha=_S8I_ALPHA, truth_discipline='self_consistent',
            mu_P=_S8I_MU_P, sig_P=_S8I_SIG_P, mu_E=_s8_meta['mu_E'], sig_E=_s8_meta['sig_E'],
            w_P=_S8I_W_P, r_m=_S8I_R_M, frac_woody=_S8I_FRAC_WOODY,
            occ_prev=_op8i, n_reps=1)
        _s8i_prev_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _seeds_used_8i.append(_s8i_seed); _s8i_seed += 100
results_s8i_prev = _pd_s8.DataFrame(_s8i_prev_rows)
print(f'\n8i-prevalence: {len(results_s8i_prev)} rows', flush=True)

# =============================================================================
# Arm 4 -- RESOLUTION at alpha = 1 (the alpha=1 half of 8c-fixed)
# =============================================================================
# Same design as 8c-fixed: precise 100 m points, truth fixed at 100 m, only the
# covariates (and the forest-proportion channel) coarsen; same points at every
# constant-footprint level.
# READ THIS ARM VIA THE CEILING, NOT RAW SKILL, AND VIA BOTH res_modes.
#
# fixed_patch_px: r_m is fixed in METRES while PATCH is fixed in PIXELS, so the
# window/patch ratio moves with resolution (1.00 at 100 m, 0.50 at 200 m, 0.33
# at 300 m) -- and that ratio sets how much of the configurational signal any
# patch-limited model can reach. Raw skill therefore confounds "the grid got
# coarser" with "the CNN's achievable ceiling moved", which is why
# ceiling_config_patch is recomputed per level and the plotting cell reports
# skill as a FRACTION of what was achievable.
#
# constant_footprint: PATCH*resolution is held at _S8_FOOTPRINT_M = 2*r_m, so
# window/patch is 1.00 at EVERY level and the alignment is no longer confounded
# with the treatment at all. This is the headline arm here for the same reason
# it is in 8c, and more strongly: this cell's whole subject IS the alignment
# mechanism, and the fixed-pixel convention breaks it as a side effect of the
# sweep. The cost is the CNN-capacity term (PATCH 32 -> 16 -> 8), which 8i's
# own patch arm calibrates directly.
_S8I_RES_ARMS = [
    ('constant_footprint', _S8_RES_CONSTFOOT,       0),
    ('fixed_patch_px',     RESOLUTION_RANGE_CFG_A1, 50_000),
]
_s8i_res_rows = []
for _sp_i, _sp_sd in _s8_species_iter():
    # PRECISE 100 m POINTS, TRUTH FIXED AT 100 m -- the alpha = 1 half of 8c-fixed,
    # same function (_s8_run_level_fixedtruth). The species' configurational truth
    # is built once at 100 m; only the covariates and the forest-proportion channel
    # the models see are coarsened.
    _cfg_t = _s8_config_truth_100m(env_scaled_100m, _W_8i, r_m=_S8I_R_M, sig_P=_S8I_SIG_P,
                                   mu_E=_s8_meta['mu_E'], sig_E=_s8_meta['sig_E'], w_P=_S8I_W_P)
    _truth8i = _s8_suit_blend(_s8_suit_100m, _cfg_t['S_config_100'], _S8I_ALPHA)
    _xcols8i = dict(alpha=_S8I_ALPHA, truth_discipline='fixed_100m_points', w_source='grf',
                    corr_len_m=_S8I_CORR_LEN_M, mu_P=_cfg_t['mu_P'], mu_P_requested=_S8I_MU_P,
                    sig_P=_S8I_SIG_P, mu_E=_s8_meta['mu_E'], sig_E=_s8_meta['sig_E'],
                    w_P=_S8I_W_P, r_m=_S8I_R_M, frac_woody=_S8I_FRAC_WOODY,
                    frac_woody_realized=_cfg_t['frac_woody_realized'])
    for _res_mode, _res_levels, _mode_off in _S8I_RES_ARMS:
        # one seed per (species, arm): the constant-footprint levels share points
        _s8i_seed = _S8I_SEED_RES + _sp_i * 100000 + _mode_off
        _ladder8i = [(_r, _s8_constfoot_patch(_r) if _res_mode == 'constant_footprint' else PATCH)
                     for _r in _res_levels]
        for _res8i, _patch8i in _ladder8i:
            print(f'\n8i-resolution [{_res_mode}] | covariates at {_res8i} m  PATCH={_patch8i}px '
                  f'window/patch={2*_S8I_R_M/(_patch8i*_res8i):.2f}  (grf, alpha={_S8I_ALPHA}); '
                  f'points and truth at 100 m', flush=True)
            _mask8i = _s8_fixed_common_mask(
                env_scaled_100m,
                _ladder8i if _res_mode == 'constant_footprint' else [(_res8i, _patch8i)])
            _rows_lv = _s8_run_level_fixedtruth(
                env_scaled_100m, _res8i, _S8_N_POINTS, PRES_RATIO, _patch8i,
                '8i_resolution_grf_alpha1', _res8i, _s8i_seed, _mask8i, _truth8i,
                n_reps=1, cfg=_cfg_t, extra_cols=_xcols8i)
            for _r in _rows_lv:
                _r['res_mode'] = _res_mode
            _s8i_res_rows.extend(_s8_tag(_rows_lv, _sp_i, _sp_sd))
        _seeds_used_8i.append(_s8i_seed)
    del _cfg_t, _truth8i
results_s8i_res = _pd_s8.DataFrame(_s8i_res_rows)
print(f'\n8i-resolution: {len(results_s8i_res)} rows', flush=True)

# =============================================================================
# Combine + save (own file -- 8h's output is never touched)
# =============================================================================
# res_mode is only meaningful for the resolution arm; the other three arms
# run at the baseline 100 m / PATCH 32, where both conventions coincide.
for _df8i in (results_s8i_patch, results_s8i_npts, results_s8i_prev):
    if not _df8i.empty:
        _df8i['res_mode'] = 'baseline_100m'
# spearman_rel_ceiling: skill RELATIVE TO what coarsening left recoverable.
# Only the resolution arm has a ceiling (the others run at 100 m under
# 'self_consistent', where nothing is destroyed), so it is NaN elsewhere.
# Same guards and the same caveats as 8c-fixed -- see the block there: it is a
# ratio of correlations, not a share, and it goes negative where the model is
# anti-correlated (13.9% of these rows). spearman_rel_ceiling_sq is the
# variance-style companion that IS a share.
if not results_s8i_res.empty and 'oracle_spearman_infoloss' in results_s8i_res.columns:
    _ceil_8i = results_s8i_res['oracle_spearman_infoloss']
    results_s8i_res['spearman_rel_ceiling'] = (
        results_s8i_res['spearman_true'] / _ceil_8i).where(_ceil_8i > 0.05)
    results_s8i_res['spearman_rel_ceiling_sq'] = (
        (results_s8i_res['spearman_true'] ** 2) / (_ceil_8i ** 2)).where(_ceil_8i > 0.05)
    print('\n8i-resolution info-loss ceiling by level (fixed 100 m truth):')
    print(results_s8i_res.groupby(['res_mode', 'level'])['oracle_spearman_infoloss']
          .mean().round(4).to_string(), flush=True)

results_s8i = _pd_s8.concat([results_s8i_patch, results_s8i_npts,
                             results_s8i_prev, results_s8i_res],
                            ignore_index=True)
_S8I_VERSION = _S8_VERSION
_suffix_8i = '_smoke' if SMOKE_TEST_S8 else ''
_s8i_out = _os_8i.path.join(FIGURES_DIR,
                            f'section8i_alpha_axes_{_S8I_VERSION}{_suffix_8i}.csv')
results_s8i.to_csv(_s8i_out, index=False)
print(f'\nSaved: {_s8i_out}  ({len(results_s8i)} rows)', flush=True)

# -- The registered prediction, now measured on the real raster --------------
# ceiling_config_patch is computed inside _s8_run_level_config per level, so
# this table is the prediction realised on the actual data rather than the
# synthetic field it was registered on. Compare its shape (peak at the aligned
# patch) with the CNN's own curve in the plotting cell.
# REPORTING IS NON-FATAL. The fits above cost hours; a KeyError in a
# summary print must never discard them. The traceback is printed in
# full so the bug is still obvious.
try:
    print('\n-- patch arm: achievable ceiling per level (measured on the real raster) --')
    _ceil_tab_8i = (results_s8i_patch.groupby('level')['ceiling_config_patch']
                    .mean().rename('ceiling'))
    _ceil_tab_8i = _ceil_tab_8i.to_frame()
    _ceil_tab_8i['window_over_patch'] = [2 * _S8I_R_M / (float(p) * 100.0)
                                         for p in _ceil_tab_8i.index]
    print(_ceil_tab_8i.round(3).to_string())
    print('  (registered prediction was 0.209 / 0.391 / 0.930 / 0.383 for '
          'PATCH 8 / 16 / 32 / 64, at mu_E = sig_E = 1; species now draw their own)')
except Exception as _rep_err:
    import traceback as _tb_rep
    print(f'\n[REPORTING FAILED -- results are intact] {type(_rep_err).__name__}: {_rep_err}', flush=True)
    _tb_rep.print_exc()


_s8_drop_report('8i')

# -- Seed check on what ACTUALLY ran (see the note above) ---------------------
assert len(set(_seeds_used_8i)) == len(_seeds_used_8i), (
    f'8i seed collision: {len(_seeds_used_8i)} draws but only '
    f'{len(set(_seeds_used_8i))} distinct -- two conditions shared a point draw '
    f'while being reported as independent conditions')
print(f'8i seeds actually used: {len(_seeds_used_8i)} draws, all distinct '
      f'({min(_seeds_used_8i)}..{max(_seeds_used_8i)})', flush=True)


### 8.9 — Results: Configurational Axes (`8i`)
Raw skill is not comparable across patch sizes, because the ceiling itself varies there; each figure shows skill, ceiling and their ratio, plus the α = 0 counterpart.

In [ ]:
# =============================================================================
# 8i -- Plots (run after the 8i sweep cell)
#
# Same conventions as the Section-8 and 8h plotting cells: colour = model
# (Okabe-Ito), arm = linestyle, faint dots = individual reps shown raw (never
# reduced to an SE/CI, because reps within a level share a species), CNN
# init-SD as the noise band, degenerate fits flagged not dropped, no p-values.
#
# TWO THINGS THIS CELL IS FOR, neither of which the raw skill curve shows:
#
# 1. THE PATCH ARM IS A MECHANISTIC TEST. The achievable ceiling varies with
#    patch size here (unlike every other sweep, where it is fixed), so RAW
#    SKILL IS THE WRONG READOUT: at PATCH=8 the ceiling is ~0.21, so a CNN
#    scoring 0.15 has captured ~70% of what exists and is doing well, while
#    the same 0.15 against the ~0.93 ceiling at PATCH=32 would be a failure.
#    The figure therefore plots skill AND ceiling AND their ratio, and the
#    claim to check is whether the CNN curve PEAKS where the ceiling peaks.
#
# 2. THE FACTORIAL. 8a and 8e are the alpha=0 halves of these same axes, so
#    they are loaded and overlaid. The question is not "is the CNN better"
#    (8h settled that) but "does the axis behave DIFFERENTLY in the two
#    regimes" -- e.g. does the CNN need more data before its configurational
#    advantage appears than the RF needs for its pointwise one.
# =============================================================================
import pandas as pd
import matplotlib.pyplot as _p8i
import matplotlib.ticker as _tk8i

_sfx8i = '_smoke' if SMOKE_TEST_S8 else ''
_S8I_VERSION = _S8_VERSION
_csv8i = os.path.join(FIGURES_DIR,
                      f'section8i_alpha_axes_{_S8I_VERSION}{_sfx8i}.csv')
if not os.path.exists(_csv8i):
    raise FileNotFoundError(f'{os.path.basename(_csv8i)} not found -- run the 8i sweep cell first.')
_i = pd.read_csv(_csv8i)
print(f'8i results loaded: {len(_i)} rows from {os.path.basename(_csv8i)}')
print(f'  sub_exp = {sorted(_i.sub_exp.unique())}')

# -- ONE res_mode PER PANEL ----------------------------------------------------
# The 8i resolution arm runs both resolution conventions, which use DIFFERENT
# patch sizes at the same resolution; the panels below key on `level`, so
# pooling them would average an 8 px CNN with a 32 px one. Keep the headline
# arm (constant_footprint -- here window/patch is 1.00 at every level, so the
# alignment mechanism this cell is about is NOT confounded with the sweep) and
# retain the full frame for the res_mode comparison figure at the END OF THIS
# CELL. (It used to say the 8c plotting cell would use it -- that was wrong:
# that cell never loads the 8i CSV, so the fixed_patch_px rows appeared in no
# figure at all despite being computed.)
_i_all = _i.copy()
if 'res_mode' in _i.columns:
    _keep8i = _i.res_mode.isna() | _i.res_mode.isin(
        ['constant_footprint', 'baseline_100m'])
    print(f'  res_mode present: keeping constant_footprint as headline '
          f'({int(_keep8i.sum())} of {len(_i)} rows)')
    _i = _i[_keep8i].reset_index(drop=True)

# the alpha=0 halves of the same two axes, for the factorial panels
_a0_csv = os.path.join(FIGURES_DIR, f'section8_two_regime_{_S8_VERSION}{_sfx8i}.csv')
_A0 = pd.read_csv(_a0_csv) if os.path.exists(_a0_csv) else None
# 8c-fixed lives in its own file; append it so the resolution axis has an
# alpha=0 reference like every other axis. Only the columns _A0 already has are
# kept, so nothing new is injected into the frame.
_cf0_csv = os.path.join(FIGURES_DIR, f'section8c_fixed_truth_{_S8_VERSION}{_sfx8i}.csv')
if _A0 is not None and os.path.exists(_cf0_csv):
    _cf0 = pd.read_csv(_cf0_csv)
    _cf0 = _cf0[[c for c in _A0.columns if c in _cf0.columns]]
    _A0 = pd.concat([_A0, _cf0], ignore_index=True)
    print(f'  alpha=0 frame: +{len(_cf0)} rows from 8c-fixed (resolution axis)')
elif _A0 is not None:
    print('  (8c-fixed CSV absent -- the resolution axis will have no alpha=0 curve)')
if _A0 is None:
    print('  (section8_two_regime CSV absent -- alpha=0 comparison panels skipped)')

_CN8I = 'cnn_mean' if (_i.model == 'cnn_mean').any() else 'cnn'
_M8I = {'rf':               ('#0072B2', 'RF (center pixel)',  'RF-center'),
        'rf_patch_summary': ('#009E73', 'RF (patch summary)', 'RF-patch'),
        'rf_oracle_metrics': ('#CC79A7', 'RF (oracle metrics)', 'RF-oracle'),
        _CN8I:              ('#D55E00', 'CNN (full patch)',   'CNN')}
_ORD8I = ['rf', 'rf_patch_summary', 'rf_oracle_metrics', _CN8I]
# Same convention as the 8a-8e cell: arm encoded by BOTH line style and
# marker, and neither arm solid (skill was measured only at the plotted
# levels, so a solid line would overstate what was tested).
# Same encoding convention as the 8a-8e cell: MODEL = colour + marker shape,
# ARM = line style + marker fill. With FOUR models here, colour alone was the
# only separator -- and #009E73 (RF-patch) against #D55E00 (CNN) is exactly the
# red-green pair a colourblind reader cannot split. Marker shape now carries the
# model; the arm moves to fill.
_LS8I  = {'random': ':',  'extrap': '--'}
_FILL8I = {'random': 'full', 'extrap': 'none'}
_MKM8I = {'rf': 'o', 'rf_patch_summary': 's',
          'rf_oracle_metrics': 'D', _CN8I: '^'}   # model -> marker shape
_ARMS8I = ['random', 'extrap']
_AXES8I = [('8i_patch_grf_alpha1',      'level', 'Patch size (px)',   '8e', 'Patch size'),
           ('8i_npoints_grf_alpha1',    'level', 'N training points', '8a', 'Sample size'),
           ('8i_prevalence_grf_alpha1', 'level', 'Landscape occupancy', '8b', 'Prevalence'),
           ('8i_resolution_grf_alpha1', 'level', 'Resolution (m)',    '8c-fixed', 'Resolution')]
# The alpha=0 counterpart of the resolution axis is 8c-fixed (the self-consistent
# variant, key 8c, was retired and removed). 8c-fixed writes its OWN csv -- the combined
# two-regime file deliberately excludes it, because its extra columns would
# inject NaNs there -- so it is loaded separately just below and appended to the
# alpha=0 frame for this one axis.


# REPLICATION UNIT. Section 8 now replicates over SPECIES with one rep each,
# so `rep` is 0 for every row and grouping on it would pool all species into
# one group -- which would inflate the CNN init-SD with between-species
# variance and destroy the pairing in every paired contrast. Older CSVs
# predate the column, so fall back to `rep` for those.
_UNIT = 'species_idx' if 'species_idx' in _i.columns else 'rep'
print(f'replication unit: {_UNIT}')

def _sd8i(sub_exp, arm):
    d = _i[(_i.sub_exp == sub_exp) & (_i.arm == arm) & (_i.model == 'cnn')]
    if d.empty:
        return pd.Series(dtype=float)
    return d.groupby(['level', _UNIT])['spearman_true'].std().groupby(level=0).mean()


# -- CONFOUND CHECK -----------------------------------------------------------
# The patch arm must hold N and prevalence fixed; the N arm is SUPPOSED to move
# n_tr. Anything else moving is a confound.
print('\n-- confound check --')
for _se, _xc, _xl, _a0se, _nm in _AXES8I:
    for _arm in _ARMS8I:
        _g = _i[(_i.sub_exp == _se) & (_i.arm == _arm) & (_i.model == 'rf')]
        if _g.empty: continue
        _parts = []
        for k in ['n_tr', 'n_te', 'n_vl', 'real_prev_tr']:
            lo, hi = float(_g[k].min()), float(_g[k].max())
            _parts.append(f'{k}={lo:g}' if abs(hi - lo) < 1e-9 else f'{k}={lo:g}..{hi:g} VARIES')
        print(f'  {_se:<24} {_arm:<7} ' + '  '.join(_parts))
print('  (the n_points arm is expected to vary n_tr/n_te/n_vl; the patch arm'
      ' must not.)')

# -- CNN FIT HEALTH -----------------------------------------------------------
# Same prevalence-aware rule as the other two plotting cells.
_N_SE_ABOVE_CHANCE = 2.0
def _floor8i(n_vl, prev_vl):
    n  = np.asarray(n_vl, float)
    p  = np.clip(np.asarray(prev_vl, float), 1e-6, 1 - 1e-6)
    n1 = np.maximum(n * p, 1.0); n0 = np.maximum(n * (1 - p), 1.0)
    return 0.5 + _N_SE_ABOVE_CHANCE * np.sqrt((n + 1.0) / (12.0 * n1 * n0))

_DEG8I = set()
if 'val_auc_best' in _i.columns and _i.val_auc_best.notna().any():
    _cfi = _i[(_i.model == 'cnn') & _i.val_auc_best.notna()].copy()
    _cfi['auc_floor']  = _floor8i(_cfi.n_vl, _cfi.real_prev_vl)
    _cfi['degenerate'] = _cfi.val_auc_best < _cfi.auc_floor
    _H8I = _cfi.groupby(['sub_exp', 'arm', 'level']).agg(
        n_fits=('degenerate', 'size'), frac_degen=('degenerate', 'mean'),
        val_auc_best=('val_auc_best', 'mean'), auc_floor=('auc_floor', 'mean'),
        best_epoch=('early_stop_ep', 'mean'))
    print(f'\n-- CNN fit health (degenerate = val AUC not '
          f'{_N_SE_ABOVE_CHANCE:.0f} SE above chance) --')
    print(_H8I.round(3).to_string())
    _DEG8I = set(_H8I[_H8I.frac_degen > 0.5].index)
    if _DEG8I:
        print(f'\n  WARNING: {len(_DEG8I)} condition(s) where >50% of CNN fits never '
              f'learned;\n  those points are hollow with a red x. In the patch arm this '
              f'is expected\n  at the extreme patch sizes -- the ceiling there is low, so '
              f'there is little\n  to learn. Read it together with the ceiling, not alone.')
else:
    _H8I = None
    print('\n  (no val_auc_best column -- results predate the fit-health diagnostic)')


# -- One figure per axis ------------------------------------------------------
for _se, _xc, _xlab, _a0se, _nm in _AXES8I:
    _g = _i[_i.sub_exp == _se]
    if _g.empty:
        continue
    _lv = sorted(_g[_xc].dropna().unique())
    _fig, _ax = _p8i.subplots(1, 3, figsize=(16, 4.6))
    _fig.suptitle(f'{_nm} at $\\alpha=1$ (configurational DGP)',
                  fontsize=12, fontweight='bold')

    # (a) absolute skill + the achievable ceiling
    _a = _ax[0]
    for _m in _ORD8I:
        _clr, _lbl, _shrt = _M8I[_m]
        for _arm in _ARMS8I:
            _d = _g[(_g.model == _m) & (_g.arm == _arm)]
            if _d.empty: continue
            _pr = _d.groupby([_xc, _UNIT])['spearman_true'].mean()
            _mu = _pr.groupby(level=_xc).mean()
            _a.plot(_pr.index.get_level_values(_xc).astype(float), _pr.values,
                    _MKM8I[_m], color=_clr, ms=3.4, alpha=0.30, ls='none',
                    fillstyle=_FILL8I[_arm], zorder=1)
            _a.plot(_mu.index.astype(float), _mu.values, ls=_LS8I[_arm],
                    marker=_MKM8I[_m], color=_clr, ms=5, lw=1.7,
                    fillstyle=_FILL8I[_arm], label=f'{_shrt} {_arm}', zorder=3)
            if _m == _CN8I:
                # init-SD band removed for the same reason as in the 8a-8e cell:
                # it applied to one model only and made panels look inconsistent.
                # The band still governs verdicts in the contrast figure.
                _dg = [v for v in _mu.index if (_se, _arm, float(v)) in _DEG8I]
                if _dg:
                    _dv = _mu.reindex(_dg)
                    _a.plot([float(v) for v in _dg], _dv.values, 'o', ms=9,
                            mfc='white', mec=_clr, mew=1.8, zorder=4)
                    _a.plot([float(v) for v in _dg], _dv.values, 'x', ms=7,
                            color='crimson', mew=2.0, zorder=5)
    _ceil = _g.groupby(_xc)['ceiling_config_patch'].mean().dropna()
    if not _ceil.empty:
        _a.plot(_ceil.index.astype(float), _ceil.values, 'k:', lw=1.5, marker='s',
                ms=5, label='achievable ceiling', zorder=6)
    _a.set_ylabel('Spearman $\\rho$ vs truth')
    _a.set_title('Skill and what was achievable', fontsize=10)

    # (b) the ratio -- the readout that is comparable across levels
    _b = _ax[1]
    _gc = _g.copy()
    _gc['frac'] = (_gc.spearman_true / _gc.ceiling_config_patch
                   ).where(_gc.ceiling_config_patch > 0.05)
    # RF-ORACLE IS EXCLUDED HERE, DELIBERATELY. ceiling_config_patch is what a
    # PATCH-LIMITED model could achieve, but RF-oracle is handed the true
    # landscape metrics at the truth's own radius r_m and never looks at a
    # patch at all. Dividing its skill by that ceiling is a category error: it
    # produced ratios of 3.2-3.6 at patch 8, which read as "captured 360% of
    # what was achievable" when the real content is that the ceiling does not
    # apply. Its near-constant 0.70-0.80 across every patch size (panel a) is
    # exactly the evidence that it is not patch-limited.
    _ORD_FRAC = [_m for _m in _ORD8I if _m != 'rf_oracle_metrics']
    for _m in _ORD_FRAC:
        _clr, _lbl, _shrt = _M8I[_m]
        for _arm in _ARMS8I:
            _s = _gc[(_gc.model == _m) & (_gc.arm == _arm)].groupby(_xc)['frac'].mean().dropna()
            if _s.empty: continue
            _b.plot(_s.index.astype(float), _s.values, ls=_LS8I[_arm], marker=_MKM8I[_m],
                    fillstyle=_FILL8I[_arm],
                    color=_clr, ms=5, lw=1.7, label=f'{_shrt} {_arm}')
    _b.axhline(1.0, color='gray', lw=0.8, ls=':')
    _b.set_ylabel('captured / achievable')
    _b.set_title('Share of the achievable signal\n(patch-limited models only)',
                 fontsize=10)

    # (c) the factorial: same axis at alpha=0 (8a/8e) vs alpha=1 (here)
    _c = _ax[2]
    if _A0 is not None and _a0se in _A0.sub_exp.unique():
        _cn0 = 'cnn_mean' if (_A0.model == 'cnn_mean').any() else 'cnn'
        # MATCH THE res_mode ON BOTH SIDES. The alpha=1 frame (_i) is filtered
        # to the headline convention, but _A0 was not, so for the resolution
        # axis the alpha=0 curve pooled BOTH conventions and showed six x
        # positions (100/200/300/400/500/1000) against the alpha=1 curve's
        # three. The two curves were then not the same experiment at the same
        # x, which is the entire point of a factorial panel.
        _A0p = _A0
        if 'res_mode' in _A0.columns:
            _A0p = _A0[_A0.res_mode.isna()
                       | _A0.res_mode.isin(['constant_footprint', 'baseline_100m'])]
        for _m0, _m1 in [('rf', 'rf'), (_cn0, _CN8I)]:
            _clr = _M8I['rf' if _m1 == 'rf' else _CN8I][0]
            _shrt = _M8I['rf' if _m1 == 'rf' else _CN8I][2]
            _s0 = (_A0p[(_A0p.sub_exp == _a0se) & (_A0p.model == _m0)
                        & (_A0p.arm == 'random')]
                   .groupby('level')['spearman_true'].mean())
            _s1 = (_g[(_g.model == _m1) & (_g.arm == 'random')]
                   .groupby(_xc)['spearman_true'].mean())
            if not _s0.empty:
                _c.plot(_s0.index.astype(float), _s0.values, ls=':',
                        marker=_MKM8I.get(_m1, 'o'), color=_clr, ms=4.5, lw=1.5,
                        alpha=0.75, fillstyle='none',
                        label=f'{_shrt} $\\alpha$=0')
            if not _s1.empty:
                _c.plot(_s1.index.astype(float), _s1.values, ls='--',
                        marker=_MKM8I.get(_m1, 'o'), color=_clr, ms=5, lw=1.8,
                        label=f'{_shrt} $\\alpha$=1')
        _c.set_title('Same axis, both regimes (random arm)', fontsize=10)
        _c.set_ylabel('Spearman $\\rho$ vs truth')
    else:
        _c.text(0.5, 0.5, '(alpha=0 counterpart not available)', ha='center',
                va='center', transform=_c.transAxes)

    for _axx in _ax:
        _axx.set_xlabel(_xlab); _axx.set_xscale('log')
        _axx.set_xticks(_lv); _axx.set_xticklabels([f'{v:g}' for v in _lv])
        _axx.xaxis.set_minor_formatter(_tk8i.NullFormatter())
        _axx.grid(alpha=0.25)
        # legend OUTSIDE the axes: with 4 models x 2 arms it covered the very
        # curves it labels, in every panel.
        _axx.legend(fontsize=6, ncol=1, handlelength=3.0, frameon=True,
                    framealpha=0.9, loc='center left', bbox_to_anchor=(1.01, 0.5))

    # PATCH ARM: a second x-axis in units of the generative window.
    # A patch size in pixels says nothing on its own -- what decides whether the
    # CNN can see the configuration is how the patch compares with the window
    # the species was generated on. That window has radius r_m and therefore
    # DIAMETER 2*r_m, so the ratio is patch_m / (2*r_m) and it reads exactly
    # 1.0 where the patch just covers the window. Measured skill peaks there
    # (ceiling 0.950 at 32 px vs 0.218 / 0.403 / 0.396 at 8 / 16 / 64), so this
    # axis is the one that carries the mechanism.
    if _xc == 'level' and _se == '8i_patch_grf_alpha1':
        # r_m and resolution are READ FROM THE DATA, not from the sweep cell's
        # variables. This cell must be runnable from the CSVs alone -- reaching
        # for _S8I_R_M made it silently depend on the 8i sweep cell having been
        # executed in the same kernel, which is exactly the coupling that makes
        # a plotting cell unusable for re-plotting an old result.
        _rm_d  = float(_g['r_m'].dropna().iloc[0]) if 'r_m' in _g.columns else np.nan
        _res_d = float(_g['resolution'].dropna().iloc[0]) if 'resolution' in _g.columns else np.nan
        if np.isfinite(_rm_d) and np.isfinite(_res_d):
            _win_m = 2.0 * _rm_d
            for _axx in _ax:
                _tw = _axx.secondary_xaxis(
                    'top',
                    functions=(lambda p, _r=_res_d, _w=_win_m: p * _r / _w,
                               lambda f, _r=_res_d, _w=_win_m: f * _w / _r))
                _tw.set_xticks([v * _res_d / _win_m for v in _lv])
                _tw.set_xticklabels([f'{v * _res_d / _win_m:g}' for v in _lv],
                                    fontsize=8)
                _tw.set_xlabel(f'patch / generative window  '
                               f'(2$\\cdot r_m$ = {_win_m:.0f} m)', fontsize=8.5)
    _fig.text(0.5, -0.02, 'Dotted black = achievable ceiling (varies with patch '
              'size, so raw skill is not comparable across levels there). '
              'Hollow + red x = CNN did not fit.',
              ha='center', fontsize=7.5, color='0.35')
    _p8i.tight_layout()
    _fp = os.path.join(FIGURES_DIR, f's8i_{_nm.lower().replace(" ", "_")}.png')
    _p8i.savefig(_fp, dpi=150, bbox_inches='tight')
    _p8i.show(); _p8i.close()
    print(f'Saved: {_fp}')


# -- Verdict table ------------------------------------------------------------
# CNN - RF-patch is the contrast of interest (RF-patch is the composition
# ceiling); it is judged against the CNN init-SD band. The ceiling column is
# printed alongside because a small contrast at a level where little was
# achievable means something different from the same contrast where a lot was.
print('\n' + '=' * 100)
print('8i VERDICTS -- CNN - RF-patch paired contrast vs the CNN init-SD band')
print('=' * 100)
print(f"{'axis':<24} {'arm':<7} {'level':>7} {'ceiling':>8} {'CNN':>7} "
      f"{'CNN-RFpatch':>12} {'initSD':>7} {'%degen':>7}  verdict")
print('-' * 100)
for _se, _xc, _xlab, _a0se, _nm in _AXES8I:
    _g = _i[_i.sub_exp == _se]
    if _g.empty: continue
    for _arm in _ARMS8I:
        _sd = _sd8i(_se, _arm)
        _d = _g[_g.arm == _arm]
        _hi = _d[_d.model == _CN8I].groupby([_xc, _UNIT])['spearman_true'].mean()
        _lo = _d[_d.model == 'rf_patch_summary'].groupby([_xc, _UNIT])['spearman_true'].mean()
        _con = (_hi - _lo).dropna().groupby(level=_xc).mean()
        _cnn = _d[_d.model == _CN8I].groupby(_xc)['spearman_true'].mean()
        _cl  = _d.groupby(_xc)['ceiling_config_patch'].mean()
        for _lv_ in _con.index:
            _v = float(_con.loc[_lv_]); _b = float(_sd.get(_lv_, np.nan))
            _fd = np.nan
            if _H8I is not None and (_se, _arm, _lv_) in _H8I.index:
                _fd = float(_H8I.loc[(_se, _arm, _lv_), 'frac_degen'])
            if not np.isfinite(_b):
                _vd = 'no noise band (needs _S8_N_INIT > 1)'
            elif np.isfinite(_fd) and _fd > 0.5:
                _vd = f'UNRELIABLE -- {_fd:.0%} of CNN fits never learned'
            elif abs(_v) < _b:
                _vd = 'INCONCLUSIVE (inside init-SD band)'
            else:
                _vd = 'CNN above composition ceiling' if _v > 0 else 'CNN below it'
            _fds = f'{_fd:.0%}' if np.isfinite(_fd) else '  --'
            print(f'{_se:<24} {_arm:<7} {float(_lv_):>7.4g} {_cl.get(_lv_, np.nan):>8.3f} '
                  f'{_cnn.get(_lv_, np.nan):>7.3f} {_v:>12.3f} {_b:>7.3f} {_fds:>7}  {_vd}')
print('=' * 100)
print('Contrasts are paired within rep (same drawn data) before averaging.')
print('No p-values: reps within a level share a species and are not independent.')
# -- ESTIMATOR ROBUSTNESS: mean-of-k vs best-of-k ---------------------------
# Same check as the other two plotting cells: keep the restart with the best
# VALIDATION AUC instead of averaging over failed ones. Selection never
# touches test data.
if 'val_auc_best' in _i.columns and _i.val_auc_best.notna().any():
    _bki = (_i[_i.model == 'cnn'].sort_values('val_auc_best')
              .groupby(['sub_exp', 'arm', 'level', _UNIT], as_index=False).tail(1))
    print('\n' + '=' * 88)
    print('8i ESTIMATOR ROBUSTNESS -- CNN - RF-patch, both estimators')
    print('=' * 88)
    print(f"{'axis':<24} {'arm':<7} {'level':>7} {'mean-of-k':>11} "
          f"{'best-of-k':>11} {'shift':>7}")
    print('-' * 88)
    for _se_r, _xc_r, _xl_r, _a0_r, _nm_r in _AXES8I:
        for _arm_r in _ARMS8I:
            _d_r = _i[(_i.sub_exp == _se_r) & (_i.arm == _arm_r)]
            if _d_r.empty: continue
            _rp_r = _d_r[_d_r.model == 'rf_patch_summary'].groupby(_xc_r)['spearman_true'].mean()
            _mn_r = _d_r[_d_r.model == _CN8I].groupby(_xc_r)['spearman_true'].mean()
            _bk_r = (_bki[(_bki.sub_exp == _se_r) & (_bki.arm == _arm_r)]
                     .groupby(_xc_r)['spearman_true'].mean())
            for _lv_r in _mn_r.index:
                _vm = _mn_r.get(_lv_r, np.nan) - _rp_r.get(_lv_r, np.nan)
                _vb = _bk_r.get(_lv_r, np.nan) - _rp_r.get(_lv_r, np.nan)
                print(f'{_se_r:<24} {_arm_r:<7} {float(_lv_r):>7.4g} '
                      f'{_vm:>11.3f} {_vb:>11.3f} {_vb - _vm:>7.3f}')
    print('=' * 88)


# -- Fig: the two resolution conventions at alpha = 1 -------------------------
# COMPANION TO s8_8c_res_mode_comparison.png, which asks the same question in
# the POINTWISE regime. Here the DGP is configurational, which is the case the
# section exists to test, so this is the version that matters.
#
# The two conventions answer "what does coarsening do?" differently because
# they hold different things fixed:
#   fixed_patch_px      PATCH stays 32 px, so the window grows with the grid
#                       (3.2 km -> 9.6 km) and the required point separation
#                       grows with it, taking n_tr down as a side effect.
#   constant_footprint  PATCH*resolution stays at _S8_FOOTPRINT_M, so extent and
#                       separation are constant and n_tr holds -- but PATCH in
#                       pixels shrinks (32 -> 16 -> 10 -> 8) and the CNN's
#                       pooling window shrinks with it.
# Neither is neutral. Agreement in the SIGN of the CNN-RF gap is what would
# make the resolution conclusion robust to the choice; divergence is itself the
# result, and a more interesting one.
#
# THE CEILING IS DRAWN PER CONVENTION, not once. ceiling_config_patch is a
# function of the patch, and the patch differs between the arms at the same
# resolution -- a single ceiling line would be wrong for one of them.
_cmp_i = _i_all[_i_all.res_mode.isin(['constant_footprint', 'fixed_patch_px'])] \
    if 'res_mode' in _i_all.columns else _i_all.iloc[0:0]
_cmp_i = _cmp_i[_cmp_i.sub_exp == '8i_resolution_grf_alpha1']
if _cmp_i.empty:
    print('\n(no 8i resolution rows with res_mode -- comparison figure skipped)')
else:
    _LS_RM = {'constant_footprint': ':', 'fixed_patch_px': '--'}   # never solid
    _figR, _axR = _p8i.subplots(2, 2, figsize=(11.5, 8.4))
    _figR.suptitle('Resolution at $\\alpha=1$: does the conclusion survive the '
                   'resolution convention?', fontsize=12, fontweight='bold')
    for _ci_r, _arm_r in enumerate(_ARMS8I):
        _aA, _aB = _axR[0][_ci_r], _axR[1][_ci_r]
        _dA = _cmp_i[_cmp_i.arm == _arm_r]
        for _rm in ('constant_footprint', 'fixed_patch_px'):
            _dr = _dA[_dA.res_mode == _rm]
            if _dr.empty: continue
            # (top) absolute skill, per model
            for _m in _ORD8I:
                _dm = _dr[_dr.model == _m]
                if _dm.empty: continue
                _clr, _lbl, _sh = _M8I[_m]
                _pr = _dm.groupby(['level', _UNIT])['spearman_true'].mean()
                _mu = _pr.groupby(level='level').mean().sort_index()
                _aA.plot(_mu.index.astype(float), _mu.values, ls=_LS_RM[_rm],
                         marker=_MKM8I[_m], ms=4.5, lw=1.7, color=_clr,
                         label=f'{_sh} {_rm.split("_")[0]}')
            # ceiling for THIS convention
            _ce = _dr.groupby('level')['ceiling_config_patch'].mean().dropna().sort_index()
            if not _ce.empty:
                _aA.plot(_ce.index.astype(float), _ce.values, ls=_LS_RM[_rm],
                         color='0.25', lw=1.2, marker='s', ms=3.5,
                         label=f'ceiling {_rm.split("_")[0]}')
            # (bottom) captured / achievable -- patch-limited models only, since
            # RF-oracle is handed the true metrics and never reads a patch
            _dc = _dr[_dr.model != 'rf_oracle_metrics'].copy()
            _dc['frac'] = (_dc.spearman_true / _dc.ceiling_config_patch
                           ).where(_dc.ceiling_config_patch > 0.05)
            for _m in [x for x in _ORD8I if x != 'rf_oracle_metrics']:
                _dm = _dc[_dc.model == _m]
                if _dm.empty: continue
                _clr, _lbl, _sh = _M8I[_m]
                _fr = _dm.groupby('level')['frac'].mean().dropna().sort_index()
                if _fr.empty: continue
                _aB.plot(_fr.index.astype(float), _fr.values, ls=_LS_RM[_rm],
                         marker=_MKM8I[_m], ms=4.5, lw=1.7, color=_clr,
                         label=f'{_sh} {_rm.split("_")[0]}')
        # n_tr of the fixed-pixel arm: the confound the other convention removes
        _nt = (_dA[_dA.res_mode == 'fixed_patch_px']
               .groupby('level')['n_tr'].mean().sort_index())
        for _lv, _nv in _nt.items():
            _aA.annotate(f'{int(_nv)}', xy=(float(_lv), 0.015),
                         xycoords=('data', 'axes fraction'), fontsize=6,
                         color='#888888', ha='center', va='bottom', clip_on=False)
        _aB.axhline(1.0, color='gray', lw=0.8, ls=':')
        _aA.set_title(f'{_arm_r} arm -- skill and its ceiling', fontsize=10)
        _aB.set_title(f'{_arm_r} arm -- captured / achievable '
                      f'(patch-limited models)', fontsize=10)
        for _ax in (_aA, _aB):
            _lv_all = sorted(_cmp_i.level.dropna().unique())
            _ax.set_xscale('log'); _ax.set_xticks(_lv_all)
            _ax.set_xticklabels([f'{v:g}' for v in _lv_all], fontsize=8)
            _ax.xaxis.set_minor_formatter(_tk8i.NullFormatter())
            _ax.xaxis.set_minor_locator(_tk8i.NullLocator())
            _ax.grid(alpha=0.25)
            _ax.legend(fontsize=5.8, ncol=2, handlelength=3.0,
                       framealpha=0.9, loc='best')
            _ax.set_xlabel('Resolution (m)   ' + r'$grey$: $n_{tr}$ fixed-pixel')
    _axR[0][0].set_ylabel('Spearman $\\rho$ vs truth')
    _axR[1][0].set_ylabel('captured / achievable')
    _p8i.tight_layout()
    _fpR = os.path.join(FIGURES_DIR, 's8i_resolution_res_mode_comparison.png')
    _p8i.savefig(_fpR, dpi=150, bbox_inches='tight'); _p8i.show(); _p8i.close()
    print(f'Saved: {_fpR}')

    # Does the CNN - RF gap agree in sign between the conventions?
    print('\n-- 8i resolution: CNN - RF-center gap under both conventions --')
    print(f"{'arm':<8} {'res':>6} {'const_foot':>11} {'fixed_px':>9}  agree")
    for _arm_r in _ARMS8I:
        _dA = _cmp_i[_cmp_i.arm == _arm_r]
        for _lv in sorted(_dA.level.dropna().unique()):
            _g = {}
            for _rm in ('constant_footprint', 'fixed_patch_px'):
                _x = _dA[(_dA.res_mode == _rm) & (_dA.level == _lv)]
                _g[_rm] = (_x[_x.model == _CN8I]['spearman_true'].mean()
                           - _x[_x.model == 'rf']['spearman_true'].mean())
            _a, _b = _g['constant_footprint'], _g['fixed_patch_px']
            _ok = 'n/a' if not (np.isfinite(_a) and np.isfinite(_b)) else \
                  ('yes' if np.sign(_a) == np.sign(_b) else 'NO')
            print(f'{_arm_r:<8} {_lv:>6.0f} {_a:>11.3f} {_b:>9.3f}  {_ok}')
    # DERIVED, NOT HARDCODED -- the ladders have changed before and this line
    # named "320 m", a level that turned out to be 300 m and has since been removed.
    _only_r = {}
    for _rm_o in ('constant_footprint', 'fixed_patch_px'):
        # _cmp_i, NOT _dA: _dA is whatever the arm loop above left behind, so a
        # level missing from that one arm would be reported as missing from the
        # whole convention. The level SETS are a property of the ladders, not of
        # an arm.
        _only_r[_rm_o] = set(_cmp_i.loc[_cmp_i.res_mode == _rm_o, 'level'].astype(float))
    _cf_o = sorted(_only_r['constant_footprint'] - _only_r['fixed_patch_px'])
    _fp_o = sorted(_only_r['fixed_patch_px'] - _only_r['constant_footprint'])
    print('  (levels present in only one convention read NaN by construction: '
          + (f'{[f"{v:g}" for v in _cf_o]} m constant-footprint only; ' if _cf_o else '')
          + (f'{[f"{v:g}" for v in _fp_o]} m fixed-pixel only' if _fp_o else '')
          + ')')


### 8.10 — Generalisation across Species (`8j`)
K = 8 independent species over α ∈ {0, 0.25, 0.5, 0.75, 1} for both refuge-field sources, plus a correlation-length sweep and an oracle-validation bound.

In [ ]:
_s8_check_hp()   # refuse to sweep on hyperparameters tuned for another stack
# =============================================================================
# 8j -- Does the configurational result GENERALISE ACROSS SPECIES?
#
# THE GAP THIS CLOSES. Every result in Section 8 so far rests on ONE virtual
# species: cell 27 draws it once and reuses it for every sub-experiment, level,
# rep and arm. That is the right choice WITHIN an experiment -- holding the
# species fixed isolates the data-condition effect -- but it means "the CNN
# wins when the DGP is configurational" is demonstrated for a single niche
# realisation. It also leaves Section 8 unable to use the statistics
# "aggregate across species (median + bootstrap CI, Wilcoxon on
# per-species values)", because n_species = 1.
#
# THE DESIGN. K independent species, each run across the full alpha sweep on
# two W sources plus a corr-len sweep, one rep each. SPECIES IS THE
# REPLICATION UNIT -- one rep is correct rather than cheap: reps within a
# species share a niche and are NOT independent, species are. K species x 1
# rep is therefore MORE usable replication than 1 species x K reps, and it is
# the kind the prescribed statistics actually apply to.
#
# THIS CELL SUPERSEDES 8h. Every arm 8h ran at n=1 species is here at
# ensemble level; see the arm list further down.
#
# WHAT VARIES AND WHAT DOES NOT. Only the niche changes. The environmental
# raster, the PCA, the W field, and the extrapolative fold thresholds
# (_S8_PC1_TR/VAL/TE, which come from the PCA and not the species) are all
# held fixed, so every species is evaluated on the same environmental split.
#
# WHY THE PAIRED QUANTITY IS THE ONE TO AGGREGATE. Species differ enormously
# in intrinsic difficulty, so absolute skill will scatter. cfgGain is paired
# WITHIN species -- each species is its own alpha=0 control -- so that scatter
# cancels and what is left is the effect of the DGP regime.
#
# HOW THE SPECIES SWAP WORKS. _s8_suit_fn resolves _s8_meta as a GLOBAL at
# call time, so rebinding _s8_meta re-points the suitability surface with no
# change to _s8_suit_fn or _s8_run_level_config. Only the derived raster
# _s8_suit_100m has to be refreshed alongside it. Both are restored at the end
# so the rest of the notebook is unaffected by having run this cell.
# =============================================================================
import os as _os_8j

# Constants come from _S8_CFG (cell 27) so 8i and 8j cannot drift apart.
# EIGHT species. At K=3 the cross-species Wilcoxon signed-rank test could not
# reach p<0.05 at all (its minimum two-sided p at n=3 is 0.25), so 8j could
# only DESCRIBE the across-species spread, never test it. At n=8 the minimum
# two-sided p is 0.0078, so the test becomes usable and the
# prescription (median + bootstrap CI, Wilcoxon on per-species values) has
# enough replication to mean something.
#
# The K=3 run is preserved as ..._K3.csv. Its three species are the first
# three admitted here (screening is deterministic, so a larger `want` admits a
# SUPERSET), and their per-species .part.csv checkpoints are reloaded rather
# than refitted -- only the 5 new species are actually computed.
# SPECIES IN THE ENSEMBLE. 8 for v5, and the number is a statistical
# requirement rather than a preference: the Wilcoxon signed-rank floor is
# 2^-(K-1), so the smallest attainable two-sided p is 0.125 at K=4, 0.0625 at
# K=5 and 0.0078 at K=8. Below K=6 the primary endpoint cannot clear 0.05 no
# matter how large the effect, which makes the test unanswerable rather than
# negative. At K=8 it can, for a single declared endpoint (see _S8J_PRIMARY_ARM
# in the plotting cell); a fully Holm-corrected family of all 8 cells would
# need K>=9, which is why one endpoint is declared instead of eight tested.
# Cost, measured on the v4 runs: ~2.3 h per species for the full 8j grid.
_S8J_K            = 2 if SMOKE_TEST_S8 else 8   # the smoke pool holds only 6 candidates
_S8J_ALPHAS       = ([0.0, 1.0] if SMOKE_TEST_S8
                     else [0.0, 0.25, 0.5, 0.75, 1.0])
_S8J_SEED_BASE    = 40000
_S8J_R_M = _S8_CFG['r_m']
_S8J_CORR_LEN_M = _S8_CFG['corr_len_m']
_S8J_FRAC_WOODY = _S8_CFG['frac_woody']
_S8J_MU_P, _S8J_SIG_P = _S8_CFG['mu_P'], _S8_CFG['sig_P']
# mu_E / sig_E: per species, read from _s8_meta at each call (_s8_config_niche).
_S8J_W_P = _S8_CFG['w_P']
_S8J_RESOLUTION = _S8_CFG['resolution']

# -- W field ------------------------------------------------------------------
# Built here unconditionally (8h, which used to supply it, is retired).
# _s8_make_W is deterministic in (shape, corr_len_px, frac, seed), so this is
# bit-identical to what the 8i cell builds.
_W_8j = _s8_make_W(_s8_suit_100m.shape, _S8_CFG['corr_len_m'] / 100.0,
                   _S8_CFG['frac_woody'], seed=_S8_CFG['grf_seed'])
print('8j: GRF W built (corr_len={:.0f} m, seed={})'.format(
    _S8_CFG['corr_len_m'], _S8_CFG['grf_seed']), flush=True)

# SEED is species 0, so the single-species results this supersedes are nested
# SPECIES. 8j draws from the SAME candidate pool as 8a-8e/8i and admits more of
# them (_S8J_K vs _S8_N_SPECIES), so the smaller sweeps' ensemble is a NESTED
# SUBSET of this one rather than a different set of species: screening is
# deterministic, so the first _S8_N_SPECIES admitted here are exactly the ones
# those cells ran. SEED is candidate 0, so the original single-species result is
# nested inside the ensemble rather than sitting beside it.
assert _S8J_K <= _S8_SPECIES_POOL_N, (
    f'8j asks for {_S8J_K} species but the pool only holds '
    f'{_S8_SPECIES_POOL_N} candidates -- raise _S8_SPECIES_POOL_N')

# =============================================================================
# ARMS. 8h is superseded by this cell, so everything it tested at n=1 species
# is here at ensemble level:
#   (a) pointwise baseline -- no W anywhere. The true null.
#   (b) alpha sweep, GRF W.
#   (c) alpha sweep, LANDUSE W -- real spatial structure, the ecological
#       validity check on (b).
#   (d) corr-len sweep at alpha=1 -- does the effect depend on habitat grain.
# The channel-only null check is NOT a separate arm any more: it is exactly
# the alpha=0 point of (b) and (c), where W is stacked as a channel but the
# truth ignores it. Comparing that against (a) is the same test 8h ran
# separately, now with species-level replication.
# =============================================================================
_S8J_CORRLEN_ALL = ([300.0] if SMOKE_TEST_S8 else [150.0, 200.0, 300.0, 400.0])
# corr_len 300 is the alpha-sweep default, so it is already covered by (b) at
# alpha=1; only the others need their own runs.
_S8J_CORRLEN_EXTRA = [x for x in _S8J_CORRLEN_ALL if abs(x - _S8J_CORR_LEN_M) > 1e-9]   # excludes 300 (already computed)

# One W per correlation length, built once.
_W8J = {_cl: (_W_8j if abs(_cl - _S8J_CORR_LEN_M) < 1e-9
              else _s8_make_W(_s8_suit_100m.shape, _cl / 100.0,
                              _S8_CFG['frac_woody'], seed=_S8_CFG['grf_seed']))
        for _cl in _S8J_CORRLEN_ALL}

# Landuse W, from `landuse` / WOODY_CLASSES as extracted in the loader cell.
if 'landuse' not in dir() or 'WOODY_CLASSES' not in dir():
    raise RuntimeError('8j landuse arm needs `landuse` and `WOODY_CLASSES` -- '
                       'run the loader cell (section 2) first -- it extracts both.')
_W_landuse_8j = np.isin(landuse, list(WOODY_CLASSES)).astype(np.float32)
print('8j: landuse W built, woody fraction={:.3f}'.format(
    float(_W_landuse_8j[~np.isnan(_s8_suit_100m)].mean())), flush=True)

_S8J_LANDUSE_ALPHAS = ([0.0, 1.0] if SMOKE_TEST_S8 else _S8J_ALPHAS)

# -- Fit count ----------------------------------------------------------------
_n_cond_8j = (1                                    # pointwise baseline
              + len(_S8J_ALPHAS)                   # grf alpha sweep
              + len(_S8J_LANDUSE_ALPHAS)           # landuse alpha sweep
              + len(_S8J_CORRLEN_EXTRA)            # corr-len at alpha=1
              + 2)                                 # oracle-val bound (pointwise + alpha=1)
_fits_8j = (_S8J_K * 2 *
            ((_n_cond_8j - 1) * (3 + _S8_N_INIT) + (1 + _S8_N_INIT)))
print('\n8j: {} conditions/species x {} species x 2 arms'.format(
    _n_cond_8j, _S8J_K))
print('  config conditions fit 3 RF (center/patch/oracle) + {} CNN;'.format(_S8_N_INIT))
print('  the pointwise baseline has no W, so 1 RF + {} CNN.'.format(_S8_N_INIT))
print('Total model fits: {}'.format(_fits_8j), flush=True)


def _run8j(sub_exp, level, seed, W, w_source, corr_len, alpha, oracle_val=False):
    return _s8_run_level_config(
        env_scaled_100m, resolution=_S8J_RESOLUTION, n_points=_S8_N_POINTS,
        pres_ratio=PRES_RATIO, patch=PATCH,
        sub_exp=sub_exp, level=level, seed_base=seed,
        W_100m=W, w_source=w_source, corr_len_m=corr_len,
        alpha=alpha, truth_discipline='self_consistent',
        mu_P=_S8J_MU_P, sig_P=_S8J_SIG_P, mu_E=_s8_meta['mu_E'], sig_E=_s8_meta['sig_E'],
        w_P=_S8J_W_P, r_m=_S8J_R_M, frac_woody=_S8J_FRAC_WOODY, n_reps=1,
        oracle_val=oracle_val)


# =============================================================================
# Run
# =============================================================================
# Seed layout, per species block of 2000: baseline 0, grf alpha 100+, landuse
# 300+, corr-len 500+. Species stride 2000 keeps blocks disjoint, and the whole
# range stays clear of the +100000 species offsets the 8a-8e/8i sweeps use.
# CHECKPOINT PER SPECIES (the analysis plan requires checkpointing long runs).
# 8j is a multi-hour cell that previously wrote nothing until the very end, so a
# machine crash partway through discarded every completed species. Each species
# now writes its own .part.csv the moment it finishes, and a re-run reloads
# those instead of refitting them. The partials are kept after the final
# concat -- they cost little and they are the only record if the final write
# itself is interrupted.
#
# STALE-CHECKPOINT GUARD: a partial is only trusted if its species_seed matches
# the seed the iterator just yielded for that index. Change _S8J_K or the pool
# and the indices shift; without this check a stale partial would be silently
# adopted under the wrong species and the ensemble would be quietly corrupt.
_S8J_VERSION = _S8_VERSION
_suffix_8j = '_smoke' if SMOKE_TEST_S8 else ''


def _s8j_part_path(sp_i):
    return _os_8j.path.join(
        FIGURES_DIR, 'section8j_species_ensemble_{}{}_sp{}.part.csv'.format(
            _S8J_VERSION, _suffix_8j, sp_i))


_S8_DROPS.clear()   # per-cell; see _s8_drop_report at the end
_s8j_rows = []
for _sp_i, _sp_sd in _s8_species_iter(n_admit=_S8J_K):
    _pp8j = _s8j_part_path(_sp_i)
    if _os_8j.path.exists(_pp8j):           # check checkpoint to avaid recomputing existing species
        _prev8j = _pd_s8.read_csv(_pp8j)
        _seed_ok = (not _prev8j.empty
                    and int(_prev8j['species_seed'].iloc[0]) == int(_sp_sd))
        if _seed_ok:
            _s8j_rows.extend(_prev8j.to_dict('records'))
            print('  species {} (seed {}): RESTORED from checkpoint, {} rows '
                  '-- not re-run'.format(_sp_i, _sp_sd, len(_prev8j)), flush=True)
            continue
        print('  species {}: checkpoint is for seed {}, not {} -- STALE, '
              'recomputing'.format(_sp_i,
                                   _prev8j['species_seed'].iloc[0] if not _prev8j.empty else '?',
                                   _sp_sd), flush=True)
    _sp_rows = []
    _b = _S8J_SEED_BASE + _sp_i * 2000

    # (a) pointwise baseline -- no W at all, so _s8_run_level, not the config twin
    _sp_rows.extend(_s8_tag(_s8_run_level(
        env_scaled_100m, _S8J_RESOLUTION, _S8_N_POINTS, PRES_RATIO, PATCH,
        '8j_pointwise_baseline', 0.0, _b, n_reps=1), _sp_i, _sp_sd))

    # (b) alpha sweep, GRF
    for _ai, _a in enumerate(_S8J_ALPHAS):
        print('  [grf] alpha={:.2f}'.format(_a), flush=True)
        _sp_rows.extend(_s8_tag(_run8j(
            '8j_alpha_grf', _a, _b + 100 + _ai * 10, _W8J[_S8J_CORR_LEN_M],
            'grf', _S8J_CORR_LEN_M, _a), _sp_i, _sp_sd))

    # (c) alpha sweep, LANDUSE
    for _ai, _a in enumerate(_S8J_LANDUSE_ALPHAS):
        print('  [landuse] alpha={:.2f}'.format(_a), flush=True)
        _sp_rows.extend(_s8_tag(_run8j(
            '8j_alpha_landuse', _a, _b + 300 + _ai * 10, _W_landuse_8j,
            'landuse', None, _a), _sp_i, _sp_sd))

    # (d) corr-len at alpha=1
    for _ci, _cl in enumerate(_S8J_CORRLEN_EXTRA):
        print('  [corr-len] {:.0f} m at alpha=1'.format(_cl), flush=True)
        _sp_rows.extend(_s8_tag(_run8j(
            '8j_corrlen_grf_alpha1', _cl, _b + 500 + _ci * 10, _W8J[_cl],
            'grf', _cl, 1.0), _sp_i, _sp_sd))

    # (e) ORACLE MODEL SELECTION BOUND -- pointwise and configurational.
    # Identical to (a) and to the alpha=1 point of (b) in every respect except
    # where the validation fold comes from: the TEST band rather than the mid
    # band, so early stopping sees the shift it is scored at (D=1.30 -> 1.99 in
    # the primary design). It assumes labels in the target domain, which real
    # transfer does not have, so it is an UPPER BOUND and is labelled as one --
    # the same status as the RF-oracle arm.
    #
    # Paired within species against its own primary counterpart, so the
    # difference isolates model selection from model capability. Only the
    # extrapolative arm is informative here; the random arm has no mismatch to
    # correct (D(val,train)=D(test,train)=1.00) and is run only so the pairing
    # is complete.
    print('  [oracle-val] pointwise baseline', flush=True)
    _sp_rows.extend(_s8_tag(_s8_run_level(
        env_scaled_100m, _S8J_RESOLUTION, _S8_N_POINTS, PRES_RATIO, PATCH,
        '8j_oracleval_pointwise', 0.0, _b + 700, n_reps=1,
        oracle_val=True), _sp_i, _sp_sd))
    print('  [oracle-val] alpha=1 grf', flush=True)
    _sp_rows.extend(_s8_tag(_run8j(
        '8j_oracleval_alpha1', 1.0, _b + 800, _W8J[_S8J_CORR_LEN_M],
        'grf', _S8J_CORR_LEN_M, 1.0, oracle_val=True), _sp_i, _sp_sd))

    # Species complete -- checkpoint it before starting the next one.
    _pd_s8.DataFrame(_sp_rows).to_csv(_pp8j, index=False)
    print('  species {} (seed {}): checkpointed {} rows -> {}'.format(
        _sp_i, _sp_sd, len(_sp_rows), _os_8j.path.basename(_pp8j)), flush=True)
    _s8j_rows.extend(_sp_rows)

_s8_drop_report('8j')
results_s8j = _pd_s8.DataFrame(_s8j_rows)
_s8j_out = _os_8j.path.join(FIGURES_DIR,
                            'section8j_species_ensemble_{}{}.csv'.format(
                                _S8J_VERSION, _suffix_8j))
results_s8j.to_csv(_s8j_out, index=False)
print('\nSaved: {}  ({} rows)'.format(_s8j_out, len(results_s8j)), flush=True)

# -- Sanity: the species really did differ ------------------------------------
# If the swap silently failed, every species would share one suitability
# surface and auc_oracle would be identical across them. This is the check that
# the rebinding worked at all -- without it a broken swap would produce a
# beautiful, entirely fake cross-species result.
_chk8j = results_s8j.groupby('species_idx')['auc_oracle'].mean()
print('\nauc_oracle by species (must differ -- identical values would mean the '
      'species swap silently failed):')
print(_chk8j.round(4).to_string())
assert _chk8j.nunique() > 1, (
    'every species produced the same auc_oracle -- the _s8_meta rebinding did '
    'not take effect and these are not independent species')
print('\narms produced:')
print(results_s8j.groupby('sub_exp').size().to_string())


### 8.11 — Results: Cross-Species Statistic (`8j`)
The only place in section 8 that reports confidence intervals and p-values, because species are the independent unit: median and bootstrap CI of the per-species cfgGain, and a Wilcoxon signed-rank test on one pre-declared primary endpoint (α = 1, extrap arm).

In [ ]:
# =============================================================================
# 8j -- Plots + the cross-species statistic
#
# This is the only cell in Section 8 that can legitimately produce an interval
# and a p-value, and the reason is the design, not the code: SPECIES is the
# replication unit here, and species ARE independent of one another. Everywhere
# else in Section 8 the replication is reps-within-a-species, which share a
# niche and a raster, so the analysis plan forbids pooling them into an SE or a test --
# hence per-rep scatter and noise bands there, and a real statistic only here.
#
# THE QUANTITY. Per species, arm and ALPHA:
#     cfgGain(a) = [CNN - RF-patch](a) - [CNN - RF-patch](alpha=0)
# referenced to that species' OWN alpha=0 run, so each species is its own
# control and the large species-to-species differences in intrinsic difficulty
# cancel. Subtracting the alpha=0 gap also removes the CNN's fixed
# architecture penalty, which is present even where configuration is worth
# nothing -- what is left is the effect of the DGP regime.
# Aggregation follows the analysis plan exactly: median across species, percentile
# bootstrap CI on the per-species values, Wilcoxon signed-rank on those same
# values -- computed SEPARATELY AT EACH ALPHA, because the whole point of
# sweeping alpha here is that the effect need not be monotone in it.
#
# READ THE N HONESTLY. With K species the Wilcoxon cannot return a p below
# 2^-(K-1) however large the effect is (K=8 -> 0.0078), so the test is reported
# with its floor stated. The median and CI carry more information than the p.
# =============================================================================
import pandas as pd
import matplotlib.pyplot as _p8j
from scipy.stats import wilcoxon as _wilcoxon_8j

_sfx8j = '_smoke' if SMOKE_TEST_S8 else ''
_S8J_VERSION = _S8_VERSION
_csv8j = os.path.join(FIGURES_DIR,
                      f'section8j_species_ensemble_{_S8J_VERSION}{_sfx8j}.csv')
if not os.path.exists(_csv8j):
    raise FileNotFoundError(f'{os.path.basename(_csv8j)} not found -- run the 8j sweep first.')
_j = pd.read_csv(_csv8j)
# -- CNN FIT HEALTH (this cell had none) -------------------------------------
# Same rule as the 8a-8e and 8i cells: a fit whose best validation AUC never
# cleared chance by 2 Hanley-McNeil SE never learned, so its test score is not
# a measurement of the model. 8j was the one plotting cell without this, which
# meant a species whose CNN failed to train was averaged in as though it had
# simply scored low -- different causes, and only the second says anything
# about capability. Flagged, never dropped.
def _val_auc_floor8j(n_vl, prev_vl):
    n  = np.asarray(n_vl, float)
    p  = np.clip(np.asarray(prev_vl, float), 1e-6, 1 - 1e-6)
    n1 = np.maximum(n * p, 1.0); n0 = np.maximum(n * (1 - p), 1.0)
    return 0.5 + 2.0 * np.sqrt((n + 1.0) / (12.0 * n1 * n0))

_DEG8J = set()
if 'val_auc_best' in _j.columns and _j.val_auc_best.notna().any():
    _cf8j = _j[(_j.model == 'cnn') & _j.val_auc_best.notna()].copy()
    _cf8j['degenerate'] = _cf8j.val_auc_best < _val_auc_floor8j(_cf8j.n_vl,
                                                               _cf8j.real_prev_vl)
    _h8j = _cf8j.groupby(['sub_exp', 'arm', 'alpha'])['degenerate'].mean()
    _DEG8J = set(_h8j[_h8j > 0.5].index)
    print(f'\n8j CNN fit health: {len(_DEG8J)} of {len(_h8j)} conditions with '
          f'>50% degenerate fits')
    if _DEG8J:
        for _k in sorted(_DEG8J, key=str):
            print(f'   never learned: {_k}')
else:
    print('\n  (no val_auc_best in this CSV -- 8j fit health not available)')

_CN8J = 'cnn_mean' if (_j.model == 'cnn_mean').any() else 'cnn'
# The alpha figure and the cross-species statistic describe ONE arm at a
# time; 8j now contains four. The GRF alpha sweep is the primary one -- the
# landuse sweep is its ecological replicate and the corr-len arm has a
# different x axis entirely, so pooling them would mix incomparable rows.
_S8J_MAIN = '8j_alpha_grf'
print('arms present:', sorted(_j.sub_exp.unique()))
_SPECIES = sorted(_j.species_idx.unique())
print(f'8j loaded: {len(_j)} rows, {len(_SPECIES)} species, '
      f'alpha levels {sorted(_j.alpha.unique())}')

# -- Per-species cfgGain at EVERY alpha ---------------------------------------
# One value per (species, arm, alpha), each referenced to that species' OWN
# alpha=0 run. The pairing is what makes species-to-species differences in
# intrinsic difficulty cancel; without it the scatter would swamp the effect.
# dropna(): the 8j_pointwise_baseline rows have no alpha, so a NaN rode along
# in this list. min()/max() over it returned NaN, and the decomposition below
# then silently looked up a NaN alpha and reported an empty result as a
# finding. Alphas are a property of the alpha sweeps, not of the baseline.
_ALPHAS_J = sorted(_j.alpha.dropna().unique())
_recs = []
for _sp in _SPECIES:
    for _arm in ['random', 'extrap']:
        _g = _j[(_j.species_idx == _sp) & (_j.arm == _arm) & (_j.sub_exp == _S8J_MAIN)]
        if _g.empty: continue
        _cn = _g[_g.model == _CN8J].groupby('alpha')['spearman_true'].mean()
        _rp = _g[_g.model == 'rf_patch_summary'].groupby('alpha')['spearman_true'].mean()
        _rf = _g[_g.model == 'rf'].groupby('alpha')['spearman_true'].mean()
        if 0.0 not in _cn.index or 0.0 not in _rp.index:
            continue
        _gap0 = float(_cn[0.0] - _rp[0.0])          # the architecture penalty
        for _a in _ALPHAS_J:
            if _a not in _cn.index or _a not in _rp.index: continue
            _recs.append(dict(species_idx=_sp, arm=_arm, alpha=float(_a),
                              cfg_gain=float((_cn[_a] - _rp[_a]) - _gap0),
                              cnn=float(_cn[_a]), rf_patch=float(_rp[_a]),
                              rf=float(_rf.get(_a, np.nan))))
_PS = pd.DataFrame(_recs)
print(f'\n-- per-species values: {len(_SPECIES)} species x {len(_ALPHAS_J)} alpha '
      f'x 2 arms = {len(_PS)} rows (species is the unit) --')
print(_PS.pivot_table(index=['arm', 'alpha'], values=['cnn', 'rf_patch', 'cfg_gain'],
                      aggfunc='median').round(3).to_string())


def _boot_ci(x, n_boot=10000, seed=0, lo=2.5, hi=97.5, return_grain=False):
    """Percentile bootstrap CI of the MEDIAN over per-species values.

    Bootstrapping SPECIES is what makes this interval meaningful -- they are the
    independent units. Resampling reps would not be.

    ON THE GRANULARITY (return_grain). With K species the bootstrap median can
    only ever take one of a handful of values: resampling 5 numbers with
    replacement yields a median drawn from those same 5 (or midpoints of them),
    so the "95% CI" is a coarse lattice rather than a smooth interval, and its
    endpoints move in visible jumps. That is a property of K, not a bug, and it
    is the same honesty the Wilcoxon floor gets below -- so the number of
    DISTINCT bootstrap medians is reported alongside the interval instead of
    being left for the reader to infer.
    """
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 3:
        return (np.nan, np.nan, 0) if return_grain else (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    _bm = np.median(rng.choice(x, (n_boot, len(x)), replace=True), axis=1)
    _lo, _hi = (float(v) for v in np.percentile(_bm, [lo, hi]))
    return (_lo, _hi, int(np.unique(_bm).size)) if return_grain else (_lo, _hi)


# -- THE PRIMARY ENDPOINT, DECLARED BEFORE THE NUMBERS ARE READ ---------------
# cfgGain is tested at every alpha in both arms, which is 4 alphas x 2 arms = 8
# tests. Left uncorrected that is a multiplicity problem; corrected, it collides
# with the Wilcoxon floor. The arithmetic, for K species:
#   smallest attainable two-sided p = 2^-(K-1)   (0.125 at K=4, 0.0078 at K=8)
#   Holm threshold for the smallest p            = 0.05 / m
# At K=8 with m=8 the floor (0.0078) still exceeds the threshold (0.00625), so
# NO test in a fully corrected family of 8 can reach significance however large
# the effect. Declaring ONE primary endpoint sets m=1 (threshold 0.05, needs only
# K>=6) and makes the test answerable; the remaining cells are then reported as
# descriptive, with medians and CIs but no significance claim.
#
# The choice is alpha=1 in the EXTRAPOLATIVE arm, and it follows from what 8j
# exists to ask: whether a configurational signal the CNN can see and a
# composition-only model cannot survives the change of species, in the regime
# where the two architectures actually diverge. alpha=1 is the full
# configurational DGP (no pointwise component to dilute it) and extrap is the
# arm where the CNN-RF gap lives. Fixed here, above the table, so it is not a
# choice made after looking.
_S8J_PRIMARY_ARM   = 'extrap'
_S8J_PRIMARY_ALPHA = 1.0


print('\n' + '=' * 104)
print('8j CROSS-SPECIES STATISTIC -- cfgGain by alpha, aggregated over species')
print('=' * 104)
print(f"{'arm':<8} {'alpha':>6} {'K':>3} {'median':>8} {'95% CI (bootstrap)':>22} "
      f"{'grain':>6} {'>0':>5} {'Wilcoxon p':>11} {'role':>12}")
print('-' * 104)
_KJ = 0
_rows_stat = []
for _arm in ['random', 'extrap']:
    for _a in _ALPHAS_J:
        _v = _PS.loc[(_PS.arm == _arm) & (_PS.alpha == _a), 'cfg_gain'].dropna().values
        if len(_v) < 3:
            continue
        _KJ = max(_KJ, len(_v))
        _lo, _hi, _grain = _boot_ci(_v, return_grain=True)
        _is_primary = (_arm == _S8J_PRIMARY_ARM and float(_a) == _S8J_PRIMARY_ALPHA)
        if _a == 0.0:
            _ps, _pv = '   (ref)', np.nan
            _role = 'reference'
        else:
            try:
                _pv = float(_wilcoxon_8j(_v, alternative='two-sided').pvalue)
                _ps = f'{_pv:.4f}'
            except Exception:
                _pv, _ps = np.nan, 'n/a'
            _role = 'PRIMARY' if _is_primary else 'descriptive'
        _rows_stat.append(dict(arm=_arm, alpha=float(_a), K=len(_v), p=_pv,
                               primary=_is_primary))
        print(f'{_arm:<8} {_a:>6.2f} {len(_v):>3} {np.median(_v):>8.3f} '
              f'{f"[{_lo:.3f}, {_hi:.3f}]":>22} {_grain:>6} '
              f'{f"{int((_v > 0).sum())}/{len(_v)}":>5} {_ps:>11} {_role:>12}')
    print('-' * 104)
print('=' * 104)

# -- What the p column may and may not be used for ---------------------------
_prim = [r for r in _rows_stat if r['primary']]
if _prim:
    _pp, _kk = _prim[0]['p'], _prim[0]['K']
    _floor = 2.0 ** -(int(_kk) - 1)
    print(f'PRIMARY ENDPOINT: cfgGain at alpha={_S8J_PRIMARY_ALPHA:g}, '
          f'{_S8J_PRIMARY_ARM} arm (declared above the table, not chosen from it).')
    print(f'  K={_kk} species, Wilcoxon p={_pp:.4f}, alpha=0.05, m=1 test '
          f'-> {"REJECT the null" if np.isfinite(_pp) and _pp < 0.05 else "do NOT reject"}')
    print(f'  Attainable floor at K={_kk} is p={_floor:.4f}. '
          + ('The test can reach 0.05.' if _floor < 0.05 else
             f'IT CANNOT REACH 0.05 -- K>=6 species are needed for that, so this '
             f'endpoint is INCONCLUSIVE by design at K={_kk}, whatever the effect size.'))
_desc = [r for r in _rows_stat if not r['primary'] and np.isfinite(r['p'])]
if _desc:
    _m = len(_desc) + 1
    print(f'DESCRIPTIVE CELLS: {len(_desc)} further tests are shown. They are NOT '
          f'corrected and carry no significance claim.')
    print(f'  For reference, a Holm-corrected family of all m={_m} tests would need '
          f'the smallest p below {0.05 / _m:.5f};')
    print(f'  at K={_KJ} the attainable floor is {2.0 ** -(int(_KJ) - 1):.5f}, which is '
          + ('below that, so a corrected family would be feasible.'
             if 2.0 ** -(int(_KJ) - 1) < 0.05 / _m else
             f'ABOVE it -- a fully corrected family of {_m} is unreachable at this K '
             f'(K>={next(_k for _k in range(2, 30) if 2.0 ** -(_k - 1) < 0.05 / _m)} '
             f'would be required). This is why one endpoint is declared instead.'))
print('alpha=0 is the reference by construction (cfgGain = 0 there), not a result.')
print('"grain" = number of DISTINCT bootstrap medians behind the interval; at small')
print('  K the CI is a coarse lattice, not a smooth interval (see _boot_ci).')
print('These CIs are over SPECIES, the independent unit -- which is why they appear')
print('here and nowhere else in Section 8, where replication is reps-within-species.')
print('Inconclusive is a valid verdict here (per the analysis plan): if the primary endpoint')
print('cannot clear its own floor, that is what gets reported, not a winner.')

# -- Figure -------------------------------------------------------------------
# TWO ROWS, and the top one is not optional. cfgGain is a CONTRAST, so on its
# own it cannot distinguish "the CNN got better" from "the RF collapsed" -- and
# on this data it is overwhelmingly the latter: across species the RF falls
# from ~0.85 to ~0.01 as alpha goes 0 -> 1 while the CNN merely declines more
# slowly. Showing only the contrast would invite exactly the wrong reading, so
# the absolute curves sit directly above it.
# MODEL = colour + marker shape (see the 8a-8e cell for the full convention).
# Four models cannot be separated by colour alone: #009E73 and #D55E00 are the
# red-green confusion pair.
_MJ = {'rf':               ('#0072B2', 'RF-center',  'o'),
       'rf_patch_summary': ('#009E73', 'RF-patch',   's'),
       'rf_oracle_metrics':('#CC79A7', 'RF-oracle',  'D'),
       _CN8J:              ('#D55E00', 'CNN',        '^')}
_f, _axs = _p8j.subplots(2, 2, figsize=(13, 9), sharex=True)
_f.suptitle('The configurational advantage across independent species',
            fontsize=13, fontweight='bold')

for _k, _arm in enumerate(['random', 'extrap']):
    # ---- row 1: absolute skill, so the contrast below is interpretable ----
    _ax = _axs[0, _k]
    _d = _j[(_j.arm == _arm) & (_j.sub_exp == _S8J_MAIN)]
    for _m, (_clr, _lbl, _mk) in _MJ.items():
        _dm = _d[_d.model == _m]
        if _dm.empty: continue
        _per = _dm.groupby(['alpha', 'species_idx'])['spearman_true'].mean()
        _med = _per.groupby(level='alpha').median()
        _ax.plot(_per.index.get_level_values('alpha'), _per.values, _mk,
                 color=_clr, ms=3, alpha=0.25, ls='none', zorder=1)
        _ax.plot(_med.index, _med.values, ls='--', marker=_mk, color=_clr,
                 lw=2.0, ms=5.5, label=_lbl, zorder=3)
    _cl = _d.groupby('alpha')['ceiling_config_patch'].median().dropna()
    if not _cl.empty:
        # For alpha < 1 this bounds only the CONFIGURATIONAL COMPONENT of the
        # target (see _s8_config_ceiling); it is the ceiling of the full target
        # only at alpha = 1, and the label says so.
        _ax.plot(_cl.index, _cl.values, 'k:', lw=1.4, marker='s', ms=4,
                 label='ceiling of the configurational component\n'
                       '(= full ceiling only at $\\alpha$=1)', zorder=4)
    _ax.axhline(0, color='gray', lw=0.8, ls=':')
    _ax.set_ylabel('Spearman $\\rho$ vs truth (median over species)')
    _ax.set_title(f'{_arm} -- absolute skill', fontsize=10)
    _ax.grid(alpha=0.25); _ax.legend(fontsize=7, loc='best', framealpha=0.9)

    # ---- row 2: the paired contrast, with the cross-species CI ----
    _ax = _axs[1, _k]
    _s = _PS[_PS.arm == _arm]
    if _s.empty:
        _ax.set_visible(False); continue
    for _sp in _SPECIES:
        _q = _s[_s.species_idx == _sp].sort_values('alpha')
        if _q.empty: continue
        _ax.plot(_q.alpha, _q.cfg_gain, ls='--', color='#D55E00', alpha=0.28,
                 lw=1.0, marker='^', ms=3)
    _med = _s.groupby('alpha')['cfg_gain'].median()
    _cis = {a: _boot_ci(_s.loc[_s.alpha == a, 'cfg_gain'].dropna().values)
            for a in _med.index}
    _lo = np.array([_cis[a][0] for a in _med.index], float)
    _hi = np.array([_cis[a][1] for a in _med.index], float)
    _ok = np.isfinite(_lo) & np.isfinite(_hi)
    if _ok.any():
        _ax.fill_between(_med.index.values[_ok], _lo[_ok], _hi[_ok],
                         color='#D55E00', alpha=0.20, lw=0,
                         label='95% CI of the median (over species)')
    _ax.plot(_med.index, _med.values, ls='--', marker='^', color='#0072B2',
             lw=2.2, ms=6, label='median of per-species differences', zorder=5)

    # WHY THIS DOES NOT EQUAL THE VISIBLE GAP ABOVE. The top row shows, per
    # model, the MEDIAN OVER SPECIES; the eye then reads the gap between two
    # medians. This row shows the MEDIAN OF THE PER-SPECIES DIFFERENCES, which
    # is the correct paired statistic (the analysis plan: pair within species, then
    # aggregate) but is NOT the difference of the medians -- the median is not a
    # linear operator. On this data the two disagree by up to 0.125 and at
    # extrap alpha=0.50 they even have OPPOSITE SIGNS (+0.061 vs -0.064).
    # The difference-of-medians is drawn faintly so the gap between the rows is
    # visible rather than looking like an error.
    _dm_ref = []
    for _a in _med.index:
        _t = _d[np.isclose(_d.alpha, _a)]
        _cnm = _t[_t.model == _CN8J].groupby('species_idx')['spearman_true'].mean()
        _rpm = _t[_t.model == 'rf_patch_summary'].groupby('species_idx')['spearman_true'].mean()
        _sh = sorted(set(_cnm.index) & set(_rpm.index))
        _g0 = _s.loc[np.isclose(_s.alpha, 0.0), 'cfg_gain']
        _dm_ref.append(np.median([_cnm[i] for i in _sh]) - np.median([_rpm[i] for i in _sh])
                       if _sh else np.nan)
    _dm_ref = np.asarray(_dm_ref, float)
    _off = _dm_ref[0] if np.isfinite(_dm_ref[0]) else 0.0
    _ax.plot(_med.index, _dm_ref - _off, ls=':', color='0.45', lw=1.3,
             marker='v', ms=4, zorder=4,
             label='difference of medians (what the top row looks like)')

    _ax.axhline(0, color='gray', lw=1.0, ls='--')
    _ax.set_xlabel('$\\alpha$  (0 = pointwise, 1 = configurational)')
    _ax.set_ylabel('cfgGain (median of PER-SPECIES differences)')
    _ax.set_title(f'{_arm} -- paired contrast', fontsize=10)
    _ax.grid(alpha=0.25); _ax.legend(fontsize=7, loc='best', framealpha=0.9)

_f.text(0.5, -0.01, 'Top row: faint dots are individual species. Bottom row: faint '
        'lines are individual species -- the independent replication unit. Read the '
        'rows together: a rising contrast driven by a collapsing RF is a different '
        'claim from one driven by an improving CNN.',
        ha='center', fontsize=7.5, color='0.35')
_p8j.tight_layout()
_fp8j = os.path.join(FIGURES_DIR, 's8j_species_ensemble.png')
_p8j.savefig(_fp8j, dpi=150, bbox_inches='tight'); _p8j.show(); _p8j.close()
print(f'\nSaved: {_fp8j}')

# -- What actually drives the contrast ----------------------------------------
# Decomposes each arm's alpha=0 -> alpha=1 change into the CNN's own movement
# and the RF-patch movement, because the contrast alone cannot separate them.
print('\n' + '=' * 74)
print('WHAT DRIVES cfgGain: CNN movement vs RF-patch movement (median over species)')
print('=' * 74)
print(f"{'arm':<8} {'d(CNN)':>9} {'d(RF-patch)':>13} {'cfgGain':>9}  reading")
print('-' * 74)
for _arm in ['random', 'extrap']:
    _d = _j[(_j.arm == _arm) & (_j.sub_exp == _S8J_MAIN)]
    _a_lo, _a_hi = min(_ALPHAS_J), max(_ALPHAS_J)
    _cn = _d[_d.model == _CN8J].groupby('alpha')['spearman_true'].median()
    _rp = _d[_d.model == 'rf_patch_summary'].groupby('alpha')['spearman_true'].median()
    _dc = float(_cn.get(_a_hi, np.nan) - _cn.get(_a_lo, np.nan))
    _dr = float(_rp.get(_a_hi, np.nan) - _rp.get(_a_lo, np.nan))
    # Guard FIRST: with non-finite deltas every comparison below is False and
    # the chain would fall through to a positive-sounding default, stating a
    # conclusion the data does not contain.
    if not (np.isfinite(_dc) and np.isfinite(_dr)):
        print(f'{_arm:<8} {"n/a":>9} {"n/a":>13} {"n/a":>9}  '
              f'alpha={_a_lo}/{_a_hi} not both present -- NOT INTERPRETED')
        continue
    if _dc > 0 and _dr < 0:
        _msg = 'CNN improves AND RF falls'
    elif _dc > 0 and _dr >= 0:
        _msg = 'both rise; CNN rises more' if _dc > _dr else 'both rise; RF rises more'
    elif _dc <= 0 and _dr < _dc:
        _msg = 'both fall; CNN falls LESS (RF collapse dominates)'
    else:
        _msg = 'CNN falls at least as fast -- no CNN advantage'
    print(f'{_arm:<8} {_dc:>9.3f} {_dr:>13.3f} {_dc - _dr:>9.3f}  {_msg}')
print('=' * 74)
print('cfgGain is the right PAIRED quantity, but it is a difference: state in the')
print('text which term moved, or a reader will assume the CNN got better.')


# =============================================================================
# THE THREE SIDE ARMS. 8j computes six sub-experiments but this cell used to
# plot only two of them (_S8J_MAIN and the pointwise baseline) -- 408 of 648
# rows reached no figure at all. The arms below are not footnotes: the landuse
# sweep is the ecological-validity check on the whole configurational claim,
# and the oracle-val arm is the bound that separates model selection from model
# capability. They were computed and then invisible.
# =============================================================================

def _cfg_tab(sub_exp, keycol='alpha', gap0_from=None):
    """Per-species cfgGain for one arm.

    cfgGain = (CNN - RF-patch) at this level, MINUS the same gap at alpha=0.
    Subtracting the alpha=0 gap removes the architecture penalty that exists
    even with no configuration in the truth, so what is left is attributable to
    the DGP regime rather than to the CNN simply being worse at pointwise
    problems.

    gap0_from: where to take the alpha=0 reference. An alpha sweep has its own
    alpha=0 point and uses it. The corr-len arm runs at alpha=1 ONLY, so it
    borrows the reference from the GRF alpha sweep for the same species and arm
    -- legitimate because the architecture penalty is a property of the species
    and the fold, not of the habitat grain being swept.
    """
    ref = gap0_from or sub_exp
    out = []
    for _sp in _SPECIES:
        for _arm in ['random', 'extrap']:
            _r = _j[(_j.species_idx == _sp) & (_j.arm == _arm) & (_j.sub_exp == ref)]
            _rc = _r[_r.model == _CN8J].groupby('alpha')['spearman_true'].mean()
            _rp = _r[_r.model == 'rf_patch_summary'].groupby('alpha')['spearman_true'].mean()
            if 0.0 not in _rc.index or 0.0 not in _rp.index:
                continue
            _gap0 = float(_rc[0.0] - _rp[0.0])
            _g = _j[(_j.species_idx == _sp) & (_j.arm == _arm) & (_j.sub_exp == sub_exp)]
            if _g.empty: continue
            _cn = _g[_g.model == _CN8J].groupby(keycol)['spearman_true'].mean()
            _pa = _g[_g.model == 'rf_patch_summary'].groupby(keycol)['spearman_true'].mean()
            for _k in _cn.index:
                if _k not in _pa.index: continue
                out.append(dict(species_idx=_sp, arm=_arm, key=float(_k),
                                cfg_gain=float((_cn[_k] - _pa[_k]) - _gap0),
                                cnn=float(_cn[_k]), rf_patch=float(_pa[_k])))
    return pd.DataFrame(out)


def _med_ci(df, arm, keys):
    """median + bootstrap CI of cfgGain per key, species as the unit."""
    _s = df[df.arm == arm]
    med, lo, hi, n = [], [], [], []
    for _k in keys:
        _v = _s.loc[_s.key == _k, 'cfg_gain'].dropna().values
        if len(_v) == 0:
            med.append(np.nan); lo.append(np.nan); hi.append(np.nan); n.append(0); continue
        _l, _h = _boot_ci(_v)
        med.append(float(np.median(_v))); lo.append(_l); hi.append(_h); n.append(len(_v))
    return np.array(med), np.array(lo), np.array(hi), n


# -- Fig: GRF vs LANDUSE ------------------------------------------------------
# The validity check. The GRF field is synthetic by construction: smoothed noise
# thresholded to a target cover. Real land use is not smoothed noise -- it has
# roads, field boundaries, and patches with a geometry no GRF produces. If the
# configurational advantage only exists on the synthetic field, it is a property
# of the simulation and not of landscapes.
_T_GRF = _cfg_tab('8j_alpha_grf')
_T_LU  = _cfg_tab('8j_alpha_landuse')
if _T_LU.empty:
    print('\n(no landuse arm in this CSV -- GRF/landuse figure skipped)')
else:
    _KS = sorted(set(_T_GRF.key) & set(_T_LU.key))
    _figW, _axW = _p8j.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    _figW.suptitle('Does the configurational advantage survive real land use?',
                   fontsize=12, fontweight='bold')
    for _ax, _arm in zip(_axW, ['random', 'extrap']):
        for _T, _lab, _clr, _ls in ((_T_GRF, 'GRF (synthetic)', '#0072B2', '-'),
                                    (_T_LU,  'land use (real)', '#D55E00', '--')):
            _m, _lo, _hi, _n = _med_ci(_T, _arm, _KS)
            _ax.fill_between(_KS, _lo, _hi, color=_clr, alpha=0.13, lw=0)
            _ax.plot(_KS, _m, ls=_ls, marker='o', ms=5, lw=1.9, color=_clr,
                     label=f'{_lab}  (n={max(_n)} species)')
            # per-species lines: species is the replication unit, so they are shown
            for _sp in _SPECIES:
                _v = (_T[(_T.arm == _arm) & (_T.species_idx == _sp)]
                      .set_index('key')['cfg_gain'].reindex(_KS))
                _ax.plot(_KS, _v.values, ls=_ls, color=_clr, lw=0.7, alpha=0.30, zorder=1)
        _ax.axhline(0, color='k', lw=0.9, ls=':')
        _ax.set_xlabel(r'$\alpha$   (0 = pointwise, 1 = configurational)')
        _ax.set_title(f'{_arm} arm', fontsize=10)
        _ax.grid(alpha=0.25); _ax.legend(fontsize=7, loc='best', framealpha=0.9)
    _axW[0].set_ylabel('cfgGain = (CNN $-$ RF-patch) $-$ its own $\\alpha$=0 value')
    _p8j.tight_layout()
    _fpW = os.path.join(FIGURES_DIR, 's8j_grf_vs_landuse.png')
    _p8j.savefig(_fpW, dpi=150, bbox_inches='tight'); _p8j.show(); _p8j.close()
    print(f'Saved: {_fpW}')
    print('\n-- cfgGain at alpha=1, GRF vs landuse (median over species) --')
    for _arm in ['random', 'extrap']:
        _a = _T_GRF[(_T_GRF.arm == _arm) & (_T_GRF.key == 1.0)]['cfg_gain']
        _b = _T_LU[(_T_LU.arm == _arm) & (_T_LU.key == 1.0)]['cfg_gain']
        print(f'  {_arm:<8} GRF {np.median(_a):>6.3f}   landuse {np.median(_b):>6.3f}   '
              f'{"same sign" if np.sign(np.median(_a)) == np.sign(np.median(_b)) else "SIGN FLIP"}')


# -- Fig: correlation-length sweep at alpha = 1 -------------------------------
# Does the configurational advantage depend on habitat GRAIN? corr_len sets the
# typical blob size of the woody field; r_m (the truth's window) stays fixed, so
# sweeping corr_len sweeps `spans = 2*r_m / corr_len`, the number of correlation
# lengths the ecological window covers: 21.3 / 16.0 / 10.7 / 8.0 at
# 150 / 200 / 300 / 400 m.
#
# THE 300 m POINT COMES FROM THE ALPHA SWEEP, not from this arm. corr_len=300 is
# the alpha-sweep default, so its alpha=1 point IS the 300 m corr-len condition
# -- identical species, identical W (same seed), identical alpha. The sweep cell
# deliberately excludes it from _S8J_CORRLEN_EXTRA to avoid fitting it twice;
# splicing it back in here is what makes the x-axis complete rather than leaving
# a hole where the default sits.
_T_CL = _cfg_tab('8j_corrlen_grf_alpha1', keycol='corr_len_m',
                 gap0_from='8j_alpha_grf')
if _T_CL.empty:
    print('\n(no corr-len arm in this CSV -- figure skipped)')
else:
    # the alpha=1 row of the GRF sweep, relabelled to its corr_len
    _T300 = _T_GRF[_T_GRF.key == 1.0].copy()
    _T300['key'] = float(_S8J_CORR_LEN_M) if '_S8J_CORR_LEN_M' in dir() else 300.0
    _T_CLfull = pd.concat([_T_CL, _T300], ignore_index=True)
    _KC = sorted(_T_CLfull.key.unique())

    _figC, _axC = _p8j.subplots(1, 2, figsize=(11, 4.4), sharey=True)
    _figC.suptitle('Does the configurational advantage depend on habitat grain? '
                   '($\\alpha=1$)', fontsize=12, fontweight='bold')
    for _ax, _arm in zip(_axC, ['random', 'extrap']):
        _m, _lo, _hi, _n = _med_ci(_T_CLfull, _arm, _KC)
        _ax.fill_between(_KC, _lo, _hi, color='#009E73', alpha=0.15, lw=0,
                         label='95% CI of the median (over species)')
        _ax.plot(_KC, _m, ls='--', marker='o', ms=5.5, lw=1.9, color='#009E73',
                 label=f'median (n={max(_n)} species)')
        for _sp in _SPECIES:
            _v = (_T_CLfull[(_T_CLfull.arm == _arm) & (_T_CLfull.species_idx == _sp)]
                  .set_index('key')['cfg_gain'].reindex(_KC))
            _ax.plot(_KC, _v.values, color='#009E73', lw=0.7, alpha=0.32, zorder=1)
        # mark which point was borrowed from the alpha sweep
        _d300 = float(_S8J_CORR_LEN_M) if '_S8J_CORR_LEN_M' in dir() else 300.0
        if _d300 in _KC:
            _ax.plot([_d300], [_m[_KC.index(_d300)]], 'o', ms=11, mfc='none',
                     mec='0.35', mew=1.4, zorder=5,
                     label='from the $\\alpha$ sweep (not refitted)')
        _ax.axhline(0, color='k', lw=0.9, ls=':')
        _ax.set_xlabel('correlation length (m)   ' + r'$grey$: spans $=2r_m/$corr_len')
        _ax.set_title(f'{_arm} arm', fontsize=10)
        _ax.set_xticks(_KC); _ax.set_xticklabels([f'{v:g}' for v in _KC])
        _rm_v = float(_S8J_R_M) if '_S8J_R_M' in dir() else 1600.0
        for _k in _KC:
            _ax.annotate(f'{2*_rm_v/_k:.0f}', xy=(_k, 0.015),
                         xycoords=('data', 'axes fraction'), fontsize=6.5,
                         color='#888888', ha='center', va='bottom', clip_on=False)
        _ax.grid(alpha=0.25); _ax.legend(fontsize=7)
    _axC[0].set_ylabel('cfgGain at $\\alpha=1$')
    _p8j.tight_layout()
    _fpC = os.path.join(FIGURES_DIR, 's8j_corrlen_sweep.png')
    _p8j.savefig(_fpC, dpi=150, bbox_inches='tight'); _p8j.show(); _p8j.close()
    print(f'Saved: {_fpC}')
    print('\n-- cfgGain at alpha=1 by correlation length (median over species) --')
    print(f"{'corr_len':>9} {'spans':>6} {'random':>8} {'extrap':>8}")
    _rm_v = float(_S8J_R_M) if '_S8J_R_M' in dir() else 1600.0
    for _k in _KC:
        _r = _T_CLfull[(_T_CLfull.arm == 'random') & (_T_CLfull.key == _k)]['cfg_gain']
        _e = _T_CLfull[(_T_CLfull.arm == 'extrap') & (_T_CLfull.key == _k)]['cfg_gain']
        print(f'{_k:>9.0f} {2*_rm_v/_k:>6.1f} {np.median(_r):>8.3f} {np.median(_e):>8.3f}')


# -- Fig: oracle model selection, an UPPER BOUND ------------------------------
# WHAT IT ANSWERS. In the primary design the validation fold sits BETWEEN train
# and test in PC1, so it is necessarily closer to train: measured
# D(val,train)=1.30 against D(test,train)=1.99. The CNN's early stopping picks
# the epoch that is best under a shift of 1.30 and is then scored at 1.99, while
# the RF -- which performs no model selection at all -- carries no such penalty.
# That asymmetry sits in exactly the arm where the CNN does worst, so the
# primary design alone cannot separate "the CNN is worse at transfer" from "the
# CNN was model-selected for the wrong shift".
#
# The oracle arm draws val from the TEST band instead, so selection and scoring
# see the same shift (verified: ratio 0.69 -> 0.98). It assumes labels in the
# target domain, which real extrapolative transfer does not have -- it is the
# standard domain-adaptation upper bound, reported as a bound and never as the
# headline. Paired within species against its own primary counterpart, so the
# difference isolates selection from capability.
#
# The random arm is plotted for completeness and is expected to show nothing:
# there D(val,train) = D(test,train) = 1.00, so there is no mismatch to remove.
_PAIRS_OV = [('$\\alpha$=1 (configurational)', '8j_alpha_grf', 1.0,
              '8j_oracleval_alpha1', 1.0),
             ('$\\alpha$=0 (pointwise)', '8j_pointwise_baseline', 0.0,
              '8j_oracleval_pointwise', 0.0)]

def _cnn_by_species(sub_exp, alpha, arm):
    _g = _j[(_j.sub_exp == sub_exp) & (_j.arm == arm) & (_j.model == _CN8J)]
    if 'alpha' in _g.columns and _g.alpha.notna().any():
        _g = _g[np.isclose(_g.alpha.fillna(-999), alpha)]
    return _g.groupby('species_idx')['spearman_true'].mean()

if _j[_j.sub_exp.str.startswith('8j_oracleval')].empty:
    print('\n(no oracle-val arm in this CSV -- figure skipped)')
else:
    _figO, _axO = _p8j.subplots(1, 2, figsize=(11, 4.6), sharey=True)
    _figO.suptitle('Is the CNN model-selected on the same problem it is scored on?',
                   fontsize=12, fontweight='bold')
    for _ax, _arm in zip(_axO, ['random', 'extrap']):
        _xt, _lbls = [], []
        for _xi, (_nm, _se_p, _a_p, _se_o, _a_o) in enumerate(_PAIRS_OV):
            _pri = _cnn_by_species(_se_p, _a_p, _arm)
            _ora = _cnn_by_species(_se_o, _a_o, _arm)
            _sh = sorted(set(_pri.index) & set(_ora.index))
            if not _sh: continue
            for _sp in _sh:      # one connected pair per species
                _ax.plot([_xi - 0.16, _xi + 0.16], [_pri[_sp], _ora[_sp]],
                         ls='--', color='0.55', lw=0.9, alpha=0.7, zorder=1)
            _ax.plot([_xi - 0.16] * len(_sh), [_pri[s] for s in _sh], 'o',
                     color='#0072B2', ms=6, zorder=3,
                     label='primary (val between train and test)' if _xi == 0 else None)
            _ax.plot([_xi + 0.16] * len(_sh), [_ora[s] for s in _sh], '^',
                     color='#D55E00', ms=6, zorder=3,
                     label='oracle val (val from the test band)' if _xi == 0 else None)
            _d = float(np.median([_ora[s] - _pri[s] for s in _sh]))
            _ax.annotate(f'{_d:+.3f}', xy=(_xi, 0.965), xycoords=('data', 'axes fraction'),
                         ha='center', fontsize=8, color='0.3')
            _xt.append(_xi); _lbls.append(_nm)
        _ax.set_xticks(_xt); _ax.set_xticklabels(_lbls, fontsize=9)
        _ax.set_xlim(-0.55, len(_PAIRS_OV) - 0.45)
        _ax.set_title(f'{_arm} arm', fontsize=10)
        _ax.grid(alpha=0.25, axis='y'); _ax.legend(fontsize=7.5, loc='lower right')
    _axO[0].set_ylabel('CNN Spearman $\\rho$ vs truth')
    _figO.text(0.5, -0.03, 'Each grey line is one species. The number above each pair is '
               'the median shift from primary to oracle selection.',
               ha='center', fontsize=7.5, color='0.35')
    _p8j.tight_layout()
    _fpO = os.path.join(FIGURES_DIR, 's8j_oracle_val.png')
    _p8j.savefig(_fpO, dpi=150, bbox_inches='tight'); _p8j.show(); _p8j.close()
    print(f'Saved: {_fpO}')
    print('\n-- how much of the CNN deficit is model selection, not capability? --')
    print(f"{'arm':<8} {'condition':<26} {'primary':>8} {'oracle':>8} {'delta':>8}  {'n':>3}")
    for _arm in ['random', 'extrap']:
        for _nm, _se_p, _a_p, _se_o, _a_o in _PAIRS_OV:
            _pri = _cnn_by_species(_se_p, _a_p, _arm)
            _ora = _cnn_by_species(_se_o, _a_o, _arm)
            _sh = sorted(set(_pri.index) & set(_ora.index))
            if not _sh: continue
            _nm_txt = _nm.replace('$\\alpha$', 'alpha')
            print(f'{_arm:<8} {_nm_txt:<26} {np.median([_pri[s] for s in _sh]):>8.3f} '
                  f'{np.median([_ora[s] for s in _sh]):>8.3f} '
                  f'{np.median([_ora[s]-_pri[s] for s in _sh]):>+8.3f}  {len(_sh):>3}')
    print('  A large positive delta in the extrap arm means part of the CNN deficit')
    print('  was the selection/evaluation mismatch. A near-zero delta means the')
    print('  primary result stands as a capability statement.')
